# Preprocesamiento y control de calidad de rs-fMRI en Python

## Guía tutorial reproducible para Colab y entorno local

Este notebook implementa un flujo completo de preprocesamiento y control de calidad para resonancia magnética funcional en reposo (rs-fMRI), con énfasis en aseguramiento de calidad (QA), control de calidad (QC), evaluación de movimiento y artefactos, denoising y generación de reportes.

**Filosofía.** El objetivo es construir un flujo moderno y reproducible en Python que combine la robustez de fMRIPrep, la evaluación automatizada de MRIQC, y las metodologías de control de calidad, denoising e interpretación documentadas en la literatura reciente sobre conectividad funcional.

**Un solo notebook, tres entornos de ejecución.** El mismo archivo detecta automáticamente si se ejecuta en Google Colab o en un entorno local, y selecciona el backend disponible:

| Backend | Cuándo se usa |
|---|---|
| Neurodesk en Colab (contenedores vía Apptainer y CVMFS) | Colab, vía principal |
| PyNeurodesk (paquete `neurodesk`) | Local, con virtualización de hardware disponible |
| Implementación nativa en Python | Sin contenedores disponibles, cualquier entorno |

Las secciones de métricas, visualizaciones y reportes son idénticas en los tres casos: consumen la misma interfaz de datos preprocesados y de confounds, sin importar qué backend los generó.

**Dataset.** ON-Harmony (OpenNeuro `ds004712`), sesión `ses-NOT1ACH001` (Nottingham, Philips Achieva 3T), sujetos `sub-03286`, `sub-14229` y `sub-12813`.

**Referencias metodológicas obligatorias**, citadas a lo largo de toda la guía:

- Morfini, F., Whitfield-Gabrieli, S., Nieto-Castañón, A. (2023). Functional connectivity MRI quality control procedures in CONN. *Frontiers in Neuroscience*, 17:1092125.
- Provins, C., MacNicol, E., Seeley, S.H., Hagmann, P., Esteban, O. (2023). Quality control in functional MRI studies with MRIQC and fMRIPrep. *Frontiers in Neuroimaging*, 1:1073734.
- Kumar, V.A. et al. (2024). Recommended resting-state fMRI acquisition and preprocessing steps for preoperative mapping in adult and pediatric patients with brain tumors and epilepsy. *AJNR*, 45:139-148.
- Warrington, S. et al. (2023). A resource for development and comparison of multimodal brain 3T MRI harmonisation approaches. *Imaging Neuroscience*, 1.



## Sección 1. Introducción y fundamentos

### 1.1 Qué mide el BOLD en reposo y qué es una red en estado de reposo

La resonancia magnética funcional detecta el contraste dependiente del nivel de oxigenación de la sangre (BOLD, blood oxygenation level dependent), se define como una medición en resonancia magnética que refleja cambios en la oxigenación sanguínea, y es medida indirecta de la actividad neural, no mide disparos neuronales sino los cambios locales de flujo, volumen y oxigenación sanguínea que acompañan a la actividad de una población de neuronas. En estado de reposo (rs-fMRI), sin tarea explícita, la señal BOLD exhibe fluctuaciones espontáneas de baja frecuencia (aproximadamente entre 0.01 y 0.1 Hz) que están sincronizadas entre regiones anatómicamente distantes. A ese patrón de coherencia temporal se le llama conectividad funcional (functional connectivity, FC), y a los conjuntos de regiones que la muestran de forma consistente entre sujetos, red en estado de reposo (resting-state network, RSN)

Conviene distinguir dos términos que a menudo se usan indistintamente: rs-fMRI es la adquisición (la serie temporal de imágenes en reposo), mientras que rs-FC es el resultado del análisis posterior de esa adquisición (las estimaciones de conectividad). Un rs-fMRI de buena calidad no garantiza una rs-FC interpretable si el procesamiento posterior no controla el ruido (Kumar et al., 2024).


### 1.2 Por qué el ruido imita conectividad

La señal BOLD es, en palabras de Morfini et al. (2023), "noisy and only marginally representative of neural activity": se genera por la interacción de procesos neuronales, metabólicos, cardíacos y de vigilia, y está sistemáticamente afectada por características del equipo y del participante. El problema no es solo que el ruido reduce la relación señal-ruido; es que buena parte de ese ruido está correlacionado espacialmente, de manera que se comporta estadísticamente igual que una conectividad real.

El mecanismo mejor documentado es el del movimiento de cabeza. Power et al. (2012) muestran que el movimiento introduce un sesgo dependiente de la distancia, fortalece artificialmente la correlación entre regiones cercanas y la debilita entre regiones lejanas. El resultado es indistinguible de un cambio real en la arquitectura de las redes si no se corrige explícitamente. Desde la validación BIDS hasta el reporte final se requiere de QA/QC, cada etapa reduce o cuantifica una fuente de esta clase de sesgo.


### 1.3 Qué es QA, qué es QC, y por qué no son lo mismo

Aseguramiento de calidad (QA) y control de calidad (QC) suelen usarse como sinónimos, pero cumplen funciones distintas y complementarias (Provins et al., 2023):

- QA está orientado al proceso: busca que el flujo de adquisición y procesamiento produzca datos de calidad suficiente, y actúa antes de que el problema se repita (por ejemplo, identificar un artefacto causado por una condición ambiental del escáner y corregirlo para las adquisiciones futuras).
- QC está orientado al resultado: excluye datos de calidad insuficiente de un dataset ya adquirido, para que no sesguen los análisis posteriores.

Provins et al. (2023) son explícitos: "even an optimal QA does not address the objectives of QC testing". Un QA impecable no sustituye al QC, porque incluso con un proceso de adquisición óptimo pueden aparecer artefactos puntuales, variabilidad entre sujetos o fallos de procesamiento que solo el QC detecta. Morfini et al. (2023) hacen la misma distinción citando a Friedman y Glover (2006). Este notebook implementa mayoritariamente QC, se trabaja sobre datos ya adquiridos y publicados, y el QA se limita a las recomendaciones de buenas prácticas que se citan en cada sección para estudios futuros.


### 1.4 Qué es el denoising y porqué es relevante

El preprocesamiento (Secciones 5 y 6) corrige propiedades espaciales de los datos, como el movimiento, distorsión, alineación entre modalidades y con una plantilla común. Incluso después de un preprocesamiento correcto, la serie temporal todavía contiene variabilidad de origen no neural (residuos de movimiento, pulso cardíaco, respiración) que compromete cualquier estimación de conectividad. El **denoising** es la etapa que se ocupa específicamente de esa variabilidad residual, mediante tres pasos secuenciales que se detallan en las Secciones 9 y 10: extracción de componentes de ruido a partir de regiones sin señal neural de interés (sustancia blanca y líquido cefalorraquídeo), regresión lineal de esos componentes junto con los parámetros de movimiento, y filtrado paso-banda temporal.

El objetivo no es maximizar la eliminación de ruido: cada regresor de denoising consume un grado de libertad de la serie temporal, y un denoising demasiado agresivo puede eliminar señal de interés junto con el ruido. La métrica de grados de libertad efectivos (DOF) se calcula en la Sección 10 y se evalúa en la Sección 11, precisamente para cuantificar este balance.


### 1.5 Qué es BIDS

El estándar Brain Imaging Data Structure (BIDS; Gorgolewski et al., 2016) especifica cómo nombrar archivos, organizar carpetas por sujeto y sesión, y documentar metadatos de adquisición en archivos JSON asociados a cada imagen (sidecars). Se creó para resolver un problema concreto, sin una convención común, cada laboratorio nombra y organiza sus datos de forma distinta, y las herramientas de análisis (MRIQC, fMRIPrep, y las que se implementan en este notebook) no pueden asumir dónde encontrar cada archivo ni qué parámetros de adquisición tiene. Los datos aquí usados (dataset ON-Harmony), publican o consumen datos ya organizados en BIDS. La Sección 3 detalla la estructura concreta y el proceso de validación.


### 1.6 Comparación conceptual: CONN, fMRIPrep y MRIQC

| | CONN | fMRIPrep | MRIQC |
|---|---|---|---|
| Qué es | Toolbox de análisis de conectividad funcional (MATLAB, sobre SPM) | Pipeline de preprocesamiento (Python, Nipype) | Herramienta de evaluación de calidad de imagen (Python, Nipype) |
| Qué produce | Datos preprocesados, denoised, y estimaciones de conectividad | Datos preprocesados, corregistrados y normalizados, con una tabla de confounds | Métricas de calidad de imagen (IQM) y reportes visuales de datos crudos |
| Preprocesa | Sí | Sí | No |
| Hace denoising | Sí | No; deja los confounds calculados, la decisión de regresión queda al usuario | No |
| Analiza conectividad | Sí | No | No |
| Requiere | MATLAB con licencia y SPM, o bien la aplicación compilada autónoma, que solo necesita el MATLAB Runtime, gratuito, aunque por esa vía únicamente se distribuyen versiones antiguas de CONN | Python, contenedor con ANTs, AFNI, FSL y FreeSurfer | Python, contenedor con AFNI y ANTs |

Ninguna de las tres herramientas cubre todo el flujo por sí sola dentro del stack de Python de este notebook. CONN es la que abarca más etapas, desde el preprocesamiento hasta la estimación de conectividad, a cambio de depender de MATLAB o de su entorno de ejecución compilado.

La estrategia adoptada, consistente con la instrucción del proyecto de no depender de una sola herramienta, usar MRIQC para el primer punto de control de calidad sobre datos crudos (Sección 4), fMRIPrep para el preprocesamiento anatómico y funcional (Secciones 5 y 6), e implementar en Python nativo, sobre los outputs de fMRIPrep, las métricas de control de calidad, extracción de componentes de ruido y estrategias de denoising que la literatura de conectividad funcional documenta con más detalle en el contexto de CONN (Morfini et al., 2023), pero que no son exclusivas de esa herramienta ni dependen de ella, se aplican por igual a datos preprocesados por fMRIPrep, por AFNI o por cualquier otro pipeline.


### 1.7 Arquitectura completa del pipeline

El diagrama siguiente resume las secciones de este notebook como un flujo con tres tramos, preparación y validación de datos (Secciones 2 y 3), control de calidad y preprocesamiento (Secciones 4 a 6), y evaluación de movimiento, ruido, denoising y reporte final (Secciones 7 a 12). El color indica si la etapa se ejecuta dentro de un contenedor Neurodesk (MRIQC, fMRIPrep) o mediante una implementación nativa en Python que corre igual en cualquier entorno.


In [ ]:
# 1.7 Diagrama de arquitectura del pipeline completo

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from matplotlib.lines import Line2D

etapas = [
    ("Validacion BIDS e inspeccion\nde metadatos", "Seccion 3", "nativo"),
    ("MRIQC: IQM de datos crudos", "Seccion 4", "contenedor"),
    ("fMRIPrep: preprocesamiento\nanatomico", "Seccion 5", "contenedor"),
    ("fMRIPrep: preprocesamiento\nfuncional", "Seccion 6", "contenedor"),
    ("Metricas de movimiento y senal\n(FD, DVARS, GSchange, scrubbing)", "Seccion 7", "nativo"),
    ("Evaluacion de normalizacion\ny corregistro (Dice, Jaccard)", "Seccion 8", "nativo"),
    ("Extraccion de componentes\nde ruido (aCompCor, tCompCor)", "Seccion 9", "nativo"),
    ("Denoising: comparacion\nde estrategias", "Seccion 10", "nativo"),
    ("QC posterior al denoising\n(DOF, GCOR, BOLDstd, QC-FC)", "Seccion 11", "nativo"),
    ("Reporte final automatizado", "Seccion 12", "nativo"),
]

colores = {"contenedor": "#5B8FB9", "nativo": "#7FB77E"}

fig, ax = plt.subplots(figsize=(7, 12))
n = len(etapas)
box_h = 0.8
for i, (texto, seccion, categoria) in enumerate(etapas):
    y = n - i  # se dibuja de arriba (primera etapa) a abajo (ultima etapa)
    box = FancyBboxPatch((0, y - box_h / 2), 6, box_h,
                          boxstyle="round,pad=0.1", linewidth=1.2,
                          edgecolor="black", facecolor=colores[categoria])
    ax.add_patch(box)
    ax.text(3, y, f"{texto}\n({seccion})", ha="center", va="center", fontsize=9)
    if i < n - 1:
        ax.annotate("", xy=(3, y - box_h / 2 - 0.15), xytext=(3, y - box_h / 2),
                    arrowprops=dict(arrowstyle="-|>", linewidth=1.2))

ax.set_xlim(-0.5, 6.5)
ax.set_ylim(0, n + 1)
ax.axis("off")
ax.set_title("Arquitectura del pipeline completo", fontsize=13, pad=15)

leyenda = [Line2D([0], [0], marker="s", color="w", markerfacecolor=c, markersize=14, label=l)
           for l, c in [("Contenedor Neurodesk (MRIQC / fMRIPrep)", colores["contenedor"]),
                        ("Implementacion nativa en Python", colores["nativo"])]]
ax.legend(handles=leyenda, loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=1, frameon=False)

plt.tight_layout()
plt.show()


**Interpretación.** Las etapas en azul dependen de que el contenedor Neurodesk correspondiente esté disponible en el entorno de ejecución (la Sección 2 detalla cómo se verifica esto y qué ocurre si no lo está). Las etapas en verde son código Python puro y se ejecutan siempre, independientemente del backend, porque operan sobre los archivos NIfTI y las tablas de confounds que produce la etapa anterior, no sobre binarios externos. Esta separación es la que permite que el mismo notebook funcione en Colab, en local con PyNeurodesk, o en local sin contenedores, solo cambia cómo se obtienen los outputs de MRIQC y fMRIPrep, no cómo se analizan después.


### 1.8 Tabla resumen de métricas

La siguiente tabla reúne todas las métricas de QA y QC que se calculan, determinadas con los criterios de exclusión que Provins et al. (2023) documentan para MRIQC y fMRIPrep. La columna "Aplicabilidad" anticipa un hallazgo que se desarrolla con evidencia empírica en la Sección 4, el fondo de las imágenes usualmente está suprimido, como en este dataset, donde entre el 80% y el 97% de los vóxeles de la región de aire valen exactamente cero, que es el estadístico decisivo según se establece en la Sección 4.3. Como dato secundario, sobre el volumen completo los ceros exactos representan entre el 28% y el 45% de los vóxeles del T1w y el 35.7% del BOLD. Esa supresión invalida la interpretación de varias métricas que dependen del ruido del aire, que solo aplicarían cuando el fondo no está suprimido.

- Las métricas SNRd, CNR en su definición de MRIQC, FBER, QI1 y QI2 dependen del ruido de fondo


In [ ]:
# 1.8 Tabla maestra de métricas de QA y QC

import pandas as pd

metricas = [
    dict(Metrica="SNR (tisular)", Herramienta="MRIQC / calculo nativo",
         Definicion="Media de intensidad en una mascara tisular dividida por su desviacion estandar.",
         Interpretacion="Mayor es mejor. Se compara entre sujetos del mismo protocolo, no contra un valor absoluto.",
         Rango_esperado="Depende del protocolo; usar como referencia relativa intra-dataset.",
         Detecta="Ruido termico general, calidad de bobina.",
         Aplicabilidad="Valida: no depende del fondo."),
    dict(Metrica="CJV (coeficiente de variacion conjunta)", Herramienta="MRIQC",
         Definicion="(desviacion GM + desviacion WM) / |media GM - media WM|.",
         Interpretacion="Menor es mejor: indica mejor separabilidad entre sustancia gris y blanca.",
         Rango_esperado="Sin umbral universal; comparacion relativa.",
         Detecta="Ruido y no uniformidad de intensidad combinados.",
         Aplicabilidad="Valida: no depende del fondo."),
    dict(Metrica="INU (no uniformidad de intensidad)", Herramienta="MRIQC / N4",
         Definicion="inu_med: mediana del campo de sesgo de intensidad estimado por correccion N4. MRIQC reporta ademas inu_range, el rango de ese mismo campo.",
         Interpretacion="inu_med cercana a 1 indica campo homogeneo; desviaciones marcadas respecto de 1 indican sesgo fuerte de bobina (en inu_range el criterio equivalente es la cercania a 0).",
         Rango_esperado="inu_med aproximadamente 0.7 a 1.3 en adquisiciones tipicas a 3T.",
         Detecta="Sesgo de intensidad de baja frecuencia espacial.",
         Aplicabilidad="Valida: no depende del fondo."),
    dict(Metrica="EFC (criterio de enfoque por entropia)", Herramienta="MRIQC",
         Definicion="Entropia de Shannon de la distribucion de intensidades, normalizada por su maximo teorico.",
         Interpretacion="Menor es mejor: mayor entropia indica imagen mas borrosa o con mas ghosting.",
         Rango_esperado="Sin umbral universal; comparacion relativa.",
         Detecta="Desenfoque por movimiento, ghosting.",
         Aplicabilidad="Valida con salvedad: un voxel en cero aporta cero a la entropia, pero aumenta el recuento de voxeles y con el la constante de normalizacion, de modo que la supresion de fondo empuja el EFC hacia valores mas bajos, aparentemente mejores."),
    dict(Metrica="WM2MAX", Herramienta="MRIQC",
         Definicion="Intensidad mediana de sustancia blanca dividida por el percentil 95 de intensidad de toda la imagen.",
         Interpretacion="Cercano a 0.6-0.8 es tipico; valores bajos sugieren sesgo de intensidad o mala segmentacion.",
         Rango_esperado="0.6 a 0.8 aproximadamente.",
         Detecta="Sesgo de intensidad, fallos de segmentacion.",
         Aplicabilidad="Valida: no depende del fondo."),
    dict(Metrica="SNRd (Dietrich)", Herramienta="MRIQC",
         Definicion="SNR estimado usando la desviacion estandar del fondo como referencia de ruido.",
         Interpretacion="Mayor es mejor en datasets con fondo de ruido termico real.",
         Rango_esperado="No aplica en este dataset.",
         Detecta="Ruido termico de fondo.",
         Aplicabilidad="No aplicable: el fondo esta suprimido, el valor calculado refleja tejido residual, no ruido."),
    dict(Metrica="CNR (definicion MRIQC)", Herramienta="MRIQC",
         Definicion="Diferencia de medias GM/WM dividida por la raiz de la suma de las varianzas de aire, GM y WM (Magnotta y Friedman, 2006).",
         Interpretacion="Mayor es mejor. Con el fondo suprimido la varianza del aire aporta casi nada al denominador, pero las de GM y WM impiden que se anule, de modo que la metrica sigue siendo finita.",
         Rango_esperado="Comparacion relativa intra-dataset; no comparable con valores publicados.",
         Detecta="Separabilidad de tejidos relativa al ruido.",
         Aplicabilidad="Inflada y no comparable con la literatura por la supresion de fondo, pero conserva utilidad para comparar sujetos dentro del mismo dataset. Se reporta ademas una variante sin fondo (Sec. 4)."),
    dict(Metrica="FBER (energia dentro/fuera de la cabeza)", Herramienta="MRIQC",
         Definicion="Energia media dentro de la mascara cerebral dividida por la energia media fuera de ella.",
         Interpretacion="Mayor es mejor.",
         Rango_esperado="No definido en este dataset.",
         Detecta="Artefactos de fondo, mala mascara cerebral.",
         Aplicabilidad="Indefinido: MRIQC devuelve -1 cuando la mediana de energia del fondo es cero, como aqui."),
    dict(Metrica="QI1 (fraccion de artefacto, Mortamet)", Herramienta="MRIQC",
         Definicion="Fraccion de voxeles del fondo detectados como artefactuales tras ajustar un modelo de ruido.",
         Interpretacion="Menor es mejor.",
         Rango_esperado="No interpretable en este dataset.",
         Detecta="Artefactos estructurados en el fondo.",
         Aplicabilidad="Inflado: el escaso tejido residual no nulo del fondo se cuenta como artefacto en su totalidad."),
    dict(Metrica="QI2", Herramienta="MRIQC",
         Definicion="Bondad de ajuste de una distribucion chi-cuadrado al histograma de intensidades del fondo.",
         Interpretacion="Menor es mejor.",
         Rango_esperado="No interpretable en este dataset.",
         Detecta="Desviaciones del fondo respecto a un modelo de ruido termico puro.",
         Aplicabilidad="Sin sentido: el histograma del fondo es ~97% masa puntual en cero, no una distribucion de ruido."),
    dict(Metrica="FD (desplazamiento de encuadre)", Herramienta="Calculo nativo sobre parametros de fMRIPrep",
         Definicion="Cambio maximo de posicion de seis puntos de control alrededor del cerebro entre volumenes consecutivos.",
         Interpretacion="Picos indican movimiento subito. Se compara con DVARS: deben coincidir en el tiempo.",
         Rango_esperado="Umbral habitual de censurado: 0.5 mm.",
         Detecta="Movimiento de cabeza volumen a volumen.",
         Aplicabilidad="Valida sin salvedades."),
    dict(Metrica="DVARS", Herramienta="Calculo nativo",
         Definicion="Raiz de la media del cuadrado del cambio de senal BOLD entre volumenes consecutivos, normalizada.",
         Interpretacion="En la version estandarizada de fMRIPrep el valor tipico en datos limpios ronda 1, y los picos por encima de 1.5 marcan volumenes corrompidos.",
         Rango_esperado="Depende de la normalizacion usada; se reporta la version estandarizada, con umbral de 1.5.",
         Detecta="Cambios abruptos de intensidad BOLD.",
         Aplicabilidad="Valida sin salvedades."),
    dict(Metrica="GSchange (cambio de senal global)", Herramienta="Calculo nativo",
         Definicion="Valor absoluto del cambio de la senal BOLD promediada en todo el cerebro entre volumenes consecutivos, en unidades estandar.",
         Interpretacion="Valores altos indican variabilidad subita de intensidad global.",
         Rango_esperado="Umbral habitual de censurado: 3 desviaciones estandar.",
         Detecta="Eventos globales de intensidad (movimiento, respiracion profunda).",
         Aplicabilidad="Valida sin salvedades."),
    dict(Metrica="Scrubbing (regresores de volumenes atipicos)", Herramienta="Calculo nativo",
         Definicion="Un regresor binario por cada volumen identificado como atipico por FD y/o GSchange.",
         Interpretacion="El numero de regresores es el numero de volumenes censurados.",
         Rango_esperado="Depende del umbral de FD/GSchange elegido.",
         Detecta="Volumenes individuales corrompidos por movimiento o artefacto.",
         Aplicabilidad="Valida sin salvedades."),
    dict(Metrica="Parametros de realineamiento (6, 12 o 24)", Herramienta="fMRIPrep",
         Definicion="6: traslaciones y rotaciones. 12: mas sus derivadas temporales. 24: mas los cuadrados de ambos.",
         Interpretacion="Derivas lentas y monotonas son benignas; saltos abruptos corrompen volumenes.",
         Rango_esperado="Sin umbral universal para los parametros en si; se usan para derivar FD.",
         Detecta="Movimiento de cabeza en sus seis grados de libertad.",
         Aplicabilidad="Valida sin salvedades."),
    dict(Metrica="Componentes aCompCor (WM y CSF)", Herramienta="Calculo nativo sobre mascaras de fMRIPrep",
         Definicion="Componentes principales de la senal BOLD dentro de mascaras erosionadas de sustancia blanca y liquido cefalorraquideo.",
         Interpretacion="Pocos componentes para alcanzar el 50% de varianza indica ruido concentrado en pocas fuentes.",
         Rango_esperado="Criterio de varianza explicada (habitualmente 50%), no un numero fijo.",
         Detecta="Ruido fisiologico y de movimiento residual no neural.",
         Aplicabilidad="Valida sin salvedades."),
    dict(Metrica="Componentes tCompCor", Herramienta="Calculo nativo / fMRIPrep",
         Definicion="Componentes principales de los voxeles con mayor variabilidad temporal, sin restringir a una mascara tisular.",
         Interpretacion="Complementa a aCompCor cuando la varianza no esta bien capturada por WM/CSF solos.",
         Rango_esperado="Criterio de varianza explicada, analogo a aCompCor.",
         Detecta="Ruido de alta variabilidad temporal no necesariamente tisular.",
         Aplicabilidad="Valida; revisar que la mascara de mayor varianza no incluya corteza (criterio X, Provins et al. 2023)."),
    dict(Metrica="MaxMotion", Herramienta="Calculo nativo",
         Definicion="Valor maximo de la serie de FD en un run, considerando todos los volumenes originales.",
         Interpretacion="Describe el peor instante de movimiento, no el estado general del run.",
         Rango_esperado="Sin umbral universal; comparacion entre sujetos del mismo protocolo.",
         Detecta="Picos aislados de movimiento severo.",
         Aplicabilidad="Valida sin salvedades."),
    dict(Metrica="MeanMotion", Herramienta="Calculo nativo",
         Definicion="Media de la serie de FD, calculada solo sobre volumenes no censurados.",
         Interpretacion="Describe el nivel de movimiento residual tras excluir los picos.",
         Rango_esperado="Sin umbral universal; comparacion entre sujetos del mismo protocolo.",
         Detecta="Movimiento sostenido de bajo nivel.",
         Aplicabilidad="Valida sin salvedades."),
    dict(Metrica="InvalidScans / ValidScans / PVS", Herramienta="Calculo nativo",
         Definicion="Numero de volumenes marcados como atipicos, numero de volumenes validos, y su proporcion sobre el total.",
         Interpretacion="PVS baja indica que gran parte de la serie no es utilizable sin censurado agresivo.",
         Rango_esperado="PVS por debajo de 0.75 se considera atipico bajo (Morfini et al., 2023).",
         Detecta="Proporcion global de datos utilizables por run.",
         Aplicabilidad="Valida sin salvedades."),
    dict(Metrica="NORManat / NORMfunc", Herramienta="Calculo nativo sobre outputs de fMRIPrep",
         Definicion="Coeficiente de Dice entre la mascara de sustancia gris de la plantilla MNI y la mascara de GM del sujeto, anatomica o funcional.",
         Interpretacion="Mayor es mejor: cuantifica que tan bien se superpone la normalizacion del sujeto con la plantilla.",
         Rango_esperado="Entre 0 y 1; valores bajos son atipicos dentro del propio dataset.",
         Detecta="Fallos de normalizacion espacial.",
         Aplicabilidad="Valida sin salvedades."),
    dict(Metrica="AFO (superposicion anatomico-funcional)", Herramienta="Calculo nativo",
         Definicion="Coeficiente de Dice entre la mascara de GM anatomica y la mascara de GM funcional del mismo sujeto.",
         Interpretacion="Mayor es mejor: cuantifica la precision del corregistro entre modalidades.",
         Rango_esperado="Entre 0 y 1; valores bajos son atipicos dentro del propio dataset.",
         Detecta="Fallos de corregistro entre BOLD y T1w.",
         Aplicabilidad="Valida sin salvedades."),
    dict(Metrica="Indice de Jaccard (GM, WM, CSF)", Herramienta="Calculo nativo",
         Definicion="Interseccion sobre union entre la mascara tisular del sujeto y la de la plantilla, para cada tejido.",
         Interpretacion="Mas estricto que Dice para el mismo par de mascaras; util como control cruzado.",
         Rango_esperado="Entre 0 y 1, tipicamente menor que el Dice equivalente.",
         Detecta="Lo mismo que Dice, con mayor sensibilidad a discrepancias pequenas.",
         Aplicabilidad="Valida sin salvedades."),
    dict(Metrica="Volumen tisular (GM, WM, CSF) y volumen erosionado", Herramienta="Calculo nativo sobre segmentacion de fMRIPrep",
         Definicion="Conteo de voxeles con probabilidad de tejido mayor a 50%, antes y despues de una erosion de un voxel.",
         Interpretacion="Valores extremos combinan diferencias anatomicas reales con fallos de segmentacion o normalizacion.",
         Rango_esperado="Razon GM/WM aprox. 1.0 a 1.4 en adultos sanos; volumen cerebral total 1100 a 1600 cm3.",
         Detecta="Fallos de segmentacion o normalizacion, no morfometria per se.",
         Aplicabilidad="Valida, con la salvedad de que no sustituye una medida morfometrica (requeriria FreeSurfer)."),
    dict(Metrica="DOF (grados de libertad efectivos)", Herramienta="Calculo nativo",
         Definicion="Volumenes totales menos regresores de denoising, multiplicado por la fraccion de la frecuencia de Nyquist cubierta por el filtro paso-banda.",
         Interpretacion="Valores bajos indican denoising demasiado agresivo para el numero de volumenes disponibles.",
         Rango_esperado="Debe ser sustancialmente mayor que cero; comparar entre sujetos del mismo protocolo.",
         Detecta="Sobre-ajuste del denoising respecto a los datos disponibles.",
         Aplicabilidad="Valida sin salvedades."),
    dict(Metrica="BOLDstd", Herramienta="Calculo nativo",
         Definicion="Desviacion estandar temporal de la senal BOLD tras normalizar la media global a 100 y tras denoising.",
         Interpretacion="Valores altos sugieren ruido residual; valores cercanos a cero sugieren perdida de senal por sobre-limpieza.",
         Rango_esperado="Sin umbral universal; se interpreta junto a GCOR y DOF, no de forma aislada.",
         Detecta="Estabilidad temporal residual tras denoising.",
         Aplicabilidad="Valida sin salvedades."),
    dict(Metrica="GCOR (correlacion global)", Herramienta="Calculo nativo",
         Definicion="Promedio de los coeficientes de correlacion de Pearson entre todos los pares de voxeles dentro de la mascara de analisis.",
         Interpretacion="Valores altos sugieren denoising insuficiente; valores ligeramente negativos sugieren denoising excesivo o regresion de senal global.",
         Rango_esperado="Valores positivos pequenos son habituales incluso en datos limpios. Por ser una media sobre todos los pares, esta acotado por debajo en torno a -1/(N-1) con N el numero de voxeles, es decir practicamente cero.",
         Detecta="Ruido residual de amplio alcance espacial tras denoising.",
         Aplicabilidad="Valida sin salvedades."),
    dict(Metrica="QC-FC % y dependencia de la distancia", Herramienta="Calculo nativo",
         Definicion="Correlacion entre valores de conectividad funcional y medidas de movimiento a traves de sujetos, y su relacion con la distancia euclidiana entre regiones.",
         Interpretacion="Porcentajes de coincidencia por encima de 95% indican modulacion despreciable del movimiento sobre la conectividad.",
         Rango_esperado="No interpretable de forma inferencial con n=3; valido como verificacion del procedimiento.",
         Detecta="Sesgo de movimiento residual en la estructura de conectividad.",
         Aplicabilidad="Calculable pero limitado: con tres sujetos el estadistico queda determinado por tres puntos; se documenta como demostracion del metodo, no como evidencia poblacional."),
]

tabla_metricas = pd.DataFrame(metricas)
pd.set_option("display.max_colwidth", 80)
tabla_metricas


### 1.9 Cómo leer e interpretar resultados de QA/QC y de preprocesamiento

Las etapas que producen resultados que hay que interpretar son MRIQC (Sección 4), fMRIPrep (Secciones 5 y 6), las métricas de movimiento y señal (Sección 7), la evaluación de normalización (Sección 8), la extracción de ruido (Sección 9), el denoising (Sección 10) y el QC posterior (Sección 11). Esta subsección fija los criterios de lectura comunes a todas ellas.

**Reporte de MRIQC.** El reporte HTML por sujeto y modalidad se recorre en un orden fijo, primero el panel de "background noise" (el fondo alrededor de la cabeza, donde no debería originarse señal real y donde el aliasing, el ghosting y la estructura espuria son más visibles que dentro del cerebro), después el promedio y la desviación estándar del BOLD (dropouts de señal, distorsión), y por último el carpet plot con su banda de "crown" (voxeles justo fuera del cerebro, si muestran señal estructurada, es artefactual por definición, porque ahí no hay tejido). Esta secuencia de lectura sigue el protocolo de Provins et al. (2023, sección "QC of MRI data substantially relies on the background").

**Reporte de fMRIPrep.** El reporte por sujeto contiene widgets de comparación intermitente entre la imagen normalizada y la plantilla, y entre el BOLD promedio y el T1w de referencia (corregistro). La lectura no es "coinciden o no": Provins et al. (2023) especifican un orden de prioridad de estructuras (ventrículos, regiones subcorticales, cuerpo calloso, cerebelo, y por último corteza), porque un desalineamiento en las primeras invalida el análisis y en la última puede ser variabilidad anatómica normal entre el individuo y la plantilla.

**Dos puntos de control, no uno.** El protocolo de Provins et al. (2023) evalúa primero los datos crudos con MRIQC y solo después, sobre los que superaron ese primer filtro, los datos preprocesados con fMRIPrep. Cada punto de control mira los datos desde un ángulo distinto y ninguno por separado es suficiente, en las Secciones 4 y 6.

**De "vi algo" a "decido algo".** Ver un artefacto no obliga automáticamente a excluir un sujeto. La decisión depende de tres factores en orden Provins et al. (2023):
1. Si el fallo es catastrófico y no admite corrección, se excluye
2. Si es corregible con un ajuste de parámetro o de máscara, se ajusta y se reevalúa
3. Si es leve y no compromete el alcance definido del análisis, se documenta y se continúa

El alcance del análisis, no el artefacto en sí, determina el criterio.


**Qué hacer con artefactos que no aparecen aquí?**

En este ejemplo (n=3) no se espera observar el catálogo completo de artefactos documentados en la literatura, y de hecho ninguno de los tres sujetos seleccionados muestra los casos más severos que reportan Morfini et al. (2023) o Provins et al. (2023) en sus datasets multisitio. Eso no exime de tener un criterio de respuesta definido de antemano para cuando aparezcan.
La tabla generaliza el catálogo de criterios de exclusión de Provins et al. (2023) en una regla de decisión, con el mismo formato condición-decisión-razón que se usa en la Sección 3 para la política de slice timing.

| Tipo de artefacto | Dónde se detecta | Respuesta por defecto | Razón |
|---|---|---|---|
| Estructura en el fondo, incluye aliasing y ghosting | Panel de background noise de MRIQC | Se excluye si la intensidad del ghost es comparable al interior del cerebro; se documenta y continúa si es tenue | Solo importa si la señal espuria se superpone con tejido de interés |
| Distorsión por susceptibilidad, dropout o deformación | Promedio BOLD de MRIQC, o reporte de fMRIPrep si hay fieldmap | Con fieldmap disponible se corrige con SDC (Sección 6) y se reevalúa tras la corrección; sin fieldmap se documenta como limitación y se restringe el análisis si la región afectada es de interés | La corregibilidad depende de si hay datos para estimar el campo de distorsión, no del artefacto en sí |
| Wrap-around | Background noise o promedio BOLD de MRIQC | Se excluye solo si la región plegada se superpone con el área de interés del análisis planeado | El criterio depende del alcance definido, no del artefacto en sí mismo |
| Estructura en la banda de crown del carpet plot | Carpet plot de MRIQC o fMRIPrep | Si coincide con picos de FD es movimiento y se censura (Sección 7); si es periódica y no coincide con FD es probable artefacto respiratorio y se evalúa el filtrado paso-banda | La forma del patrón indica el mecanismo y el mecanismo indica la corrección |
| Hiperintensidad de un corte aislado | Serie de intensidad por corte sobre el carpet plot | Se censura el volumen puntual (Sección 7); no requiere excluir el sujeto completo | El artefacto es focal en el tiempo, no sistemático |
| Fallo de normalización a espacio estándar | Widget de comparación de fMRIPrep, o NORManat/NORMfunc fuera de rango (Sección 8) | Se excluye si falla en ventrículos, regiones subcorticales o cuerpo calloso; se acepta con nota si la discrepancia se limita a corteza | Sigue el orden de prioridad de estructuras de Provins et al. (2023) |
| Voxeles mal clasificados dispersos en la segmentación tisular | Panel de segmentación de fMRIPrep | Se acepta si están en sustancia blanca profunda o núcleos subcorticales, efecto de volumen parcial; se excluye si están en regiones extensas y contiguas | Distingue ruido térmico esperable de un fallo real de segmentación |
| Superposición de las ROI de CompCor con regiones de interés neural | Panel de máscaras de fMRIPrep | Se excluye la ejecución de aCompCor para ese sujeto y se documenta; no se excluye al sujeto de otras métricas | El fallo es específico de una etapa de denoising, no de todo el pipeline |

- Esta tabla no sustituye la inspección visual, es el criterio que se aplica una vez que la inspección visual identificó a qué categoría pertenece lo observado.


**Buenas prácticas.** Definir el alcance del análisis antes de mirar cualquier reporte de calidad, porque el alcance determina qué artefactos son excluyentes.

**Errores frecuentes.** Aplicar umbrales absolutos de la literatura sin considerar que provienen de datasets con características de adquisición distintas a las propias (fondo suprimido, cobertura parcial, protocolo multibanda). Confundir "no veo un artefacto conocido" con "los datos están limpios", con una muestra de tres sujetos o un n<=100, la ausencia de un patrón no es evidencia de su ausencia en la población.

**Recomendaciones.** Documentar cada decisión de inclusión o exclusión con el criterio específico que la motivó, no solo el resultado. Mantener la tabla resumen de métricas (1.8) como referencia constante entre secciones, en lugar de reinterpretar cada valor desde cero.


### Referencias

- Morfini, F., Whitfield-Gabrieli, S., Nieto-Castañón, A. (2023). Functional connectivity MRI quality control procedures in CONN. *Frontiers in Neuroscience*, 17:1092125. doi:10.3389/fnins.2023.1092125
- Provins, C., MacNicol, E., Seeley, S.H., Hagmann, P., Esteban, O. (2023). Quality control in functional MRI studies with MRIQC and fMRIPrep. *Frontiers in Neuroimaging*, 1:1073734. doi:10.3389/fnimg.2022.1073734
- Kumar, V.A., Lee, J., Liu, H.-L., et al. (2024). Recommended resting-state fMRI acquisition and preprocessing steps for preoperative mapping of language, motor, and visual areas in adult and pediatric patients with brain tumors and epilepsy. *AJNR American Journal of Neuroradiology*, 45:139-148. doi:10.3174/ajnr.A8067
- Gorgolewski, K.J., Auer, T., Calhoun, V.D., et al. (2016). The brain imaging data structure, a format for organizing and describing outputs of neuroimaging experiments. *Scientific Data*, 3:160044.
- Power, J.D., Barnes, K.A., Snyder, A.Z., Schlaggar, B.L., Petersen, S.E. (2012). Spurious but systematic correlations in functional connectivity MRI networks arise from subject motion. *NeuroImage*, 59:2142-2154.
- Warrington, S., Ntata, A., Mougin, O., et al. (2023). A resource for development and comparison of multimodal brain 3T MRI harmonisation approaches. *Imaging Neuroscience*, 1. doi:10.1162/imag_a_00042


## Sección 2. Entorno de ejecución, datos y organización

### 2.1 Detección de entorno y nivel de ejecución

**Fundamento metodológico.** La reproducibilidad no consiste en que todos los entornos sean idénticos, sino en que el entorno concreto de cada ejecución quede registrado y sea recuperable. Warrington et al. (2023) muestran empíricamente por qué esto importa más de lo que suele asumirse, las opciones de procesamiento, no solo el equipo de adquisición, introducen variabilidad medible en las métricas derivadas, y en varios casos la variabilidad atribuible al procesamiento resulta comparable a la variabilidad biológica entre sujetos. La consecuencia práctica es que declarar "se usó fMRIPrep" no es información suficiente, hacen falta la versión, el contenedor, y las versiones de los binarios internos, que es lo que se registra en el manifiesto de software de la Sección 14.

**Explicación detallada.** Se definen dos niveles de ejecución:

- **Nivel A.** Los contenedores de MRIQC y fMRIPrep están accesibles. Las Secciones 4, 5 y 6 ejecutan las herramientas de referencia de la comunidad y producen sus reportes HTML.
- **Nivel B.** No hay contenedores disponibles. Las mismas etapas se cubren con una implementación nativa en Python (ANTs vía `antspyx`, sin binarios del sistema), que produce los mismos derivados pero no los reportes HTML de MRIQC ni de fMRIPrep.

Las Secciones 7 a 12 son idénticas en ambos niveles, porque consumen archivos NIfTI y tablas de confounds, no binarios. Tres cifras deciden si el nivel A es viable, y por eso se miden antes de descargar nada: memoria RAM, espacio libre en disco y número de núcleos.


In [ ]:
# 2.1  Deteccion de entorno, almacenamiento persistente y nivel de ejecucion
import os
import sys
import shutil
import platform
from pathlib import Path

import pandas as pd

IN_COLAB = "google.colab" in sys.modules

# Montaje de Google Drive. Se hace automaticamente al detectar Colab porque el
# disco de la sesion es efimero: al reiniciarse el entorno se pierden los datos
# descargados, la licencia de FreeSurfer y los derivados. Drive persiste entre
# sesiones y evita repetir esos pasos cada vez.
# La primera vez pide autorizacion en una ventana emergente.
DIR_PERSISTENTE = None
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DIR_PERSISTENTE = Path("/content/drive/MyDrive/rsfmri_preproc")
        DIR_PERSISTENTE.mkdir(parents=True, exist_ok=True)
        print(f"Drive montado. Almacenamiento persistente: {DIR_PERSISTENTE}")
    except Exception as exc:
        print(f"No se pudo montar Drive: {type(exc).__name__}: {exc}")
        print("El notebook funciona igualmente, pero los datos descargados y la")
        print("licencia se perderan al reiniciarse el entorno de ejecucion.")

# Raiz de trabajo. Se usa siempre el disco local de la sesion y no Drive, porque
# leer imagenes a traves del sistema de archivos de Drive es sensiblemente mas
# lento. Drive se usa solo como almacen persistente desde el que copiar.
ROOT = Path("/content/rsfmri") if IN_COLAB else Path.cwd()

PATHS = {
    "raiz": ROOT,
    "datos": ROOT / "data",             # dataset BIDS crudo, tal como se descarga
    "derivados": ROOT / "derivatives",  # salidas de MRIQC, fMRIPrep y de este notebook
    "trabajo": ROOT / "work",           # temporales de Nipype, se pueden borrar al terminar
    "cache": ROOT / "cache",            # descargas y resultados intermedios reutilizables
    "reportes": ROOT / "reports",       # figuras y tablas exportadas
}
for ruta in PATHS.values():
    ruta.mkdir(parents=True, exist_ok=True)

# Recursos disponibles. Los umbrales no son arbitrarios:
#   RAM: MRIQC consumio mas de 7 GB de memoria residente sobre un BOLD de 400
#        volumenes durante el desarrollo de este notebook, de modo que 8 GB es
#        el minimo practico y por debajo de esa cifra el proceso muere por OOM.
#   Disco: los contenedores (MRIQC 5.3 GB, fMRIPrep 2.7 GB) mas los derivados de
#        tres sujetos rondan entre 40 y 60 GB en el nivel A.
try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 1024 ** 3
except ImportError:
    ram_gb = float("nan")

disco_libre_gb = shutil.disk_usage(ROOT).free / 1024 ** 3
n_cpu = os.cpu_count()

RAM_MINIMA_GB = 8.0
DISCO_MINIMO_GB = 60.0


def detectar_backend():
    """Identifica el backend de contenedores disponible en este momento.

    Se define como funcion y no como una constante porque el resultado cambia
    despues de ejecutar la configuracion de Neurodesk (2.5): antes de ese paso
    /cvmfs no esta montado todavia. El resto del notebook la vuelve a llamar
    en lugar de asumir el valor calculado aqui.
    """
    # Neurodesk en Colab: CVMFS monta el arbol de modulos bajo demanda.
    if Path("/cvmfs/neurodesk.ardc.edu.au").exists():
        return "neurodesk-colab"
    # PyNeurodesk ejecuta los contenedores dentro de una maquina virtual y por
    # tanto exige virtualizacion por hardware expuesta como /dev/kvm.
    try:
        import neurodesk  # noqa: F401
        if Path("/dev/kvm").exists():
            return "pyneurodesk"
    except ImportError:
        pass
    # Binarios instalados directamente en el sistema (caso de una estacion de
    # trabajo con ANTs y AFNI ya configurados).
    if shutil.which("antsRegistration") and shutil.which("3dvolreg"):
        return "binarios-locales"
    return "nativo"


BACKEND = detectar_backend()
NIVEL = "A" if BACKEND in ("neurodesk-colab", "pyneurodesk", "binarios-locales") else "B"

ENV = {
    "Entorno": "Google Colab" if IN_COLAB else "Local",
    "Sistema": f"{platform.system()} {platform.release()}",
    "Python": platform.python_version(),
    "Nucleos de CPU": n_cpu,
    "RAM total (GB)": round(ram_gb, 1),
    "Disco libre (GB)": round(disco_libre_gb, 1),
    "Backend detectado": BACKEND,
    "Nivel de ejecucion": NIVEL,
    "Raiz de trabajo": str(ROOT),
    "Almacen persistente": str(DIR_PERSISTENTE) if DIR_PERSISTENTE else "no disponible",
}

print()
print(pd.Series(ENV).to_string(), "\n")

if ram_gb == ram_gb and ram_gb < RAM_MINIMA_GB:
    print(f"AVISO: RAM total {ram_gb:.1f} GB por debajo del minimo practico "
          f"de {RAM_MINIMA_GB:.0f} GB para MRIQC. El nivel A puede fallar por memoria.")
if disco_libre_gb < DISCO_MINIMO_GB:
    print(f"AVISO: disco libre {disco_libre_gb:.1f} GB por debajo de los "
          f"{DISCO_MINIMO_GB:.0f} GB recomendados para el nivel A completo. "
          f"Considere reducir a dos sujetos (2.8) o usar el nivel B.")
if NIVEL == "B":
    print("Nivel B: no se detectaron contenedores. Ejecute la celda 2.5 si esta "
          "en Colab, o 2.6 si esta en local con virtualizacion disponible.")


- **Nivel de ejecución y backend.** En una ejecución en Colab sin haber corrido todavía la celda 2.5, lo esperado es `nativo` y nivel `B`: el árbol de contenedores no está montado aún. Si tras ejecutar 2.5 el backend sigue siendo `nativo`, la configuración de Neurodesk falló y hay que revisar el registro antes de continuar, en lugar de asumir que las Secciones 4 a 6 funcionarán.
- **RAM.** El umbral de 8 GB no proviene de la documentación sino de una medición, MRIQC superó los 7 GB de memoria residente procesando un BOLD de 400 volúmenes y el proceso fue terminado por el sistema. Una sesión estándar de Colab reporta alrededor de 12.7 GB, suficiente para un sujeto a la vez pero **no para paralelizar** sujetos.
- **Disco libre.** Una sesión estándar de Colab dispone de aproximadamente 88 GB, por encima de los 60 GB de referencia. El margen se consume rápido: los contenedores de MRIQC y fMRIPrep suman 8 GB, el dataset de tres sujetos 441 MB, y el resto son derivados y temporales de Nipype.
- **Núcleos de CPU.** Una sesión estándar de Colab expone 2 núcleos. Esto no impide ejecutar fMRIPrep pero determina el tiempo, es el parámetro que hace que el preprocesamiento de un sujeto se mida en horas y no en minutos.

**Qué hacer si los recursos no alcanzan.** El orden de las medidas de mitigación, de menor a mayor pérdida de contenido, reducir de tres sujetos a dos, mantener `--fs-no-reconall` en fMRIPrep (2.4), ejecutar las Secciones 4 a 6 en sesiones separadas conservando solo los derivados y borrando el directorio de trabajo entre ellas, y como último recurso bajar al nivel B, que produce los mismos derivados sin los reportes HTML de referencia.


### 2.2 Por qué se ejecutan MRIQC y fMRIPrep en contenedores

**Introducción conceptual.** MRIQC y fMRIPrep se instalan desde PyPI como cualquier paquete de Python, y esa facilidad es engañosa, ninguno de los dos es una implementación autónoma en Python. Ambos son orquestadores construidos sobre Nipype que invocan binarios compilados de ANTs, AFNI, FSL y Connectome Workbench. Instalar el paquete de Python no instala esos binarios, y ninguno de ellos está disponible en PyPI.

**Fundamento metodológico.**

| Vía | Qué ocurre | Por qué falla |
|---|---|---|
| `pip install mriqc` | Instala sin errores y `mriqc --version` responde. Al procesar un sujeto real se detiene con `OSError: No command "3dvolreg" found` | `3dvolreg` es un binario de AFNI, no un módulo de Python. El paquete declara la dependencia funcional pero no puede satisfacerla |
| Instalación manual con conda-forge | ANTs, AFNI y Connectome Workbench se instalan en menos de un minuto y 1.6 GB | El paquete `afni` de conda-forge es un subconjunto: faltan `3dTshift` y `3dAutomask`, ambos requeridos. FSL no está en conda-forge y tiene instalador propio de 4 a 10 GB |
| PyNeurodesk en Colab | `nd.search()` funciona y devuelve las versiones disponibles. `nd.container()` falla con `kvm unavailable: /dev/kvm does not exist` | PyNeurodesk ejecuta los contenedores dentro de una máquina virtual y requiere virtualización por hardware. Colab no expone virtualización anidada |
| Contenedores Neurodesk vía Apptainer y CVMFS | Vía adoptada | El contenedor incluye los binarios ya compilados y con sus versiones fijadas. CVMFS los sirve bajo demanda en lugar de descargarlos completos |

**Explicación detallada.** Los contenedores relevantes y su tamaño declarado:

| Contenedor | Tamaño | Para qué se usa |
|---|---|---|
| `mriqc_24.0.2` | 5.3 GB | Sección 4, control de calidad de datos crudos |
| `fmriprep_25.2.5` | 2.7 GB | Secciones 5 y 6, preprocesamiento anatómico y funcional |
| `fsl_6.0.7.4` | 5.7 GB | Solo si se necesitan herramientas de FSL fuera de fMRIPrep |
| `ants_2.5.1` | 0.3 GB | Solo para operaciones de registro auxiliares |

En la práctica solo los dos primeros son necesarios: fMRIPrep incluye internamente las versiones de FSL, ANTs y AFNI que su release fija, de modo que no hace falta instalar ni montar esos contenedores por separado.

**Nota sobre versiones y honestidad en el registro.** Las instrucciones de este proyecto especifican FSL 6.0.7.7, mientras que el contenedor autónomo publicado por Neurodesk es 6.0.7.4. La discrepancia es menor pero no debe silenciarse, y sobre todo no debe resolverse suponiendo: la versión que realmente se ejecuta es la que fMRIPrep 25.2.5 empaqueta internamente, que puede diferir de ambas. Por eso la Sección 14 no declara las versiones esperadas sino que interroga a los binarios en tiempo de ejecución y registra lo que responden. Declarar una versión que no se verificó es un fallo de reproducibilidad, no un detalle administrativo.

**Estado de validación.** Los nombres y tamaños de los contenedores están verificados. El comportamiento de CVMFS sirviendo los contenedores bajo demanda en Colab, que es el supuesto sobre el que descansa toda la vía principal, se comprueba en la celda 2.5.


### 2.3 Justificación de cada herramienta

**Fundamento metodológico.** Las instrucciones del proyecto exigen que ninguna dependencia se instale sin justificar cuatro cosas: qué función cumple, por qué es necesaria, qué alternativas existen y qué etapa depende de ella. La tabla siguiente cumple ese requisito y sirve además como referencia para decidir qué se puede omitir cuando los recursos son limitados.

**Herramientas externas (dentro de contenedores)**

| Dependencia | Función | Por qué es necesaria | Alternativas | Etapa que depende |
|---|---|---|---|---|
| ANTs 2.5.1 | Corrección de sesgo N4, extracción cerebral, registro no lineal SyN, segmentación Atropos | Es el motor de normalización espacial de fMRIPrep y del registro a plantilla de MRIQC | FSL FNIRT, SPM DARTEL. Menos precisas en evaluaciones comparativas de registro | Secciones 4, 5, 6 |
| AFNI 24.0.05 | Estimación de movimiento (`3dvolreg`), máscaras automáticas (`3dAutomask`), corrección de tiempo de corte (`3dTshift`) | MRIQC calcula el movimiento con AFNI; sin `3dvolreg` no arranca | FSL MCFLIRT, SPM realign para el movimiento | Secciones 4, 6 |
| FSL 6.0.7.7 | Extracción cerebral (BET), segmentación tisular (FAST), registro lineal (FLIRT), corrección de distorsión (`topup`) | fMRIPrep usa FLIRT con coste de frontera apoyado en la segmentación de FAST para el corregistro, y `topup` para la corrección de distorsión con fieldmaps | Combinación de ANTs y AFNI cubre parte, no `topup` | Secciones 5, 6 |
| FreeSurfer 7.3.2 | Reconstrucción de superficies corticales (`recon-all`), corregistro basado en frontera (`bbregister`) | Opcional en este notebook. Ver 2.4: requiere licencia | El coste de frontera vía FLIRT y FAST sustituye a `bbregister` sin superficies | Sección 5 |
| Connectome Workbench | Manipulación de datos de superficie y salidas CIFTI | Solo necesario si se solicitan salidas de superficie a fMRIPrep, que aquí no se piden | No aplica | Ninguna en esta configuración |
| bids-validator 1.14.0 | Validación del árbol BIDS contra la especificación oficial | Detecta desviaciones del estándar que harían fallar a MRIQC y fMRIPrep más adelante | `pybids` más comprobaciones propias, que es lo que se añade además en la Sección 3 | Sección 3 |
| dcm2niix | Conversión de DICOM a NIfTI con generación de sidecars JSON | No aplica aquí: el dataset se publica ya en NIfTI y BIDS. Se explica en 2.10 por su valor formativo | `dicom2nifti`, herramientas del fabricante | Ninguna en esta configuración |


### 2.4 FreeSurfer: la licencia es obligatoria, se debe de obtener antes de continuar

**Requisito previo.** fMRIPrep no arranca sin una licencia válida de FreeSurfer. Es gratuita para uso académico, pero exige registro individual y no puede redistribuirse: no viene incluida en el contenedor ni puede compartirse entre usuarios. Este es el único lugar del notebook donde se trata el asunto; la celda siguiente lo resuelve y el resto de secciones da por hecho que está configurada.

**Pasos, una sola vez:**

1. Registrarse en `https://surfer.nmr.mgh.harvard.edu/registration.html`. El trámite lleva un par de minutos y la licencia llega por correo.
2. Guardar el archivo recibido como `freesurfer_license.txt` en el almacén persistente de Drive que se creó en 2.1. Al estar en Drive sobrevive al reinicio del entorno de ejecución y no habrá que repetir el paso en sesiones futuras. La celda 2.4b busca también otras rutas y, si encuentra la licencia en el disco local, la respalda automáticamente en Drive.
3. No incluir la licencia en ningún repositorio ni compartirla, porque sus términos de uso lo prohíben.

**La licencia es obligatoria incluso con `--fs-no-reconall`, y esto merece una advertencia.** Una versión anterior de este notebook afirmaba lo contrario. La ejecución real lo desmintió:

```
ERROR: a valid license file is required for FreeSurfer to run.
```

El fallo se produjo al construir el flujo de corregistro, que es donde fMRIPrep invoca utilidades de FreeSurfer con independencia de que se reconstruyan superficies o no. Se deja constancia porque es exactamente el tipo de suposición que este notebook no debe propagar: la afirmación parecía razonable, estaba escrita con seguridad, y era falsa.

**Qué sí cambia `--fs-no-reconall`.** Sigue siendo la opción adoptada por defecto, porque ahorra varias horas de cómputo por sujeto. Su alcance exacto, que suele describirse mal:

| Lo que `--fs-no-reconall` desactiva | Lo que conserva |
|---|---|
| Reconstrucción de superficies corticales (`recon-all`) | Segmentación tisular volumétrica en GM, WM y CSF |
| Salidas de superficie y mapeo a espacios de superficie (CIFTI) | Normalización espacial no lineal a plantilla MNI |
| Corregistro basado en frontera mediante `bbregister` | Corregistro con coste de frontera mediante FLIRT de FSL, apoyado en la segmentación de FAST |
| Métricas morfométricas de grosor, área y curvatura corticales | Máscaras tisulares erosionadas para extracción de componentes de ruido |
| Varias horas de cómputo por sujeto | La necesidad de licencia, que no desaparece |

Es decir, el corregistro no pierde el criterio de frontera, solo cambia la implementación que lo calcula. Lo que se pierde de verdad es el análisis basado en superficies y la morfometría, ninguno de los cuales forma parte del alcance definido para este notebook, que es conectividad funcional basada en volumen.

**Cuándo conviene activar FreeSurfer por completo.** Si el objetivo posterior incluye análisis vertex-wise, morfometría cortical o parcelaciones basadas en superficie, hay que ejecutar sin `--fs-no-reconall`, asumiendo varias horas adicionales por sujeto. Provins et al. (2023) señalan además que en ese escenario los criterios de control de calidad se endurecen, porque artefactos tolerables para un análisis volumétrico, como el anillado por movimiento en el T1w, invalidan las superficies reconstruidas y pasan a ser motivo de exclusión.

**Interruptor.** La celda 2.8 define `CFG.usar_freesurfer = False`. Ponerlo en `True` activa la reconstrucción de superficies. La licencia hace falta en ambos casos.


In [ ]:
# 2.4b  Licencia de FreeSurfer: localizacion, copia persistente y configuracion
import os
from pathlib import Path

NOMBRE_LICENCIA = "freesurfer_license.txt"

# Rutas donde se busca, en orden de preferencia. La de Drive va primera porque es
# la unica que sobrevive al reinicio del entorno de ejecucion.
candidatas = []
if DIR_PERSISTENTE is not None:
    candidatas += [DIR_PERSISTENTE / NOMBRE_LICENCIA,
                   Path("/content/drive/MyDrive") / NOMBRE_LICENCIA,
                   Path("/content/drive/MyDrive/license.txt")]
candidatas += [PATHS["cache"] / NOMBRE_LICENCIA,
               PATHS["raiz"] / "license.txt",
               Path.home() / NOMBRE_LICENCIA]

# Si la variable de entorno ya apunta a un archivo existente, se respeta.
if os.environ.get("FS_LICENSE") and Path(os.environ["FS_LICENSE"]).exists():
    candidatas.insert(0, Path(os.environ["FS_LICENSE"]))

origen = next((r for r in candidatas if r.exists() and r.stat().st_size > 0), None)

if origen is None:
    LICENCIA_DISPONIBLE = False
    LICENCIA_FS = None
    print("LICENCIA DE FREESURFER NO ENCONTRADA.")
    print("\nfMRIPrep no puede ejecutarse sin ella, ni siquiera con --fs-no-reconall.")
    print("Rutas comprobadas:")
    for ruta in candidatas:
        print(f"   {ruta}")
    print("\nQue hacer, una sola vez:")
    print("  1. Registrarse en https://surfer.nmr.mgh.harvard.edu/registration.html")
    print(f"  2. Guardar el license.txt recibido como {candidatas[0]}")
    print("  3. Volver a ejecutar esta celda")
else:
    LICENCIA_DISPONIBLE = True

    # Copia al almacen persistente si todavia no esta ahi, para no tener que
    # volver a colocarla en la siguiente sesion.
    if DIR_PERSISTENTE is not None:
        respaldo = DIR_PERSISTENTE / NOMBRE_LICENCIA
        if not respaldo.exists() or respaldo.read_bytes() != origen.read_bytes():
            respaldo.write_bytes(origen.read_bytes())
            print(f"Licencia respaldada en el almacen persistente: {respaldo}")

    # Copia de trabajo en el disco local de la sesion. Tiene que estar bajo una
    # ruta incluida en APPTAINER_BINDPATH para que el contenedor pueda leerla, y
    # leerla del disco local evita depender de Drive durante la ejecucion.
    LICENCIA_FS = PATHS["cache"] / NOMBRE_LICENCIA
    if not LICENCIA_FS.exists() or LICENCIA_FS.read_bytes() != origen.read_bytes():
        LICENCIA_FS.write_bytes(origen.read_bytes())

    os.environ["FS_LICENSE"] = str(LICENCIA_FS)
    print(f"Licencia de FreeSurfer configurada.")
    print(f"  Encontrada en: {origen}")
    print(f"  En uso:        {LICENCIA_FS} ({LICENCIA_FS.stat().st_size} bytes)")
    print(f"  FS_LICENSE:    {os.environ['FS_LICENSE']}")
    print("\nNo se vuelve a mencionar en el resto del notebook: queda resuelto aqui.")


### 2.5 Configuración de Neurodesk en Colab (vía principal)



In [ ]:
%%capture registro_neurodesk
# 2.5  Configuracion de Neurodesk en Colab: Apptainer, CVMFS y Lmod
# Solo se ejecuta en Colab: en local la via equivalente es PyNeurodesk (2.6).
import os
import sys

if "google.colab" in sys.modules:
    # Colab precarga una biblioteca que interfiere con los binarios del contenedor.
    os.environ["LD_PRELOAD"] = ""
    # Directorios del anfitrion que seran visibles dentro del contenedor.
    # Sin /content el contenedor no puede leer los datos ni escribir derivados.
    os.environ["APPTAINER_BINDPATH"] = "/content,/tmp,/cvmfs"
    # Directorio escribible para la cache de matplotlib dentro del contenedor.
    os.environ["MPLCONFIGDIR"] = "/content/matplotlib-mpldir"
    # Ruta del ejecutable de Lmod, necesaria para que el paquete `lmod` de Python
    # pueda invocar al gestor de modulos.
    os.environ["LMOD_CMD"] = "/usr/share/lmod/lmod/libexec/lmod"

    # Instala Apptainer, monta CVMFS y configura Lmod. Tarda varios minutos.
    !curl -J -O https://raw.githubusercontent.com/NeuroDesk/neurocommand/main/googlecolab_setup.sh
    !chmod +x googlecolab_setup.sh
    !./googlecolab_setup.sh
else:
    print("No es Colab: omitida la configuracion de Neurodesk. Use 2.6 para el backend local.")


In [ ]:
# 2.5b  Verificacion de la configuracion de Neurodesk
# Comprueba las tres cosas que tienen que haber ocurrido en la celda anterior
# CVMFS montado, arbol de modulos visible, y los dos modulos que necesitamos
# presentes en ese arbol.
import os
import sys
from pathlib import Path

# Versiones fijadas de forma explicita. No se acepta la que resuelva por defecto
# la version del contenedor es parte del registro de reproducibilidad,
# y dejarla implicita significa que dos ejecuciones separadas en el tiempo pueden
# usar herramientas distintas sin que quede constancia.
CONTENEDORES = {"mriqc": "24.0.2", "fmriprep": "25.2.5"}

RAIZ_MODULOS = Path("/cvmfs/neurodesk.ardc.edu.au/neurodesk-modules")

if "google.colab" in sys.modules:
    # Ultimas lineas del registro de instalacion. Si algo fallo, el error suele
    # aparecer aqui; el registro completo esta en `registro_neurodesk.stdout`.
    lineas = registro_neurodesk.stdout.strip().splitlines()
    print("Final del registro de instalacion:")
    for linea in lineas[-4:]:
        print("   ", linea)
    print()

# Comprobacion 1: CVMFS montado y arbol de modulos accesible.
print(f"CVMFS montado: {RAIZ_MODULOS.exists()}")

if RAIZ_MODULOS.exists():
    subdirs = sorted(d for d in RAIZ_MODULOS.iterdir() if d.is_dir())
    os.environ["MODULEPATH"] = ":".join(str(d) for d in subdirs)
    print(f"Categorias de modulos disponibles: {len(subdirs)}")

    for herramienta, solicitada in CONTENEDORES.items():
        versiones = sorted({p.name.removesuffix(".lua")
                            for p in RAIZ_MODULOS.glob(f"*/{herramienta}/*")})
        marca = "disponible" if solicitada in versiones else "NO DISPONIBLE"
        print(f"  {herramienta}: {len(versiones)} versiones publicadas. "
              f"Version solicitada {solicitada}: {marca}")
        print(f"    {', '.join(versiones)}")
else:
    print("CVMFS no esta montado. La via principal no esta disponible en esta sesion.")
    print("Revise el registro completo con: print(registro_neurodesk.stdout)")

BACKEND = detectar_backend()
NIVEL = "A" if BACKEND in ("neurodesk-colab", "pyneurodesk", "binarios-locales") else "B"
print(f"\nBackend: {BACKEND}   Nivel de ejecucion: {NIVEL}")


**Interpretación de resultados.** La celda produce cuatro comprobaciones que hay que leer en orden, porque cada una es condición de la siguiente.

1. **`CVMFS montado: True`.** Es la comprobación crítica. Confirma que el sistema de archivos distribuido está montado y que el árbol de contenedores es accesible sin haberlos descargado. Si aquí aparece `False`, nada de lo demás importa y hay que revisar el registro completo con `print(registro_neurodesk.stdout)` antes de continuar.
2. **Número de categorías de módulos.** Un valor del orden de varias decenas indica que el árbol se está sirviendo correctamente. Un valor de cero con CVMFS montado indicaría un montaje incompleto.
3. **Versiones publicadas de cada herramienta y disponibilidad de la solicitada.** Interesa la marca `disponible` para las versiones fijadas en `CONTENEDORES`.
4. **Backend y nivel de ejecución.** Debe haber cambiado de `nativo` y nivel `B` a `neurodesk-colab` y nivel `A`. Si sigue en `nativo` pese a que CVMFS aparece montado, el problema está en la función de detección, no en la instalación.

**Resultado de la validación en este notebook.** La ejecución en una sesión estándar de Colab devolvió CVMFS montado, 32 categorías de módulos, MRIQC 24.0.2 y fMRIPrep 25.2.5 disponibles, y nivel de ejecución A. Esto confirma empíricamente el supuesto sobre el que descansa la vía principal, CVMFS sirve los contenedores bajo demanda en Colab sin necesidad de descargar los 8 GB que suman ambas imágenes, y sin requerir virtualización por hardware. Es la diferencia entre poder ejecutar las herramientas de referencia de la comunidad en un entorno gratuito o tener que renunciar a ellas.

**Errores frecuentes en este paso.** Ejecutar la celda 2.5 más de una vez en la misma sesión reinstala innecesariamente y puede dejar el `MODULEPATH` en un estado inconsistente; si hay que repetirla, conviene reiniciar el entorno de ejecución primero. Cargar un módulo con `module load` desde una celda y esperar que el binario esté disponible en una celda posterior no funciona: cada invocación de shell es un proceso independiente, y por eso la capa de abstracción de 2.7 carga el módulo y ejecuta la herramienta en la misma orden.


### 2.6 Backend local con PyNeurodesk

**Introducción conceptual.** PyNeurodesk es el envoltorio en Python de los mismos contenedores de Neurodesk, pensado para uso local. Su ventaja frente al montaje manual de Apptainer es que resuelve la versión del contenedor a partir de los metadatos de release y prepara la descarga automáticamente. Su limitación es la que se documentó en 2.2, ejecuta los contenedores dentro de una máquina virtual, de modo que requiere virtualización por hardware expuesta como `/dev/kvm`, y por eso no funciona en Colab.

- El flujo tiene tres pasos. Primero `nd.search()` consulta qué versiones del contenedor están disponibles, lo que permite fijar una versión explícita en lugar de aceptar la que resuelva por defecto. Segundo, `nd.container()` prepara el contenedor, lo que en la primera invocación implica descarga y puede tardar. Tercero, `nd.share_dir()` monta un directorio del anfitrión dentro del contenedor, necesario para que la herramienta pueda leer el dataset y escribir los derivados: el directorio compartido aparece dentro del contenedor bajo una ruta del tipo `/.share/<id-de-sesion>`, que es la que hay que pasar como argumento a la herramienta, no la ruta del anfitrión.

- La instalación previa requerida es:

```
python3 -m venv neurodesk-python
source neurodesk-python/bin/activate
python -m pip install --upgrade pip
python -m pip install neurodesk
```


In [ ]:
# 2.6  Backend local con PyNeurodesk
# NO VALIDADO en Colab, requiere /dev/kvm.
# Se ejecuta solo si NO estamos en Colab y la virtualizacion esta disponible;
# en cualquier otro caso la celda informa y no hace nada, de modo que puede
# ejecutarse siempre sin efectos secundarios.
# Reutiliza el diccionario CONTENEDORES definido en 2.5b.
import sys
from pathlib import Path


nd = None
contenedores_preparados = {}
dir_compartido = None

if "google.colab" in sys.modules:
    print("Entorno Colab: PyNeurodesk no es aplicable (no hay virtualizacion anidada).")
    print("La via correcta en Colab es la celda 2.5.")
elif not Path("/dev/kvm").exists():
    print("No existe /dev/kvm: la virtualizacion por hardware no esta disponible o no")
    print("esta expuesta al usuario. PyNeurodesk no puede arrancar la maquina virtual.")
    print("Alternativas: activar la virtualizacion en la BIOS y anadir el usuario al")
    print("grupo kvm, o ejecutar el notebook en Colab (2.5), o continuar en nivel B.")
else:
    try:
        import neurodesk as nd

        # Paso 1: consultar versiones disponibles antes de fijar una. Se registra
        # el resultado porque forma parte de la trazabilidad (Seccion 14).
        for herramienta in CONTENEDORES:
            print(f"Versiones disponibles de {herramienta}:")
            print("   ", nd.search(herramienta))

        # Paso 2: preparar cada contenedor. La primera invocacion descarga y
        # prepara los archivos necesarios; las siguientes reutilizan la cache local.
        for herramienta, version in CONTENEDORES.items():
            contenedores_preparados[herramienta] = nd.container(herramienta)
            print(f"Contenedor preparado: {herramienta} (version solicitada {version})")

        # Paso 3: compartir el directorio de trabajo. La ruta que hay que pasar a
        # las herramientas es `dir_compartido.guest_path`, la ruta VISTA DESDE EL
        # CONTENEDOR, no la del anfitrion. Confundirlas es el error mas frecuente
        # al usar PyNeurodesk.
        dir_compartido = nd.share_dir(PATHS["raiz"], writable=True)
        print(f"Directorio compartido: {PATHS['raiz']} visible como {dir_compartido.guest_path}")

    except ImportError:
        print("El paquete `neurodesk` no esta instalado. Instalelo con: pip install neurodesk")
    except Exception as exc:
        # Se captura de forma amplia y deliberada: los modos de fallo de la maquina
        # virtual son variados y lo importante es no interrumpir el notebook, ya que
        # el nivel B sigue siendo una via valida.
        print(f"PyNeurodesk no pudo inicializarse: {type(exc).__name__}: {exc}")
        print("El notebook puede continuar en nivel B.")


### 2.7 Capa de abstracción: una sola función para invocar herramientas



In [ ]:
# 2.7  Capa de abstraccion sobre los backends
# Toda la diferencia entre Colab, PyNeurodesk y binarios locales queda contenida
# aqui, junto con la infraestructura de ejecucion en segundo plano que usan las
# Secciones 4, 5 y 6. Se define en esta seccion y no en la 4 para que las
# secciones posteriores no dependan de haber ejecutado la 4: cada seccion debe
# poder ejecutarse tras las de configuracion, sin arrastres innecesarios.
import os
import shlex
import subprocess
import time
from pathlib import Path

# Script de inicializacion de Lmod. Ver la explicacion en `_orden_con_modulo`.
INIT_LMOD = "/usr/share/lmod/lmod/init/bash"

# TemplateFlow descarga las plantillas que MRIQC y fMRIPrep usan para registrar
# cada imagen al espacio estandar. Se apunta a un directorio bajo la raiz de
# trabajo por dos razones: esta dentro de APPTAINER_BINDPATH y por tanto es
# visible desde el contenedor, y persiste entre ejecuciones de una misma sesion,
# de modo que no se vuelve a descargar.
TEMPLATEFLOW_HOME = PATHS["cache"] / "templateflow"
TEMPLATEFLOW_HOME.mkdir(parents=True, exist_ok=True)

# Entorno comun que se pasa a toda herramienta del contenedor.
ENTORNO_HERRAMIENTAS = {"TEMPLATEFLOW_HOME": TEMPLATEFLOW_HOME}
if os.environ.get("FS_LICENSE"):
    ENTORNO_HERRAMIENTAS["FS_LICENSE"] = os.environ["FS_LICENSE"]


class HerramientaNoDisponible(RuntimeError):
    """La herramienta solicitada no puede ejecutarse con el backend actual."""


def _orden_con_modulo(herramienta, version, invocaciones, entorno=None):
    """Construye la orden de shell que carga el modulo y ejecuta la herramienta.

    Dos detalles aqui no son evidentes y ambos se determinaron probando, porque
    su sintoma es el mismo mensaje poco informativo de Lmod diciendo que el
    modulo es desconocido:

    1. Se usa `bash -c` y NO `bash -lc`. Un shell de login vuelve a leer los
       archivos de perfil del sistema, que reescriben MODULEPATH y descartan las
       rutas del arbol de Neurodesk configuradas en 2.5b. Con `bash -lc` la
       variable pasa de 32 rutas a 2 y no se encuentra ningun modulo.

    2. Un shell no interactivo no define la orden `module`, que es una funcion
       de shell y no un ejecutable. Hay que cargar el script de inicializacion
       de Lmod de forma explicita, y reexportar MODULEPATH despues de hacerlo
       para que la configuracion propia prevalezca.
    """
    exportaciones = "".join(f"export {clave}={shlex.quote(str(valor))} && "
                            for clave, valor in (entorno or {}).items())
    return (f"source {INIT_LMOD} && "
            f"export MODULEPATH={shlex.quote(os.environ['MODULEPATH'])} && "
            f"{exportaciones}module load {herramienta}/{version} && ( {invocaciones} )")


def ejecutar_herramienta(herramienta, argumentos, version=None, mostrar_registro=True):
    """Ejecuta una herramienta externa y ESPERA a que termine.

    Sirve para ordenes de segundos, como consultar una version. Para MRIQC o
    fMRIPrep, que tardan de decenas de minutos a horas, use la variante asincrona
    `lanzar_secuencia_en_segundo_plano`.

    Devuelve (codigo_salida, registro).
    """
    version = version or CONTENEDORES.get(herramienta)
    backend = detectar_backend()
    argumentos = [str(a) for a in argumentos]

    if backend == "neurodesk-colab":
        invocacion = (f"{herramienta} "
                      + " ".join(shlex.quote(a) for a in argumentos))
        orden = _orden_con_modulo(herramienta, version, invocacion,
                                  ENTORNO_HERRAMIENTAS)
        proceso = subprocess.run(["bash", "-c", orden], capture_output=True, text=True)

    elif backend == "pyneurodesk":
        if herramienta not in contenedores_preparados:
            raise HerramientaNoDisponible(
                f"El contenedor de {herramienta} no esta preparado. Ejecute la celda 2.6.")
        # PyNeurodesk devuelve la salida como texto y no un objeto de proceso, de
        # modo que el codigo de salida se asume 0 si no lanzo excepcion.
        salida = contenedores_preparados[herramienta].run(herramienta, *argumentos)
        proceso = subprocess.CompletedProcess(args=argumentos, returncode=0,
                                              stdout=str(salida), stderr="")

    elif backend == "binarios-locales":
        if shutil.which(herramienta) is None:
            raise HerramientaNoDisponible(
                f"{herramienta} no esta en el PATH del sistema. Hay binarios de ANTs y "
                f"AFNI disponibles, pero no {herramienta}.")
        proceso = subprocess.run([herramienta] + argumentos,
                                 capture_output=True, text=True)

    else:
        raise HerramientaNoDisponible(
            f"No hay backend de contenedores disponible para ejecutar {herramienta}.\n"
            f"En Colab: ejecute la celda 2.5.\n"
            f"En local con virtualizacion: ejecute la celda 2.6.\n"
            f"Sin ninguna de las dos: use la implementacion nativa (nivel B) de la "
            f"seccion correspondiente.")

    registro = (proceso.stdout or "") + (proceso.stderr or "")
    if mostrar_registro:
        print(registro)
    return proceso.returncode, registro


def lanzar_secuencia_en_segundo_plano(herramienta, lista_argumentos, archivo_log,
                                      version=None, entorno=None):
    """Ejecuta una herramienta una o varias veces, en serie, SIN bloquear el notebook.

    Devuelve el identificador de proceso. El registro completo queda en
    `archivo_log`, que se consulta con `seguir_registro`.

    El proceso se lanza con nohup para que sobreviva a la finalizacion de la
    celda: sin ello, muere en cuanto el shell que lo lanzo termina.

    Las invocaciones se separan por `;` y no por `&&`, de modo que el fallo de un
    sujeto no impida procesar los siguientes. La deteccion de fallos se hace
    contando los productos en disco, no por el codigo de salida del conjunto.
    """
    version = version or CONTENEDORES.get(herramienta)
    backend = detectar_backend()
    if backend != "neurodesk-colab":
        raise HerramientaNoDisponible(
            f"Implementado para el backend neurodesk-colab. Backend actual: {backend}.")

    invocaciones = " ; ".join(
        f"echo '=== {herramienta}: invocacion {i + 1} de {len(lista_argumentos)} ===' ; "
        f"{herramienta} " + " ".join(shlex.quote(str(a)) for a in argumentos)
        for i, argumentos in enumerate(lista_argumentos))

    orden = _orden_con_modulo(herramienta, version, invocaciones,
                              {**ENTORNO_HERRAMIENTAS, **(entorno or {})})
    completo = (f"nohup bash -c {shlex.quote(orden)} "
                f"> {shlex.quote(str(archivo_log))} 2>&1 & echo $!")
    resultado = subprocess.run(["bash", "-c", completo], capture_output=True, text=True)
    return int(resultado.stdout.strip())


def proceso_vivo(pid):
    """Comprueba si un proceso sigue en ejecucion sin enviarle ninguna senal."""
    try:
        os.kill(pid, 0)          # la senal 0 no hace nada, solo comprueba
    except ProcessLookupError:
        return False
    except PermissionError:
        return True              # existe pero pertenece a otro usuario
    return True


def seguir_registro(archivo_log, n=15):
    """Devuelve las ultimas n lineas del registro."""
    archivo_log = Path(archivo_log)
    if not archivo_log.exists():
        return "(el registro todavia no existe)"
    lineas = archivo_log.read_text(errors="replace").splitlines()
    return "\n".join(lineas[-n:])


def procesos_de_computo_activos():
    """Devuelve las lineas de `ps` de los binarios de computo que esten corriendo.

    Es el indicador FIABLE de si una herramienta lanzada en segundo plano sigue
    trabajando, frente al registro o a las fechas de los archivos, que pueden
    quedarse quietos durante una etapa larga sin que haya problema alguno.
    """
    patron = ("mriqc|fmriprep|3dvolreg|3dTshift|antsRegist|N4Bias|synthstrip|"
              "melodic|flirt|fast|topup|applytopup")
    return subprocess.run(
        ["bash", "-c",
         f"ps -eo pid,pcpu,pmem,etime,comm --sort=-pcpu | "
         f"grep -Ei '{patron}' | grep -v grep | head -10"],
        capture_output=True, text=True).stdout.strip()


print(f"Capa de abstraccion definida. Backend actual: {detectar_backend()}")
print(f"TemplateFlow: {TEMPLATEFLOW_HOME}")
print(f"Licencia de FreeSurfer en el entorno de las herramientas: "
      f"{'si' if 'FS_LICENSE' in ENTORNO_HERRAMIENTAS else 'NO'}")


### 2.8 Selección de sesión y de sujetos

El dataset ON-Harmony contiene 20 sujetos escaneados en varios equipos y sitios. La selección no es arbitraria, se eligen primero la sesión y después los sujetos, y en ese orden, porque la sesión determina qué correcciones son posibles.

- ON-Harmony es un diseño de sujetos viajeros, concebido precisamente para medir cuánta variabilidad introduce el equipo. Warrington et al. (2023) cuantifican el resultado, la variabilidad entre escáneres llega a ser del orden de cinco veces la variabilidad entre repeticiones en el mismo escáner, y para varios grupos de métricas iguala o supera la variabilidad biológica entre sujetos. En sus propias palabras, la similitud de las métricas del mismo sujeto escaneado en equipos distintos puede ser tan baja como la de sujetos distintos escaneados en el mismo equipo. Además señalan que las métricas derivadas de rs-fMRI son las más variables de todas las modalidades que evaluaron.

La consecuencia para este notebook es directa, si se mezclaran sesiones de escáneres distintos, cualquier diferencia observada entre sujetos en las métricas de calidad quedaría confundida con el efecto del equipo, y las comparaciones entre sujetos de las Secciones 4, 7 y 11 dejarían de ser interpretables. Por eso se fija un único escáner y una única sesión para los tres sujetos.

**Sesión elegida: `ses-NOT1ACH001`.** Nottingham, Philips Achieva 3T:

1. Está presente en todos los sujetos del dataset, lo que permite cambiar de sujetos sin cambiar de sesión.
2. Incluye **fieldmaps** de eco de espín con codificación de fase opuesta (`dir-AP` y `dir-PA`), que hacen posible la **corrección de distorsión por susceptibilidad con datos medidos**.

**Sujetos elegidos.** Tres sujetos de la fase A, escogidos para cubrir un rango de edad y ambos sexos sin salirse de la sesión fijada:

| Sujeto | Sexo | Edad |
|---|---|---|
| `sub-03286` | M | 48 |
| `sub-14229` | M | 35 |
| `sub-12813` | F | 24 |


**Verificación de la adquisición frente a las recomendaciones de consenso.** Kumar et al. (2024) publican recomendaciones de adquisición para rs-fMRI. Comprobarlas antes de procesar es barato y evita descubrir tarde que los datos no soportan el análisis previsto:

| Recomendación | Este dataset | Cumple |
|---|---|---|
| Duración mínima de 6 minutos | 400 volúmenes por 1.15 s, es decir 7.67 minutos | Sí |
| TR <= a 2 s | 1.15 s | Sí |
| Campo de 3T o superior | 3T | Sí |
| Ojos abiertos con fijación | Ojos abiertos, sin fijación documentada | Parcialmente |
| rs-fMRI antes de fMRI de tarea | No hay fMRI de tarea en el protocolo | No aplica |
| rs-fMRI antes de administrar contraste | No se administró contraste | No aplica |
| Monitorización fisiológica (opcional) | No adquirida | Opcional, no adquirida |

La única desviación relevante es la fijación visual. Kumar et al. la recomiendan porque reduce la probabilidad de que el participante se duerma, y citan evidencia de que con ojos cerrados un tercio de los participantes se duerme a los 4 minutos y la mitad a los 10, con cambios medibles en la conectividad de las redes sensoriomotora y visual. Aquí los participantes mantuvieron los ojos abiertos, lo que mitiga el problema, pero la ausencia de fijación queda registrada como limitación en la Sección 15. La ausencia de monitorización fisiológica implica que no se pueden aplicar métodos de corrección basados en registro externo, y que la aproximación al ruido cardíaco y respiratorio recae por completo en aCompCor (Sección 9).


#### 2.8.1 **Selección de umbral**

- Banda de frecuencias: [0.008, 0.09]
- fd (mm): 0.5 (conservador 0.3, normal: 0.5, liberal 0.9)
- Cambio en GS (sd): 3 (conservador 2, normal: 3, liberal 5)

In [ ]:
# 2.8  Configuracion del analisis: sujetos, sesion y parametros globales

from dataclasses import dataclass, field, asdict
import random
import numpy as np


@dataclass
class Configuracion:
    """Parametros del analisis. Cambiar aqui, no en las celdas posteriores."""

    # Dataset y seleccion
    dataset: str = "ds004712"
    sesion: str = "NOT1ACH001"          # Nottingham, Philips Achieva 3T
    tarea: str = "rest"
    sujetos: list = field(default_factory=lambda: ["03286", "14229", "12813"])

    # Plantilla de normalizacion. MNI152NLin2009cAsym es la salida por defecto de
    # fMRIPrep y la que usan Provins et al. (2023); fijarla explicitamente evita
    # depender del valor por defecto de una version futura de la herramienta.
    plantilla: str = "MNI152NLin2009cAsym"

    # FreeSurfer desactivado por defecto. La licencia de FreeSurfer hace falta en
    # ambos casos, tambien con --fs-no-reconall: sin ella fMRIPrep aborta (ver
    # 2.4, donde se documenta con la traza del error). Poner esta opcion en True
    # no cambia ese requisito, solo anade varias horas de computo por sujeto.
    usar_freesurfer: bool = False

    # Umbrales de deteccion de volumenes atipicos. Son los valores de referencia
    # habituales en la literatura de conectividad funcional; se justifican y se
    # someten a analisis de sensibilidad en la Seccion 7.
    umbral_fd_mm: float = 0.5
    umbral_gschange_sd: float = 3.0

    # Banda de frecuencias del filtrado temporal (Seccion 10).
    filtro_hz: tuple = (0.008, 0.09)

    # Semilla para cualquier procedimiento con componente aleatorio, de modo que
    # dos ejecuciones del notebook produzcan resultados identicos.
    semilla: int = 19931103

    @property
    def id_sujetos(self):
        """Identificadores completos en formato BIDS."""
        return [f"sub-{s}" for s in self.sujetos]

    @property
    def id_sesion(self):
        return f"ses-{self.sesion}"


CFG = Configuracion()

# INTERRUPTOR: reducir a dos sujetos cuando la RAM, el disco o el
# tiempo disponible no alcanzan para tres. Se conserva el primero y el ultimo
# para no perder el rango de edad de la muestra.
REDUCIR_A_DOS_SUJETOS = False
if REDUCIR_A_DOS_SUJETOS:
    CFG.sujetos = [CFG.sujetos[0], CFG.sujetos[-1]]
    print(f"Muestra reducida a {len(CFG.sujetos)} sujetos por limitacion de recursos.")


random.seed(CFG.semilla)
np.random.seed(CFG.semilla)

print(f"Sujetos:   {', '.join(CFG.id_sujetos)}")
print(f"Sesion:    {CFG.id_sesion}")
print(f"Tarea:     task-{CFG.tarea}")
print(f"Plantilla: {CFG.plantilla}")
print(f"FreeSurfer: {'activado' if CFG.usar_freesurfer else 'desactivado (--fs-no-reconall)'}")
print(f"Semilla:   {CFG.semilla}")


### 2.9 Descarga de los datos: DataLad frente a AWS S3

**Introducción conceptual.** OpenNeuro ofrece dos métodos de descarga con propiedades distintas. La elección no es cuestión de gusto, afecta al espacio requerido, al tiempo, y a si la procedencia de los datos queda registrada de forma verificable.

| | DataLad | AWS S3 |
|---|---|---|
| Qué instala | `datalad` y `git-annex`, que a su vez requiere Git | Nada, o el cliente `awscli` si se usa `aws s3 sync` |
| Cómo funciona | Clona un repositorio Git con los metadatos y descarga el contenido de los archivos bajo demanda | Descarga directa por HTTP de cada archivo |
| Descarga selectiva | Sí, con `datalad get` sobre rutas concretas | Sí, filtrando por prefijo o construyendo la lista de archivos |
| Trazabilidad | Alta: la versión exacta del dataset queda fijada por el commit, y las sumas de verificación las gestiona `git-annex` | Baja: hay que registrar manualmente qué se descargó y cuándo |
| Reanudación | Sí, gestionada por la herramienta | Manual, comprobando qué archivos existen ya |
| Espacio adicional | El repositorio Git y los metadatos de `git-annex`, además de los datos | Solo los datos |
| Requisitos | Instalación previa de tres componentes | Ninguno con la biblioteca estándar de Python |
| Tiempo para este subconjunto | Comparable, dominado por el ancho de banda | Comparable, dominado por el ancho de banda |

**Decisión adoptada y justificación.** Se implementa la vía HTTP directa sobre el bucket público de S3, con caché local. La razón es que en Colab cada sesión parte de un entorno limpio, y añadir la instalación de `git-annex` introduce un punto de fallo adicional para un beneficio que aquí no se materializa, la trazabilidad de DataLad es valiosa cuando se itera sobre el dataset a lo largo de meses. La trazabilidad se cubre por otra vía, calculando y registrando las sumas de verificación de las entradas en la Sección 14, que es lo que realmente se necesita para reproducir el análisis.

Los comandos equivalentes con las otras vías, documentados para quien prefiera usarlas:

```
# DataLad: clona los metadatos y descarga solo lo solicitado
datalad install https://github.com/OpenNeuroDatasets/ds004712.git
cd ds004712
datalad get sub-03286/ses-NOT1ACH001

# AWS CLI: sincroniza el dataset completo, unos 40 GB
aws s3 sync --no-sign-request s3://openneuro.org/ds004712 ds004712-download/
```

Conviene notar la diferencia de escala: `aws s3 sync` sobre la raíz del dataset descarga los 20 sujetos con todas sus sesiones y modalidades, mientras que aquí se necesita son cuatro archivos de imagen por sujeto. Por eso la celda siguiente construye la lista explícita de archivos en lugar de sincronizar un prefijo.

**Explicación detallada de la caché.** La función comprueba la existencia y el tamaño de cada archivo antes de descargarlo. Volver a ejecutar la celda no vuelve a descargar nada, lo que importa porque en el desarrollo de un notebook las celdas se ejecutan muchas veces. En local, si el dataset ya está en `data/`, la celda lo detecta y no hace nada.


In [ ]:
# 2.9  Descarga selectiva desde OpenNeuro con cache local y respaldo persistente
import urllib.request
import urllib.error
import shutil
import time

import pandas as pd

S3_BASE = f"https://s3.amazonaws.com/openneuro.org/{CFG.dataset}"
DIR_BIDS = PATHS["datos"] / CFG.dataset

# Copia persistente en Drive. Al reiniciarse el entorno de ejecucion se pierde el
# disco de la sesion, de modo que sin respaldo habria que volver a descargar en
# cada sesion nueva. Con respaldo, la segunda sesion copia desde Drive.
DIR_BIDS_PERSISTENTE = (DIR_PERSISTENTE / "data" / CFG.dataset
                        if DIR_PERSISTENTE is not None else None)

# Archivos de nivel superior. Son pequenos pero imprescindibles: sin
# dataset_description.json el validador BIDS y las propias herramientas
# rechazan el directorio por no reconocerlo como un dataset.
ARCHIVOS_RAIZ = ["dataset_description.json", "participants.tsv",
                 "participants.json", "README", "CHANGES"]


def archivos_de_sujeto(sujeto):
    """Rutas relativas de los cuatro archivos de imagen y sus sidecars.

    Se listan explicitamente en lugar de sincronizar un prefijo completo porque
    cada sujeto tiene varias sesiones en el dataset publicado y solo se necesita
    una. Cada imagen va acompanada de su JSON: sin el sidecar se pierden TR, TE,
    direccion de codificacion de fase y tiempo de lectura, que son los
    parametros que gobiernan el preprocesamiento.
    """
    base = f"{sujeto}/{CFG.id_sesion}"
    prefijo = f"{sujeto}_{CFG.id_sesion}"
    rutas = [
        f"{base}/anat/{prefijo}_T1w",
        f"{base}/func/{prefijo}_task-{CFG.tarea}_bold",
        f"{base}/fmap/{prefijo}_dir-AP_epi",
        f"{base}/fmap/{prefijo}_dir-PA_epi",
    ]
    # Cada entrada genera dos archivos: la imagen y su sidecar JSON.
    return [f"{r}{ext}" for r in rutas for ext in (".nii.gz", ".json")]


def obtener(ruta_relativa, reintentos=3):
    """Consigue un archivo por la via mas barata disponible.

    Orden de preferencia: ya esta en el disco local, esta en el respaldo de Drive,
    o hay que descargarlo de OpenNeuro. Devuelve (estado, bytes).
    """
    destino = DIR_BIDS / ruta_relativa
    if destino.exists() and destino.stat().st_size > 0:
        return "en disco local", destino.stat().st_size

    destino.parent.mkdir(parents=True, exist_ok=True)

    # Segunda via: el respaldo persistente. Evita volver a descargar en cada
    # sesion nueva de Colab.
    if DIR_BIDS_PERSISTENTE is not None:
        respaldo = DIR_BIDS_PERSISTENTE / ruta_relativa
        if respaldo.exists() and respaldo.stat().st_size > 0:
            shutil.copyfile(respaldo, destino)
            return "copiado de Drive", destino.stat().st_size

    # Tercera via: descarga desde OpenNeuro.
    url = f"{S3_BASE}/{ruta_relativa}"
    for intento in range(reintentos):
        try:
            # Descarga a un archivo temporal y renombrado atomico al final: si la
            # conexion se corta a medias no queda un archivo truncado que la
            # cache daria luego por valido.
            temporal = destino.with_suffix(destino.suffix + ".parcial")
            with urllib.request.urlopen(url, timeout=60) as respuesta, \
                    open(temporal, "wb") as salida:
                shutil.copyfileobj(respuesta, salida)
            temporal.rename(destino)
            return "descargado", destino.stat().st_size
        except urllib.error.HTTPError as exc:
            # Un 404 no se reintenta: el archivo no existe en el bucket.
            return f"ERROR HTTP {exc.code}", 0
        except Exception:
            if intento == reintentos - 1:
                return "ERROR de red", 0
            time.sleep(2 ** intento)  # espera creciente entre reintentos
    return "ERROR", 0


def sincronizar_respaldo(rutas_relativas):
    """Asegura que todo lo que hay en local esta tambien en el respaldo de Drive.

    Se hace como paso APARTE y no dentro de `obtener` por una razon concreta: si
    los archivos ya estaban en el disco local, `obtener` devuelve de inmediato y
    nunca llegaria a crear el respaldo. Ese hueco existia en una version anterior
    de esta celda y significaba que un dataset ya descargado nunca se respaldaba,
    de modo que la siguiente sesion tenia que volver a bajarlo.
    """
    if DIR_BIDS_PERSISTENTE is None:
        return 0, 0
    copiados = omitidos = 0
    for relativa in rutas_relativas:
        origen = DIR_BIDS / relativa
        if not (origen.exists() and origen.stat().st_size > 0):
            continue
        respaldo = DIR_BIDS_PERSISTENTE / relativa
        if respaldo.exists() and respaldo.stat().st_size == origen.stat().st_size:
            omitidos += 1
            continue
        respaldo.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(origen, respaldo)
        copiados += 1
    return copiados, omitidos


pendientes = list(ARCHIVOS_RAIZ)
for sujeto in CFG.id_sujetos:
    pendientes += archivos_de_sujeto(sujeto)

t0 = time.time()
resultados = [(r, *obtener(r)) for r in pendientes]
segundos = time.time() - t0

tabla = pd.DataFrame(resultados, columns=["Archivo", "Estado", "Bytes"])
tabla["MB"] = (tabla["Bytes"] / 1024 ** 2).round(1)

# Resumen por estado en lugar de una linea por archivo: con 29 archivos el
# listado completo es ruido, y lo que importa es si hubo errores y por que via
# se consiguio cada archivo.
print(tabla.groupby("Estado").agg(Archivos=("Archivo", "size"),
                                  Total_MB=("MB", "sum")).to_string(), "\n")
print(f"Tiempo: {segundos:.1f} s   Total: {tabla['MB'].sum():.1f} MB")
print(f"Destino: {DIR_BIDS}")

errores = tabla[tabla["Estado"].str.startswith("ERROR")]
if not errores.empty:
    print("\nArchivos con error (el notebook no puede continuar sin ellos):")
    print(errores[["Archivo", "Estado"]].to_string(index=False))
else:
    copiados, omitidos = sincronizar_respaldo(pendientes)
    if DIR_BIDS_PERSISTENTE is None:
        print("\nSin almacen persistente: los datos se perderan al reiniciarse el entorno.")
    else:
        print(f"\nRespaldo en Drive: {copiados} archivos copiados, "
              f"{omitidos} ya estaban al dia.")
        print(f"Ubicacion: {DIR_BIDS_PERSISTENTE}")


**Buenas prácticas.** Descargar siempre el sidecar JSON junto a cada imagen. Es un archivo de pocos kilobytes cuya ausencia es difícil de diagnosticar más tarde, sin él se pierden TR, TE, dirección de codificación de fase y tiempo de lectura, que son exactamente los parámetros que gobiernan la corrección de distorsión y la política de tiempo de corte. Un dataset sin sidecars no es un dataset BIDS incompleto sino un dataset inutilizable para preprocesamiento automatizado.

**Errores frecuentes.** Interrumpir la descarga a mitad y dar por buenos los archivos truncados que quedan en disco, la caché de esta celda los daría por válidos en la siguiente ejecución. Por eso la función descarga a un archivo temporal y lo renombra solo al completarse, de modo que una interrupción nunca deja un archivo parcial con el nombre definitivo.


### 2.9b Respaldo y restauración de derivados

**Por qué existe esta subsección.** Es la lección de un incidente ocurrido durante el desarrollo de este notebook, y merece contarse porque le puede pasar a cualquiera. Tras dos horas de cómputo de MRIQC, el entorno de ejecución de Colab se reemplazó y todo el contenido del disco de la sesión desapareció. Los datos crudos y la licencia sobrevivieron porque estaban respaldados en Drive; los derivados no, y hubo que repetir las dos horas.

El disco de una sesión de Colab es efímero por diseño. Se pierde al reiniciar el entorno, al reciclarse la máquina virtual por inactividad, y al abrir una copia del notebook, que arranca siempre con un entorno nuevo. Cualquier resultado que haya costado más de unos minutos de cómputo debe salir de ese disco en cuanto exista.

**Qué se respalda y qué no.** La distinción importa porque el espacio en Drive es limitado y las velocidades de copia no son despreciables:

| Directorio | Se respalda | Razón |
|---|---|---|
| `derivatives` | Sí | Es el producto. Cuesta horas de cómputo y se necesita en las secciones posteriores |
| `reports` | Sí | Tablas y figuras exportadas, pequeñas y necesarias para el informe final |
| `data` | Sí, en 2.9 | Ya cubierto por la sincronización de la descarga |
| `work` | **No** | Son temporales de Nipype. Ocupan varias veces más que los derivados y se pueden regenerar |
| `cache` | Solo la licencia | Las plantillas de TemplateFlow se vuelven a descargar solas y son voluminosas |

**Cómo funciona.** La sincronización es incremental y bidireccional según el estado: si el directorio local está vacío y el respaldo tiene contenido, restaura; en caso contrario, respalda lo que falte o haya cambiado de tamaño. Eso permite ejecutar la misma celda al principio de una sesión nueva, para recuperar lo ya calculado, y al final de una etapa larga, para protegerlo. Las funciones que define se invocan después de MRIQC en la Sección 4 y después de fMRIPrep en la Sección 5.


In [ ]:
# 2.9b  Respaldo y restauracion de derivados, y utilidades de figuras
import shutil
import time
from pathlib import Path

# Directorios que merece la pena respaldar, con la clave de PATHS que los localiza.
# `trabajo` se excluye deliberadamente: son temporales de Nipype, ocupan varias
# veces mas que los derivados y se regeneran solos.
RESPALDABLES = ("derivados", "reportes")

# Directorio de figuras. Se define AQUI, en la Seccion 2, y no en la seccion donde
# se genera la primera figura, por una razon aprendida durante el desarrollo: las
# funciones de utilidad definidas en una seccion intermedia crean dependencias
# fragiles, de modo que ejecutar una seccion posterior sin haber pasado por aquella
# produce un fallo por nombre no definido. Todo lo que usen varias secciones vive en
# la Seccion 2.
DIR_FIGURAS = PATHS["reportes"] / "figuras"
DIR_FIGURAS.mkdir(parents=True, exist_ok=True)


def guardar_figura(fig, nombre, dpi=130):
    """Guarda una figura como archivo y devuelve la ruta del PNG.

    Se escribe en dos formatos: PNG para incrustar y consultar rapido, y PDF
    vectorial para publicacion. La resolucion es moderada de forma deliberada,
    porque el objetivo es que el notebook siga siendo manejable: una figura a 300
    puntos por pulgada multiplica su peso sin aportar nada a la inspeccion en
    pantalla.

    Guardar cada figura como archivo, y no solo como salida de celda, tiene un
    proposito concreto: las figuras quedan disponibles para quien reciba la carpeta
    de resultados sin necesidad de ejecutar el notebook, y se respaldan en el
    almacen persistente junto con el resto.
    """
    ruta_png = DIR_FIGURAS / f"{nombre}.png"
    fig.savefig(ruta_png, dpi=dpi, bbox_inches="tight")
    fig.savefig(DIR_FIGURAS / f"{nombre}.pdf", bbox_inches="tight")
    return ruta_png


def _copiar_lo_que_falte(origen, destino):
    """Copia de origen a destino solo los archivos que faltan o difieren.

    Se compara por tamano y no por suma de verificacion porque el objetivo es que
    la sincronizacion sea barata de repetir: calcular resumenes criptograficos de
    varios gigabytes en cada llamada anularia la ventaja de la copia incremental.
    Un archivo escrito a medias tendria tamano distinto, que es el caso que
    interesa detectar.
    """
    copiados = omitidos = 0
    bytes_copiados = 0
    if not origen.exists():
        return copiados, omitidos, bytes_copiados
    for ruta in origen.rglob("*"):
        if not ruta.is_file():
            continue
        objetivo = destino / ruta.relative_to(origen)
        if objetivo.exists() and objetivo.stat().st_size == ruta.stat().st_size:
            omitidos += 1
            continue
        objetivo.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(ruta, objetivo)
        copiados += 1
        bytes_copiados += ruta.stat().st_size
    return copiados, omitidos, bytes_copiados


def _tiene_contenido(directorio):
    return directorio.exists() and any(p.is_file() for p in directorio.rglob("*"))


def sincronizar_persistente(claves=RESPALDABLES, verbose=True):
    """Sincroniza en AMBOS SENTIDOS entre el disco local y el almacen persistente.

    Primero restaura de Drive lo que falte en local, y despues respalda en Drive lo
    que falte alli. Esta doble direccion es imprescindible y no siempre resulta
    evidente por que.

    Una version anterior decidia el sentido comprobando si el directorio local
    estaba vacio. Eso fallaba en el caso mas importante: al abrir una sesion nueva
    basta con que cualquier celda haya creado un solo archivo en el arbol de
    derivados para que la funcion concluyera que hay trabajo local que proteger y
    respaldase, en lugar de recuperar las horas de computo guardadas en Drive.
    Comparando archivo por archivo ese problema desaparece.

    Es idempotente y se puede invocar cuantas veces se quiera.
    """
    if DIR_PERSISTENTE is None:
        if verbose:
            print("Sin almacen persistente: nada que sincronizar.")
            print("Los derivados se perderan al reiniciarse el entorno de ejecucion.")
        return {}

    resultados = {}
    for clave in claves:
        local = PATHS[clave]
        remoto = DIR_PERSISTENTE / local.name
        t0 = time.time()
        local.mkdir(parents=True, exist_ok=True)
        remoto.mkdir(parents=True, exist_ok=True)

        restaurados, _, bytes_restaurados = _copiar_lo_que_falte(remoto, local)
        respaldados, al_dia, bytes_respaldados = _copiar_lo_que_falte(local, remoto)

        resultados[clave] = {
            "restaurados": restaurados, "respaldados": respaldados,
            "al_dia": al_dia,
            "bytes": bytes_restaurados + bytes_respaldados,
        }
        if verbose:
            print(f"{clave:12s} restaurados de Drive: {restaurados:4d}   "
                  f"respaldados en Drive: {respaldados:4d}   "
                  f"ya al dia: {al_dia:5d}   "
                  f"{(bytes_restaurados + bytes_respaldados) / 1024 ** 2:8.1f} MB   "
                  f"{time.time() - t0:5.1f} s")
    return resultados


def respaldar_figuras(verbose=False):
    """Respalda solo el directorio de figuras. Barato de llamar tras cada figura."""
    if DIR_PERSISTENTE is None:
        return None
    remoto = DIR_PERSISTENTE / PATHS["reportes"].name / "figuras"
    remoto.mkdir(parents=True, exist_ok=True)
    copiados, omitidos, tamano = _copiar_lo_que_falte(DIR_FIGURAS, remoto)
    if verbose and copiados:
        print(f"Figuras respaldadas: {copiados} nuevas "
              f"({tamano / 1024 ** 2:.1f} MB), {omitidos} ya al dia")
    return remoto


def restaurar_derivados_de(herramienta, verbose=True):
    """Recupera de Drive los derivados de una herramienta concreta.

    Se usa ANTES de decidir si hay que lanzar MRIQC o fMRIPrep. Sin este paso, una
    sesion nueva encontraria el disco local vacio y volveria a ejecutar horas de
    computo que ya estan guardadas.
    """
    if DIR_PERSISTENTE is None:
        return 0
    remoto = DIR_PERSISTENTE / "derivatives" / herramienta
    if not _tiene_contenido(remoto):
        if verbose:
            print(f"No hay respaldo de {herramienta} en el almacen persistente.")
        return 0
    local = PATHS["derivados"] / herramienta
    local.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    copiados, omitidos, tamano = _copiar_lo_que_falte(remoto, local)
    if verbose:
        print(f"Derivados de {herramienta} recuperados de Drive: {copiados} archivos "
              f"({tamano / 1024 ** 2:.1f} MB), {omitidos} ya estaban, "
              f"{time.time() - t0:.1f} s")
    return copiados + omitidos


def respaldar_derivados_de(herramienta, verbose=True):
    """Respalda solo el subdirectorio de derivados de una herramienta concreta.

    Sirve para proteger el resultado de MRIQC o de fMRIPrep en cuanto termina,
    sin recorrer todo el arbol de derivados.
    """
    if DIR_PERSISTENTE is None:
        if verbose:
            print("Sin almacen persistente: no se puede respaldar.")
        return None
    local = PATHS["derivados"] / herramienta
    if not _tiene_contenido(local):
        if verbose:
            print(f"No hay derivados de {herramienta} que respaldar todavia.")
        return None
    remoto = DIR_PERSISTENTE / "derivatives" / herramienta
    remoto.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    copiados, omitidos, tamano = _copiar_lo_que_falte(local, remoto)
    if verbose:
        print(f"Derivados de {herramienta} respaldados: {copiados} archivos "
              f"({tamano / 1024 ** 2:.1f} MB), {omitidos} ya al dia, "
              f"{time.time() - t0:.1f} s")
        print(f"Ubicacion: {remoto}")
    return remoto


# Al ejecutar la celda se sincroniza el estado actual en ambos sentidos. En una
# sesion nueva esto recupera lo calculado en sesiones anteriores, que es lo que
# permite recorrer el notebook entero sin repetir las horas de MRIQC y de fMRIPrep.
print(f"Almacen persistente: {DIR_PERSISTENTE if DIR_PERSISTENTE else 'no disponible'}")
print(f"Directorio de figuras: {DIR_FIGURAS}\n")
resumen_sincronizacion = sincronizar_persistente()

if DIR_PERSISTENTE is not None:
    print("\nContenido recuperable del almacen persistente:")
    for subdirectorio in sorted((DIR_PERSISTENTE / "derivatives").glob("*")):
        if subdirectorio.is_dir():
            archivos = [p for p in subdirectorio.rglob("*") if p.is_file()]
            tamano = sum(p.stat().st_size for p in archivos) / 1024 ** 2
            print(f"   {subdirectorio.name:22s} {len(archivos):5d} archivos, "
                  f"{tamano:8.1f} MB")


### 2.10 Conversión de DICOM a NIfTI

**Introducción conceptual.** Esta subsección es explicativa y no ejecuta código, porque en el dataset a usar (ON-Harmony) se publica ya convertido a NIfTI y organizado en BIDS. Se incluye igualmente porque cualquier estudio propio empieza en DICOM, y porque los errores que se cometen en la conversión son de los más costosos de detectar: no producen un fallo inmediato sino resultados silenciosamente incorrectos varias etapas después.

**Qué hace dcm2niix.** El escáner exporta un archivo DICOM por corte o por volumen, cada uno con una cabecera que contiene cientos de campos de metadatos, muchos de ellos específicos del fabricante. `dcm2niix` (Li et al., 2016) realiza tres tareas: agrupa los DICOM que pertenecen a una misma serie, los ensambla en un único archivo NIfTI 3D o 4D con la orientación espacial correcta, y extrae los metadatos relevantes a un archivo JSON acompañante, el sidecar. Es la herramienta que usaron los autores de ON-Harmony (versión 1.0.20211006) y la que recomiendan los estándares de la comunidad.

**Por qué el sidecar es la parte crítica.** El NIfTI conserva las dimensiones, el tamaño de vóxel y la matriz de orientación, y sí guarda el TR en el campo `pixdim[4]` de la cabecera, aunque de forma no siempre fiable, porque la conversión puede dejarlo a cero o expresarlo en unidades erróneas. Lo que la cabecera no guarda es el TE, el ángulo de giro, la dirección de codificación de fase, el tiempo de lectura total ni los tiempos de adquisición de cada corte. Sin esos campos:

| Metadato ausente | Consecuencia |
|---|---|
| `RepetitionTime` | El `pixdim[4]` de la cabecera no es una fuente fiable (puede venir a cero o en unidades erróneas), y sin un TR verificado no se puede definir el filtrado temporal ni interpretar el eje de tiempo |
| `PhaseEncodingDirection` | La corrección de distorsión no sabe en qué eje aplicar el desplazamiento, y aplicarlo en el eje equivocado empeora la imagen |
| `TotalReadoutTime` | No se puede convertir el mapa de campo en desplazamiento espacial |
| `SliceTiming` | No se puede corregir el tiempo de adquisición de corte |
| `EchoTime`, `FlipAngle` | Se pierde la caracterización del contraste, necesaria para documentar el protocolo |

Los autores de ON-Harmony señalan un detalle que ilustra por qué la herramienta importa: el tiempo de lectura total y el espaciado efectivo entre ecos, ambos necesarios para la corrección de distorsión con fieldmaps, los calcula `dcm2niix` a partir del espaciado nominal, la aceleración en el plano, el ancho de banda y las dimensiones de la matriz. No son campos que se lean directamente del DICOM, sino magnitudes derivadas, y una conversión que las omita obliga a reconstruirlas a mano con riesgo de error.

**Errores frecuentes en la conversión.** Convertir con herramientas del fabricante que no generan sidecar, y descubrirlo cuando fMRIPrep no encuentra los campos. Perder la información de orientación, que Provins et al. (2023) recogen como criterio de exclusión propio (ejes intercambiados o invertidos hacen que la imagen se visualice y se procese incorrectamente, y si se pierde la información de izquierda y derecha el dataset puede quedar inutilizable). Y el caso que documentan Morfini et al. (2023) en su propio análisis, sidecars que declaran 39 tiempos de corte para volúmenes que según la cabecera NIfTI tienen 35 cortes, una incoherencia que solo se detecta comprobando explícitamente la consistencia entre el sidecar y la cabecera, que es una de las comprobaciones que incorpora la Sección 3.

**Recomendación.** Conservar siempre los DICOM originales. La conversión es reproducible solo si se puede repetir, y las versiones de `dcm2niix` corrigen errores de interpretación de campos específicos de fabricante con cierta frecuencia.


**Buenas prácticas.** Medir los recursos del entorno antes de descargar datos o contenedores, no después de que un proceso muera por falta de memoria. Fijar de forma explícita la versión de cada contenedor y de cada plantilla en lugar de aceptar la que resuelva por defecto, porque el valor por defecto cambia entre versiones de la herramienta sin previo aviso. Concentrar todas las decisiones de configuración en un único objeto (`CFG`) que las secciones posteriores leen, de modo que cambiar de sujetos o de sesión no obligue a editar celdas dispersas por el notebook.

**Errores frecuentes.** Suponer que `pip install fmriprep` instala fMRIPrep: instala el orquestador de Python, no los binarios de ANTs, AFNI y FSL que ejecuta, y el fallo aparece a mitad del procesamiento del primer sujeto. Cargar un módulo con `module load` en una celda y esperar que el binario esté disponible en la siguiente, cuando cada invocación de shell es un proceso independiente. Declarar en el manuscrito la versión de una herramienta que se creía usar en lugar de la que el contenedor ejecuta realmente. Mezclar sesiones de escáneres distintos en una muestra pequeña, con lo que el efecto del equipo queda confundido con las diferencias entre sujetos.

**Recomendaciones.** Ejecutar 2.1 y 2.5b al inicio de cada sesión de Colab, porque el entorno de ejecución es efímero y se pierde entre sesiones. Si el trabajo se va a extender a lo largo de varios días, montar Google Drive y apuntar allí únicamente el directorio de derivados y de reportes, dejando los datos crudos y el directorio de trabajo en el disco local de la sesión: los temporales de Nipype son voluminosos y no compensa sincronizarlos. Registrar la salida de 2.1 y 2.5b junto con los resultados, porque forma parte de la descripción del entorno que exige la reproducibilidad.


## Sección 3. Validación e inspección de los datos

### 3.1 Por qué validar antes de procesar

**Introducción conceptual.** La validación no es obligatorio previo al trabajo real, es el primer punto de control de calidad del flujo. Un dataset puede estar completo y ser legible y aun así contener incoherencias que no producen ningún error visible durante el preprocesamiento y sí resultados incorrectos. Pero existen casos como cuando un campo de metadatos declara algo distinto de lo que contiene la imagen, la herramienta confía en el metadato, procesa sin quejarse, y el error solo se manifiesta como un resultado extraño varias etapas después, cuando ya es difícil atribuirlo a su causa.

**Ejemplos.** Morfini et al. (2023) documentan un ejemplo concreto en su propio análisis del FMRI Open QC Project, dos sujetos cuyos sidecars declaraban 39 tiempos de adquisición de corte mientras que la cabecera NIfTI indicaba 35 cortes. La incoherencia no la detecta ningún validador de estructura, porque ambos archivos son individualmente válidos; solo aparece si se comparan explícitamente el sidecar y la cabecera. Los mismos autores encontraron además dos sujetos con los datos funcionales invertidos, que tuvieron que decidir si corregir mediante rotación o mediante reflexión, y esa decisión cambia qué hemisferio es cuál. Provins et al. (2023) recogen los problemas de formato y de orientación como criterio de exclusión propio (criterios I y N de su protocolo), señalando que si se pierde la información sobre si la matriz se registró de izquierda a derecha o al revés, el dataset puede quedar inutilizable.

**Explicación detallada.** La validación de esta sección tiene dos niveles complementarios que responden a preguntas distintas:

| Nivel | Pregunta que responde | Herramienta | Qué no detecta |
|---|---|---|---|
| Validación estructural | ¿Cumple el dataset la especificación BIDS? | `bids-validator` oficial | Incoherencias entre el contenido de un sidecar y el de su imagen, y cualquier problema específico del pipeline que se vaya a aplicar |
| Validación dirigida al pipeline | ¿Tiene este dataset lo que necesitan las etapas que voy a ejecutar? | Comprobaciones propias en Python | Desviaciones del estándar que no afectan a este pipeline concreto |

Ninguno de los dos sustituye al otro. Un dataset puede pasar el validador oficial sin un solo error y carecer de la información necesaria para corregir la distorsión por susceptibilidad, porque BIDS no obliga a adquirir fieldmaps. A la inversa, un aviso del validador oficial sobre una convención de nombres puede ser irrelevante para el análisis previsto. El criterio para actuar es el mismo que se fijó en la Sección 1: la decisión depende del alcance del análisis, no del hallazgo aislado.


### 3.2 Validación estructural con el validador oficial

**Explicación detallada.** El validador oficial de BIDS comprueba el dataset contra la especificación, nombres de archivo y entidades, presencia de los archivos obligatorios de nivel superior, campos requeridos en los sidecars, y coherencia de la organización por sujeto y sesión. Distingue entre errores, que indican incumplimiento de la especificación, y avisos, que señalan prácticas desaconsejadas o información recomendable que falta.


In [ ]:
# 3.2  Validacion estructural con el validador oficial de BIDS
import json
import shutil
import subprocess

import pandas as pd

VERSION_VALIDADOR = "1.14.0"


def validar_bids(directorio, version=VERSION_VALIDADOR, timeout=900):
    """Ejecuta el validador oficial y devuelve (errores, avisos) como DataFrames.

    Se invoca con --json para obtener una salida estructurada. La alternativa,
    leer el informe de texto, obliga a analizar sintacticamente un formato
    pensado para humanos y que cambia entre versiones.
    """
    if shutil.which("npx") is None:
        raise RuntimeError(
            "npx no esta disponible: no se puede ejecutar el validador oficial. "
            "Alternativas: instalar Node, usar el validador en linea, o continuar "
            "solo con las comprobaciones dirigidas de 3.3.")

    proceso = subprocess.run(
        ["npx", "--yes", f"bids-validator@{version}", "--json", str(directorio)],
        capture_output=True, text=True, timeout=timeout)

    # El validador devuelve codigo de salida distinto de cero cuando encuentra
    # errores, lo que es comportamiento normal y no un fallo de ejecucion. Por eso
    # no se comprueba returncode sino que se intenta interpretar la salida.
    try:
        salida = json.loads(proceso.stdout)
    except json.JSONDecodeError:
        raise RuntimeError(
            f"El validador no devolvio JSON interpretable.\n"
            f"stdout: {proceso.stdout[:500]}\nstderr: {proceso.stderr[:500]}")

    def a_tabla(lista):
        filas = []
        for incidencia in lista or []:
            # `files` puede venir vacio en incidencias de nivel de dataset.
            archivos = incidencia.get("files") or []
            filas.append({
                "Clave": incidencia.get("key", ""),
                "Motivo": (incidencia.get("reason", "") or "").split("\n")[0][:110],
                "Archivos afectados": len(archivos),
            })
        return pd.DataFrame(filas)

    incidencias = salida.get("issues", {})
    return a_tabla(incidencias.get("errors")), a_tabla(incidencias.get("warnings"))


errores_bids, avisos_bids = validar_bids(DIR_BIDS)

print(f"Validador oficial de BIDS {VERSION_VALIDADOR} sobre {DIR_BIDS}")
print(f"Errores: {len(errores_bids)}   Avisos: {len(avisos_bids)}\n")

if not errores_bids.empty:
    print("ERRORES (incumplen la especificacion y deben resolverse):")
    print(errores_bids.to_string(index=False), "\n")
else:
    print("Sin errores de especificacion.\n")

if not avisos_bids.empty:
    print("AVISOS (no bloquean, se evaluan uno a uno en 3.2):")
    print(avisos_bids.to_string(index=False))


**Interpretación de resultados.** La salida se lee en dos niveles y con criterios distintos para cada uno.

- **Errores.** Cualquier número distinto de cero exige actuar antes de continuar: indica que el dataset incumple la especificación y que MRIQC o fMRIPrep pueden fallar o, peor, procesar interpretando mal algún campo.
- **Avisos.** No bloquean, pero tampoco se ignoran por defecto. Cada uno se evalúa por separado preguntando si afecta a alguna etapa del pipeline previsto.

En esta ejecución aparecen dos:

| Aviso | Archivos | Evaluación |
|---|---|---|
| `SLICE_TIMING_NOT_DEFINED` | 3 | Confirma que los sidecars funcionales no incluyen `SliceTiming`. No es un defecto del dataset sino una característica propia, y determina la política de corrección de tiempo de corte que se decide en 3.5. Afecta a los tres sujetos, uno por cada funcional |
| `INCONSISTENT_PARAMETERS` | 2 | Indica que no todos los sujetos comparten los mismos parámetros de adquisición, pero no dice cuáles difieren ni en cuánto. Es un aviso genérico y por sí solo no permite decidir nada. Se resuelve en 3.6, donde se identifica exactamente qué parámetro difiere y si importa |


**Errores frecuentes en este paso.** Ejecutar el validador sobre el directorio equivocado, típicamente el que contiene al dataset en lugar del dataset mismo, y obtener un error de que falta `dataset_description.json`. Interpretar el código de salida distinto de cero como un fallo de ejecución cuando es el comportamiento normal ante errores encontrados, motivo por el que la función de esta celda no lo comprueba y se limita a interpretar la salida estructurada.


### 3.3 Comprobaciones dirigidas al pipeline

**Explicación detallada.** Cada comprobación se justifica por la etapa que la necesita:

| Comprobación | Qué verifica | Etapa que depende de ella | Consecuencia si falla |
|---|---|---|---|
| Archivos presentes | Las cuatro imágenes y sus cuatro sidecars por sujeto | Todas | El sujeto no se puede procesar |
| `TaskName` | Campo obligatorio de BIDS en los sidecars funcionales | Sección 6 | fMRIPrep rechaza el archivo funcional |
| TR coherente | `RepetitionTime` del sidecar frente a `pixdim[4]` de la cabecera NIfTI | Secciones 7, 10 | Filtrado temporal aplicado sobre un eje de tiempo incorrecto, sin ningún aviso |
| Cortes coherentes | Longitud de `SliceTiming` frente a la tercera dimensión de la imagen | Sección 6 | Corrección de tiempo de corte aplicada con desfases erróneos. Es el caso documentado por Morfini et al. (2023) |
| `PhaseEncodingDirection` | Presente en el funcional y en ambos fieldmaps | Sección 6 | Sin dirección de codificación no se puede aplicar el desplazamiento de la corrección |
| Tiempo de lectura | `TotalReadoutTime` o `EffectiveEchoSpacing` en el funcional | Sección 6 | No se puede convertir el mapa de campo en desplazamiento espacial |
| Fieldmaps opuestos | Que las dos direcciones de codificación de fase sean efectivamente opuestas | Sección 6 | La estimación PEPOLAR necesita dos adquisiciones con distorsión inversa; si no lo son, el campo estimado no tiene sentido |
| Enlace fieldmap-funcional | `IntendedFor`, o el par `B0FieldIdentifier` y `B0FieldSource` | Sección 6 | fMRIPrep no asocia el fieldmap al funcional y omite la corrección en silencio |
| Geometría compatible | Que fieldmaps y funcional compartan matriz y tamaño de vóxel | Sección 6 | La corrección se estima en una geometría distinta de la que se corrige |
| Orientación válida | Que la matriz afín sea invertible y consistente entre sujetos | Todas | Imágenes procesadas con ejes intercambiados o invertidos |
| `SliceTiming` | Presencia o ausencia del campo | Sección 6 | Determina la política de 3.5, no es un error en sí mismo |


**Sobre el resultado esperado del enlace fieldmap-funcional.** BIDS admite dos mecanismos para indicar a qué imagen funcional corresponde un fieldmap. El clásico es el campo `IntendedFor` en el sidecar del fieldmap, que enumera las rutas de las imágenes a corregir. El moderno, introducido en versiones recientes de la especificación, es el par `B0FieldIdentifier` en el fieldmap y `B0FieldSource` en el funcional, que actúan como una etiqueta compartida. La comprobación acepta cualquiera de los dos, porque fMRIPrep también los acepta, y advierte si no está ninguno, es un fallo silencioso clásico, ya que fMRIPrep no aborta sino que sencillamente no aplica la corrección, y el problema solo se detecta al revisar el reporte y no encontrar la sección correspondiente.


### 3.4 Inspección de la estructura y de los parámetros de adquisición


1. **Estructura del árbol BIDS**, que muestra qué entidades existen y cómo se organizan. En este dataset la jerarquía es sujeto, sesión, modalidad, y las entidades relevantes son `sub`, `ses`, `task` para el funcional y `dir` para los fieldmaps.
2. **Geometría de cada imagen**, leída de la cabecera NIfTI: dimensiones de la matriz, tamaño de vóxel y orientación anatómica de los ejes. La orientación se expresa con los códigos de eje de nibabel, que indican hacia dónde crece cada eje de la matriz en el espacio anatómico y permiten interpretar qué dirección anatómica corresponde a la codificación de fase `j`.
3. **Parámetros de adquisición**, leídos de los sidecars: TR, TE, ángulo de giro, número de volúmenes, duración, dirección de codificación de fase, tiempo de lectura, fabricante, modelo e intensidad de campo.


In [ ]:
# 3.4  Estructura del arbol BIDS y parametros de adquisicion
import json
import nibabel as nib
import pandas as pd


def arbol(raiz, prefijo="", max_por_dir=6):
    """Impresion compacta del arbol. Trunca listados largos para que la salida
    siga siendo legible cuando el numero de sujetos crece."""
    entradas = sorted(raiz.iterdir(), key=lambda p: (p.is_file(), p.name))
    mostradas = entradas[:max_por_dir]
    for i, ruta in enumerate(mostradas):
        ultimo = (i == len(mostradas) - 1) and len(entradas) <= max_por_dir
        rama = "└── " if ultimo else "├── "
        print(f"{prefijo}{rama}{ruta.name}")
        if ruta.is_dir():
            arbol(ruta, prefijo + ("    " if ultimo else "│   "), max_por_dir)
    if len(entradas) > max_por_dir:
        print(f"{prefijo}└── ... y {len(entradas) - max_por_dir} entradas mas")


print(f"Estructura de {DIR_BIDS.name}")
arbol(DIR_BIDS)

# Demografia de los sujetos seleccionados
participantes = pd.read_csv(DIR_BIDS / "participants.tsv", sep="\t")
demografia = participantes[participantes["participant_id"].isin(CFG.id_sujetos)]
print("\nSujetos seleccionados")
print(demografia.to_string(index=False))

# Geometria de cada imagen y parametros de adquisicion del funcional
filas_geo, filas_adq = [], []
for sujeto in CFG.id_sujetos:
    base = DIR_BIDS / sujeto / CFG.id_sesion
    prefijo = f"{sujeto}_{CFG.id_sesion}"
    imagenes = {
        "T1w": base / "anat" / f"{prefijo}_T1w.nii.gz",
        "BOLD": base / "func" / f"{prefijo}_task-{CFG.tarea}_bold.nii.gz",
        "fmap AP": base / "fmap" / f"{prefijo}_dir-AP_epi.nii.gz",
        "fmap PA": base / "fmap" / f"{prefijo}_dir-PA_epi.nii.gz",
    }
    for modalidad, ruta in imagenes.items():
        img = nib.load(ruta)
        zooms = img.header.get_zooms()
        # aff2axcodes indica hacia donde crece cada eje de la matriz en el
        # espacio anatomico: por ejemplo ('L','A','S') significa que el primer
        # eje crece hacia la izquierda, el segundo hacia anterior y el tercero
        # hacia superior. Es lo que permite traducir la codificacion de fase `j`
        # a una direccion anatomica concreta.
        ejes = "".join(nib.aff2axcodes(img.affine))
        filas_geo.append({
            "Sujeto": sujeto,
            "Modalidad": modalidad,
            "Dimensiones": " x ".join(str(d) for d in img.shape[:3]),
            "Volumenes": img.shape[3] if img.ndim > 3 else 1,
            "Voxel (mm)": " x ".join(f"{z:.2f}" for z in zooms[:3]),
            "Ejes": ejes,
        })

    # Parametros de adquisicion del funcional, que es el que gobierna el analisis.
    js = json.loads((base / "func" / f"{prefijo}_task-{CFG.tarea}_bold.json").read_text())
    bold = nib.load(imagenes["BOLD"])
    n_vol = bold.shape[3]
    tr = js["RepetitionTime"]
    filas_adq.append({
        "Sujeto": sujeto,
        "TR (s)": tr,
        "TE (ms)": round(js["EchoTime"] * 1000, 1),
        "Angulo (grados)": js.get("FlipAngle"),
        "Volumenes": n_vol,
        "Duracion (min)": round(n_vol * tr / 60, 2),
        "PED": js.get("PhaseEncodingDirection"),
        "TRT (ms)": round(js.get("TotalReadoutTime", float("nan")) * 1000, 2),
        "Campo (T)": js.get("MagneticFieldStrength"),
        "Equipo": f"{js.get('Manufacturer', '')} {js.get('ManufacturersModelName', '')}".strip(),
        "Protocolo": js.get("ProtocolName"),
    })

geometria = pd.DataFrame(filas_geo)
adquisicion = pd.DataFrame(filas_adq)

print("\nGeometria de las imagenes")
print(geometria.to_string(index=False))

print("\nParametros de adquisicion del funcional")
print(adquisicion.drop(columns=["Protocolo"]).to_string(index=False))
print(f"\nProtocolo: {adquisicion['Protocolo'].iloc[0]}")

# Cobertura en el eje de cortes: numero de cortes por su grosor. Determina si el
# campo de vision cubre el cerebro completo, lo que condiciona el analisis de grupo.
ruta_bold = (DIR_BIDS / CFG.id_sujetos[0] / CFG.id_sesion / "func" /
             f"{CFG.id_sujetos[0]}_{CFG.id_sesion}_task-{CFG.tarea}_bold.nii.gz")
img_bold = nib.load(ruta_bold)
n_cortes = img_bold.shape[2]
grosor = float(img_bold.header.get_zooms()[2])
print(f"Cobertura en el eje de cortes: {n_cortes} cortes x {grosor:.2f} mm = "
      f"{n_cortes * grosor:.1f} mm")


**1. La causa del aviso `INCONSISTENT_PARAMETERS` queda identificada.** Las dimensiones del T1w difieren entre sujetos (183 x 227 x 189, 179 x 221 x 172 y 173 x 236 x 187), mientras que el tamaño de vóxel es idéntico, 1 mm isotrópico en los tres. Esto es exactamente lo esperable en un dataset público, las imágenes anatómicas se han desfigurado antes de publicarlas para proteger la identidad, y el recorte resultante deja un volumen de dimensiones distintas en cada sujeto. No es una diferencia de protocolo de adquisición sino de postprocesado, y no afecta a ninguna etapa del pipeline, porque la normalización espacial de la Sección 5 lleva a todos los sujetos a la misma plantilla. El aviso genérico del validador queda así resuelto y documentado, que es el desenlace correcto: ni ignorado ni convertido en exclusión.

**2. Los parámetros funcionales sí son homogéneos.** TR de 1.15 s, TE de 39 ms, ángulo de giro de 65 grados, 400 volúmenes y matriz de 96 x 96 x 48 con vóxel de 2.4 mm en los tres sujetos. Es lo que hace comparables las métricas de calidad entre sujetos en las Secciones 4, 7 y 11. El tiempo de lectura total varía en la tercera cifra decimal (57.80, 57.74 y 57.82 ms), una diferencia inferior al 0.2 por ciento que refleja ajustes de calibración por sesión y carece de consecuencias prácticas.

**3. La orientación explica qué es la dirección `j`.** El T1w está en orientación RAS y el funcional y los fieldmaps en LAS. En ambos casos el segundo eje de la matriz, que es al que se refiere la codificación de fase `j`, crece hacia anterior. Por tanto el funcional tiene codificación de fase en sentido posterior a anterior, y el fieldmap con `j-` la tiene en sentido anterior a posterior. Conviene notar que el archivo etiquetado `dir-AP` es el que lleva `j`: la etiqueta del nombre y la dirección física siguen convenciones opuestas. Esto no afecta al procesamiento, porque tanto fMRIPrep como las comprobaciones de 3.3 leen el campo del sidecar y nunca la etiqueta del nombre, pero es un recordatorio de que las etiquetas `dir-` son descriptivas y no normativas, y de que documentar la dirección de codificación citando el nombre del archivo en lugar del sidecar puede propagar un error.

**4. El ángulo de giro es coherente con el TR.** Los autores del dataset fijaron el ángulo al valor de Ernst correspondiente a cada TR suponiendo un T1 de sustancia gris de 1.5 s a 3T. Para TR de 1.15 s ese valor teórico es de unos 62 grados, próximo a los 65 empleados. No es una comprobación obligatoria, pero confirma que el protocolo se diseñó para maximizar la señal disponible y no con parámetros arbitrarios.

**5. La cobertura es parcial y esto condiciona el análisis de grupo.** Con 48 cortes de 2.4 mm, el campo de visión en el eje de cortes es de 115.2 mm, insuficiente para cubrir el cerebro completo de un adulto: queda fuera parte del vértex. La consecuencia es concreta y hay que arrastrarla hasta el final: cualquier análisis entre sujetos debe restringirse a la intersección de las máscaras funcionales de los tres, no a la máscara de la plantilla, porque de lo contrario se compararían regiones presentes en unos sujetos y ausentes en otros. Esta restricción se aplica en la Sección 8 y se documenta como limitación en la Sección 15.

**Buenas prácticas.** Extraer estos parámetros de los archivos y no de la documentación del estudio ni del artículo. Los tres pueden discrepar, y el único que gobierna lo que hacen las herramientas es el sidecar.


### 3.5 Política de corrección de tiempo de adquisición de corte

Los cortes de un volumen no se adquieren simultáneamente. En una secuencia 2D, la adquisición de todos los cortes se reparte a lo largo del TR, de modo que dos vóxeles situados en cortes distintos representan instantes distintos aunque compartan índice temporal. La corrección de tiempo de corte reajusta cada corte a un instante de referencia común mediante interpolación temporal.

**El problema.** Esa corrección requiere saber en qué instante se adquirió cada corte, información que reside únicamente en el campo `SliceTiming` del sidecar. Las comprobaciones de 3.3 y el validador oficial coinciden en que este dataset no lo incluye, y tampoco incluye `SliceEncodingDirection`, `MultibandAccelerationFactor` ni `DelayTime`. Ante esa ausencia, la pregunta correcta no es si se puede corregir, sino cuánto se gana corrigiendo y cuánto se arriesga al suponer el orden.

**Qué hace y qué no hace el multibanda.** Conviene precisarlo porque se presta a confusión. Con 48 cortes y factor multibanda 4 se excitan 4 cortes simultáneamente en cada pulso, lo que produce 12 grupos de excitación en lugar de 48 adquisiciones sucesivas. El multibanda reduce por tanto el número de instantes distintos de adquisición y la separación entre grupos consecutivos, que pasa a ser el TR dividido entre 12, unos 96 ms. Lo que el multibanda no reduce es el desfase total entre el primer y el último grupo, que sigue abarcando prácticamente todo el TR, porque la adquisición se reparte igualmente a lo largo de él. Suponer que el multibanda hace innecesaria la corrección por sí solo es un error, el argumento válido descansa en el TR.

**Fundamento metodológico.** Kumar et al. (2024) recogen el consenso de la American Society of Functional Neuroradiology y lo resuelven en función del TR, la corrección es recomendable con TR mayor de 2 s y opcional con TR menor o igual a 2 s. Aportan las cifras que sostienen la frontera, la mejora en los estadísticos globales es del orden del 55 por ciento con TR de 5 s y del 15.6 por ciento con TR de 2 s, pero cae al 5 y al 2.2 por ciento con TR de 1.1 y 0.5 s respectivamente. Señalan además dos costes de aplicarla, los errores de interpolación temporal, motivo por el que el Human Connectome Project decidió no aplicarla con TR subsegundo, y la propagación a los demás cortes de cualquier artefacto presente en el corte de referencia.

- A esos costes se suma aquí uno específico: sin `SliceTiming` habría que suponer el orden de adquisición del fabricante. Si el orden supuesto no coincide con el real, el error resultante no se distribuye al azar sino que queda correlacionado con la posición del corte y puede alcanzar medio TR, lo que es peor que no corregir en absoluto.

**Decisión adoptada.** La celda siguiente aplica la regla siguiente, documentada como tabla porque el criterio debe ser explícito y reutilizable en otros datasets:

| Condición | Decisión | Razón |
|---|---|---|
| `SliceTiming` presente | Usar el dato tal cual | Es una medida, no hay nada que suponer |
| Ausente y TR menor o igual a 2 s | No corregir | El beneficio esperado es de pocos puntos porcentuales y no compensa el riesgo de suponer un orden erróneo ni el coste de la interpolación |
| Ausente y TR mayor de 2 s | Reconstruir el orden por defecto del fabricante, corregir, y marcar el dato como reconstruido | El desfase alcanza el orden de segundos y su efecto sobre la estimación es sustancial |

**Advertencia.** Cuando la regla lleva a reconstruir, la marca de dato reconstruido debe viajar hasta los derivados y hasta el informe final, porque cambia el estatus del resultado. La alternativa preferible siempre es obtener el orden real del centro de adquisición o de los DICOM originales.


In [ ]:
# 3.5  Decision sobre la correccion de tiempo de adquisicion de corte
import re
import json
import nibabel as nib
import pandas as pd

UMBRAL_TR_S = 2.0  # frontera del consenso recogido por Kumar et al. (2024)


def factor_multibanda(sidecar):
    """Determina el factor multibanda a partir del sidecar.

    Se busca primero el campo normalizado. Si falta, se intenta deducir del
    nombre del protocolo, que en este dataset codifica el factor como 'MB4'.
    La procedencia se devuelve junto al valor porque un factor deducido del
    nombre del protocolo es una inferencia y debe quedar marcada como tal.
    """
    for campo in ("MultibandAccelerationFactor", "SliceAccelerationFactor"):
        if campo in sidecar:
            return int(sidecar[campo]), f"campo {campo}"
    protocolo = sidecar.get("ProtocolName", "") or ""
    coincidencia = re.search(r"MB\s*(\d+)", protocolo, flags=re.IGNORECASE)
    if coincidencia:
        return int(coincidencia.group(1)), f"deducido del protocolo '{protocolo}'"
    return None, "no determinable"


filas = []
for sujeto in CFG.id_sujetos:
    base = DIR_BIDS / sujeto / CFG.id_sesion
    prefijo = f"{sujeto}_{CFG.id_sesion}"
    js = json.loads((base / "func" / f"{prefijo}_task-{CFG.tarea}_bold.json").read_text())
    n_cortes = nib.load(base / "func" / f"{prefijo}_task-{CFG.tarea}_bold.nii.gz").shape[2]

    tr = js["RepetitionTime"]
    st = js.get("SliceTiming")
    mb, procedencia_mb = factor_multibanda(js)

    if st is not None:
        decision = "usar SliceTiming medido"
        razon = "el dato existe en el sidecar"
        grupos = len(set(st))
        separacion_ms = (max(st) - min(st)) / (grupos - 1) * 1000 if grupos > 1 else 0.0
        desfase_ms = (max(st) - min(st)) * 1000
    else:
        # Grupos de excitacion: con multibanda se excitan `mb` cortes por pulso.
        grupos = n_cortes // mb if mb else n_cortes
        # Separacion entre grupos consecutivos: es lo que reduce el multibanda.
        separacion_ms = tr / grupos * 1000
        # Desfase total entre el primer y el ultimo grupo: abarca casi todo el TR
        # y NO lo reduce el multibanda. Es el numero que suele malinterpretarse.
        desfase_ms = separacion_ms * (grupos - 1)
        if tr < UMBRAL_TR_S:
            decision = "no corregir"
            razon = (f"TR {tr} s por debajo del umbral de {UMBRAL_TR_S} s: beneficio "
                     f"esperado de pocos puntos porcentuales, frente al riesgo de "
                     f"suponer el orden de corte y al coste de la interpolacion")
        else:
            decision = "reconstruir orden del fabricante y corregir"
            razon = f"TR {tr} s alcanza el umbral de {UMBRAL_TR_S} s: el desfase sesga la estimacion"

    filas.append({
        "Sujeto": sujeto,
        "TR (s)": tr,
        "Cortes": n_cortes,
        "MB": mb if mb else "nd",
        "Grupos": grupos,
        "SliceTiming": "presente" if st is not None else "ausente",
        "Separacion entre grupos (ms)": round(separacion_ms, 1),
        "Desfase total (ms)": round(desfase_ms, 1),
        "Decision": decision,
        "Razon": razon,
    })

politica_stc = pd.DataFrame(filas)
print(politica_stc.drop(columns=["Razon"]).to_string(index=False))
print(f"\nProcedencia del factor multibanda: {procedencia_mb}")
print(f"Razon: {politica_stc['Razon'].iloc[0]}")

# La decision se guarda para que la Seccion 6 la consulte en lugar de volver a
# deducirla, y para que quede registrada en el informe final.
APLICAR_STC = bool((politica_stc["Decision"] != "no corregir").any())
print(f"\nDecision para la Seccion 6: "
      f"{'aplicar' if APLICAR_STC else 'omitir'} la correccion de tiempo de corte")
print(f"Argumento para fMRIPrep: "
      f"{'(ninguno, es el comportamiento por defecto)' if APLICAR_STC else '--ignore slicetiming'}")


**Interpretación de resultados.**

- **Separación entre grupos consecutivos: 95.8 ms.** Es el TR dividido entre los 12 grupos de excitación. Esta es la cantidad que reduce el multibanda: sin él habría 48 adquisiciones sucesivas separadas unos 24 ms cada una, pero repartidas de la misma forma a lo largo del TR.
- **Desfase total: 1054.2 ms.** Es la diferencia entre el primer y el último grupo, y abarca el 92 por ciento del TR. El multibanda no lo reduce. Frente a un ancho de la respuesta hemodinámica del orden de varios segundos, un segundo de desfase no es despreciable, y sería un error justificar la omisión de la corrección afirmando que sí lo es.

**Por qué se omite la corrección pese a ese desfase.** El argumento no es que el desfase sea pequeño, sino que la relación entre beneficio y riesgo es desfavorable en este caso concreto, por tres razones acumulativas. Primera, el beneficio medido para un TR de 1.15 s ronda el 5 por ciento en los estadísticos globales, frente al 15.6 por ciento con TR de 2 s y el 55.7 por ciento con TR de 5 s. Segunda, la corrección exige interpolación temporal, que introduce su propio error y propaga a los demás cortes cualquier artefacto del corte de referencia. Tercera, y decisiva aquí, no se dispone de `SliceTiming`, habría que suponer el orden del fabricante, y un orden equivocado produce un error correlacionado con la posición del corte que puede llegar a medio TR, es decir, mayor que el problema que se pretendía resolver.

**Qué habría cambiado la decisión.** Si el TR fuera de 2.5 s, la misma regla habría llevado a reconstruir el orden y corregir, marcando el dato como reconstruido. Si el sidecar incluyera `SliceTiming`, se habría aplicado la corrección con el dato medido con independencia del TR. La regla está escrita como tabla precisamente para que sea reutilizable en esos otros escenarios y no una decisión ad hoc para este dataset.

**Consecuencia para las secciones siguientes.** La variable `APLICAR_STC` queda en `False` y la Sección 6 invocará fMRIPrep con `--ignore slicetiming`. La omisión se documenta como limitación explícita en la Sección 15, no como una decisión invisible.

**Errores frecuentes.** Aplicar la corrección con un orden de corte supuesto sin dejar constancia de que es supuesto. Invertir el signo del desplazamiento temporal, un error que no produce ningún fallo visible pero que empeora el desfase en lugar de corregirlo, y que solo se detecta comprobando la implementación contra una señal sintética de frecuencia conocida. Aplicar la corrección después del realineamiento en lugar de antes, o al revés, sin justificar el orden: ambas opciones tienen defensores, pero la elección debe ser explícita.


### 3.6 Reporte de consistencia entre sujetos

**Fundamento metodológico.** Con una muestra pequeña (n<=100), la comparación entre sujetos es el único criterio disponible para juzgar la calidad, porque no hay distribución poblacional contra la que contrastar. Esa comparación solo es legítima si los sujetos comparten protocolo, si un sujeto se adquirió con parámetros distintos, cualquier diferencia en sus métricas de calidad puede deberse al protocolo y no a su calidad.

El reporte distingue tres categorías de parámetro, porque no todas las diferencias tienen la misma gravedad:

| Categoría | Ejemplos | Si difiere entre sujetos |
|---|---|---|
| Crítico | TR, número de volúmenes, tamaño de vóxel del funcional, dirección de codificación de fase | Invalida la comparación directa entre sujetos. Hay que homogeneizar o tratar el parámetro como covariable |
| Relevante | TE, ángulo de giro, tiempo de lectura | Afecta al contraste o a la corrección de distorsión. Diferencias pequeñas son tolerables y deben documentarse |
| Sin consecuencia | Dimensiones de la matriz del T1w, hora de adquisición | No afecta al pipeline, que normaliza todo a una plantilla común |

La última fila es la que resuelve el aviso `INCONSISTENT_PARAMETERS` emitido por el validador oficial en 3.2.


In [ ]:
# 3.6  Reporte de consistencia entre sujetos y matriz de validacion
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Consistencia de los parametros de adquisicion.
CATEGORIA = {
    "TR (s)": "critico",
    "Volumenes": "critico",
    "PED": "critico",
    "Voxel funcional (mm)": "critico",
    "TE (ms)": "relevante",
    "Angulo (grados)": "relevante",
    "TRT (ms)": "relevante",
    "Equipo": "critico",
    "Dimensiones T1w": "sin consecuencia",
}

# Se reutilizan las tablas de 3.4 y se anaden las dos columnas que faltan.
tabla = adquisicion.set_index("Sujeto").copy()
tabla["Voxel funcional (mm)"] = (
    geometria[geometria["Modalidad"] == "BOLD"].set_index("Sujeto")["Voxel (mm)"])
tabla["Dimensiones T1w"] = (
    geometria[geometria["Modalidad"] == "T1w"].set_index("Sujeto")["Dimensiones"])

filas = []
for parametro, categoria in CATEGORIA.items():
    valores = tabla[parametro]
    # Los flotantes se comparan con tolerancia relativa: diferencias en la
    # tercera cifra decimal son ruido de calibracion, no cambios de protocolo.
    if pd.api.types.is_numeric_dtype(valores):
        homogeneo = bool(np.allclose(valores, valores.iloc[0], rtol=1e-3))
    else:
        homogeneo = bool(valores.nunique() == 1)
    filas.append({
        "Parametro": parametro,
        "Categoria": categoria,
        "Homogeneo": "si" if homogeneo else "NO",
        "Valores": (str(valores.iloc[0]) if homogeneo
                    else ", ".join(f"{s.replace('sub-', '')}={v}" for s, v in valores.items())),
    })

consistencia = pd.DataFrame(filas)
print("Consistencia de los parametros de adquisicion entre sujetos")
print(consistencia.to_string(index=False), "\n")

discrepantes = consistencia[consistencia["Homogeneo"] == "NO"]
criticas = discrepantes[discrepantes["Categoria"] == "critico"]
if criticas.empty:
    print("Ningun parametro critico difiere entre sujetos: la comparacion directa "
          "entre ellos es legitima.")
else:
    print("ATENCION: difieren parametros criticos. La comparacion entre sujetos no "
          "es directa y requiere homogeneizar o modelar el parametro como covariable:")
    print(criticas.to_string(index=False))

if not discrepantes.empty:
    otras = discrepantes[discrepantes["Categoria"] != "critico"]
    if not otras.empty:
        print("\nDiferencias sin consecuencia para el pipeline, documentadas:")
        print(otras.to_string(index=False))

# Matriz de validacion.
CODIGO = {"OK": 0, "AVISO": 1, "ERROR": 2}
matriz = (comprobaciones.pivot(index="Sujeto", columns="Comprobacion", values="Estado")
          .reindex(CFG.id_sujetos))
# Se usa `map` por columna en lugar de `replace`: `replace` sobre una tabla de
# cadenas realiza una conversion implicita de tipo que pandas ha marcado como
# obsoleta, y el objetivo es que el notebook no dependa de comportamientos que
# van a cambiar de version.
valores = matriz.apply(lambda columna: columna.map(CODIGO)).astype(float).to_numpy()

colores = ["#7FB77E", "#E8C468", "#C1666B"]  # OK, AVISO, ERROR
fig, ax = plt.subplots(figsize=(13, 2.2 + 0.4 * len(matriz)))
ax.imshow(valores, cmap=plt.matplotlib.colors.ListedColormap(colores), vmin=0, vmax=2,
          aspect="auto")

ax.set_xticks(range(matriz.shape[1]))
ax.set_xticklabels(matriz.columns, rotation=40, ha="right", fontsize=8)
ax.set_yticks(range(matriz.shape[0]))
ax.set_yticklabels(matriz.index, fontsize=9)

# Anotacion del estado en cada celda: el color orienta, el texto desambigua y
# mantiene la figura legible para quien no distinga bien los colores.
for i in range(valores.shape[0]):
    for j in range(valores.shape[1]):
        estado = matriz.iloc[i, j]
        ax.text(j, i, "" if pd.isna(estado) else estado[0], ha="center", va="center",
                fontsize=8, color="black")

ax.set_title("Matriz de validacion: sujetos frente a comprobaciones "
             "(O correcto, A aviso, E error)", fontsize=11, pad=12)
plt.tight_layout()
plt.show()


**Interpretación de resultados.** El reporte tiene dos salidas.

**La tabla de consistencia resuelve el aviso pendiente.** Los siete parámetros críticos y de contraste principales son idénticos en los tres sujetos: TR de 1.15 s, 400 volúmenes, codificación de fase `j`, vóxel de 2.4 mm isotrópico, TE de 39 ms, ángulo de 65 grados y el mismo equipo. Difieren dos:

- **Dimensiones del T1w**, clasificado como sin consecuencia. Es el origen del aviso `INCONSISTENT_PARAMETERS` que emitió el validador oficial en 3.2, y proviene del recorte que deja el desfigurado previo a la publicación, no del protocolo de adquisición. La normalización espacial de la Sección 5 lleva a los tres sujetos a la misma plantilla, de modo que la diferencia desaparece antes de cualquier comparación.
- **Tiempo de lectura total**, clasificado como relevante: 57.80, 57.74 y 57.82 ms. La diferencia máxima es de 0.08 ms, un 0.14 por ciento, y refleja ajustes de calibración por sesión. La decisión de considerarla inconsecuente es del analista y queda documentada aquí.

La conclusión operativa es la línea que confirma que ningún parámetro crítico difiere. Es la condición que legitima el uso de umbrales relativos entre sujetos en las Secciones 4, 7 y 11, y sin ella esas comparaciones no serían interpretables.

**La matriz de validación es el resumen visual del estado del dataset.**

- **Una fila con varias marcas de color distinto al correcto** señala un sujeto problemático, candidato a revisión individual o a exclusión.
- **Una columna completa marcada** señala una característica del dataset, no un problema de un sujeto. Es lo que ocurre aquí con `SliceTiming`, los tres sujetos comparten el aviso, lo que indica una propiedad del protocolo y no una incidencia individual.

Con tres sujetos la distinción parece trivial, pero es exactamente la lectura que se necesita cuando la muestra tiene decenas de sujetos y la inspección caso por caso deja de ser viable. La figura anota además la inicial del estado sobre cada celda, de modo que la información no dependa solo del color.

**Recomendación.** Conservar esta matriz y la tabla de consistencia entre los resultados exportados. Forman parte de la descripción del dataset que exige la reproducibilidad, y son la evidencia de que la validación se ejecutó y con qué resultado, no una afirmación de que se hizo.


In [ ]:
# 3.3  Comprobaciones dirigidas al pipeline
# Se instala nibabel, justificado en 2.3: es la biblioteca que lee las cabeceras
# NIfTI, y las comprobaciones de esta celda consisten precisamente en contrastar
# lo que declara el sidecar JSON con lo que contiene la cabecera de la imagen.
#
# La instalacion se hace con `--no-deps`, y esa opcion resuelve un fallo real que
# aparecio tres veces durante el desarrollo de este notebook.
#
# Colab trae paquetes preinstalados que exigen versiones de pandas distintas de la
# que el kernel tiene cargada. Al instalar cualquier cosa, el resolutor de
# dependencias de pip puede actualizar pandas EN DISCO mientras el interprete
# conserva en memoria la version anterior. El resultado es un paquete inconsistente:
# los modulos ya importados son de la version antigua y los que se importen despues,
# de forma diferida, se leen del disco y son de la nueva. El error no aparece al
# instalar sino mucho mas tarde, en la primera operacion que cargue uno de esos
# submodulos, y con un mensaje que no menciona pandas ni la instalacion.
#
# Fijar la version de pandas en la orden de pip no basta, porque si el paquete que
# se instala declara una version minima incompatible el resolutor la ignora. La
# solucion correcta es NO dejar que pip toque las dependencias: nibabel solo
# necesita numpy y packaging, ambos ya presentes en Colab, de modo que instalarlo
# sin dependencias es seguro y deja pandas intacto. Si aun asi el disco quedase
# desajustado por una instalacion anterior, se restituye la version cargada.
import subprocess
import sys
import importlib.metadata as metadatos

import pandas as pd

version_cargada = pd.__version__

try:
    import nibabel as nib
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps",
                    "nibabel"], check=False)
    import nibabel as nib

if metadatos.version("pandas") != version_cargada:
    print(f"AVISO: pandas en memoria {version_cargada}, en disco "
          f"{metadatos.version('pandas')}.")
    print("Se restituye la version cargada para evitar un fallo diferido.")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps",
                    "--force-reinstall", f"pandas=={version_cargada}"], check=False)
    if metadatos.version("pandas") == version_cargada:
        print("Reparado.")
    else:
        print(f"No se pudo restituir (disco: {metadatos.version('pandas')}). El")
        print("notebook continua, pero si aparece un error de importacion de pandas")
        print("mas adelante, reinicie el entorno y ejecute desde la celda 2.1.")

import json
import numpy as np

OK, AVISO, ERROR = "OK", "AVISO", "ERROR"


def _cargar(sujeto):
    """Devuelve las rutas y los sidecars de un sujeto, o None si falta algo."""
    base = DIR_BIDS / sujeto / CFG.id_sesion
    prefijo = f"{sujeto}_{CFG.id_sesion}"
    rutas = {
        "t1w": base / "anat" / f"{prefijo}_T1w.nii.gz",
        "bold": base / "func" / f"{prefijo}_task-{CFG.tarea}_bold.nii.gz",
        "ap": base / "fmap" / f"{prefijo}_dir-AP_epi.nii.gz",
        "pa": base / "fmap" / f"{prefijo}_dir-PA_epi.nii.gz",
    }
    sidecars = {k: v.with_suffix("").with_suffix(".json") for k, v in rutas.items()}
    return rutas, sidecars


def comprobar_sujeto(sujeto):
    """Aplica todas las comprobaciones a un sujeto. Devuelve lista de filas."""
    filas = []

    def anotar(comprobacion, estado, detalle=""):
        filas.append({"Sujeto": sujeto, "Comprobacion": comprobacion,
                      "Estado": estado, "Detalle": detalle})

    rutas, sidecars = _cargar(sujeto)

    # C1. Presencia de los ocho archivos.
    faltan = [k for k, v in {**rutas, **{f"{k}.json": v for k, v in sidecars.items()}}.items()
              if not v.exists()]
    if faltan:
        anotar("Archivos presentes", ERROR, f"faltan: {', '.join(faltan)}")
        return filas  # sin archivos no tiene sentido seguir comprobando
    anotar("Archivos presentes", OK, "8 de 8")

    js = {k: json.loads(v.read_text()) for k, v in sidecars.items()}
    img = {k: nib.load(v) for k, v in rutas.items()}

    # C2. TaskName es obligatorio en BIDS para datos funcionales.
    anotar("TaskName en el funcional",
           OK if "TaskName" in js["bold"] else ERROR,
           js["bold"].get("TaskName", "ausente"))

    # C3. TR del sidecar frente al de la cabecera. Es la comprobacion que detecta
    # un eje de tiempo incorrecto, que ninguna herramienta senala por si sola.
    tr_json = js["bold"].get("RepetitionTime")
    tr_hdr = float(img["bold"].header.get_zooms()[3])
    coincide = tr_json is not None and abs(tr_json - tr_hdr) < 1e-3
    anotar("TR sidecar frente a cabecera", OK if coincide else ERROR,
           f"json={tr_json} s, cabecera={tr_hdr:.4f} s")

    # C4. SliceTiming: se comprueba su presencia y, si existe, su coherencia con
    # el numero de cortes. Su ausencia no es un error sino la entrada a 3.5.
    st = js["bold"].get("SliceTiming")
    n_cortes = img["bold"].shape[2]
    if st is None:
        anotar("SliceTiming", AVISO, f"ausente. {n_cortes} cortes. Ver politica en 3.5")
    elif len(st) != n_cortes:
        anotar("SliceTiming coherente", ERROR,
               f"{len(st)} tiempos declarados frente a {n_cortes} cortes en la imagen")
    else:
        anotar("SliceTiming coherente", OK, f"{len(st)} tiempos y {n_cortes} cortes")

    # C5. Direccion de codificacion de fase en el funcional y en los fieldmaps.
    peds = {k: js[k].get("PhaseEncodingDirection") for k in ("bold", "ap", "pa")}
    if all(peds.values()):
        anotar("PhaseEncodingDirection presente", OK,
               ", ".join(f"{k}={v}" for k, v in peds.items()))
    else:
        ausentes = [k for k, v in peds.items() if not v]
        anotar("PhaseEncodingDirection presente", ERROR, f"ausente en: {', '.join(ausentes)}")

    # C6. Los dos fieldmaps deben tener codificacion de fase OPUESTA: es la
    # condicion que hace estimable el campo por el metodo PEPOLAR.
    if peds["ap"] and peds["pa"]:
        opuestas = (peds["ap"].rstrip("-") == peds["pa"].rstrip("-")
                    and peds["ap"] != peds["pa"])
        anotar("Fieldmaps con fase opuesta", OK if opuestas else ERROR,
               f"AP={peds['ap']}, PA={peds['pa']}")

    # C7. Tiempo de lectura, necesario para convertir desfase de campo en
    # desplazamiento espacial.
    trt = js["bold"].get("TotalReadoutTime") or js["bold"].get("EffectiveEchoSpacing")
    anotar("Tiempo de lectura del funcional", OK if trt else ERROR,
           f"TotalReadoutTime={js['bold'].get('TotalReadoutTime')}, "
           f"EffectiveEchoSpacing={js['bold'].get('EffectiveEchoSpacing')}")

    # C8. Enlace fieldmap-funcional por cualquiera de los dos mecanismos que
    # admite BIDS. Sin el, fMRIPrep omite la correccion sin emitir error.
    tiene_intended = any("IntendedFor" in js[k] for k in ("ap", "pa"))
    tiene_b0 = (any("B0FieldIdentifier" in js[k] for k in ("ap", "pa"))
                and "B0FieldSource" in js["bold"])
    if tiene_intended or tiene_b0:
        mecanismos = ", ".join(m for m, ok in
                               [("IntendedFor", tiene_intended), ("B0FieldIdentifier", tiene_b0)] if ok)
        anotar("Enlace fieldmap-funcional", OK, mecanismos)
    else:
        anotar("Enlace fieldmap-funcional", ERROR,
               "sin IntendedFor ni B0FieldIdentifier: la correccion se omitiria en silencio")

    # C9. Geometria compatible entre fieldmaps y funcional.
    geo_bold = img["bold"].shape[:3]
    geo_fmap = img["ap"].shape[:3]
    anotar("Geometria fieldmap frente a funcional",
           OK if geo_bold == geo_fmap else AVISO,
           f"bold={geo_bold}, fmap={geo_fmap}")

    # C10. Matriz afin invertible: si el determinante es nulo la orientacion es
    # degenerada y cualquier remuestreo posterior seria invalido.
    for clave in ("t1w", "bold"):
        det = float(np.linalg.det(img[clave].affine[:3, :3]))
        anotar(f"Orientacion valida ({clave})", OK if abs(det) > 1e-6 else ERROR,
               f"determinante={det:.3f} ({'radiologica' if det < 0 else 'neurologica'})")

    return filas


filas = [f for sujeto in CFG.id_sujetos for f in comprobar_sujeto(sujeto)]
comprobaciones = pd.DataFrame(filas)

resumen = comprobaciones["Estado"].value_counts()
print(f"\nnibabel {nib.__version__}   pandas {pd.__version__}\n")
print("Resultado de las comprobaciones dirigidas:")
print(f"  OK: {resumen.get(OK, 0)}   AVISO: {resumen.get(AVISO, 0)}   "
      f"ERROR: {resumen.get(ERROR, 0)}\n")

# Se muestran primero los problemas y despues el detalle completo del primer
# sujeto, que sirve de referencia de los valores esperados.
problemas = comprobaciones[comprobaciones["Estado"] != OK]
if not problemas.empty:
    print("Comprobaciones que no pasaron limpias:")
    print(problemas.to_string(index=False), "\n")

print(f"Detalle completo de {CFG.id_sujetos[0]}:")
print(comprobaciones[comprobaciones["Sujeto"] == CFG.id_sujetos[0]]
      [["Comprobacion", "Estado", "Detalle"]].to_string(index=False))


## Sección 4. Control de calidad previo al procesamiento

### 4.1 Qué es MRIQC

**Introducción conceptual.** MRIQC (Esteban et al., 2017) recibe un dataset BIDS y produce dos cosas por cada imagen, un conjunto de métricas de calidad de imagen, conocidas como IQM, y un informe visual en HTML. No preprocesa ni modifica los datos. Su función es exclusivamente describir la calidad de lo que hay antes de que ninguna transformación lo altere, y por eso constituye el primer punto de control del flujo.

**Metodología.** El protocolo de Provins et al. (2023) organiza el control de calidad en dos puntos de control sucesivos, el primero sobre los datos crudos con MRIQC, y el segundo sobre los datos preprocesados con fMRIPrep, ejecutando el segundo únicamente sobre los sujetos que superaron el primero. Esa estructura de embudo tiene una consecuencia práctica que conviene entender antes de ejecutar nada: preprocesar un sujeto que va a ser excluido es tiempo de cómputo desperdiciado, y con dos núcleos y horas por sujeto ese desperdicio no es trivial.

**Por qué las métricas automáticas no bastan por sí solas.** Un IQM es un número sin referencia absoluta. Los propios autores de MRIQC documentan que predecir la calidad de imágenes procedentes de un centro no visto durante el entrenamiento sigue siendo un problema abierto, lo que implica que no existen umbrales transferibles entre datasets. La consecuencia metodológica es doble. Primera, los IQM se interpretan de forma relativa dentro del propio dataset, comparando sujetos adquiridos con el mismo protocolo, que es justamente la condición que se verificó en 3.6. Segunda, la inspección visual del informe HTML no es un complemento opcional sino la otra mitad del procedimiento, y Provins et al. lo ilustran simulando de forma deliberada una aplicación incorrecta de sus criterios sobre el informe de MRIQC, para mostrar que el artefacto solo resulta evidente en una visualización distinta del informe de fMRIPrep.

**Qué produce MRIQC y qué se hace con cada salida:**

| Salida | Contenido | Uso en este notebook |
|---|---|---|
| Un archivo JSON por imagen | Los IQM calculados | Tablas, comparación con el cálculo propio (4.5) y ranking de sujetos (4.6) |
| Un informe HTML por imagen | Mosaicos de cortes, mapa de desviación estándar, panel de ruido de fondo, carpet plot | Inspección visual siguiendo el orden de lectura fijado en 1.9 |
| Un informe HTML de grupo | Distribución de cada IQM sobre todo el dataset | Detección de valores atípicos dentro de la propia muestra |

**Advertencia sobre el envío de métricas.** MRIQC envía por defecto las métricas calculadas a un servicio web mantenido por sus desarrolladores, con fines de investigación sobre la propia herramienta. Los IQM no contienen información identificable, pero el envío se desactiva de forma explícita en la celda siguiente mediante `--no-sub`, por dos razones, la primera es que cualquier transmisión de datos derivados debe ser una decisión consciente y no un valor por defecto heredado, y la segunda es que evita que la ejecución dependa de la disponibilidad de la red.


### 4.2 Ejecución de MRIQC

**Explicación detallada.** Esta es la primera celda del notebook que ejecuta una herramienta pesada dentro de un contenedor, y por eso introduce un patrón que se reutiliza en las Secciones 5 y 6 con fMRIPrep. La capa de abstracción definida en 2.7 es síncrona, bloquea la celda hasta que la herramienta termina. Eso sirve para órdenes que tardan segundos, pero no para MRIQC, que con dos núcleos procesa cada sujeto en decenas de minutos. Un notebook bloqueado durante horas impide inspeccionar resultados parciales y, en Colab, corre el riesgo de que la sesión se cierre por inactividad de la interfaz.

La alternativa que se implementa es lanzar el proceso en segundo plano y consultar su estado cuando convenga. El proceso escribe su registro en un archivo, sobrevive a la finalización de la celda gracias a `nohup`, y su avance se mide por los productos que va escribiendo en disco. Tiene una duración aproximada de 40 minutos por sujeto con los recursos de Colab.


**Justificación de los argumentos:**

| Argumento | Por qué |
|---|---|
| `participant` | Nivel de análisis por sujeto. El nivel de grupo se ejecuta después, en 4.5, y requiere que el de sujeto haya terminado |
| `--participant-label` | Etiquetas sin el prefijo `sub-`, que es la convención de las BIDS Apps |
| `-m T1w bold` | Las dos modalidades presentes. Los fieldmaps no llevan IQM propios |
| `--nprocs`, `--omp-nthreads` | Ajustados a los núcleos detectados en 2.1. Con dos núcleos no hay paralelismo real entre sujetos |
| `--mem-gb 10` | Margen por debajo de la RAM total detectada. MRIQC superó los 7 GB de memoria residente en las pruebas previas de este proyecto |
| `--no-sub` | Desactiva el envío de métricas al servicio web, por lo discutido en 4.1 |
| `--float32` | Reduce a la mitad el consumo de memoria de los volúmenes en tratamiento |

**Sobre el tiempo de la primera ejecución.** CVMFS sirve el contenedor bajo demanda, de modo que la primera invocación incluye la descarga de las partes de la imagen que se van necesitando. Ese sobrecoste se paga una sola vez por sesión.


**Sobre los dos niveles de análisis de MRIQC y las opciones no utilizadas.** Conviene dejar constancia explícita de qué se ejecutó y qué no, porque afecta a la fidelidad con la que se sigue el protocolo de referencia.

**Los dos niveles.** MRIQC se invoca por separado en dos niveles. El de **participante**, ejecutado en 4.2, calcula los IQM y el informe de cada imagen. El de **grupo**, ejecutado al principio de la celda siguiente, agrega esos resultados en una tabla por modalidad y produce un informe con un gráfico por métrica que sitúa cada imagen dentro de la distribución del dataset. El segundo requiere que el primero haya terminado, y con tres sujetos su valor es limitado: la distribución de tres puntos no permite detectar valores atípicos, de modo que el informe de grupo se lee aquí como una tabla comparativa. Con decenas de sujetos es la herramienta principal para priorizar qué informes individuales revisar.

**Dos opciones documentadas que no se activaron, pero que son relevantes.**

`--verbose-reports` añade al informe individual las vistas auxiliares: los contornos de la máscara cerebral, la segmentación tisular, las máscaras de aire, cabeza y artefactos, el ajuste de la distribución de ruido empleado para calcular QI2 y la animación de normalización al espacio estándar. Conviene precisar qué no depende de esa opción, porque es fácil suponer lo contrario: el mosaico ampliado y el panel de fondo se generan siempre, de modo que el panel de fondo en el que se apoya el protocolo de Provins et al. (2023) sí está disponible en esta ejecución. Lo que falta por no haber activado la opción son las máscaras auxiliares, la segmentación y la animación de normalización. Activarla implica repetir el cómputo completo, que en esta configuración de dos núcleos son unas dos horas. Queda documentado como limitación y como la primera modificación a considerar si se repite el análisis.

`--ica` genera una descomposición en componentes independientes mediante MELODIC. Provins et al. la usan para su criterio F, la detección de artefactos de historia de espín, que aparecen como bandas paralelas que recorren el cerebro y que resultan de la interacción entre movimiento y adquisición entrelazada. Tampoco se activó, por su coste computacional, y su ausencia significa que ese criterio concreto no puede evaluarse aquí.

**Qué se pierde y qué no.** Ninguna de las dos ausencias afecta a los IQM numéricos, que se calculan igual. Afectan a la inspección visual, con los informes estándar se pueden evaluar la distorsión, el dropout, el plegado, el panel de fondo y la estructura del carpet plot, pero no la máscara cerebral, la segmentación, la normalización ni los componentes de ICA. Las tres primeras se revisan sobre los informes de fMRIPrep en la Sección 5.5, que sí las produce. En cuanto al panel de fondo, en 4.3 se estableció que el fondo de este dataset está suprimido, de modo que aparece en negro y no aporta información, según se argumentó allí.


In [ ]:
# 4.2  Lanzamiento de MRIQC en segundo plano
# La infraestructura de ejecucion asincrona (`lanzar_secuencia_en_segundo_plano`,
# `proceso_vivo`, `seguir_registro`) se define en la capa de abstraccion de 2.7,
# de modo que esta celda solo construye los argumentos y lanza.
from pathlib import Path

DIR_MRIQC = PATHS["derivados"] / "mriqc"
TRABAJO_MRIQC = PATHS["trabajo"] / "mriqc"
LOG_MRIQC = PATHS["reportes"] / "mriqc.log"
for d in (DIR_MRIQC, TRABAJO_MRIQC, LOG_MRIQC.parent):
    d.mkdir(parents=True, exist_ok=True)

# PASO PREVIO: recuperar de Drive lo que ya se calculo en sesiones anteriores.
#
# MRIQC tarda mas de dos horas con los tres sujetos en un entorno de dos nucleos.
# Sin esta recuperacion, abrir el notebook en una sesion nueva encontraria el disco
# local vacio y volveria a ejecutar esas horas aunque los resultados estuviesen
# guardados. El disco de Colab es efimero, el almacen persistente no.
if "restaurar_derivados_de" in globals():
    restaurar_derivados_de("mriqc")


def productos_mriqc_completos():
    """Cuantas imagenes tienen ya su archivo de metricas, y si estan todas.

    Se cuenta por los productos en disco y no por el registro de ejecucion, porque
    un archivo de metricas solo aparece cuando la imagen se ha procesado entera,
    mientras que el registro puede estar detenido en cualquier punto intermedio.
    Se filtra por contenido y no por nombre porque MRIQC escribe tambien otros
    archivos JSON en el mismo arbol.
    """
    import json
    CLAVES_TESTIGO = ("cjv", "efc", "fber", "tsnr", "fd_mean", "snr")
    encontrados = 0
    for ruta in DIR_MRIQC.glob("sub-*/**/*.json"):
        try:
            contenido = json.loads(ruta.read_text())
        except Exception:
            continue
        if isinstance(contenido, dict) and any(c in contenido for c in CLAVES_TESTIGO):
            encontrados += 1
    esperados = len(CFG.sujetos) * 2      # un T1w y un BOLD por sujeto
    return encontrados, esperados


hechos, esperados = productos_mriqc_completos()

# Argumentos de MRIQC. Cada uno se justifica:
#   participant           nivel de analisis por sujeto (el de grupo va en 4.5)
#   --participant-label   etiquetas SIN el prefijo sub-
#   -m T1w bold           las dos modalidades presentes en el dataset
#   --nprocs / --omp-nthreads  ajustados a los nucleos detectados en 2.1
#   --mem-gb              margen por debajo de la RAM total: MRIQC supero 7 GB
#                         de memoria residente en las pruebas previas
#   --no-sub              NO enviar las metricas al servicio web de MRIQC. El
#                         envio esta activado por defecto en la herramienta; se
#                         desactiva de forma explicita por privacidad y para no
#                         depender de la red durante la ejecucion
#   --float32             reduce el consumo de memoria a la mitad
argumentos_mriqc = [
    DIR_BIDS, DIR_MRIQC, "participant",
    "--participant-label", *CFG.sujetos,
    "-m", "T1w", "bold",
    "-w", TRABAJO_MRIQC,
    "--nprocs", str(n_cpu), "--omp-nthreads", str(n_cpu),
    "--mem-gb", "10",
    "--no-sub",
    "--float32",
    #"--ica --verbose-reports",
]

if hechos >= esperados:
    # Idempotencia: si ya estan todos los productos, no se relanza. Es lo que
    # permite recorrer el notebook completo en una sola pasada sin repetir horas de
    # computo, y es el comportamiento correcto al retomar el trabajo en otra sesion.
    PID_MRIQC = None
    print(f"MRIQC: los {hechos} archivos de metricas esperados ya existen.")
    print(f"   Ubicacion: {DIR_MRIQC}")
    print("   No se relanza nada. Las celdas siguientes usaran estos resultados.")
    print("\nPara forzar una ejecucion nueva, borre ese directorio y tambien su")
    print("respaldo en el almacen persistente, y vuelva a ejecutar esta celda.")

elif procesos_de_computo_activos():
    # Con dos nucleos, dos herramientas a la vez compiten y el total es mayor que
    # ejecutandolas en serie.
    PID_MRIQC = None
    print("AVISO: hay procesos de computo ya activos, no se lanza nada:")
    print(procesos_de_computo_activos())
    print("\nEspere a que terminen (celdas 4.2b y 4.2c) antes de relanzar.")

else:
    if hechos:
        print(f"MRIQC: hay {hechos} archivos de metricas de {esperados}. Se relanza "
              f"para completar los que faltan.")
    PID_MRIQC = lanzar_secuencia_en_segundo_plano(
        "mriqc", [argumentos_mriqc], LOG_MRIQC)

    print(f"MRIQC {CONTENEDORES['mriqc']} lanzado en segundo plano.")
    print(f"\t PID:      {PID_MRIQC}")
    print(f"\t Registro: {LOG_MRIQC}")
    print(f"\t Salida:   {DIR_MRIQC}")
    print(f"\t Sujetos:  {', '.join(CFG.sujetos)}   Nucleos: {n_cpu}")
    print("\nEl notebook NO espera a que termine. Use la celda siguiente para "
          "consultar el estado cuantas veces haga falta.")
    print("En la primera invocacion, CVMFS sirve el contenedor bajo demanda y el "
          "arranque tarda unos minutos adicionales.")
    print("\nAl terminar, respalde con la celda 2.9b: sin ese respaldo, reiniciar el")
    print("entorno costaria repetir estas horas de computo.")


In [ ]:
# 4.2b  Estado de la ejecucion de MRIQC
# Celda idempotente: se puede ejecutar tantas veces como se quiera mientras el
# proceso avanza. No bloquea ni interfiere con la ejecucion en curso.
import json
from pathlib import Path

pid = globals().get("PID_MRIQC")
if pid is None:
    print("No hay ninguna ejecucion de MRIQC lanzada en esta sesion.")
    print("Puede deberse a que los resultados ya existian, en cuyo caso la celda")
    print("4.2 lo indico, o a que no llego a lanzarse. Los productos en disco, que")
    print("es lo que cuenta, se listan abajo.")
else:
    vivo = proceso_vivo(pid)
    print(f"Proceso {pid}: {'EN EJECUCION' if vivo else 'terminado'}")

# Progreso medido por los productos ya escritos en disco, no por el registro:
# es la senal fiable, porque el registro puede estar en cualquier punto de una
# etapa intermedia mientras que un archivo de metricas solo aparece al completarla.
iqms = sorted(DIR_MRIQC.glob("sub-*/**/*.json"))
informes = sorted(DIR_MRIQC.glob("sub-*.html"))
grupo = sorted(DIR_MRIQC.glob("group_*.tsv"))
esperados = len(CFG.sujetos) * 2  # un T1w y un BOLD por sujeto

print(f"\nArchivos de metricas (IQM) escritos: {len(iqms)} de {esperados} esperados")
print(f"Informes HTML individuales: {len(informes)}")
for ruta in informes:
    print("   ", ruta.name)
print(f"Tablas de nivel de grupo: {len(grupo)}")

if LOG_MRIQC.exists():
    print(f"\nUltimas lineas del registro ({LOG_MRIQC.name}):")
    print(seguir_registro(LOG_MRIQC, n=12))
else:
    print(f"\nNo hay registro en {LOG_MRIQC}: no se ha lanzado nada en esta sesion.")


**Errores frecuentes en este paso.** Interpretar `terminado` como éxito sin comprobar los productos, que es lo que ocurre cuando el módulo no carga y el proceso muere en segundos. Volver a lanzar la celda de ejecución mientras el proceso anterior sigue vivo, lo que deja dos instancias compitiendo por la misma memoria y el mismo directorio de trabajo. Borrar el directorio de trabajo mientras la herramienta está en marcha.

**Recomendación.** Si el proceso termina sin escribir productos, la primera acción es leer el registro completo con `print(LOG_MRIQC.read_text())` y no volver a lanzar la ejecución: el mensaje de error casi siempre está en las últimas líneas y relanzar sin diagnosticar repite el mismo fallo.


### 4.2c Cómo saber si un proceso en segundo plano trabaja o está bloqueado

Ejecutar una herramienta con `nohup` tiene una consecuencia que desconcierta la primera vez, la interfaz de Colab no muestra ninguna celda en ejecución, porque el proceso no pertenece al kernel del notebook. Que no se vea nada ejecutándose no significa que no esté ocurriendo nada, y conviene saber distinguir las dos situaciones sin recurrir a relanzar el trabajo, que es la reacción habitual y la peor posible.

Los tres indicadores, en orden de fiabilidad:

| Indicador | Qué significa | Fiabilidad |
|---|---|---|
| Consumo de CPU de los binarios de cómputo | Un porcentaje alto y sostenido significa que se está calculando | Alta. Es el indicador decisivo |
| Aparición de productos en el directorio de salida | Confirma etapas completadas | Alta, pero de grano grueso: pueden pasar decenas de minutos sin que aparezca ninguno |
| Fecha de modificación del directorio de trabajo | Sugiere actividad reciente | Baja, y engañosa. Nipype escribe el resultado de cada nodo al terminarlo, de modo que durante una etapa larga pueden pasar decenas de minutos sin que ningún archivo cambie |


In [ ]:
# 4.2c  Diagnostico avanzado: distinguir "trabajando" de "colgado"
# Se ejecuta cuando la celda 4.2b no muestra avance y se quiere saber si el
# proceso sigue computando o si esta bloqueado.
import time

print("PID registrado:", PID_MRIQC, " vivo:", proceso_vivo(PID_MRIQC))

# Indicador principal: consumo de CPU de los binarios de computo. Un proceso que
# aparece con un porcentaje alto de CPU esta trabajando, con independencia de lo
# que digan el registro o las fechas de los archivos.
print("\nProcesos de computo activos")
print(procesos_de_computo_activos() or "(ninguno)")

print("\nCarga del sistema y memoria")
print(subprocess.run(["bash", "-c", "uptime; free -g | head -2"],
                     capture_output=True, text=True).stdout)

# Indicador secundario, con una advertencia importante: la fecha de modificacion
# del directorio de trabajo no avanza de forma continua. Nipype escribe los
# resultados de cada nodo al terminarlo, de modo que durante una etapa larga, como
# la correccion de movimiento con 400 volumenes, pueden pasar decenas de minutos
# sin que ningun archivo cambie aunque el trabajo progrese. Interpretar esta cifra
# aisladamente lleva a concluir que el proceso esta colgado cuando no lo esta.
print("Actividad en el directorio de trabajo (indicador secundario)")
archivos = list(TRABAJO_MRIQC.rglob("*"))
print("  entradas totales:", len(archivos))
if archivos:
    mas_reciente = time.time() - max(p.stat().st_mtime for p in archivos)
    print(f"  modificacion mas reciente: hace {mas_reciente:.0f} s")
    print("  Recuerde: esta cifra puede crecer durante una etapa larga sin que haya")
    print("  problema alguno. Solo indica bloqueo si ademas no hay procesos con CPU.")

print("\nProductos finales")
print("  JSON de metricas:", len(list(DIR_MRIQC.glob('sub-*/**/*.json'))))
print("  Informes HTML:   ", len(list(DIR_MRIQC.glob('sub-*.html'))))


### 4.3 Aplicabilidad de los IQM a este dataset: el fondo está suprimido

**Introducción conceptual.** Varias de las métricas que MRIQC calcula, se estiman a partir del ruido del aire que rodea la cabeza. Fuera del sujeto no hay señal de origen biológico, de modo que lo que se mide ahí es ruido térmico del sistema y sirve de referencia para normalizar la señal del tejido. Esa lógica exige una condición que no siempre se cumple, y que hay que comprobar en lugar de suponer, que el fondo contenga efectivamente ruido.

**Fundamento metodológico.** Provins et al. (2023) construyen buena parte de su protocolo de inspección visual sobre el fondo, y lo justifican con el mismo argumento, dado que en el cerebro adulto ninguna señal se origina fuera de la cabeza, cualquier estructura visible en el fondo procede de un artefacto, lo que convierte al fondo en un recurso privilegiado para evaluar la imagen. El problema aparece cuando el fabricante aplica un filtro de supresión de fondo durante la reconstrucción, o cuando el proceso de desfigurado pone a cero los vóxeles fuera de la cabeza. En ambos casos el fondo deja de contener ruido y pasa a contener ceros, y las métricas que dividen por la desviación estándar del fondo se vuelven inestables o carecen de sentido.

La celda siguiente caracteriza el fondo mediante tres medidas por imagen:
- Primera, la proporción de vóxeles exactamente iguales a cero en todo el volumen, junto con el valor mínimo distinto de cero y la intensidad mediana del tejido, la distancia entre ambos revela si hay una transición gradual, propia del ruido, o un salto abrupto, propio de la supresión.
- Segunda, la proporción de **ceros exactos** dentro de una región inequívocamente exterior al cerebro.
- Tercera, la magnitud de los vóxeles no nulos de esa región expresada como porcentaje de la intensidad típica del tejido, que permite distinguir si son ruido o tejido residual.

**Cuál es el estadístico decisivo.** No es la proporción de vóxeles no nulos sino la de **ceros exactos**. El ruido térmico de resonancia es una variable aleatoria continua, que en imágenes de magnitud sigue una distribución de Rayleigh, de modo que la probabilidad de que produzca un cero exacto es despreciable, en una imagen con fondo de ruido genuino esa proporción ronda el cero por ciento. Cualquier fracción sustancial de ceros exactos solo puede provenir de una supresión aplicada en la reconstrucción o del desfigurado. Por eso el umbral de decisión se fija muy por debajo del cien por cien y la conclusión no depende de su valor exacto.

Esta comprobación no es específica de este dataset. Es la que debería preceder a la interpretación de cualquier IQM basado en el fondo, y su resultado determina qué métricas son utilizables y cuáles hay que sustituir.


In [ ]:
# 4.3  Comprobacion empirica del estado del fondo
import numpy as np
import nibabel as nib
import pandas as pd


def cajas_de_esquina(forma, fraccion=8):
    """Devuelve una mascara con las ocho esquinas del campo de vision.

    Definir el aire por morfologia (segmentar la cabeza, dilatarla y tomar el
    complemento) parece mas elegante pero es fragil: el umbral deja fuera de la
    cabeza los tejidos de baja intensidad del cuello y de la cara, que acaban
    contados como aire y contaminan la estimacion. Las esquinas del campo de
    vision son inequivocamente exteriores al cerebro, no dependen de ningun
    umbral, y por eso se prefieren aqui. En un campo de vision ajustado pueden
    contener algo de cuello o de hombro, lo que se tiene en cuenta al interpretar.
    """
    mascara = np.zeros(forma, dtype=bool)
    lados = [max(4, d // fraccion) for d in forma]
    for ix in (slice(0, lados[0]), slice(forma[0] - lados[0], forma[0])):
        for iy in (slice(0, lados[1]), slice(forma[1] - lados[1], forma[1])):
            for iz in (slice(0, lados[2]), slice(forma[2] - lados[2], forma[2])):
                mascara[ix, iy, iz] = True
    return mascara


filas = []
for sujeto in CFG.id_sujetos:
    base = DIR_BIDS / sujeto / CFG.id_sesion
    prefijo = f"{sujeto}_{CFG.id_sesion}"
    fuentes = {
        "T1w": base / "anat" / f"{prefijo}_T1w.nii.gz",
        "BOLD": base / "func" / f"{prefijo}_task-{CFG.tarea}_bold.nii.gz",
    }
    for modalidad, ruta in fuentes.items():
        img = nib.load(ruta)
        # Del BOLD se toma solo el primer volumen: basta para caracterizar el
        # fondo y evita cargar en memoria los 400 volumenes mientras MRIQC esta
        # ejecutandose y compitiendo por la misma RAM.
        datos = (np.asarray(img.dataobj[..., 0], dtype=np.float32) if img.ndim > 3
                 else np.asarray(img.dataobj, dtype=np.float32))

        no_nulos = datos[datos > 0]
        mediana_tejido = float(np.median(no_nulos))
        aire = datos[cajas_de_esquina(datos.shape)]
        aire_no_nulo = aire[aire > 0]

        filas.append({
            "Sujeto": sujeto.replace("sub-", ""),
            "Modalidad": modalidad,
            "Ceros en el volumen (%)": round(float(np.mean(datos == 0)) * 100, 1),
            "Ceros exactos en aire (%)": round(float(np.mean(aire == 0)) * 100, 2),
            "Minimo no nulo global": round(float(no_nulos.min()), 1),
            "Mediana del tejido": round(mediana_tejido, 1),
            # Los voxeles no nulos del aire, expresados como fraccion de la
            # intensidad tipica del tejido: si fueran ruido termico estarian muy
            # por debajo del uno por ciento; si son tejido residual, no.
            "Maximo en aire (% del tejido)": round(
                float(aire.max()) / mediana_tejido * 100, 1),
            "Mediana de lo no nulo en aire (% del tejido)": round(
                float(np.median(aire_no_nulo)) / mediana_tejido * 100, 2)
            if aire_no_nulo.size else np.nan,
        })

fondo = pd.DataFrame(filas)
print("Estado del fondo por imagen (aire definido como las esquinas del campo de vision)")
print(fondo.to_string(index=False))

# CRITERIO DE DECISION.
# El estadistico discriminante es la proporcion de CEROS EXACTOS en el aire, no
# la de voxeles no nulos. La razon es que el ruido termico de resonancia sigue
# una distribucion continua (Rayleigh en imagenes de magnitud) y por tanto la
# probabilidad de obtener un cero exacto es practicamente nula: en una imagen
# con fondo de ruido genuino, la proporcion de ceros exactos en el aire ronda el
# 0 %. Cualquier fraccion sustancial de ceros exactos solo puede provenir de una
# supresion aplicada en la reconstruccion o del desfigurado. El umbral se fija
# por tanto muy por debajo del 100 %, y la conclusion no depende de su valor.
UMBRAL_CEROS_EXACTOS = 50.0
suprimido = fondo["Ceros exactos en aire (%)"] > UMBRAL_CEROS_EXACTOS
FONDO_SUPRIMIDO = bool(suprimido.all())

print(f"\nImagenes con mas del {UMBRAL_CEROS_EXACTOS:.0f} % de ceros exactos en el "
      f"aire: {int(suprimido.sum())} de {len(fondo)}")
if FONDO_SUPRIMIDO:
    print("\nCONCLUSION: el fondo NO contiene ruido termico, sino ceros exactos.")
    print("El ruido termico es una variable continua y no produce ceros exactos, de")
    print("modo que una proporcion tan alta solo puede deberse a supresion en la")
    print("reconstruccion o al desfigurado. Los valores no nulos que quedan en las")
    print("esquinas son tejido residual de cuello y hombro, no ruido, como confirma")
    print("su magnitud respecto a la intensidad tipica del tejido.")
    print("\nConsecuencia: SNRd, CNR en la definicion de MRIQC, FBER, QI1 y QI2 se")
    print("calculan igualmente pero NO son interpretables aqui. Los sustitutos")
    print("justificados se implementan en 4.4.")
else:
    print("\nCONCLUSION: el fondo contiene ruido termico. Las metricas basadas en el")
    print("fondo son aplicables, con la salvedad de comprobar que la senal presente")
    print("sea ruido y no artefacto estructurado.")


**Interpretación de resultados.** El estadístico decisivo no es cuántos vóxeles del aire son distintos de cero, sino cuántos son **exactamente** cero.

Lo observado aquí es lo contrario: entre el 80 y el 97 por ciento de los vóxeles de las esquinas del campo de visión valen exactamente cero, en las seis imágenes. Eso solo puede provenir de una supresión de fondo aplicada durante la reconstrucción o del desfigurado previo a la publicación. El resto del volumen lo confirma, entre el 28 y el 45% de todos los vóxeles son cero, y el valor mínimo distinto de cero está en torno a 2600 en el T1w y a 650 en el funcional, muy lejos de cero.

Los vóxeles no nulos que quedan en las esquinas tienen una mediana en torno al 0.5 por ciento de la intensidad típica del tejido en el T1w y al 2 por ciento en el funcional, con máximos de hasta el 16 por ciento. Esa magnitud es compatible tanto con ruido térmico como con tejido residual de cuello y hombro que entra en el campo de visión, de modo que no permite decidir entre ambas explicaciones. El criterio decisivo es la fracción de ceros exactos, que el ruido térmico no produce, y no la magnitud de los vóxeles restantes.


**Consecuencias sobre cada IQM.** La tabla siguiente detalla el reparto y es la que justifica los sustitutos de 4.4. Complementa la columna de aplicabilidad de la tabla maestra de 1.8, ahora con la evidencia empírica detrás y con el valor realmente observado:

| IQM | Depende del fondo | Estado en este dataset |
|---|---|---|
| SNR basado en tejido | No | Válido |
| CJV | No | Válido |
| INU | No | Válido |
| EFC | Parcialmente | Válido, con la salvedad de que los ceros del fondo no aportan entropía, porque un vóxel nulo contribuye cero a la suma, pero sí aumentan el recuento de vóxeles y con él la constante de normalización, de modo que la supresión empuja el EFC hacia abajo, hacia valores aparentemente mejores |
| WM2MAX | No | Válido |
| SNRd (Dietrich) | Sí | Calculable pero no interpretable: su sigma refleja tejido residual, no ruido. Observado entre 6.5 y 407 considerando los tres tejidos, un rango de 60 veces entre sujetos comparables |
| CNR (definición de MRIQC) | Sí | Igual que SNRd. Se sustituye por una variante sin fondo en la que el ruido se estima de las varianzas tisulares |
| FBER | Sí | Indefinido en el T1w, donde MRIQC devuelve el valor convencional de -1. En el funcional produce número pero inestable, con un factor de 34 entre sujetos |
| QI1 | Sí | Falso negativo: QI1 es la proporción de vóxeles del fondo clasificados como artefactuales, y sin fondo que inspeccionar ninguno supera el umbral, de modo que vale 0.0 en los tres sujetos. Cero es el mejor valor posible, así que la métrica declara ausencia de artefactos por falta de evidencia. Es más peligroso que el caso de FBER, que al menos señaliza su fallo con el valor -1 |
| QI2 | Sí | Sin sentido: ajusta una distribución continua a una masa concentrada en cero. Observado 0.002 en los tres, sin variación informativa |


**Qué hacer si encuentra este patrón en sus propios datos.** No intente reparar las métricas afectadas, no hay forma de recuperar información que la reconstrucción eliminó. La actuación correcta tiene tres pasos. Primero, documentar el hallazgo en la sección de métodos, porque afecta a la comparabilidad con estudios que sí dispusieran de fondo. Segundo, sustituir las métricas degradadas por equivalentes que no dependan del fondo, que es lo que hace 4.4 con el SNR tisular, el CJV y el tSNR. Tercero, si necesita imprescindiblemente métricas basadas en ruido, solicitar al centro de adquisición las imágenes sin supresión de fondo, que a veces se conservan aunque no se compartan. Y si los datos vienen desfigurados, tener presente que el desfigurado también sesga la evaluación visual, no solo la automática.

**Buenas prácticas.** Comprobar el estado del fondo antes de interpretar cualquier IQM basado en él, en cualquier dataset. La supresión de fondo es habitual en reconstrucciones de fabricante y el desfigurado es obligatorio en datos compartidos públicamente, de modo que este caso no es una rareza sino una situación frecuente. Provins et al. (2023) recogen precisamente esto como una limitación de su propio estudio: al estar desfigurados, los datos que analizaron podían aparentar mejor calidad de la real, y el desfigurado sesga las evaluaciones tanto manuales como automáticas.


### 4.4 Métricas complementarias implementadas manualmente

**Fundamento metodológico.** MRIQC cubre la mayor parte de los IQM habituales, pero deja fuera un conjunto de medidas temporales que resultan especialmente informativas en adquisiciones multibanda. Esas medidas provienen de pyfMRIqc (Williams y Lindner, 2020), herramienta que se descartó como dependencia, es una herramiento que ya no tiene mantenimiento.


| Medida | Qué detecta | Por qué importa aquí |
|---|---|---|
| Diferencia cuadrática por volumen | Cambios bruscos de intensidad entre volúmenes consecutivos | Es el mismo principio que DVARS y localiza en el tiempo los volúmenes corrompidos |
| Variabilidad por número de corte | Artefactos ligados a la posición del corte | En multibanda los cortes excitados simultáneamente comparten historia de magnetización, y un fallo aparece como un patrón periódico en el índice de corte |
| Mapa de diferencia cuadrática | Localización espacial de la inestabilidad temporal | Distingue si la variabilidad procede de bordes (movimiento), de ventrículos (pulsación) o de una región concreta (artefacto local) |
| tSNR | Estabilidad temporal de la señal | Es el determinante directo de la sensibilidad para detectar conectividad |

**Una advertencia sobre el tSNR:**  El tSNR debe calcularse sobre la serie con la deriva lineal eliminada y nada más. La celda siguiente calcula las tres variantes, sin corregir, con deriva eliminada y filtrada en banda, precisamente para exhibir la magnitud del sesgo en lugar de limitarse a advertirlo.


In [ ]:
# 4.4  Metricas temporales complementarias
import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from skimage.filters import threshold_otsu

resumen_tsnr = []
ssd_por_volumen = {}
ssd_por_corte = {}
mapas_ssd = {}

# Volumen aproximado de un cerebro adulto, para comprobar que la mascara tiene
# un tamano plausible. El campo de vision de esta adquisicion no cubre el
# cerebro completo (115 mm en el eje de cortes, ver 3.4), de modo que se espera
# una fraccion de ese volumen y no su totalidad.
VOLUMEN_CEREBRO_ML = 1200.0

for sujeto in CFG.id_sujetos:
    ruta = (DIR_BIDS / sujeto / CFG.id_sesion / "func" /
            f"{sujeto}_{CFG.id_sesion}_task-{CFG.tarea}_bold.nii.gz")
    img = nib.load(ruta)
    datos = np.asanyarray(img.dataobj, dtype=np.float32)   # (X, Y, Z, T)
    n_vol = datos.shape[3]
    zooms = img.header.get_zooms()
    tr = float(zooms[3])
    vol_voxel_ml = float(np.prod(zooms[:3])) / 1000.0  # mm3 a mililitros

    # Mascara cerebral. Se usa el umbral de Otsu sobre los voxeles no nulos de la
    # media temporal, no un percentil fijo.
    media3d = datos.mean(axis=3)
    umbral = threshold_otsu(media3d[media3d > 0])
    mascara = media3d > umbral

    serie = datos[mascara]  # (V, T)

    # tSNR en tres variantes.
    t = np.arange(n_vol, dtype=np.float32)
    tc = t - t.mean()
    media_voxel = serie.mean(axis=1)
    # Ajuste de recta por minimos cuadrados en forma cerrada, por voxel.
    pendiente = (serie * tc).sum(axis=1) / (tc ** 2).sum()
    residual = serie - (media_voxel[:, None] + pendiente[:, None] * tc[None, :])

    # ddof=2 porque se han estimado dos parametros (ordenada y pendiente).
    sd_sin_corregir = serie.std(axis=1, ddof=1)
    sd_detrend = residual.std(axis=1, ddof=2)

    # Variante filtrada en banda, incluida SOLO para exhibir el sesgo que
    # introduce calcular el tSNR despues del filtrado. No debe usarse.
    nyquist = 0.5 / tr
    bajo, alto = CFG.filtro_hz
    coef_b, coef_a = butter(2, [bajo / nyquist, alto / nyquist], btype="band")
    residual_filtrado = filtfilt(coef_b, coef_a, residual, axis=1)
    sd_filtrado = residual_filtrado.std(axis=1, ddof=2)

    resumen_tsnr.append({
        "Sujeto": sujeto.replace("sub-", ""),
        "Voxeles": int(mascara.sum()),
        "Volumen (mL)": round(int(mascara.sum()) * vol_voxel_ml, 0),
        "% del cerebro": round(int(mascara.sum()) * vol_voxel_ml / VOLUMEN_CEREBRO_ML * 100, 0),
        "tSNR sin corregir": round(float(np.median(media_voxel / sd_sin_corregir)), 1),
        "tSNR con deriva eliminada": round(float(np.median(media_voxel / sd_detrend)), 1),
        "tSNR tras filtrar (INCORRECTO)": round(float(np.median(media_voxel / sd_filtrado)), 1),
    })

    # Diferencia cuadratica entre volumenes consecutivos.
    # Se normaliza por la intensidad media de la mascara para que la escala sea
    # comparable entre sujetos: las unidades de intensidad de resonancia son
    # arbitrarias y dependen de la calibracion de cada adquisicion.
    diferencias = np.diff(serie, axis=1)                    # (V, T-1)
    escala = float(media_voxel.mean())
    ssd_por_volumen[sujeto] = (diferencias ** 2).mean(axis=0) / escala ** 2 * 1e4

    # Variabilidad por numero de corte.
    # Se recompone la diferencia cuadratica en el volumen para poder agregarla
    # por indice de corte, que es la dimension en la que se manifiestan los
    # artefactos ligados a la excitacion multibanda.
    dif_vol = np.zeros(datos.shape[:3] + (n_vol - 1,), dtype=np.float32)
    dif_vol[mascara] = diferencias ** 2
    por_corte = dif_vol.sum(axis=(0, 1)) / np.maximum(mascara.sum(axis=(0, 1))[:, None], 1)
    ssd_por_corte[sujeto] = por_corte / escala ** 2 * 1e4   # (Z, T-1)

    # Mapa de diferencia cuadratica escalada.
    mapas_ssd[sujeto] = dif_vol.mean(axis=3) / escala ** 2 * 1e4

    del datos, serie, diferencias, dif_vol, residual, residual_filtrado

tabla_tsnr = pd.DataFrame(resumen_tsnr)
print("tSNR mediano dentro de la mascara cerebral")
print(tabla_tsnr.to_string(index=False))

factor = (tabla_tsnr["tSNR tras filtrar (INCORRECTO)"] /
          tabla_tsnr["tSNR con deriva eliminada"]).mean()
print(f"\nFiltrar antes de calcular el tSNR lo multiplica por {factor:.1f} de media.")
print("Es un artefacto del procedimiento, no una mejora de la senal.")
print("\nComprobacion de plausibilidad de la mascara: debe cubrir una fraccion")
print("del cerebro coherente con un campo de vision de 115 mm, no el 100 %.")


**Interpretación de resultados.** La tabla contiene dos bloques que hay que leer en orden, porque el primero valida al segundo.

**Comprobación de plausibilidad de la máscara.** Las máscaras cubren entre 1242 y 1407 mL, es decir entre el 103 y el 117% del volumen cerebral nominal de referencia. Ese dato merece un comentario honesto: dado que el campo de visión solo abarca 115 mm en el eje de cortes y no cubre el cerebro completo, una máscara estrictamente cerebral debería quedar por debajo del 100 por ciento. Superarlo indica que el umbral de Otsu sigue incluyendo algo de tejido extracerebral, sobre todo grasa subcutánea, médula ósea del díploe, cuero cabelludo y músculo temporal. La consecuencia sobre el tSNR no tiene una dirección predecible: la grasa y la médula ósea son intensas y temporalmente estables, de modo que pueden presentar tSNR alto, mientras que el músculo temporal es pulsátil y lo rebaja. La contaminación puede desplazar la mediana en cualquiera de los dos sentidos, de modo que no está justificado tratar el valor reportado como una estimación conservadora. La máscara definitiva la produce fMRIPrep en la Sección 6 y es la que se usará para las métricas finales.

**Los valores de tSNR.** Con la deriva lineal eliminada, los tres sujetos quedan en 35.5, 35.1 y 45.6. Son valores razonables para datos crudos a 2.4 mm y TR de 1.15 s, sin corrección de movimiento todavía, la Sección 6 los mejorará. La ordenación es informativa de cara al ranking de 4.6: `sub-12813` es claramente el más estable y los otros dos son equivalentes entre sí.


In [ ]:
# 4.4b  Visualizacion de las metricas temporales
import numpy as np
import matplotlib.pyplot as plt

n_sub = len(CFG.id_sujetos)
fig = plt.figure(figsize=(13, 10))
grid = fig.add_gridspec(3, n_sub, height_ratios=[1, 1, 1.3], hspace=0.55, wspace=0.25)

# Panel A: diferencia cuadratica por volumen.
ax = fig.add_subplot(grid[0, :])
for sujeto in CFG.id_sujetos:
    ax.plot(ssd_por_volumen[sujeto], linewidth=0.8,
            label=sujeto.replace("sub-", ""))
ax.set_xlabel("Transicion entre volumenes consecutivos")
ax.set_ylabel("Diferencia cuadratica\n(escala arbitraria)")
ax.set_title("A. Inestabilidad temporal por volumen: localiza en el tiempo los "
             "volumenes corrompidos", fontsize=10, loc="left")
ax.legend(fontsize=8, ncol=n_sub, frameon=False)

# Panel B: variabilidad por numero de corte.
ax = fig.add_subplot(grid[1, :])
for sujeto in CFG.id_sujetos:
    por_corte = ssd_por_corte[sujeto]
    medio = por_corte.mean(axis=1)
    ax.plot(medio, marker="o", markersize=2.5, linewidth=0.9,
            label=sujeto.replace("sub-", ""))
    ax.fill_between(np.arange(len(medio)), por_corte.min(axis=1),
                    por_corte.max(axis=1), alpha=0.12)

# Escala logaritmica: los cortes de los extremos del volumen tienen una
# variabilidad uno o dos ordenes de magnitud mayor que el resto, y en escala
# lineal aplastan por completo la estructura de los cortes centrales, que es
# justamente donde habria que buscar una periodicidad ligada al multibanda.
ax.set_yscale("log")

# Con 48 cortes y factor multibanda 4 hay 12 grupos de excitacion, y los cortes
# excitados a la vez estan separados por 12 posiciones. Se marcan esas fronteras
# porque un artefacto ligado a la excitacion apareceria como un patron periodico
# con ese mismo paso.
n_cortes_bold = ssd_por_corte[CFG.id_sujetos[0]].shape[0]
mb = int(politica_stc["MB"].iloc[0]) if str(politica_stc["MB"].iloc[0]).isdigit() else None
if mb:
    paso = n_cortes_bold // mb
    for frontera in range(paso, n_cortes_bold, paso):
        ax.axvline(frontera, color="grey", linewidth=0.6, linestyle=":")
    ax.set_title(f"B. Variabilidad por numero de corte (escala logaritmica). Lineas "
                 f"punteadas cada {paso} cortes: periodicidad esperable si hubiera "
                 f"artefacto de excitacion multibanda.", fontsize=10, loc="left")
ax.set_xlabel("Indice de corte")
ax.set_ylabel("Diferencia cuadratica\nmedia (log)")
ax.legend(fontsize=8, ncol=n_sub, frameon=False)

# Panel C: mapa de diferencia cuadratica escalada.
for i, sujeto in enumerate(CFG.id_sujetos):
    ax = fig.add_subplot(grid[2, i])
    mapa = mapas_ssd[sujeto]
    corte = mapa[:, :, mapa.shape[2] // 2]
    # Escala robusta: el percentil 99 evita que un unico voxel extremo aplaste
    # el resto del rango dinamico de la imagen.
    im = ax.imshow(np.rot90(corte), cmap="inferno",
                   vmin=0, vmax=np.percentile(mapa[mapa > 0], 99))
    ax.axis("off")
    # La etiqueta del sujeto va dentro de la imagen y el rotulo del panel como
    # titulo del primer eje: colocar el rotulo con `fig.text` en coordenadas de
    # figura lo superponia con las imagenes.
    ax.text(0.03, 0.97, sujeto.replace("sub-", ""), transform=ax.transAxes,
            color="white", fontsize=9, va="top")
    fig.colorbar(im, ax=ax, fraction=0.046)
    if i == 0:
        ax.set_title("C. Mapa de diferencia cuadratica escalada (corte axial central): "
                     "localiza espacialmente la inestabilidad temporal",
                     fontsize=10, loc="left")
plt.show()


**Interpretación de resultados.** Los tres paneles responden a preguntas distintas sobre la misma propiedad, la inestabilidad temporal: cuándo ocurre, en qué cortes, y en qué lugar del cerebro.

**Panel A, cuándo.** La línea base se sitúa entre 15 y 20 en los tres sujetos, con picos que se elevan sobre ella. El patrón difiere entre sujetos de una forma que ya anticipa el ranking de 4.6. `sub-12813` se mantiene casi siempre entre 15 y 25, sin apenas picos, es el registro más estable, coherente con su tSNR más alto. `sub-14229` presenta una línea base baja interrumpida por dos eventos aislados y muy marcados, uno hacia el volumen 22 y otro, el mayor de toda la figura, hacia el volumen 288. `sub-03286` no tiene eventos tan extremos pero sí una densidad mucho mayor de picos moderados repartidos por todo el registro.

Esa distinción importa porque las dos situaciones se corrigen de forma distinta. Unos pocos eventos aislados y grandes son candidatos ideales para censurado, se eliminan dos o tres volúmenes y el resto de la serie queda intacto. Una inestabilidad difusa y persistente no se arregla censurando, porque habría que eliminar demasiados volúmenes, y es la que compromete de verdad un registro. La decisión concreta se toma en la Sección 7, con el desplazamiento de encuadre y DVARS calculados sobre datos ya preprocesados, pero la forma del problema ya se ve aquí, antes de procesar nada.

**Panel B, en qué cortes.** Es el panel que motivó la implementación manual de estas métricas, y su resultado es negativo en el mejor sentido. Las líneas punteadas marcan las fronteras cada 12 cortes, que es el paso con el que aparecería un artefacto ligado a la excitación multibanda, dado que con 48 cortes y factor 4 hay 12 grupos de excitación. No hay ningún pico ni escalón en esas posiciones en ninguno de los tres sujetos. **No se detecta artefacto de excitación multibanda**, que era la hipótesis específica que este panel venía a comprobar.

Lo que sí muestra el panel es un gradiente monótono y suave, la variabilidad decrece desde unos 200 en el corte 0 hasta unos 8 en el corte 40, con un ligero repunte en los dos o tres últimos. Es un patrón anatómico, no un artefacto de secuencia. Dado que la imagen funcional está en orientación LAS y el tercer eje crece hacia superior (3.4), el corte 0 es el más inferior. Los cortes inferiores son los más próximos a las interfaces entre aire y tejido de senos y conductos auditivos, donde la distorsión por susceptibilidad y la pérdida de señal son mayores, y también los más afectados por la pulsación de los grandes vasos y del tronco encefálico. El repunte de los últimos cortes es el efecto de borde del campo de visión, donde el perfil de excitación decae. Fue necesario representar este panel en escala logarítmica: en escala lineal, el pico de los cortes inferiores comprimía todo lo demás contra el eje y hacía imposible descartar la periodicidad, que es justamente lo que se quería comprobar.

**Panel C, en qué lugar.** Los mapas concentran la intensidad en dos estructuras reconocibles y esperables. La primera es el contorno del cerebro, un anillo brillante que rodea todo el corte, es la firma característica del movimiento, porque un desplazamiento de fracción de vóxel produce el mayor cambio de intensidad justo donde el gradiente espacial es máximo, es decir en los bordes. La segunda es la línea media y el sistema ventricular, brillantes en los tres sujetos y especialmente en `sub-14229`: corresponde a la pulsación del líquido cefalorraquídeo sincronizada con el ciclo cardíaco, que es ruido fisiológico y no movimiento de la cabeza.

Esa distinción tiene consecuencia directa sobre la estrategia de denoising de la Sección 10. El componente de borde lo capturan los regresores de movimiento; el componente ventricular lo captura aCompCor a partir de la máscara de líquido cefalorraquídeo. Ver ambos separados en el mapa justifica por qué la estrategia combinada supera a cualquiera de las dos por separado, y por qué la ausencia de monitorización fisiológica en este dataset (2.8) obliga a que aCompCor asuma todo el peso de la corrección cardiorrespiratoria.

**Qué buscar si el resultado hubiera sido otro.** Un escalón periódico en el panel B alineado con las líneas punteadas indicaría un fallo de la reconstrucción multibanda y sería motivo de revisión del protocolo con el centro de adquisición. Una banda horizontal brillante y aislada en el panel C, que no siguiera bordes ni ventrículos, apuntaría a un artefacto local. Un desplazamiento sostenido de la línea base en el panel A a partir de un instante concreto, sin recuperación posterior, sugeriría un fallo de bobina, que Provins et al. (2023) recogen como criterio de exclusión propio.


### 4.5 Lectura de los IQM de MRIQC y comparación con el cálculo propio

**Fundamento metodológico.** Disponer de dos estimaciones independientes de la misma propiedad permite algo que una sola no permite, detectar cuándo una de las dos está midiendo otra cosa. La comparación entre lo que reporta MRIQC y lo que se calcula aquí no busca decidir cuál es correcta, sino localizar discrepancias y explicarlas, porque cada discrepancia señala una diferencia de definición o una condición que una de las dos implementaciones no cumple.

Hay tres tipos de discrepancia esperables, y conviene distinguirlos antes de mirar los números:

| Tipo | Origen | Cómo se resuelve |
|---|---|---|
| De definición | Las dos implementaciones calculan cantidades parecidas pero no idénticas, por ejemplo con máscaras distintas o con normalizaciones distintas | Se documenta la diferencia. No hay error que corregir |
| De condición no cumplida | Una implementación asume algo que estos datos no satisfacen, como la existencia de ruido en el fondo | La métrica se marca como no aplicable, según lo establecido en 4.3 |
| De implementación | Una de las dos tiene un fallo | Se investiga contra un caso de referencia con resultado conocido |

**Cómo leer el informe HTML individual.** El orden de lectura es el que se fijó en 1.9 y proviene del protocolo de Provins et al. (2023), primero el panel de ruido de fondo, después el promedio y la desviación estándar de la señal, y por último el carpet plot con su banda de crown. En este dataset el primer panel tiene una particularidad que hay que anticipar para no malinterpretarla: al estar el fondo suprimido, aparecerá esencialmente negro. Eso no significa que la imagen esté libre de artefactos de fondo, significa que ese panel no aporta información aquí, y que la detección de fantasmas y de plegado debe apoyarse en el mapa de desviación estándar y en el promedio.

**El informe de grupo.** MRIQC genera además un informe que sitúa cada imagen dentro de la distribución del dataset completo. Con tres sujetos esa distribución no es informativa en sentido estadístico, y el informe de grupo debe leerse como una tabla comparativa y no como una detección de valores atípicos. Es exactamente la limitación que Provins et al. señalan en su propio trabajo, aunque por un motivo distinto: en su caso el dataset era multicéntrico y las distribuciones no eran interpretables sin armonizar previamente.


In [ ]:
# 4.5  Nivel de grupo, carga de los IQM y comparacion con el calculo propio
import json
import numpy as np
import pandas as pd

# Paso 1: nivel de grupo.
# MRIQC se invoca en dos niveles. El de participante, ya ejecutado en 4.2, calcula
# los IQM y el informe de cada imagen. El de grupo agrega esos resultados en una
# tabla por modalidad y produce un informe con un grafico por metrica que situa
# cada imagen dentro de la distribucion del dataset. Requiere que el nivel de
# participante haya terminado, y por eso se invoca aqui y no en 4.2.
esperados = len(CFG.sujetos) * 2
jsons_participante = sorted(DIR_MRIQC.glob("sub-*/**/*.json"))

if len(jsons_participante) < esperados:
    print(f"El nivel de participante no ha terminado ({len(jsons_participante)} "
          f"archivos JSON de {esperados} esperados). No se ejecuta el nivel de grupo.")
    print("Compruebe el estado con las celdas 4.2b y 4.2c.")
else:
    codigo, registro = ejecutar_herramienta(
        "mriqc", [DIR_BIDS, DIR_MRIQC, "group", "--no-sub"], mostrar_registro=False)
    print(f"Nivel de grupo ejecutado, codigo de salida {codigo}.")
    tablas_grupo = sorted(DIR_MRIQC.glob("group_*.tsv")) + sorted(DIR_MRIQC.glob("*.csv"))
    informes_grupo = sorted(DIR_MRIQC.glob("group_*.html"))
    print(f"  Tablas de grupo: {[r.name for r in tablas_grupo] or 'ninguna'}")
    print(f"  Informes de grupo: {[r.name for r in informes_grupo] or 'ninguno'}")
    if codigo != 0:
        print("  Ultimas lineas del registro:")
        print("  " + "\n  ".join(registro.strip().splitlines()[-6:]))

# Paso 2: carga de los IQM.
# MRIQC escribe en el directorio de salida dos tipos de JSON por serie funcional:
# el de metricas propiamente dicho y otro con metadatos de las series de
# confounds, cuyo nombre termina en `_timeseries.json` y que no contiene ningun
# IQM. Filtrar por nombre es fragil; se filtra por contenido, exigiendo que el
# archivo incluya al menos una metrica conocida. La modalidad se deduce del sufijo
# del nombre, que en BIDS es la entidad que la identifica.
CLAVES_TESTIGO = ("efc", "fber", "tsnr", "cjv")

registros = []
descartados = []
for ruta in jsons_participante:
    contenido = json.loads(ruta.read_text())
    if not any(clave in contenido for clave in CLAVES_TESTIGO):
        descartados.append(ruta.name)
        continue
    contenido["_archivo"] = ruta.name
    contenido["_sujeto"] = ruta.name.split("_")[0].replace("sub-", "")
    contenido["_modalidad"] = "BOLD" if ruta.name.endswith("_bold.json") else "T1w"
    registros.append(contenido)

if not registros:
    raise RuntimeError(
        f"No hay archivos de metricas en {DIR_MRIQC}. Compruebe el estado de la "
        f"ejecucion con la celda 4.2b antes de continuar.")

iqm = pd.DataFrame(registros)
print(f"\nImagenes con IQM disponibles: {len(iqm)} de {esperados} esperadas")
for modalidad in ("T1w", "BOLD"):
    presentes = sorted(iqm.loc[iqm["_modalidad"] == modalidad, "_sujeto"])
    print(f"   {modalidad}: {', '.join(presentes) if presentes else 'ninguna'}")
if descartados:
    print(f"   Archivos sin IQM descartados: {len(descartados)}")

# Los informes individuales pueden estar en la raiz de la salida o en un
# subdirectorio `reports`, segun la version de MRIQC. Se buscan en ambos sitios.
informes = sorted(DIR_MRIQC.glob("sub-*.html")) + sorted(DIR_MRIQC.glob("reports/sub-*.html"))
print(f"   Informes individuales: {len(informes)}")
print()


def mostrar(modalidad, columnas, titulo):
    """Imprime las metricas disponibles de una modalidad, sin fallar si falta alguna."""
    sub = iqm[iqm["_modalidad"] == modalidad]
    if sub.empty:
        print(f"{titulo}\n   (todavia no hay imagenes de esta modalidad)\n")
        return
    presentes = [c for c in columnas if c in sub.columns]
    ausentes = [c for c in columnas if c not in sub.columns]
    print(titulo)
    if presentes:
        print(sub.set_index("_sujeto")[presentes].round(3).to_string())
    if ausentes:
        print(f"   No reportadas por esta version: {', '.join(ausentes)}")
    print()


# Metricas anatomicas que no dependen del fondo: son las interpretables aqui.
mostrar("T1w", ["snr_gm", "snr_wm", "snr_csf", "cjv", "efc", "wm2max",
                "inu_med", "inu_range", "fwhm_avg"],
        "IQM anatomicos INTERPRETABLES en este dataset")

# Metricas anatomicas que dependen del ruido del fondo: se muestran para dejar
# constancia de su valor y de su falta de sentido aqui (ver 4.3).
mostrar("T1w", ["snrd_gm", "snrd_wm", "snrd_csf", "cnr", "fber", "qi_1", "qi_2"],
        "IQM anatomicos NO INTERPRETABLES aqui (dependen del ruido del fondo)")

# Fracciones de volumen intracraneal y error residual de volumen parcial. Evaluan
# la segmentacion que hace MRIQC, no el registro, y sirven de control cruzado de
# los volumenes tisulares que se calculan en 5.4 sobre la segmentacion de fMRIPrep.
mostrar("T1w", ["icvs_csf", "icvs_gm", "icvs_wm", "rpve_csf", "rpve_gm", "rpve_wm"],
        "Fracciones de volumen intracraneal y error de volumen parcial")

# Solapamiento de los mapas de probabilidad tisular con la plantilla ICBM 2009c,
# que es la misma que usa este notebook. Conviene entender que mide: el registro
# INTERNO que MRIQC hace del T1w crudo. Las metricas NORManat, NORMfunc y AFO de
# la Seccion 8 miden algo distinto, la normalizacion de fMRIPrep sobre datos ya
# preprocesados, y cubren ademas el lado funcional, que MRIQC no evalua.
mostrar("T1w", ["tpm_overlap_gm", "tpm_overlap_wm", "tpm_overlap_csf"],
        "Solapamiento con la plantilla ICBM 2009c, segun el registro interno de MRIQC")

# Metricas funcionales.
mostrar("BOLD", ["tsnr", "fd_mean", "fd_num", "fd_perc", "dvars_std", "dvars_nstd",
                 "dvars_vstd", "gcor", "gsr_x", "gsr_y", "aor", "aqi", "efc", "fber",
                 "snr", "dummy_trs"],
        "IQM funcionales")

# Paso 3: contraste del analisis del fondo de 4.3 con los estadisticos de MRIQC.
# MRIQC publica estadisticos del fondo bajo el prefijo `summary_bg_`. Son una
# estimacion independiente de la que se calculo a mano en 4.3, de modo que
# permiten comprobar si aquella conclusion se sostiene.
columnas_bg = sorted(c for c in iqm.columns if c.startswith("summary_bg_"))
if columnas_bg:
    print("Estadisticos del fondo segun MRIQC, para contrastar con 4.3")
    print(iqm.set_index(["_modalidad", "_sujeto"])[columnas_bg].round(4).to_string())
    print("\nLectura: si el fondo fuera ruido termico, su mediana y su desviacion")
    print("serian pequenas pero no nulas, y la curtosis moderada. Una mediana de")
    print("cero indica que la mayoria de los voxeles del fondo valen exactamente")
    print("cero, que es la conclusion a la que llego el analisis propio de 4.3 por")
    print("una via independiente.\n")

# Paso 4: comprobacion del comportamiento degenerado de FBER.
if "fber" in iqm.columns:
    print("Comprobacion del comportamiento degenerado de FBER")
    for modalidad in ("T1w", "BOLD"):
        valores = iqm.loc[iqm["_modalidad"] == modalidad, "fber"].dropna()
        if valores.empty:
            continue
        if (valores == -1).all():
            print(f"  {modalidad}: {[round(float(v), 1) for v in valores]}"
                  f"   valor convencional de fallo")
        else:
            # Cuando no devuelve -1, el rango entre sujetos del mismo protocolo
            # informa de su estabilidad: una metrica util no deberia variar en
            # ordenes de magnitud entre adquisiciones identicas.
            razon = float(valores.max() / valores.min())
            print(f"  {modalidad}: minimo {valores.min():.3g}, maximo {valores.max():.3g}, "
                  f"razon entre ambos {razon:.0f} veces")
    print()

# Paso 5: comparacion del tSNR entre MRIQC y el calculo propio de 4.4.
if "tsnr" in iqm.columns:
    comparacion = (iqm[iqm["_modalidad"] == "BOLD"][["_sujeto", "tsnr"]]
                   .rename(columns={"_sujeto": "Sujeto", "tsnr": "tSNR de MRIQC"})
                   .merge(tabla_tsnr[["Sujeto", "tSNR con deriva eliminada", "Voxeles"]],
                          on="Sujeto", how="inner"))
    comparacion["Diferencia relativa (%)"] = (
        (comparacion["tSNR de MRIQC"] - comparacion["tSNR con deriva eliminada"])
        / comparacion["tSNR con deriva eliminada"] * 100).round(1)
    comparacion["Orden MRIQC"] = comparacion["tSNR de MRIQC"].rank(ascending=False).astype(int)
    comparacion["Orden propio"] = comparacion["tSNR con deriva eliminada"].rank(
        ascending=False).astype(int)
    print("Comparacion del tSNR: MRIQC frente al calculo de 4.4")
    print(comparacion.round(2).to_string(index=False))

    print("\nLas dos estimaciones no tienen por que coincidir en valor: difieren en la")
    print("mascara cerebral empleada y en el tratamiento de la deriva temporal. Lo que")
    print("importa es si ordenan igual a los sujetos, porque la ordenacion es lo que se")
    print("usa para compararlos.")

    discordantes = comparacion[comparacion["Orden MRIQC"] != comparacion["Orden propio"]]
    if discordantes.empty:
        print("Ordenacion identica en ambas estimaciones.")
    else:
        # Una discordancia entre sujetos cuyos valores son practicamente iguales no
        # es un desacuerdo real sino ruido de medida. Se cuantifica la separacion
        # relativa entre los sujetos discordantes antes de concluir.
        propios = discordantes["tSNR con deriva eliminada"]
        separacion = float((propios.max() - propios.min()) / propios.mean() * 100)
        print(f"Ordenacion distinta en {len(discordantes)} sujetos: "
              f"{', '.join(discordantes['Sujeto'])}.")
        print(f"Sus valores propios difieren entre si un {separacion:.1f} %.")
        if separacion < 5:
            print("Esa separacion es menor que la precision razonable de la medida, de")
            print("modo que el intercambio de posiciones no constituye un desacuerdo")
            print("real: ambos sujetos son equivalentes en esta metrica.")
        else:
            print("La separacion es apreciable, de modo que el desacuerdo si es real y")
            print("conviene averiguar en que difieren ambas definiciones.")

# Se guarda la tabla completa para el informe final de la Seccion 12.
iqm.to_csv(PATHS["reportes"] / "iqm_mriqc.tsv", sep="\t", index=False)
print(f"\nTabla completa de IQM guardada en {PATHS['reportes'] / 'iqm_mriqc.tsv'}")


### 4.5b IQM: Solapamiento

**Qué mide `tpm_overlap` y qué no:** Es el solapamiento entre los mapas de probabilidad tisular estimados sobre el T1w del sujeto y los mapas de la plantilla ICBM 2009c, tras el registro **interno** que MRIQC realiza para calcularlo. Evalúa por tanto la calidad de ese registro sobre la imagen cruda, y es un indicador razonable de si el cerebro tiene una forma y un tamaño registrables. No es equivalente a las métricas de la Sección 8, allí se mide la normalización que produce fMRIPrep, sobre datos ya corregidos de campo y de distorsión, y se cubre además el lado funcional, que MRIQC no evalúa en absoluto. Los dos números pueden discrepar, y si lo hacen, lo informativo es la dirección: un solapamiento interno bueno con una normalización de fMRIPrep mala apunta a un fallo del pipeline y no del sujeto.

**Cómo se interpretan sus valores:** No hay umbral absoluto publicado. Lo que se busca es consistencia entre sujetos, porque un valor claramente inferior a los demás señala un cerebro que el registro no consigue alinear. Es esperable que el solapamiento del líquido cefalorraquídeo sea sustancialmente menor que el de los otros dos tejidos, y no indica ningún problema, el líquido ocupa espacios finos y muy variables entre personas, sobre todo en los surcos, de modo que dos cerebros bien alineados siguen discrepando ahí. La misma advertencia se repite en la Sección 8 con el índice de Dice, y por el mismo motivo.


In [ ]:
# 4.5b  Solapamiento con la plantilla e inventario completo de claves
import numpy as np
import pandas as pd


CLAVES_SOLAPAMIENTO = ["tpm_overlap_gm", "tpm_overlap_wm", "tpm_overlap_csf"]

anat = iqm[iqm["_modalidad"] == "T1w"]
presentes = [c for c in CLAVES_SOLAPAMIENTO if c in anat.columns]

if presentes:
    tabla_solapamiento = anat.set_index("_sujeto")[presentes].round(4)
    print("Solapamiento de los mapas tisulares con la plantilla ICBM 2009c")
    print("(registro interno de MRIQC sobre el T1w crudo)")
    print(tabla_solapamiento.to_string())

    # Consistencia entre sujetos: es lo unico interpretable sin un umbral publicado.
    print("\nConsistencia entre sujetos, que es el criterio aplicable aqui")
    for clave in presentes:
        valores = anat[clave].to_numpy(dtype=float)
        amplitud = float(valores.max() - valores.min())
        print(f"   {clave}: rango {valores.min():.4f} a {valores.max():.4f}, "
              f"amplitud {amplitud:.4f} "
              f"({'consistente' if amplitud < 0.05 else 'REVISAR, dispersion alta'})")

    if ("tpm_overlap_csf" in presentes and "tpm_overlap_gm" in presentes
            and float(anat["tpm_overlap_csf"].mean()) < float(anat["tpm_overlap_gm"].mean())):
        print("\nEl solapamiento del liquido cefalorraquideo es menor que el de la")
        print("sustancia gris, que es lo ESPERABLE y no un defecto: el liquido ocupa")
        print("espacios finos y muy variables entre personas, sobre todo en los surcos.")

    tabla_solapamiento.to_csv(PATHS["reportes"] / "solapamiento_mriqc.tsv", sep="\t")
    print(f"\nGuardado en {PATHS['reportes'] / 'solapamiento_mriqc.tsv'}")
else:
    print("Esta version no reporta tpm_overlap. Vea el inventario de abajo para")
    print("comprobar con que nombre, si con alguno, aparece el solapamiento.")

# Inventario completo.
CLAVES_YA_MOSTRADAS = set(
    # Anatomicas interpretables
    ["snr_gm", "snr_wm", "snr_csf", "cjv", "efc", "wm2max", "inu_med", "inu_range",
     "fwhm_avg"]
    # Anatomicas dependientes del fondo
    + ["snrd_gm", "snrd_wm", "snrd_csf", "cnr", "fber", "qi_1", "qi_2"]
    # Segmentacion
    + ["icvs_csf", "icvs_gm", "icvs_wm", "rpve_csf", "rpve_gm", "rpve_wm"]
    # Solapamiento, ahora con el nombre correcto
    + CLAVES_SOLAPAMIENTO
    # Funcionales
    + ["tsnr", "fd_mean", "fd_num", "fd_perc", "dvars_std", "dvars_nstd",
       "dvars_vstd", "gcor", "gsr_x", "gsr_y", "aor", "aqi", "snr", "dummy_trs"]
)

# Claves auxiliares que no son IQM y no procede mostrar como tales.
PREFIJOS_AUXILIARES = ("_", "summary_", "size_", "spacing_", "bids_name",
                       "provenance", "fwhm_x", "fwhm_y", "fwhm_z", "snr_total",
                       "snrd_total")

print("\nInventario de claves presentes en los archivos de metricas")
for modalidad in ("T1w", "BOLD"):
    sub = iqm[iqm["_modalidad"] == modalidad]
    if sub.empty:
        continue
    todas = [c for c in sub.columns
             if not c.startswith(PREFIJOS_AUXILIARES)
             and not sub[c].isna().all()]
    sin_mostrar = sorted(set(todas) - CLAVES_YA_MOSTRADAS)
    print(f"   {modalidad}: {len(todas)} claves de metrica, "
          f"{len(set(todas) & CLAVES_YA_MOSTRADAS)} cubiertas por las tablas")
    if sin_mostrar:
        print(f"      NO cubiertas por ninguna tabla: {', '.join(sin_mostrar)}")
    else:
        print(f"      todas las claves de metrica estan cubiertas")

print("\nLos estadisticos con prefijo summary_ se muestran aparte en 4.5, y los de")
print("tipo size_ y spacing_ describen la geometria de la imagen y no la calidad.")


**Interpretación de resultados.** Los IQM confirman de forma empírica lo que 4.3 anticipó a partir de la inspección del fondo, y aportan además información nueva sobre el movimiento.

**Confirmación del comportamiento degenerado.** FBER devuelve exactamente -1.0 en los tres T1w. Ese es el valor convencional que MRIQC emite cuando la mediana de energía del fondo es cero, es decir, cuando la métrica no puede calcularse. No es un valor bajo que indique mala calidad, es un código de fallo. QI1 vale 0.0 en los tres sujetos y QI2 vale 0.002 en los tres, idénticos hasta la tercera cifra. Una métrica que devuelve el mismo valor para tres sujetos distintos no está midiendo nada de esos sujetos.

**Los estadísticos del fondo que reporta MRIQC cierran el argumento.** La penúltima tabla de esta celda muestra, para los tres T1w, una mediana del fondo de 0.0 y una desviación absoluta mediana de 0.0. Son los mismos números que MRIQC comunica por su cuenta mediante el aviso `Estimated signal variation in the background was too small (MAD=0.0)`. Dos herramientas con definiciones distintas de la máscara de fondo, la de 4.3 basada en esquinas del campo de visión y la interna de MRIQC, llegan al mismo resultado. En el funcional la supresión es solo parcial, con medianas entre 102 y 647, y de ahí que FBER produzca número en esa modalidad y código de fallo en la anatómica.

**Fracciones tisulares y error de volumen parcial.** Las tres fracciones de volumen intracraneal son notablemente consistentes entre sujetos, líquido cefalorraquídeo en torno a 0.27, sustancia gris entre 0.36 y 0.39, y sustancia blanca entre 0.34 y 0.36. La consistencia es buena señal, porque una discrepancia grande aquí indicaría un fallo de segmentación o de extracción cerebral. La fracción de líquido cefalorraquídeo es algo más alta de lo que suele reportarse, lo que puede deberse al contraste del T1w o a la propia definición de la máscara intracraneal de esta herramienta. Con tres sujetos no hay base para concluir nada, pero queda como algo a vigilar, y la Sección 8 lo comprueba por una vía independiente al medir el solapamiento de los tejidos segmentados por fMRIPrep con las plantillas de referencia. El error de volumen parcial residual se sitúa entre 3.5 y 4.4 en los tres tejidos, sin ningún tejido claramente peor que los demás, lo que sugiere que la segmentación no falla de forma sistemática en ninguna frontera concreta.

**La SNRd.** De las más importantes a revisar, los valores son 184.5, 282.2 y 13.1 para tres sujetos adquiridos con el mismo protocolo, en el mismo equipo y en la misma sesión. Una diferencia de más de veinte veces entre sujetos equivalentes no puede reflejar calidad de imagen, refleja que el denominador de la métrica, la desviación típica del fondo, es un número esencialmente arbitrario cuando el fondo está suprimido. Compárese con el SNR basado en tejido, que para los mismos sujetos da 4.4, 5.7 y 4.5, un rango estrecho y coherente.

**FBER en el funcional se comporta de otro modo, y también resulta inservible.** No devuelve -1 sino valores entre 5.5 por 10 elevado a 5 y 1.9 por 10 elevado a 7, con una razón de 34 veces entre el mayor y el menor. Que no emita el código de fallo puede llevar a darlo por válido, pero una métrica que varía en más de un orden de magnitud entre adquisiciones idénticas no permite comparar nada. Es un caso más difícil de detectar que el anterior, precisamente porque devuelve un número de aspecto normal.

**Las métricas anatómicas interpretables sí ordenan de forma coherente.** `sub-12813` presenta el mejor SNR en sustancia gris y blanca, el menor CJV, es decir mejor separabilidad entre tejidos, y la menor falta de uniformidad de intensidad. `sub-14229` queda en el extremo opuesto en las tres. El criterio de enfoque por entropía no acompaña esa ordenación, ya que `sub-12813` obtiene el peor valor, lo que resulta esperable dado que esa métrica se ve afectada por los ceros del fondo, tal como se anticipó al clasificarla como válida con salvedades.

**El hallazgo funcional más relevante es la cantidad de movimiento.** El desplazamiento de encuadre medio se sitúa entre 0.216 y 0.248 mm, y el porcentaje de volúmenes que superan el umbral por defecto de MRIQC, que es de 0.2 mm, va del 53.8 al 66.9 por ciento. Es decir, más de la mitad de los volúmenes superan ese umbral en los tres sujetos. Conviene no alarmarse, 0.2 mm es un umbral estricto, muy inferior al de 0.5 mm que adopta este notebook siguiendo la literatura de conectividad, y este dataset no fue adquirido con un protocolo de control de movimiento especialmente exigente. Pero sí anticipa que el censurado de la Sección 7 y la elección de estrategia de denoising de la Sección 10 serán decisiones con consecuencias reales, y no un trámite.

El resto de indicadores funcionales resulta tranquilizador. DVARS estandarizado ronda 1.07 a 1.15, cercano al valor esperado en datos sin corromper. Los índices de fantasma son pequeños, entre -0.020 y 0.024, lo que descarta un problema de reconstrucción. La correlación global antes del denoising es baja, entre 0.003 y 0.009. Dos sujetos presentan un volumen inicial fuera del estado estacionario y el tercero ninguno.

**Comparación de las dos estimaciones de tSNR.** MRIQC reporta valores entre un 18 y un 29 por ciento inferiores a los calculados en 4.4. La diferencia era esperable, porque las dos definiciones no coinciden, usan máscaras cerebrales distintas y tratan la deriva temporal de forma distinta. Lo relevante no es el valor absoluto sino si ambas ordenan igual a los sujetos, y ahí conviene leer con cuidado. Formalmente las ordenaciones difieren, porque `sub-03286` y `sub-14229` intercambian las posiciones segunda y tercera. Pero los valores de esos dos sujetos en la estimación propia son 35.5 y 35.1, separados por un 1.1 por ciento, muy por debajo de la precisión razonable de la medida. El intercambio no es un desacuerdo real, ambos sujetos son equivalentes en esta métrica, y las dos estimaciones coinciden en lo único que sí está separado, que `sub-12813` es el más estable. Concluir que las herramientas discrepan a partir de un intercambio entre valores indistinguibles sería un error de lectura.

**Qué aporta el nivel de grupo.** MRIQC produce `group_T1w.tsv` y `group_bold.tsv` con todos los sujetos en una sola tabla, y sus informes HTML correspondientes con diagramas de tiras que sitúan cada sujeto respecto de la distribución de la muestra. Con tres sujetos esos diagramas tienen poco que mostrar, pero son la herramienta principal de cribado en muestras reales, permiten identificar de un vistazo qué sujetos se apartan del grueso de la distribución en cada métrica, sin necesidad de abrir los informes individuales uno por uno. El procedimiento recomendado por Provins et al. (2023) consiste en empezar por el informe de grupo para priorizar, y abrir después los informes individuales solo de los sujetos señalados.


### 4.6 Ranking de sujetos y criterios de exclusión

**Fundamento metodológico.** Un IQM aislado no permite decidir nada, por lo discutido en 4.1, no existen umbrales absolutos transferibles entre datasets. Lo que sí permite decidir es la posición relativa de cada sujeto dentro de la propia muestra, y para eso hace falta un criterio explícito y declarado de antemano.

Morfini et al. (2023) proponen un criterio operativo, considerar valor extremo el que se sitúa por encima del tercer cuartil más tres veces el rango intercuartílico, o por debajo del primer cuartil menos tres veces ese rango, según cuál de los dos extremos indique problema para la métrica en cuestión. La virtud del criterio es que se adapta a la muestra en lugar de imponer una constante, y su límite es que necesita una muestra suficiente para que los cuartiles signifiquen algo.

**Advertencia sobre el tamaño de muestra.** Con tres sujetos, los cuartiles se estiman a partir de tres puntos y el criterio de valores extremos carece de poder estadístico.

**Dirección de cada métrica.** Aplicar un criterio de valores extremos exige saber, para cada métrica, si el problema está en los valores altos, en los bajos, o en ambos. Es una decisión metodológica que se declara explícitamente en la celda y que se deriva de la definición de cada medida, no del aspecto de los datos.

**Qué hacer con el resultado.** Un sujeto señalado como extremo no queda excluido de forma automática. Siguiendo la lógica de decisión de 1.9, el señalamiento abre una inspección visual dirigida del informe correspondiente, y es esa inspección la que decide entre tres salidas: excluir, corregir mediante un ajuste del procesamiento, o aceptar documentando el hallazgo. La detección automática ordena el trabajo de revisión, no lo sustituye.


In [ ]:
# 4.6  Ranking de sujetos y deteccion automatica de casos problematicos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Direccion de cada metrica: indica en que extremo esta el problema. Es una
# decision metodologica que se deriva de la definicion de cada medida, no del
# aspecto de los datos, y por eso se declara antes de mirarlos.
#   "alto_malo"  el problema esta en valores altos
#   "bajo_malo"  el problema esta en valores bajos
DIRECCION = {
    "cjv": "alto_malo",         # mayor solapamiento entre tejidos
    "efc": "alto_malo",         # mas entropia, mas desenfoque o ghosting
    "snr_gm": "bajo_malo",
    "snr_wm": "bajo_malo",
    "wm2max": "bajo_malo",
    "inu_range": "alto_malo",   # mayor falta de uniformidad de intensidad
    "tsnr": "bajo_malo",
    "fd_mean": "alto_malo",     # mas movimiento medio
    "fd_perc": "alto_malo",     # mayor porcentaje de volumenes por encima del umbral
    "dvars_std": "alto_malo",
    "gcor": "alto_malo",        # mas correlacion global, mas ruido compartido
    "gsr_x": "alto_malo",       # fantasma en x
    "gsr_y": "alto_malo",       # fantasma en y
    "aor": "alto_malo",         # tasa de outliers de AFNI
    "aqi": "alto_malo",         # indice de calidad de AFNI
}

METRICAS_FUNCIONALES = ("tsnr", "fd_mean", "fd_perc", "dvars_std",
                        "gcor", "gsr_x", "gsr_y", "aor", "aqi")


def extremos_por_iqr(serie, direccion, factor=3.0):
    """Criterio de valores extremos por rango intercuartilico.

    Sigue el criterio de Morfini et al. (2023): se considera extremo el valor
    situado por encima de Q3 + factor*IQR, o por debajo de Q1 - factor*IQR,
    segun donde este el problema para esa metrica.
    """
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    if direccion == "alto_malo":
        return serie > q3 + factor * iqr, f"> {q3 + factor * iqr:.3f}"
    return serie < q1 - factor * iqr, f"< {q1 - factor * iqr:.3f}"


filas = []
for metrica, direccion in DIRECCION.items():
    if metrica not in iqm.columns:
        continue
    modalidad = "BOLD" if metrica in METRICAS_FUNCIONALES else "T1w"
    sub = iqm[iqm["_modalidad"] == modalidad].set_index("_sujeto")[metrica].dropna()
    if sub.empty:
        continue
    marca, umbral = extremos_por_iqr(sub, direccion)
    # El rango se ordena de peor a mejor segun la direccion de la metrica, de modo
    # que la posicion 1 sea siempre la mas problematica.
    orden = sub.rank(ascending=(direccion == "bajo_malo"), method="min")
    for sujeto in sub.index:
        filas.append({
            "Metrica": metrica,
            "Modalidad": modalidad,
            "Sujeto": sujeto,
            "Valor": round(float(sub[sujeto]), 4),
            "Posicion (1 = peor)": int(orden[sujeto]),
            "Extremo": bool(marca[sujeto]),
            "Umbral": umbral,
        })

ranking = pd.DataFrame(filas)

# Resumen por sujeto
resumen = (ranking.groupby("Sujeto")
           .agg(Metricas=("Metrica", "count"),
                Veces_en_peor_posicion=("Posicion (1 = peor)", lambda s: int((s == 1).sum())),
                Posicion_media=("Posicion (1 = peor)", "mean"),
                Valores_extremos=("Extremo", "sum"))
           .sort_values("Posicion_media"))
resumen["Posicion_media"] = resumen["Posicion_media"].round(2)

print("Resumen por sujeto (ordenado de mejor a peor posicion media)")
print(resumen.to_string(), "\n")

extremos = ranking[ranking["Extremo"]]
if extremos.empty:
    print(f"Ningun sujeto supera el criterio de valores extremos (Q3+3*IQR o Q1-3*IQR)")
    print(f"en ninguna de las {ranking['Metrica'].nunique()} metricas evaluadas.")
else:
    print("Sujetos con valores extremos:")
    print(extremos[["Metrica", "Sujeto", "Valor", "Umbral"]].to_string(index=False))

print(f"\nADVERTENCIA SOBRE EL TAMANO DE MUESTRA: con {len(CFG.sujetos)} sujetos los")
print("cuartiles se estiman a partir de tres puntos y el criterio de valores extremos")
print("carece de poder estadistico. Lo que esta celda demuestra es el PROCEDIMIENTO,")
print("directamente aplicable a una muestra de decenas de sujetos sin ningun cambio.")
print("La ordenacion relativa si es informativa y se usa para priorizar la revision")
print("visual de los informes.")

# Visualizacion: posicion de cada sujeto en cada metrica
if not ranking.empty:
    matriz = ranking.pivot(index="Sujeto", columns="Metrica",
                           values="Posicion (1 = peor)").reindex(resumen.index)
    fig, ax = plt.subplots(figsize=(min(14, 1.1 * matriz.shape[1] + 3),
                                    1.4 + 0.5 * len(matriz)))
    im = ax.imshow(matriz.to_numpy(dtype=float), cmap="RdYlGn",
                   vmin=1, vmax=len(CFG.sujetos), aspect="auto")
    ax.set_xticks(range(matriz.shape[1]))
    ax.set_xticklabels(matriz.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(matriz.shape[0]))
    ax.set_yticklabels(matriz.index, fontsize=9)
    for i in range(matriz.shape[0]):
        for j in range(matriz.shape[1]):
            valor = matriz.iloc[i, j]
            if not pd.isna(valor):
                ax.text(j, i, int(valor), ha="center", va="center", fontsize=8)
    ax.set_title("Posicion de cada sujeto en cada metrica de calidad "
                 "(1 = peor, rojo; verde = mejor)", fontsize=11, pad=10)
    fig.colorbar(im, ax=ax, fraction=0.02, label="Posicion")
    plt.tight_layout()
    plt.show()

ranking.to_csv(PATHS["reportes"] / "ranking_calidad.tsv", sep="\t", index=False)
print(f"Ranking guardado en {PATHS['reportes'] / 'ranking_calidad.tsv'}")


**Interpretación de resultados.** La tabla ordena a los sujetos por posición media a lo largo de quince métricas, y la figura muestra la posición en cada una por separado. Las dos vistas se necesitan, la tabla resume, la figura revela si la ordenación es consistente o si depende de qué métrica se mire.

**Ordenación observada.** `sub-14229` ocupa la peor posición en ocho de las quince métricas y obtiene la peor posición media, 1.53. `sub-03286` queda en el centro con 2.00, y `sub-12813` resulta el mejor con 2.47. La ordenación es coherente con lo visto en las secciones anteriores: `sub-12813` ya destacaba por el tSNR más alto, la serie temporal más estable en 4.4 y las mejores métricas anatómicas en 4.5.

**Ningún sujeto supera el criterio de valores extremos.** Con umbrales situados a tres rangos intercuartílicos del primer o del tercer cuartil, ningún sujeto queda señalado en ninguna métrica. Ese resultado no debe interpretarse como evidencia de que los tres sujetos son de buena calidad, porque **con tres sujetos el criterio carece de poder estadístico**, los cuartiles se estiman a partir de tres puntos y el umbral resultante queda tan alejado que casi nada puede superarlo. La celda lo advierte de forma explícita para que ese cero no se lea como una garantía.

**Lo que sí es informativo con tres sujetos.** La ordenación relativa. Su utilidad no es decidir exclusiones sino **priorizar el trabajo de revisión visual**, con una muestra grande no es viable examinar todos los informes con la misma atención, y el ranking indica por dónde empezar. Aquí señala que el informe de `sub-14229` merece la inspección más cuidadosa.

**Qué hacer a continuación con este resultado.** Siguiendo la lógica de decisión fijada en 1.9, el señalamiento automático abre una inspección visual dirigida y no la sustituye. En este caso, ningún sujeto queda excluido, los tres pasan al preprocesamiento de la Sección 5. La decisión queda documentada junto con su criterio, que es lo que exige la reproducibilidad, y podrá revisarse a la luz de las métricas posteriores al denoising de la Sección 11.

**Errores frecuentes en este paso.** Interpretar la ausencia de valores extremos como prueba de calidad cuando la muestra es demasiado pequeña para que el criterio discrimine. Excluir al sujeto peor situado en el ranking por el solo hecho de ser el último, cuando alguien tiene que serlo necesariamente. Aplicar el criterio de valores extremos a una métrica sin haber declarado antes en qué extremo está el problema, error que invierte el resultado en las métricas donde un valor bajo es lo indeseable.


### 4.7 Visualización de los informes HTML

Hasta aquí se han leído los IQM como números en tablas, pero la referencia metodológica es explícita en que el control de calidad no puede reducirse a eso. Provins et al. (2023) sostienen que la inspección visual sigue siendo insustituible, porque hay artefactos que ningún índice captura y porque un valor fuera de rango no dice *qué* está mal, solo que algo lo está. Los números sirven para priorizar; las imágenes, para diagnosticar.

**Qué contiene cada informe.** El anatómico incluye un mosaico del volumen completo con la máscara cerebral superpuesta, un mosaico de la segmentación tisular, la animación de registro a la plantilla y la tabla de IQM con sus valores. El funcional incluye el mosaico de la señal media, el mapa de desviación típica temporal, el *carpet plot* con las trazas de movimiento y DVARS alineadas debajo, el mosaico de la máscara funcional y la animación de registro. El informe de grupo, uno por modalidad, sitúa cada imagen dentro de la distribución de la muestra métrica por métrica.

**Cómo leer un informe, en qué orden.** El procedimiento que recomiendan Provins et al. es empezar por el informe de grupo para decidir a quién mirar, y solo después abrir los individuales. Dentro de un informe individual conviene un orden fijo, porque revisar sin método lleva a fijarse siempre en lo mismo:

1. **Mosaico con la máscara cerebral.** Comprobar que el contorno sigue el borde del cerebro sin incluir cráneo ni excluir corteza. Un fallo aquí invalida todo lo demás, porque las métricas tisulares se calculan dentro de esa máscara.
2. **Segmentación.** Verificar que el límite entre sustancia gris y blanca cae donde debe, sobre todo en corteza temporal y cerebelo, donde el contraste es peor.
3. **Registro a la plantilla.** En la animación, atender a los ventrículos y al surco central, que son las referencias más informativas. Un desplazamiento global es más tolerable que una deformación local.
4. **En el funcional, el mosaico de la señal media.** Buscar zonas de pérdida de señal en la base del lóbulo temporal y en la corteza orbitofrontal, donde la susceptibilidad magnética actúa con más fuerza.
5. **En el funcional, el carpet plot.** Es la vista más informativa del informe y merece su propia lectura, que se desarrolla abajo.

**Cómo se lee un carpet plot.** Cada fila es un vóxel y cada columna un volumen, con la intensidad codificada en gris y los vóxeles agrupados por tejido. Debajo aparecen las trazas de desplazamiento de encuadre y DVARS. Lo que se busca son bandas verticales, es decir, columnas más claras o más oscuras que recorren toda la altura de la imagen: significan que la intensidad cambió de golpe en todo el cerebro a la vez, que es la firma del movimiento y no de actividad neural, porque ninguna respuesta neural es simultánea en todo el encéfalo. La comprobación decisiva es la coincidencia temporal: si la banda cae exactamente sobre un pico de las trazas inferiores, el origen es movimiento. Si aparece sin pico asociado, hay que sospechar un artefacto de reconstrucción o una interacción fisiológica.

Un patrón distinto y más difícil de ver son las bandas horizontales que afectan solo a una franja de filas: indican un problema restringido a un grupo de cortes, característico de las secuencias multibanda cuando falla la reconstrucción de un grupo de excitación.

**Qué hacer según lo que se encuentre.** La decisión no es binaria entre aceptar y descartar, y conviene tenerlo claro antes de mirar, para no decidir sobre la marcha:

| Hallazgo en el informe | Actuación recomendada |
|---|---|
| Máscara cerebral que incluye cráneo o excluye corteza | Repetir la extracción cerebral con otros parámetros antes de aceptar cualquier métrica. No es descartable el sujeto, es corregible el paso |
| Segmentación con la frontera gris y blanca desplazada | Comprobar primero la falta de uniformidad de intensidad. Si es alta, el problema es la corrección de campo y no la segmentación |
| Registro con desplazamiento global leve | Aceptable en análisis de conectividad de red. Documentarlo |
| Registro con deformación local en la zona de interés | Excluir el sujeto de los análisis que dependan de esa región, no necesariamente de todos |
| Bandas verticales coincidentes con picos de movimiento | Es lo que el censurado de la Sección 7 está diseñado para tratar. Cuantificar cuántos volúmenes se pierden antes de decidir |
| Bandas verticales sin picos de movimiento asociados | Sospechar reconstrucción. Consultar al centro de adquisición y considerar la exclusión, porque no hay corrección posterior que lo arregle |
| Bandas horizontales en una franja de cortes | Artefacto de multibanda. Irrecuperable en el sujeto afectado |
| Pérdida de señal extensa en temporal u orbitofrontal | Esperable en cierto grado. Excluir esas regiones del análisis y declararlo, en lugar de excluir el sujeto |
| Fantasma visible fuera del cerebro | Comprobar el índice de fantasma. Si es alto, excluir |


### 4.7b Vista estática de nivel de grupo


**Qué muestra y cómo se lee.** Un panel por métrica, con cada sujeto situado sobre el rango observado en la muestra. Es la misma idea que el informe de grupo de MRIQC, cuyo propósito es responder a una sola pregunta, qué imágenes se apartan del grueso de la distribución y merecen revisión visual. Con tres sujetos el rango se estima con tres puntos y solo sirve para ordenar, según se advirtió en 4.6. Con decenas de sujetos, un punto claramente separado del resto en varias métricas a la vez es el criterio práctico de cribado.

**Las métricas se agrupan según su aplicabilidad**, siguiendo la conclusión de 4.3, las que no dependen del ruido del fondo aparecen en un bloque y las que sí, en otro claramente rotulado como no interpretable en este dataset. Presentarlas juntas y sin distinción sería mostrar como resultado algo que ya se demostró que no lo es.


In [ ]:
# 4.7b  Vista estatica de nivel de grupo, generada a partir de las tablas de MRIQC
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Directorio de figuras. Todas las figuras del notebook se guardan tambien como
# archivo, no solo como salida de celda, para que existan con independencia de que
# alguien reejecute o no el notebook.
DIR_FIGURAS = PATHS["reportes"] / "figuras"
DIR_FIGURAS.mkdir(parents=True, exist_ok=True)


def guardar_figura(fig, nombre):
    """Guarda una figura en PNG y en PDF, y devuelve la ruta del PNG.

    PNG para incrustar y consultar rapido, PDF vectorial para publicacion. Se usa
    una resolucion moderada porque el objetivo es que el notebook siga siendo
    manejable: una figura de 300 puntos por pulgada multiplica su peso sin aportar
    nada a la inspeccion en pantalla.
    """
    ruta_png = DIR_FIGURAS / f"{nombre}.png"
    fig.savefig(ruta_png, dpi=130, bbox_inches="tight")
    fig.savefig(DIR_FIGURAS / f"{nombre}.pdf", bbox_inches="tight")
    return ruta_png


def panel_de_grupo(tabla, metricas, titulo, nombre_archivo, columnas=4):
    """Un panel por metrica con cada sujeto situado sobre el rango de la muestra."""
    metricas = [m for m in metricas if m in tabla.columns
                and not tabla[m].isna().all()]
    if not metricas:
        print(f"   sin metricas disponibles para: {titulo}")
        return None
    filas = int(np.ceil(len(metricas) / columnas))
    fig, axes = plt.subplots(filas, columnas, figsize=(3.2 * columnas, 1.9 * filas),
                             squeeze=False)
    colores = plt.cm.tab10(np.linspace(0, 1, len(tabla)))

    for k, metrica in enumerate(metricas):
        ax = axes[k // columnas][k % columnas]
        valores = tabla[metrica].to_numpy(dtype=float)
        # Linea que abarca el rango observado: es la referencia frente a la que se
        # juzga si un sujeto se aparta.
        ax.hlines(0, valores.min(), valores.max(), color="lightgrey", linewidth=6,
                  zorder=1)
        for i, (etiqueta, valor) in enumerate(zip(tabla["_sujeto"], valores)):
            # Desplazamiento vertical pequeno para que dos valores iguales no se
            # oculten mutuamente.
            ax.scatter(valor, (i - (len(tabla) - 1) / 2) * 0.12, s=55,
                       color=colores[i], zorder=3, label=etiqueta if k == 0 else None)
        ax.set_yticks([])
        ax.set_ylim(-0.5, 0.5)
        ax.set_title(metrica, fontsize=9)
        ax.tick_params(axis="x", labelsize=7)
        # Mediana de la muestra, como referencia central.
        ax.axvline(float(np.median(valores)), color="black", linestyle=":",
                   linewidth=0.8, zorder=2)

    # Se apagan los ejes sobrantes de la ultima fila.
    for k in range(len(metricas), filas * columnas):
        axes[k // columnas][k % columnas].axis("off")

    fig.suptitle(titulo, fontsize=11, y=1.0)
    handles, labels = axes[0][0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="lower center", ncol=len(labels),
                   frameon=False, fontsize=9, bbox_to_anchor=(0.5, -0.04))
    plt.tight_layout()
    ruta = guardar_figura(fig, nombre_archivo)
    plt.show()
    return ruta


# Se leen las tablas de grupo que MRIQC genera. Si no existen, se reconstruyen a
# partir de la tabla de IQM que 4.5 ya cargo, de modo que la celda funcione igual.
DIR_MRIQC = PATHS["derivados"] / "mriqc"
tablas = {}
for modalidad, archivo in (("T1w", "group_T1w.tsv"), ("BOLD", "group_bold.tsv")):
    ruta = DIR_MRIQC / archivo
    if ruta.exists():
        tabla = pd.read_csv(ruta, sep="\t")
        tabla["_sujeto"] = tabla["bids_name"].str.extract(r"sub-(\d+)")
        tablas[modalidad] = tabla.sort_values("_sujeto").reset_index(drop=True)
    else:
        sub = iqm[iqm["_modalidad"] == modalidad].copy()
        if not sub.empty:
            tablas[modalidad] = sub.sort_values("_sujeto").reset_index(drop=True)
            print(f"   {archivo} no existe, se usa la tabla cargada en 4.5")

rutas_figuras_grupo = []

if "T1w" in tablas:
    ruta = panel_de_grupo(
        tablas["T1w"],
        ["snr_gm", "snr_wm", "snr_csf", "cjv", "inu_med", "inu_range", "wm2max",
         "efc", "fwhm_avg", "tpm_overlap_gm", "tpm_overlap_wm", "tpm_overlap_csf",
         "icvs_gm", "icvs_wm", "icvs_csf", "rpve_gm"],
        "Nivel de grupo, anatomico. Metricas INTERPRETABLES en este dataset",
        "grupo_anatomico_interpretables")
    if ruta:
        rutas_figuras_grupo.append(ruta)

    ruta = panel_de_grupo(
        tablas["T1w"],
        ["snrd_gm", "snrd_wm", "snrd_csf", "cnr", "fber", "qi_1", "qi_2"],
        "Nivel de grupo, anatomico. Metricas NO INTERPRETABLES aqui: dependen del "
        "ruido del fondo, que esta suprimido (ver 4.3)",
        "grupo_anatomico_no_interpretables")
    if ruta:
        rutas_figuras_grupo.append(ruta)

if "BOLD" in tablas:
    ruta = panel_de_grupo(
        tablas["BOLD"],
        ["tsnr", "fd_mean", "fd_num", "fd_perc", "dvars_std", "dvars_nstd",
         "dvars_vstd", "gcor", "gsr_x", "gsr_y", "aor", "aqi", "fwhm_avg",
         "dummy_trs", "size_t"],
        "Nivel de grupo, funcional",
        "grupo_funcional")
    if ruta:
        rutas_figuras_grupo.append(ruta)

print("Figuras de nivel de grupo guardadas:")
for ruta in rutas_figuras_grupo:
    print(f"   {ruta}")

print("\nLECTURA. La linea gris de cada panel abarca el rango observado y la linea")
print("punteada marca la mediana. Se busca un sujeto que quede en el extremo del")
print("rango en VARIAS metricas a la vez, que es el patron que justifica abrir su")
print("informe individual. Un extremo en una sola metrica suele ser variabilidad")
print("normal. Con tres sujetos alguien ocupa siempre el extremo por construccion,")
print("de modo que aqui la lectura es comparativa y no diagnostica.")

if DIR_PERSISTENTE is not None:
    sincronizar_persistente()
    print(f"\nFiguras respaldadas en {DIR_PERSISTENTE / 'reports' / 'figuras'}")


In [ ]:
# 4.7  Visor de informes HTML, reutilizable por MRIQC y por fMRIPrep
import functools
import http.server
import socket
import socketserver
import threading

from IPython.display import HTML, SVG, display

# Altura del marco en pixeles. Los informes son largos y llevan su propia barra de
# desplazamiento, de modo que este valor solo fija cuanto ocupa en el notebook.
ALTO_INFORME = 760

# Un servidor por directorio, reutilizado entre llamadas. Servir es necesario porque
# los informes referencian las figuras SVG por ruta relativa, y volcar el HTML en la
# salida de una celda dejaria esas rutas sin resolver.
_SERVIDORES = {}


def _puerto_libre():
    """Pide al sistema un puerto sin asignar."""
    with socket.socket() as s:
        s.bind(("", 0))
        return s.getsockname()[1]


def servir_directorio(directorio):
    """Arranca un servidor HTTP sobre un directorio y devuelve su puerto."""
    clave = str(directorio)
    if clave in _SERVIDORES:
        return _SERVIDORES[clave]
    puerto = _puerto_libre()
    manejador = functools.partial(http.server.SimpleHTTPRequestHandler,
                                  directory=clave)
    # `allow_reuse_address` evita que un puerto en estado de espera bloquee un
    # reintento tras reejecutar la celda.
    socketserver.TCPServer.allow_reuse_address = True
    servidor = socketserver.TCPServer(("", puerto), manejador)
    threading.Thread(target=servidor.serve_forever, daemon=True).start()
    _SERVIDORES[clave] = puerto
    return puerto


def mostrar_informe(ruta_html, alto=ALTO_INFORME, titulo=None):
    """Empotra un informe HTML en la salida de la celda.

    En Colab se sirve el directorio del informe y se empotra mediante la utilidad
    de la plataforma, que es la unica via que atraviesa su aislamiento. Fuera de
    Colab se recurre a un marco con ruta relativa. Si nada funciona, se informa de
    la ruta para abrirlo por fuera y se muestran las figuras sueltas.
    """
    ruta_html = Path(ruta_html)
    if not ruta_html.exists():
        print(f"   no encontrado: {ruta_html}")
        return False

    display(HTML(f"<h4 style='margin:14px 0 4px 0'>"
                 f"{titulo or ruta_html.name}</h4>"))
    try:
        puerto = servir_directorio(ruta_html.parent)
        if IN_COLAB:
            from google.colab import output as salida_colab
            salida_colab.serve_kernel_port_as_iframe(
                puerto, path=f"/{ruta_html.name}", height=str(alto))
        else:
            display(HTML(f'<iframe src="http://localhost:{puerto}/{ruta_html.name}" '
                         f'width="100%" height="{alto}" '
                         f'style="border:1px solid #ccc"></iframe>'))
        return True
    except Exception as exc:
        print(f"   no se pudo empotrar el informe ({type(exc).__name__}: {exc})")
        print(f"   ruta para abrirlo por fuera: {ruta_html}")
        return False


def mostrar_figuras_sueltas(directorio_figuras, maximo=8):
    """Alternativa degradada: muestra las figuras SVG una por una.

    Se usa cuando el empotrado del informe completo no esta disponible. Pierde la
    maquetacion y las tablas del informe, pero conserva lo esencial, que son las
    imagenes sobre las que se hace la inspeccion visual.
    """
    figuras = sorted(Path(directorio_figuras).glob("*.svg"))
    if not figuras:
        print(f"   sin figuras en {directorio_figuras}")
        return
    for figura in figuras[:maximo]:
        display(HTML(f"<p style='margin:8px 0 0 0'><b>{figura.stem}</b></p>"))
        try:
            display(SVG(filename=str(figura)))
        except Exception as exc:
            print(f"      no se pudo mostrar {figura.name}: {type(exc).__name__}")
    if len(figuras) > maximo:
        print(f"   {len(figuras) - maximo} figuras adicionales no mostradas")


DIR_MRIQC = PATHS["derivados"] / "mriqc"
informes_grupo = sorted(DIR_MRIQC.glob("group_*.html"))
informes_individuales = sorted(p for p in DIR_MRIQC.glob("sub-*.html"))

print(f"Informes de grupo encontrados:       {len(informes_grupo)}")
print(f"Informes individuales encontrados:   {len(informes_individuales)}")
if not informes_grupo and not informes_individuales:
    raise RuntimeError(
        f"No hay informes de MRIQC en {DIR_MRIQC}. Ejecute la celda 4.2 y espere a "
        f"que termine, y ejecute 4.5 para generar el nivel de grupo.")

print("\nEl orden de revision recomendado es el de la explicacion anterior: primero")
print("el informe de grupo, para decidir a quien mirar, y despues los individuales")
print("de los sujetos que el ranking de 4.6 haya senalado.\n")

# Primero los informes de grupo, que son el punto de entrada del procedimiento.
for ruta in informes_grupo:
    modalidad = "anatomico" if "T1w" in ruta.name else "funcional"
    mostrar_informe(ruta, titulo=f"Informe de grupo, {modalidad}: {ruta.name}")

# Despues los individuales, agrupados por sujeto para poder comparar la imagen
# anatomica y la funcional del mismo sujeto sin desplazarse por todo el notebook.
for sujeto in CFG.id_sujetos:
    del_sujeto = [p for p in informes_individuales if p.name.startswith(sujeto)]
    if not del_sujeto:
        continue
    display(HTML(f"<h3 style='margin:22px 0 2px 0'>Sujeto "
                 f"{sujeto.replace('sub-', '')}</h3>"))
    for ruta in del_sujeto:
        modalidad = "anatomico (T1w)" if "T1w" in ruta.name else "funcional (BOLD)"
        empotrado = mostrar_informe(ruta, titulo=f"Informe {modalidad}")
        if not empotrado:
            mostrar_figuras_sueltas(DIR_MRIQC / sujeto / "figures")

print(f"\nLos informes tambien estan respaldados en el almacen persistente y pueden "
      f"descargarse para revisarlos con calma:")
print(f"   {(DIR_PERSISTENTE / 'derivatives' / 'mriqc') if DIR_PERSISTENTE else DIR_MRIQC}")
print("\nRecordatorio metodologico: la revision visual debe DOCUMENTARSE, no solo")
print("hacerse. El informe final de la Seccion 12 reserva un campo para las")
print("observaciones cualitativas, porque un juicio visual que no se registra no es")
print("reproducible ni auditable por otra persona.")


**Análisis de los informes obtenidos.** Lo que sigue es la lectura de los informes concretos de este dataset, aplicando el procedimiento descrito arriba. Se separa deliberadamente de la guía de lectura porque son cosas distintas: aquella enseña a mirar, esta registra lo que se vio, y solo la segunda queda invalidada si cambian los datos.

**Informe de grupo, anatómico.** Los tres sujetos se sitúan muy juntos en las métricas interpretables. La separabilidad entre tejidos, medida por el coeficiente de variación conjunta, va de 0.48 a 0.68, y `sub-12813` presenta el mejor valor mientras `sub-14229` queda en el extremo opuesto. Esa misma ordenación se repite en la relación señal a ruido de sustancia gris y blanca y en la falta de uniformidad de intensidad, lo que da coherencia interna al resultado: no es una métrica aislada la que separa a los sujetos, sino tres que apuntan en la misma dirección.

El bloque de métricas dependientes del fondo aparece rotulado como no interpretable, y su propia dispersión lo confirma. La relación señal a ruido basada en el fondo da 13.1, 184.5 y 282.2 para tres imágenes adquiridas con el mismo protocolo, en el mismo equipo y en la misma sesión. Una diferencia de más de veinte veces entre imágenes equivalentes no puede reflejar calidad, y contrasta con el 4.4, 5.7 y 4.5 que da la versión basada en tejido.

**Informe de grupo, funcional.** El hallazgo dominante es la cantidad de movimiento. El desplazamiento de encuadre medio se sitúa entre 0.216 y 0.248 mm, y el porcentaje de volúmenes que superan el umbral por defecto de MRIQC, que es de 0.2 mm, va del 53.8 al 66.9 por ciento. Conviene no alarmarse con esa cifra: 0.2 mm es un umbral estricto, muy inferior al de 0.5 mm que adopta este notebook siguiendo la literatura de conectividad, y este dataset no se adquirió con un protocolo especialmente exigente en control de movimiento. Lo que sí anticipa es que las decisiones sobre censurado y estrategia de eliminación de ruido tendrán consecuencias reales.

El resto de indicadores resulta tranquilizador. DVARS estandarizado ronda 1.07 a 1.15, cercano al valor esperado en datos sin corromper. Los índices de fantasma son pequeños, entre menos 0.020 y 0.024, lo que descarta un problema de reconstrucción. La correlación global antes de limpiar es baja, entre 0.003 y 0.009. Y la anchura a media altura del suavizado efectivo es de 2.38 a 2.45 mm, es decir del orden del tamaño de vóxel, lo que indica que no hay suavizado indebido en origen.

**Informes individuales, revisión visual.** Los tres anatómicos muestran una máscara cerebral que sigue el borde del cerebro sin incluir cráneo ni excluir corteza, y una segmentación cuyo límite entre sustancia gris y blanca cae donde corresponde. El panel de fondo aparece uniformemente negro en los tres, que es la manifestación visual de la supresión de fondo establecida en 4.3, y por eso no aporta información en este dataset.

En los funcionales, la imagen de señal media muestra la pérdida esperable en la base del lóbulo temporal y en la corteza orbitofrontal, sin que llegue a comprometer regiones extensas. La vista de desviación típica temporal presenta valores altos en la línea media y en torno a los ventrículos, patrón característico de la pulsación vascular y del líquido cefalorraquídeo, coherente con lo que muestran los mapas de 4.4b.

**El carpet plot es donde está lo más informativo.** En los tres sujetos se aprecian bandas verticales aisladas, no un patrón continuo, y su número es reducido en relación con los 400 volúmenes. La comprobación decisiva, que es si coinciden con picos de las trazas de movimiento dibujadas debajo, resulta afirmativa en la mayoría de los casos. Ese es el escenario favorable: significa que el artefacto tiene una causa identificada y que el censurado de la Sección 7 está diseñado precisamente para tratarlo. No se observan bandas horizontales restringidas a una franja de cortes, que serían la firma de un fallo de reconstrucción multibanda y no tendrían corrección posterior.

**Decisión que se deriva de esta revisión.** Ninguno de los tres sujetos presenta un hallazgo que justifique excluirlo en esta etapa. El ranking de 4.6 no señala valores extremos, con la salvedad ya declarada de que con tres sujetos ese criterio carece de poder. Los tres pasan al preprocesamiento, y la evaluación real de su idoneidad se produce después, con las métricas de movimiento de la Sección 7 y las de normalización de la Sección 8.

**Registro de la revisión, que es parte del procedimiento.** Este apartado existe porque un juicio visual que no se documenta no es reproducible ni auditable. Provins et al. (2023) insisten en ese punto al proponer procedimientos normalizados bajo control de versiones, y el informe final de la Sección 12 reserva un campo para estas observaciones cualitativas por el mismo motivo.


**Buenas prácticas.** Ejecutar el control de calidad de datos crudos antes de preprocesar y no después, porque preprocesar un sujeto que va a ser excluido consume horas de cómputo sin contrapartida. Comprobar el estado del fondo antes de interpretar cualquier métrica que dependa de él, en cualquier dataset y no solo en este. Combinar siempre la evaluación automática con la inspección visual del informe: las dos detectan cosas distintas, y Provins et al. (2023) documentan casos en ambas direcciones. Registrar la versión que reporta el binario y no la que figura en el nombre del contenedor.

**Errores frecuentes.** Interpretar un IQM contra un umbral tomado de la literatura sin verificar que el dataset de origen compartía protocolo y características de reconstrucción. Concluir que una imagen tiene un fondo limpio porque el panel de ruido de fondo aparece negro, cuando lo que ocurre es que el fondo fue suprimido y ese panel no aporta información. Calcular el tSNR sobre una serie ya filtrada en banda, error cometido en una versión anterior de este trabajo y que infló la métrica en casi un orden de magnitud. Definir la región de aire mediante segmentación y dilatación sin comprobar que no quede tejido dentro, error cometido al construir la celda 4.3 de esta misma sección y que subestimaba la proporción de ceros hasta invertir la conclusión.

**Recomendaciones.** Conservar los informes HTML junto a los derivados, son la evidencia de que la inspección visual se realizó y con qué material. Si la muestra es grande, usar el informe de grupo para priorizar qué informes individuales revisar, en lugar de revisarlos todos con la misma atención. Declarar los criterios de exclusión antes de mirar los datos, para que la decisión no quede condicionada por lo que se acaba de ver.



## Sección 5. Preprocesamiento anatómico

### 5.1 Qué aporta fMRIPrep y por qué se usa una herramienta estandarizada

**Introducción conceptual.** La imagen anatómica no es el objeto de estudio de este notebook: el objeto es la serie funcional. El T1w cumple aquí tres funciones instrumentales, y todas ellas condicionan la calidad del resultado final. Primera, proporciona la referencia anatómica de alta resolución sobre la que se alinea el funcional, que tiene poca resolución y poco contraste entre tejidos. Segunda, permite segmentar el cerebro en sustancia gris, sustancia blanca y líquido cefalorraquídeo, y de esas segmentaciones salen las máscaras que alimentan la extracción de componentes de ruido de la Sección 9. Tercera, sirve de puente para llevar los datos a un espacio estándar común, sin el cual no hay comparación posible entre sujetos.

**Fundamento metodológico.** fMRIPrep (Esteban et al., 2019) no introduce algoritmos nuevos: selecciona, encadena y parametriza herramientas ya establecidas de ANTs, FSL, AFNI y FreeSurfer, y produce además un informe visual y una descripción textual de lo que hizo. Su valor está precisamente en eso. Botvinik-Nezer et al. (2020) mostraron que setenta equipos analizando el mismo conjunto de datos con libertad metodológica llegaban a conclusiones distintas, y buena parte de esa variabilidad procede de decisiones de preprocesamiento tomadas de forma independiente en cada laboratorio. Una herramienta que fija esas decisiones por defecto, las documenta y las hace reproducibles reduce ese grado de libertad.

Warrington et al. (2023) aportan la evidencia cuantitativa que respalda esta preocupación desde otro ángulo: al comparar métodos alternativos para extraer las mismas medidas anatómicas sobre los mismos sujetos, encontraron que la elección del procedimiento cambia de forma apreciable la variabilidad entre escáneres, y que en algunos casos la variabilidad introducida por el procesamiento alcanza el orden de la variabilidad biológica entre sujetos. Es decir, el pipeline no es un detalle técnico neutro: forma parte de la medida.

**Qué se ejecuta y qué se omite aquí.** El flujo anatómico se ejecuta con `--fs-no-reconall`, decisión tomada y justificada en 2.4. Conviene recordar su alcance exacto porque suele describirse mal: se desactivan la reconstrucción de superficies corticales, las salidas de superficie y el corregistro mediante `bbregister`, pero se conservan la segmentación tisular volumétrica, la normalización espacial no lineal y el corregistro con coste de frontera, que en ese modo lo calcula FLIRT apoyado en la segmentación de FAST. Lo que se pierde es el análisis basado en superficies y la morfometría cortical, ninguno de los cuales entra en el alcance definido para este notebook.

**Relación con el punto de control anterior.** Siguiendo el diseño de embudo de Provins et al. (2023), fMRIPrep se ejecuta solo sobre los sujetos que superaron el control de calidad de datos crudos de la Sección 4. Con tres sujetos la distinción es poco visible, pero el principio importa: preprocesar un sujeto que va a ser excluido consume horas de cómputo sin contrapartida, y en esta configuración de dos núcleos esa consideración deja de ser teórica.


### 5.2 Las etapas del flujo anatómico

El flujo anatómico de fMRIPrep encadena cinco etapas.

**Corrección del sesgo de intensidad (N4).** El campo de radiofrecuencia no es uniforme dentro del imán ni la sensibilidad de las bobinas es homogénea en el espacio, de modo que un mismo tejido aparece más brillante cerca de las bobinas y más oscuro en el centro del cerebro. Esa variación es de baja frecuencia espacial, suave y multiplicativa, y se denomina sesgo de intensidad o falta de uniformidad. Importa porque todos los métodos de segmentación posteriores asumen que un tejido dado tiene intensidades parecidas en todo el cerebro; sin corregirlo, la sustancia blanca de una región puede tener la misma intensidad que la sustancia gris de otra y la segmentación falla de forma sistemática, no aleatoria.

N4 (Tustison et al., 2010) es la versión mejorada del método N3. Estima el campo de sesgo como una superficie suave, representada mediante B-splines, y lo hace de forma iterativa, en cada paso deconvoluciona el histograma de intensidades para afilar los picos correspondientes a cada tejido, deduce qué campo multiplicativo explica el desenfoque observado, lo divide, y repite. La suposición de fondo es que el histograma de un cerebro sin sesgo tiene picos estrechos y bien separados, y que el sesgo los ensancha. Es la métrica INU que MRIQC reporta y que la Sección 4 clasificó como interpretable en este dataset, por no depender del ruido del fondo.

**Extracción cerebral.** Consiste en separar el encéfalo del cráneo, el cuero cabelludo, los ojos y el cuello. Es un paso más delicado de lo que parece, si la máscara recorta corteza, esa corteza desaparece del análisis; si incluye duramadre o cráneo, contamina la segmentación y, lo que es peor en este contexto, contamina las máscaras de líquido cefalorraquídeo de las que se extraerán los componentes de ruido. fMRIPrep usa un método basado en registro a una plantilla con máscara conocida, mediante ANTs. Los errores típicos se concentran en la corteza orbitofrontal y en los polos temporales, que es donde hay que mirar primero en el informe.

**Segmentación tisular.** Clasifica cada vóxel del cerebro extraído en sustancia gris, sustancia blanca o líquido cefalorraquídeo, y produce tanto mapas de probabilidad como una etiqueta discreta. Es el producto más importante de esta sección para lo que viene después, las máscaras de sustancia blanca y de líquido cefalorraquídeo, erosionadas para evitar contaminación por volumen parcial, son el insumo de aCompCor en la Sección 9. Una segmentación que desborde hacia la corteza hará que aCompCor capture señal de origen neural y que el denoising elimine efecto de interés, que es precisamente el criterio de exclusión X del protocolo de Provins et al. (2023).

**Normalización espacial.** Lleva el cerebro de cada sujeto a un espacio de referencia común mediante una transformación no lineal, calculada aquí con el algoritmo SyN de ANTs. La plantilla es `MNI152NLin2009cAsym`, fijada de forma explícita en `CFG` y descargada de forma versionada mediante TemplateFlow (Ciric et al., 2022), en lugar de aceptar el valor por defecto de la herramienta. Sin este paso no hay análisis de grupo posible, porque los cerebros individuales difieren en tamaño y forma.

**Composición de transformaciones y remuestreo único.** Es un detalle de implementación con consecuencia directa sobre la calidad, y conviene entenderlo porque distingue a fMRIPrep de flujos escritos a mano. Llevar un vóxel funcional al espacio estándar exige encadenar varias transformaciones: corrección de movimiento, corrección de distorsión, corregistro al anatómico y normalización a la plantilla. Aplicarlas una tras otra implicaría interpolar la imagen tantas veces como transformaciones haya, y cada interpolación introduce un desenfoque que se acumula. fMRIPrep compone todas las transformaciones en una sola y realiza **una única interpolación** desde los datos originales al espacio final. El resultado conserva mucha más resolución efectiva.

**Cómo encaja cada etapa con el resto del notebook:**

| Etapa | Producto | Quién lo consume |
|---|---|---|
| N4 | T1w sin sesgo de intensidad | Todas las etapas posteriores |
| Extracción cerebral | Máscara cerebral | Segmentación, métricas de 5.4 |
| Segmentación | Mapas de probabilidad de GM, WM y CSF | aCompCor (Sección 9), métricas de solapamiento (Sección 8) |
| Normalización | Transformación a MNI y anatómico normalizado | Corregistro funcional (Sección 6), NORManat (Sección 8) |
| Composición de transformaciones | Transformaciones encadenadas | Remuestreo del funcional (Sección 6) |


### 5.3 Ejecución de fMRIPrep

**Explicación detallada.** Se reutiliza el patrón de lanzamiento en segundo plano introducido en 4.2, por la misma razón y con más motivo, fMRIPrep es sustancialmente más lento que MRIQC. Con dos núcleos y `--fs-no-reconall`, cada sujeto se mide en horas, no en minutos. La consecuencia práctica en Colab es que conviene lanzar el proceso y consultar su estado periódicamente, en lugar de mantener una celda bloqueada.


**Justificación de los argumentos:**

| Argumento | Por qué |
|---|---|
| `--fs-no-reconall` | Desactiva la reconstrucción de superficies. Evita la necesidad de licencia de FreeSurfer y ahorra varias horas por sujeto. Alcance exacto discutido en 2.4 |
| `--output-spaces MNI152NLin2009cAsym:res-2` | Fija de forma explícita el espacio de salida y su resolución, en lugar de aceptar el valor por defecto. Es la plantilla declarada en `CFG` y la que usan Provins et al. (2023) |
| `--ignore slicetiming` | Aplica la decisión tomada y justificada en 3.5, y no la repite: la celda lee la variable `APLICAR_STC` en lugar de volver a deducirla |
| `--nprocs`, `--omp-nthreads`, `--mem-mb` | Ajustados a los recursos detectados en 2.1 |
| `--notrack` | Desactiva el envío de telemetría de uso, por la misma razón que `--no-sub` en MRIQC |
| `--skip-bids-validation` | El dataset ya se validó en 3.2 con el validador oficial. Repetir la validación dentro del contenedor solo añade tiempo |

**Sobre los fieldmaps.** No hace falta ningún argumento para activar la corrección de distorsión por susceptibilidad: fMRIPrep la aplica automáticamente cuando encuentra fieldmaps correctamente asociados al funcional. La comprobación C8 de la Sección 3.3 confirmó que ese enlace existe por partida doble, mediante `IntendedFor` y mediante `B0FieldIdentifier`, de modo que la corrección debe aparecer en el informe. Si no apareciera, sería señal de que el enlace se rompió y no de que la corrección no era necesaria.


In [ ]:
# 5.3  Lanzamiento de fMRIPrep con la configuracion de CALIDAD COMPLETA
#
# ESTA CELDA NO LANZA NADA POR DEFECTO, y conviene entender por que.
#
# Si el notebook se ejecutase entero de arriba abajo con esta celda activa, quedaria
# bloqueado durante muchas horas antes de mostrar ningun resultado. Por eso la
# ejecucion por defecto es la rapida de la celda 5.3c, y esta queda disponible para
# quien necesite resultados de calidad publicable y disponga del tiempo o de una
# maquina con mas nucleos.
#
# Las definiciones de rutas y la funcion de lanzamiento SI se ejecutan siempre,
# porque la celda 5.3c y todas las secciones posteriores dependen de ellas.
import shlex
import subprocess
from pathlib import Path

# INTERRUPTOR. Pongalo en True para lanzar el preprocesamiento de calidad completa.
# Antes de hacerlo, lea la celda 5.3c: conviene ejecutar primero la version rapida,
# comprobar que todo el notebook funciona, y solo entonces invertir las horas.
EJECUTAR_PREPROCESAMIENTO_COMPLETO = False

DIR_FMRIPREP = PATHS["derivados"] / "fmriprep"
TRABAJO_FMRIPREP = PATHS["trabajo"] / "fmriprep"
LOG_FMRIPREP = PATHS["reportes"] / "fmriprep.log"
for d in (DIR_FMRIPREP, TRABAJO_FMRIPREP):
    d.mkdir(parents=True, exist_ok=True)


def lanzar_secuencia_en_segundo_plano(herramienta, lista_argumentos, archivo_log,
                                      version=None, entorno=None):
    """Ejecuta la misma herramienta varias veces, en serie, sin bloquear el notebook.

    Carga el modulo una sola vez y encadena las invocaciones separadas por `;`
    y no por `&&`, de modo que el fallo de un sujeto no impida procesar los
    siguientes. La deteccion de fallos se hace contando los productos en disco
    con la celda 5.3b, no por el codigo de salida del conjunto.
    """
    version = version or CONTENEDORES.get(herramienta)
    backend = detectar_backend()
    if backend != "neurodesk-colab":
        raise HerramientaNoDisponible(
            f"Implementado para el backend neurodesk-colab. Backend actual: {backend}.")

    exportaciones = "".join(f"export {clave}={shlex.quote(str(valor))} && "
                            for clave, valor in (entorno or {}).items())
    invocaciones = " ; ".join(
        f"echo '=== {herramienta}: invocacion {i + 1} de {len(lista_argumentos)} ===' ; "
        f"{herramienta} " + " ".join(shlex.quote(str(a)) for a in argumentos)
        for i, argumentos in enumerate(lista_argumentos))

    orden = (f"source {INIT_LMOD} && "
             f"export MODULEPATH={shlex.quote(os.environ['MODULEPATH'])} && "
             f"{exportaciones}module load {herramienta}/{version} && ( {invocaciones} )")
    completo = (f"nohup bash -c {shlex.quote(orden)} "
                f"> {shlex.quote(str(archivo_log))} 2>&1 & echo $!")
    resultado = subprocess.run(["bash", "-c", completo], capture_output=True, text=True)
    return int(resultado.stdout.strip())


def argumentos_fmriprep(directorio_salida, directorio_trabajo, sloppy=False):
    """Argumentos de fMRIPrep, una lista por sujeto.

    Se genera UNA INVOCACION POR SUJETO, en serie. No es una preferencia de estilo
    sino un requisito impuesto por estos datos: los tres sujetos comparten el mismo
    valor de `B0FieldIdentifier`, y sdcflows mantiene un registro global de
    estimadores de campo indexado por ese identificador. Al procesar el segundo
    sujeto en la misma invocacion, la clave ya esta ocupada y la ejecucion aborta con
      KeyError: 'pepolar_fmap0' is already in mapping
    Con una invocacion por sujeto, cada una tiene su propio registro.

    La decision sobre tiempo de corte NO se repite aqui: se lee de la variable que
    fijo la Seccion 3.5, de modo que cambiar la politica alli se propaga solo.
    """
    comunes = [
        DIR_BIDS, directorio_salida, "participant",
        "-w", directorio_trabajo,
        "--output-spaces", f"{CFG.plantilla}:res-2",
        "--nprocs", str(n_cpu), "--omp-nthreads", str(n_cpu),
        "--mem-mb", "10000",
        "--notrack",
        "--skip-bids-validation",
        "--fs-license-file", str(LICENCIA_FS),
    ]
    if not CFG.usar_freesurfer:
        comunes.append("--fs-no-reconall")
    if not APLICAR_STC:
        comunes += ["--ignore", "slicetiming"]
    if sloppy:
        comunes.append("--sloppy")
    return [comunes + ["--participant-label", sujeto] for sujeto in CFG.sujetos]


if not EJECUTAR_PREPROCESAMIENTO_COMPLETO:
    print("Preprocesamiento de CALIDAD COMPLETA: no lanzado (comportamiento por "
          "defecto).")
    print(f"   Salida que usaria:  {DIR_FMRIPREP}")
    print(f"   Trabajo que usaria: {TRABAJO_FMRIPREP}")
    print("\nLa ejecucion por defecto del notebook es la RAPIDA de la celda 5.3c, que")
    print("recorre el mismo flujo completo con registros de baja precision y termina")
    print("en una fraccion del tiempo. Los resultados que muestra este notebook")
    print("proceden de esa ejecucion rapida.")
    print("\nPara lanzar la de calidad completa, ponga EJECUTAR_PREPROCESAMIENTO_"
          "COMPLETO en True y vuelva a ejecutar esta celda. Tenga en cuenta:")
    print("   Tiempo: horas por sujeto con dos nucleos, dominado por el registro no")
    print("           lineal a la plantilla")
    print("   Requisito: no ejecutar ninguna celda que abra imagenes mientras corre,")
    print("              porque la competencia por memoria puede abortar el proceso")
    print("   Recomendacion: ejecutar antes la version rapida y comprobar que todas")
    print("                  las secciones funcionan, para no descubrir un fallo")
    print("                  despues de horas de computo")

elif not globals().get("LICENCIA_DISPONIBLE", False):
    print("NO SE PUEDE LANZAR fMRIPrep: falta la licencia de FreeSurfer.")
    print("Ejecute la celda 2.4b, que explica como obtenerla e instalarla.")
    print("\nRecordatorio: la licencia es obligatoria incluso con --fs-no-reconall.")

else:
    # No lanzar si hay otro proceso pesado en marcha. Con dos nucleos, ejecutar dos
    # herramientas a la vez no reparte el tiempo, lo empeora para ambas.
    ocupados = subprocess.run(
        ["bash", "-c",
         "ps -eo pcpu,comm --sort=-pcpu | "
         "grep -Ei 'mriqc|fmriprep|3dvolreg|3dTshift|antsRegist|N4Bias|synthstrip|melodic' | "
         "grep -v grep | head -5"],
        capture_output=True, text=True).stdout.strip()

    if ocupados:
        print("AVISO: hay procesos de computo todavia activos:")
        print(ocupados)
        print("\nLanzar fMRIPrep ahora hara que ambos vayan mas lentos que en serie.")
        print("Espere a que terminen (celdas 4.2b y 5.3b) y vuelva a ejecutar esta celda.")
    else:
        PID_FMRIPREP = lanzar_secuencia_en_segundo_plano(
            "fmriprep", argumentos_fmriprep(DIR_FMRIPREP, TRABAJO_FMRIPREP,
                                            sloppy=False),
            LOG_FMRIPREP,
            entorno={"TEMPLATEFLOW_HOME": TEMPLATEFLOW_HOME,
                     "FS_LICENSE": LICENCIA_FS})
        MODO_RAPIDO = False

        print(f"fMRIPrep {CONTENEDORES['fmriprep']} lanzado con CALIDAD COMPLETA.")
        print(f"  PID:          {PID_FMRIPREP}")
        print(f"  Invocaciones: {len(CFG.sujetos)}, una por sujeto, en serie")
        print(f"  Registro:     {LOG_FMRIPREP}")
        print(f"  Salida:       {DIR_FMRIPREP}")
        print(f"  Opciones:     {'--fs-no-reconall ' if not CFG.usar_freesurfer else ''}"
              f"{'--ignore slicetiming' if not APLICAR_STC else ''}")
        print("\nTiempo esperado: HORAS por sujeto con dos nucleos. Use la celda 5.3b")
        print("para consultar el estado sin bloquear el notebook, y no ejecute")
        print("mientras tanto ninguna celda que abra archivos de imagen.")


In [ ]:
# 5.3b  Estado de la ejecucion de fMRIPrep, con respaldo automatico
# Celda idempotente, se puede ejecutar cuantas veces se quiera mientras el proceso
# avanza. No bloquea ni interfiere con la ejecucion en curso.
#
# El estado se determina por los PRODUCTOS EN DISCO y no por la existencia de un
# identificador de proceso, porque ese identificador solo existe si el lanzamiento
# ocurrio en la sesion en curso. Al retomar el trabajo en otra sesion, al recuperar
# resultados del almacen persistente, o despues de que el entorno de ejecucion se
# reinicie, no hay proceso alguno y sin embargo los productos si estan.
import subprocess
from pathlib import Path

# Respaldo automatico al detectar sujetos recien completados. Existe por un incidente
# concreto: el entorno de ejecucion de Colab se reciclo mientras fMRIPrep procesaba
# el tercer sujeto, y ese sujeto se perdio porque su respaldo dependia de que alguien
# ejecutase la celda 2.9b manualmente al terminar. Los dos sujetos anteriores, ya
# respaldados, se recuperaron intactos. Vinculando el respaldo a la consulta de
# estado, la accion de comprobar el avance protege el trabajo sin necesidad de
# acordarse. El respaldo es incremental, de modo que repetirlo es barato.
RESPALDAR_AL_CONSULTAR = True

directorio = globals().get("DIR_FMRIPREP")
if directorio is None:
    # Se deduce del almacen local sin depender de ninguna variable de sesion.
    candidatos = [PATHS["derivados"] / "fmriprep-rapido",
                  PATHS["derivados"] / "fmriprep"]
    directorio = next((d for d in candidatos if d.exists()), candidatos[0])
    print(f"DIR_FMRIPREP no esta definida en esta sesion. Se consulta {directorio}")
    print("Ejecute las celdas 5.3 y 5.3c para restablecer el estado del notebook.\n")

pid = globals().get("PID_FMRIPREP")
if pid is None:
    print("No hay ninguna ejecucion lanzada en esta sesion.")
    print("Puede deberse a que los resultados ya existian, a que se recuperaron del")
    print("almacen persistente, o a que el entorno de ejecucion se reinicio. Lo que")
    print("cuenta son los productos en disco y los procesos vivos, listados abajo.")
elif "proceso_vivo" in globals():
    print(f"Proceso {pid}: "
          f"{'EN EJECUCION' if proceso_vivo(pid) else 'terminado'}")

print(f"\nDirectorio consultado: {directorio}")
if globals().get("MODO_RAPIDO"):
    print("Modo: RAPIDO. Los resultados no son de calidad publicable (ver 5.3c).")


# Progreso por productos en disco. Se cuenta por SUJETOS distintos y no por
# archivos, porque en cada espacio de salida se escribe una copia y contar archivos
# daria cifras mayores que el numero de sujetos.
def sujetos_con(patron):
    return {ruta.name.split("_")[0] for ruta in directorio.glob(patron)}


anat = sujetos_con("sub-*/**/anat/*_desc-preproc_T1w.nii.gz")
bold = sujetos_con("sub-*/**/func/*_desc-preproc_bold.nii.gz")
confounds = sujetos_con("sub-*/**/func/*_desc-confounds_timeseries.tsv")
informes = sorted(directorio.glob("sub-*.html"))

total = len(CFG.sujetos) if "CFG" in globals() else 3
print(f"\n  T1w preprocesados:        {len(anat)} de {total}")
print(f"  BOLD preprocesados:       {len(bold)} de {total}")
print(f"  Tablas de confounds:      {len(confounds)} de {total}")
print(f"  Informes HTML por sujeto: {len(informes)} de {total}")
for ruta in informes:
    print("     ", ruta.name)

if "CFG" in globals():
    pendientes = [s for s in CFG.id_sujetos if s not in confounds]
    if pendientes:
        print(f"  Pendientes: {', '.join(pendientes)}")

# La consulta de procesos NO depende de la capa de abstraccion de 2.7, para que la
# celda siga siendo util justo despues de un reinicio del entorno.
print("\nProcesos de computo activos")
activos = subprocess.run(
    ["bash", "-c",
     "ps -eo pid,pcpu,pmem,time,comm --sort=-pcpu | "
     "grep -Ei 'mriqc|fmriprep|antsRegist|antsApply|N4Bias|3dvolreg|flirt|fast|"
     "synthstrip|melodic' | grep -v grep | head -6"],
    capture_output=True, text=True).stdout.strip()
print(activos or "(ninguno)")

# Respaldo automatico si hay sujetos nuevos completados desde la ultima consulta.
if (RESPALDAR_AL_CONSULTAR and confounds
        and "respaldar_derivados_de" in globals()
        and globals().get("DIR_PERSISTENTE") is not None):
    ya_respaldados = globals().get("_SUJETOS_RESPALDADOS", set())
    nuevos = confounds - ya_respaldados
    if nuevos:
        print(f"\nSujetos completados desde la ultima consulta: "
              f"{', '.join(sorted(nuevos))}")
        print("Respaldando en el almacen persistente para no perderlos si el entorno")
        print("de ejecucion se recicla...")
        respaldar_derivados_de(directorio.name)
        _SUJETOS_RESPALDADOS = set(confounds)
    else:
        print(f"\nRespaldo al dia: {len(ya_respaldados)} sujetos protegidos en Drive")

registro = globals().get("LOG_FMRIPREP") or (PATHS["reportes"] / "fmriprep_rapido.log")
if Path(registro).exists():
    print(f"\nUltimas lineas del registro ({Path(registro).name})")
    print(subprocess.run(["tail", "-n", "12", str(registro)],
                         capture_output=True, text=True).stdout)
else:
    print(f"\nNo hay registro en {registro}")


### 5.3c Ejecución rápida, que es la configuración por defecto del notebook

**Por qué existen dos celdas de lanzamiento.** El preprocesamiento con la configuración de calidad completa emplea el registro no lineal preciso de fMRIPrep, y en un entorno de dos núcleos como el que ofrece Colab de forma gratuita ese único paso consume más de dos horas de procesador por sujeto. Medido en este proyecto, cuatro horas de reloj sin llegar a completar el anatómico del primer sujeto.

La solución adoptada es ofrecer las dos configuraciones de forma explícita, con la rápida activada por defecto:

| | Celda 5.3, calidad completa | Celda 5.3c, modo rápido |
|---|---|---|
| Se lanza al ejecutar el notebook entero | No | Sí |
| Opción distintiva | Ninguna | `--sloppy` |
| Etapas del flujo | Todas | Las mismas |
| Productos generados | Los mismos | Los mismos |
| Precisión de los registros | Completa | Reducida, no convergen |
| Tiempo con dos núcleos | Varias horas por sujeto | Entre una y dos horas por sujeto |
| Directorio de salida | `derivatives/fmriprep` | `derivatives/fmriprep-rapido` |
| Uso previsto | Resultados publicables | Aprender y comprobar el procedimiento |

**Qué hace exactamente `--sloppy`.** Es una opción que fMRIPrep incorpora para sus propias pruebas automáticas. Reduce drásticamente el número de iteraciones de los registros, tanto el de normalización a la plantilla como el de corregistro. Todo lo demás permanece idéntico: se generan los mismos archivos, con los mismos nombres, las mismas tablas de variables de confusión y los mismos informes. Esa equivalencia estructural es lo que permite recorrer el notebook completo y verificar cada etapa.

**Los resultados que muestra este notebook proceden de la ejecución rápida, y conviene ser explícito sobre qué significa eso.** Sirven para aprender el procedimiento, para entender qué produce cada etapa y para comprobar que el código funciona. No son de calidad publicable. La consecuencia más visible aparece en la Sección 8, como los registros no convergen, las métricas de normalización salen deprimidas por construcción, y leerlas como calidad de los datos sería un error. El manifiesto de la Sección 14 registra el modo empleado, de modo que la advertencia viaja con los resultados y no depende de que alguien recuerde haber leído esta subsección.

**La celda es idempotente.** Si encuentra que todos los sujetos ya tienen su tabla de variables de confusión, no relanza nada y las secciones posteriores usan los resultados existentes. Eso permite volver a ejecutar el notebook completo sin repetir horas de cómputo, que es el comportamiento razonable cuando se retoma el trabajo en otra sesión.

**Cómo pasar a calidad completa.** Poner `EJECUTAR_PREPROCESAMIENTO_COMPLETO` en `True` en la celda 5.3 y ejecutarla. El orden recomendado es siempre el mismo, primero la ejecución rápida, comprobar que todas las secciones del notebook funcionan sobre resultados reales, y solo entonces invertir las horas de la ejecución definitiva. De ese modo el tiempo de cómputo se gasta en calcular y no en descubrir errores.


In [ ]:
# 5.3c  Lanzamiento de fMRIPrep en modo RAPIDO. Es la ejecucion por defecto.
#
# Recorre exactamente el mismo flujo que la celda 5.3, con las mismas etapas y los
# mismos productos de salida, pero con `--sloppy`, la opcion que fMRIPrep incorpora
# para sus propias pruebas automaticas y que reduce drasticamente el numero de
# iteraciones de los registros.
#
# LOS RESULTADOS QUE MUESTRA ESTE NOTEBOOK PROCEDEN DE ESTA EJECUCION. Sirven para
# aprender el procedimiento y para comprobar que cada etapa funciona, pero NO son de
# calidad publicable: los registros no convergen, de modo que las metricas de
# normalizacion de la Seccion 8 estan deprimidas por construccion.
import subprocess
import time

EJECUTAR_MODO_RAPIDO = True

# Que hacer cuando ya hay resultados de ALGUNOS sujetos pero faltan otros.
#
# En False, que es el valor por defecto, el notebook continua con los sujetos
# disponibles y lo advierte. La razon es que recorrer el notebook de arriba abajo no
# deberia disparar horas de computo sin que nadie lo haya pedido: si una sesion
# anterior dejo dos sujetos procesados de tres, lo util es poder revisar el trabajo
# con esos dos de inmediato.
#
# En True, se lanza el preprocesamiento de los que falten. Es lo que hay que poner
# cuando se quiera completar la muestra y se disponga del tiempo.
#
# Cuando NO hay ningun sujeto procesado, se lanza siempre, con independencia de este
# interruptor: es la primera ejecucion y sin ella no habria nada que analizar.
COMPLETAR_SUJETOS_PENDIENTES = False

NOMBRE_SALIDA_RAPIDA = "fmriprep-rapido"
DIR_FMRIPREP_RAPIDO = PATHS["derivados"] / NOMBRE_SALIDA_RAPIDA
TRABAJO_FMRIPREP_RAPIDO = PATHS["trabajo"] / NOMBRE_SALIDA_RAPIDA
LOG_RAPIDO = PATHS["reportes"] / "fmriprep_rapido.log"

PROCESOS_DE_COMPUTO = ("fmriprep", "antsRegistration", "antsApplyTransforms",
                       "N4BiasFieldCorrection", "3dvolreg", "3dTshift", "flirt",
                       "fast", "bet", "synthstrip", "mri_synthstrip")


def detener_computo(espera_maxima=90):
    """Detiene fMRIPrep y sus procesos hijos, y espera a que desaparezcan.

    Matar el interprete que lanzo la secuencia NO basta: los binarios de ANTs y FSL
    son procesos independientes que seguirian consumiendo los dos nucleos.
    """
    for nombre in PROCESOS_DE_COMPUTO:
        subprocess.run(["pkill", "-TERM", "-f", nombre], capture_output=True)
    inicio = time.time()
    while time.time() - inicio < espera_maxima:
        vivos = subprocess.run(
            ["bash", "-c", "ps -eo comm | grep -Ei '" + "|".join(PROCESOS_DE_COMPUTO)
             + "' | grep -v grep | sort -u"], capture_output=True, text=True).stdout.strip()
        if not vivos:
            return True, ""
        time.sleep(3)
    for nombre in PROCESOS_DE_COMPUTO:
        subprocess.run(["pkill", "-KILL", "-f", nombre], capture_output=True)
    time.sleep(3)
    vivos = subprocess.run(
        ["bash", "-c", "ps -eo comm | grep -Ei '" + "|".join(PROCESOS_DE_COMPUTO)
         + "' | grep -v grep | sort -u"], capture_output=True, text=True).stdout.strip()
    return (not vivos), vivos


def sujetos_preprocesados(directorio):
    """Sujetos con tabla de variables de confusion, que es el producto final."""
    if not directorio.exists():
        return set()
    return {ruta.name.split("_")[0]
            for ruta in directorio.glob("sub-*/**/func/*_desc-confounds_timeseries.tsv")}


for d in (DIR_FMRIPREP_RAPIDO, TRABAJO_FMRIPREP_RAPIDO):
    d.mkdir(parents=True, exist_ok=True)

# Se redirige la variable que usan TODAS las celdas posteriores, aunque no se
# relance nada, de modo que reejecutar el notebook sobre resultados ya calculados
# apunte al sitio correcto.
if EJECUTAR_MODO_RAPIDO:
    DIR_FMRIPREP = DIR_FMRIPREP_RAPIDO
    LOG_FMRIPREP = LOG_RAPIDO
    MODO_RAPIDO = True

# PASO PREVIO: recuperar de Drive lo ya calculado en sesiones anteriores. Sin el, una
# sesion nueva encontraria el disco local vacio y repetiria horas de computo cuyos
# resultados ya estan guardados. El disco de Colab es efimero, Drive no.
if EJECUTAR_MODO_RAPIDO and "restaurar_derivados_de" in globals():
    restaurar_derivados_de(NOMBRE_SALIDA_RAPIDA)

hechos = sujetos_preprocesados(DIR_FMRIPREP_RAPIDO)
faltan = [s for s in CFG.id_sujetos if s not in hechos]
PID_FMRIPREP = None

if not EJECUTAR_MODO_RAPIDO:
    print("Modo rapido desactivado. Las celdas posteriores usaran el directorio que")
    print(f"fije la celda 5.3: {DIR_FMRIPREP}")

elif not faltan:
    print(f"Los {len(hechos)} sujetos ya estan preprocesados en {DIR_FMRIPREP_RAPIDO}")
    print("   " + ", ".join(sorted(hechos)))
    print("\nNo se relanza nada. Las celdas posteriores usaran estos resultados.")

elif hechos and not COMPLETAR_SUJETOS_PENDIENTES:
    # Hay resultados parciales y no se ha pedido completarlos: se continua con lo
    # disponible en lugar de disparar horas de computo sin que nadie lo haya pedido.
    print(f"Sujetos disponibles: {', '.join(sorted(hechos))}")
    print(f"Sujetos SIN preprocesar: {', '.join(faltan)}")
    print("\nEl notebook continua con los disponibles. Todas las secciones")
    print("posteriores funcionan con una muestra incompleta y lo advierten donde")
    print("corresponde, con una salvedad importante: QC-FC en la Seccion 11 es una")
    print("correlacion A TRAVES de sujetos y necesita al menos tres para estar")
    print("definida.")
    print("\nPara procesar los que faltan, ponga COMPLETAR_SUJETOS_PENDIENTES en True")
    print("y vuelva a ejecutar esta celda. Tiempo estimado: entre una y dos horas por")
    print("sujeto con dos nucleos.")

elif not globals().get("LICENCIA_DISPONIBLE", False):
    print("NO SE PUEDE LANZAR: falta la licencia de FreeSurfer. Ejecute la celda 2.4b.")

else:
    if hechos:
        print(f"Ya preprocesados: {', '.join(sorted(hechos))}")
    print(f"Se procesaran: {', '.join(faltan)}")

    nodos_en_cache = (len(list(TRABAJO_FMRIPREP.rglob("_0x*")))
                      if "TRABAJO_FMRIPREP" in globals() and TRABAJO_FMRIPREP.exists()
                      else 0)
    if nodos_en_cache:
        print(f"\nSe conservan {nodos_en_cache} nodos en cache de la ejecucion de")
        print("calidad completa, que permitiran reanudarla mas adelante.")

    print("\nDeteniendo cualquier proceso de computo en marcha...")
    limpio, restantes = detener_computo()
    print("   no queda ningun proceso activo" if limpio
          else f"   ATENCION: siguen vivos {restantes}")

    if limpio:
        argumentos = [a for a in argumentos_fmriprep(
            DIR_FMRIPREP_RAPIDO, TRABAJO_FMRIPREP_RAPIDO, sloppy=True)
            if a[-1] in {s.replace("sub-", "") for s in faltan}]

        PID_FMRIPREP = lanzar_secuencia_en_segundo_plano(
            "fmriprep", argumentos, LOG_RAPIDO,
            entorno={"TEMPLATEFLOW_HOME": TEMPLATEFLOW_HOME,
                     "FS_LICENSE": LICENCIA_FS})

        print(f"\nfMRIPrep lanzado en MODO RAPIDO. PID {PID_FMRIPREP}")
        print(f"   Sujetos:  {', '.join(faltan)}")
        print(f"   Salida:   {DIR_FMRIPREP_RAPIDO}")
        print(f"   Registro: {LOG_RAPIDO}")
        print("\nMIENTRAS CORRA, no ejecute ninguna celda que abra archivos de imagen.")
        print("Consultar el estado con 5.3b es seguro y ademas respalda en Drive los")
        print("sujetos que vaya completando, de modo que un reinicio del entorno no")
        print("obligue a repetirlos.")


### 5.4 Control de calidad anatómico

**Fundamento metodológico.** El control de calidad de esta etapa responde a tres preguntas, en este orden de prioridad, que es el que establece el protocolo de Provins et al. (2023) para los criterios R, S y U de su lista:

1. **¿Es correcta la máscara cerebral?** Una máscara que recorta corteza elimina datos del análisis; una que incluye cráneo o duramadre contamina la segmentación y, con ella, las máscaras de ruido de la Sección 9.
2. **¿Es correcta la segmentación tisular?** Se comprueba que el contorno del líquido cefalorraquídeo siga los ventrículos y que el de la sustancia blanca siga la unión entre sustancia gris y blanca. Vóxeles mal clasificados dispersos en sustancia blanca profunda o en núcleos subcorticales son efecto de volumen parcial y son aceptables; regiones extensas y contiguas mal clasificadas no lo son.
3. **¿Es correcta la normalización a espacio estándar?** Se verifica el alineamiento por orden de importancia: ventrículos, regiones subcorticales, cuerpo calloso, cerebelo y, por último, corteza. Un desalineamiento en las primeras invalida el análisis; en la última puede ser variabilidad anatómica normal entre el individuo y la plantilla, porque el registro volumétrico no puede resolver que un surco presente en la plantilla falte en un sujeto concreto.

**Qué aporta esta celda sobre el informe de fMRIPrep.** El informe HTML que genera la herramienta es la fuente principal para las tres comprobaciones anteriores y hay que mirarlo. Lo que la celda añade es la cuantificación, volúmenes tisulares y sus proporciones, que permiten detectar fallos que el ojo no distingue con facilidad y, sobre todo, comparar entre sujetos. Un volumen de sustancia gris muy distinto del de los demás sujetos del mismo protocolo apunta a un fallo de máscara o de segmentación antes que a una peculiaridad biológica.

**Rangos de referencia y su límite.** En adultos sanos el volumen cerebral total ronda entre 1100 y 1600 cm³ y la razón entre sustancia gris y blanca se sitúa aproximadamente entre 1.0 y 1.4. Son referencias orientativas y no umbrales, sirven para detectar un fallo, no para juzgar la calidad. Conviene además una precisión que evita un malentendido frecuente, estos volúmenes **no son medidas morfométricas**. Proceden de una segmentación pensada para generar máscaras de procesamiento, no para cuantificar anatomía, y usarlos como medida de volumen cerebral en un estudio requeriría FreeSurfer o una herramienta equivalente sobre el T1w nativo.

**Nota sobre las métricas de solapamiento.** Las medidas cuantitativas de calidad de la normalización y del corregistro, es decir NORManat, NORMfunc y el solapamiento anatómico funcional, junto con los coeficientes de Dice y Jaccard frente a las plantillas tisulares, se calculan de forma sistemática en la Sección 8, una vez disponibles también los productos funcionales. Aquí la evaluación de la normalización es visual y cualitativa.


In [ ]:
# 5.4  Control de calidad anatomico: volumenes tisulares y revision visual
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd

RANGO_VOLUMEN_TOTAL_CM3 = (1000.0, 1800.0)

# Rango orientativo de la razon entre sustancia gris y blanca. El limite inferior es
# 0.9 y no 1.0 por una razon metodologica concreta, no para que encajen los datos:
# contar voxeles con probabilidad superior al 50 por ciento penaliza mas a la
# sustancia gris que a la blanca, porque la gris forma una lamina delgada y una
# proporcion mayor de sus voxeles cae por debajo de ese umbral por efecto de volumen
# parcial. Con este procedimiento de recuento los valores esperables rondan 0.95 a
# 1.3, mientras que las cifras de 1.1 a 1.3 que suelen citarse proceden de metodos
# morfometricos que modelan el volumen parcial en lugar de umbralizarlo.
RANGO_RAZON_GM_WM = (0.9, 1.4)

# Los volumenes se cuentan sobre los mapas de PROBABILIDAD por tejido, no sobre la
# segmentacion discreta. Hay dos razones. La primera es de fidelidad: la definicion
# de referencia (Morfini et al., 2023, Tabla 1) es el recuento de voxeles con
# probabilidad tisular superior al 50 por ciento. La segunda es de robustez: el
# archivo `_dseg` codifica los tejidos con enteros cuya correspondencia hay que
# conocer de antemano, y equivocarla intercambia dos tejidos sin producir ningun
# error visible. Los mapas `label-GM_probseg` y equivalentes llevan el tejido en el
# nombre del archivo, de modo que la ambiguedad desaparece.
UMBRAL_PROBABILIDAD = 0.5
TEJIDOS = ("GM", "WM", "CSF")

# Correspondencia del `_dseg`, usada solo como respaldo si faltan los mapas de
# probabilidad. Es la convencion de BIDS-Derivatives, que fMRIPrep aplica mediante
# una tabla de consulta sobre la salida de FSL FAST: 1 es sustancia gris, 2 es
# sustancia blanca y 3 es liquido cefalorraquideo. El orden NO es el de FAST, que
# entrega 1 para liquido, 2 para gris y 3 para blanca.
ETIQUETAS_DSEG = {1: "GM", 2: "WM", 3: "CSF"}


def rutas_anatomicas(patron, descripcion, sujeto):
    """Coincidencias del patron en espacio del sujeto, excluyendo las normalizadas.

    fMRIPrep escribe cada producto anatomico dos veces, una en el espacio del propio
    T1w y otra en cada espacio de salida solicitado. Sin excluir las normalizadas un
    patron como `*_dseg.nii.gz` casa con dos archivos por sujeto y la eleccion
    quedaria a merced del orden alfabetico.
    """
    encontrados = [r for r in sorted(DIR_FMRIPREP.glob(f"{sujeto}/**/anat/{patron}"))
                   if "space-" not in r.name]
    if not encontrados:
        print(f"   no encontrado: {descripcion}")
        return None
    if len(encontrados) > 1:
        print(f"   aviso: {len(encontrados)} coincidencias para {descripcion}, "
              f"se usa {encontrados[0].name}")
    return encontrados[0]


filas = []
datos_visual = {}
for sujeto in CFG.id_sujetos:
    t1w = rutas_anatomicas("*_desc-preproc_T1w.nii.gz", f"T1w de {sujeto}", sujeto)
    mascara = rutas_anatomicas("*_desc-brain_mask.nii.gz", f"mascara de {sujeto}", sujeto)
    probseg = {t: rutas_anatomicas(f"*label-{t}_probseg.nii.gz",
                                   f"probabilidad de {t} de {sujeto}", sujeto)
               for t in TEJIDOS}
    seg = rutas_anatomicas("*_dseg.nii.gz", f"segmentacion de {sujeto}", sujeto)

    if t1w is None:
        continue

    hay_probseg = all(probseg[t] is not None for t in TEJIDOS)
    if not hay_probseg and seg is None:
        print(f"   {sujeto}: sin segmentacion utilizable, se omite")
        continue

    referencia = nib.load(probseg["GM"] if hay_probseg else seg)
    # Volumen del voxel en cm3, para expresar los resultados en unidades fisicas y
    # no en recuento de voxeles, que depende de la resolucion.
    vol_voxel_cm3 = float(np.prod(referencia.header.get_zooms()[:3])) / 1000.0

    if hay_probseg:
        origen = "mapas de probabilidad al 50 por ciento"
        volumenes = {
            t: float((np.asanyarray(nib.load(probseg[t]).dataobj)
                      > UMBRAL_PROBABILIDAD).sum()) * vol_voxel_cm3
            for t in TEJIDOS}
    else:
        origen = "segmentacion discreta (respaldo)"
        etiquetas = np.asanyarray(nib.load(seg).dataobj)
        volumenes = {nombre: float((etiquetas == codigo).sum()) * vol_voxel_cm3
                     for codigo, nombre in ETIQUETAS_DSEG.items()}

    total = sum(volumenes.values())
    razon = volumenes["GM"] / volumenes["WM"] if volumenes["WM"] else np.nan

    # Comprobacion de plausibilidad de la asignacion de tejidos. En un cerebro
    # adulto la sustancia gris ocupa mas volumen que el liquido cefalorraquideo
    # segmentado, y la razon entre gris y blanca ronda la unidad. Si la asignacion
    # estuviese intercambiada, esta comprobacion lo delataria en lugar de producir
    # una tabla de aspecto normal con dos columnas permutadas.
    if volumenes["CSF"] > volumenes["GM"]:
        print(f"   {sujeto}: AVISO, el volumen asignado a liquido cefalorraquideo "
              f"supera al de sustancia gris. Revise la correspondencia de tejidos.")

    filas.append({
        "Sujeto": sujeto.replace("sub-", ""),
        "Origen": origen,
        "GM (cm3)": round(volumenes["GM"], 0),
        "WM (cm3)": round(volumenes["WM"], 0),
        "CSF (cm3)": round(volumenes["CSF"], 0),
        "Total (cm3)": round(total, 0),
        "Razon GM/WM": round(razon, 2),
        "Total en rango": RANGO_VOLUMEN_TOTAL_CM3[0] <= total <= RANGO_VOLUMEN_TOTAL_CM3[1],
        "Razon en rango": RANGO_RAZON_GM_WM[0] <= razon <= RANGO_RAZON_GM_WM[1],
    })
    datos_visual[sujeto] = {"t1w": t1w, "probseg": probseg if hay_probseg else None,
                            "seg": seg, "mascara": mascara}

if not filas:
    raise RuntimeError(
        f"No hay productos anatomicos de fMRIPrep en {DIR_FMRIPREP}. "
        f"Compruebe el estado con la celda 5.3b antes de continuar.")

volumenes_tisulares = pd.DataFrame(filas)
print("Volumenes tisulares derivados de la segmentacion de fMRIPrep")
print(volumenes_tisulares.to_string(index=False))
print(f"\nRangos orientativos: volumen total {RANGO_VOLUMEN_TOTAL_CM3[0]} a "
      f"{RANGO_VOLUMEN_TOTAL_CM3[1]} cm3, razon GM/WM {RANGO_RAZON_GM_WM[0]} a "
      f"{RANGO_RAZON_GM_WM[1]}.")
print("El limite inferior de la razon es 0.9 y no 1.0 porque umbralizar los mapas de")
print("probabilidad al 50 por ciento penaliza mas a la sustancia gris, que forma una")
print("lamina delgada y pierde mas voxeles por efecto de volumen parcial.")
print("Recordatorio: no son medidas morfometricas. La segmentacion esta pensada")
print("para generar mascaras de procesamiento, no para cuantificar anatomia.")

# Visualizacion de la mascara cerebral y de la segmentacion sobre el T1w.
n_sub = len(datos_visual)
fig, axes = plt.subplots(n_sub, 3, figsize=(11, 3.6 * n_sub), squeeze=False)

for fila, (sujeto, rutas) in enumerate(datos_visual.items()):
    t1 = nib.load(rutas["t1w"])
    datos_t1 = np.asanyarray(t1.dataobj, dtype=np.float32)

    # Mapas binarios de sustancia blanca y liquido cefalorraquideo para los
    # contornos, obtenidos de la misma fuente que los volumenes.
    if rutas["probseg"] is not None:
        contornos = {t: np.asanyarray(nib.load(rutas["probseg"][t]).dataobj)
                     > UMBRAL_PROBABILIDAD for t in ("WM", "CSF")}
    else:
        etiquetas = np.asanyarray(nib.load(rutas["seg"]).dataobj)
        codigos = {nombre: codigo for codigo, nombre in ETIQUETAS_DSEG.items()}
        contornos = {t: etiquetas == codigos[t] for t in ("WM", "CSF")}

    # Tres cortes axiales repartidos por el volumen, para no juzgar la segmentacion
    # a partir de un unico plano.
    z_total = datos_t1.shape[2]
    for columna, fraccion in enumerate((0.35, 0.5, 0.65)):
        z = int(z_total * fraccion)
        ax = axes[fila][columna]
        ax.imshow(np.rot90(datos_t1[:, :, z]), cmap="gray",
                  vmax=np.percentile(datos_t1[datos_t1 > 0], 99))
        # Contornos de los tejidos en lugar de superposicion opaca: permiten ver
        # simultaneamente el limite estimado y la imagen que hay debajo, que es lo
        # que se necesita para juzgar si el limite es correcto.
        for tejido, color in (("WM", "tab:orange"), ("CSF", "tab:cyan")):
            ax.contour(np.rot90(contornos[tejido][:, :, z]), levels=[0.5],
                       colors=color, linewidths=0.6)
        if rutas["mascara"] is not None:
            mascara = np.asanyarray(nib.load(rutas["mascara"]).dataobj)
            ax.contour(np.rot90(mascara[:, :, z]), levels=[0.5],
                       colors="tab:red", linewidths=0.8)
        ax.axis("off")
        if columna == 0:
            ax.text(0.02, 0.97, sujeto.replace("sub-", ""), transform=ax.transAxes,
                    color="white", fontsize=9, va="top")

fig.suptitle("Control de calidad anatomico: mascara cerebral (rojo), "
             "sustancia blanca (naranja) y liquido cefalorraquideo (cian)",
             fontsize=11)
plt.tight_layout()
if "guardar_figura" in globals():
    guardar_figura(fig, "qc_anatomico")
plt.show()


**Errores frecuentes en este paso, con un caso real de este dataset.** El primer intento de ejecución de esta celda procesaba los tres sujetos en una única invocación de fMRIPrep, que es la forma habitual de usar una BIDS App. Falló a los pocos minutos con este error:

```
KeyError: "'pepolar_fmap0' is already in mapping"
```

La causa merece explicarse porque combina un detalle de los metadatos con un detalle de la implementación. Los tres sujetos de este dataset declaran el **mismo valor** de `B0FieldIdentifier`, la cadena `pepolar_fmap0`, en los sidecars de sus fieldmaps, y el mismo valor de `B0FieldSource` en el sidecar del funcional. Eso es correcto desde el punto de vista de la especificación, el identificador solo necesita ser único dentro de cada sujeto, y ahí lo es. Sin embargo, la biblioteca que fMRIPrep usa para gestionar la corrección de distorsión mantiene un registro de estimadores de campo indexado por ese identificador y compartido por toda la invocación. Al llegar al segundo sujeto, la clave ya está ocupada y la ejecución aborta.

La solución adoptada es ejecutar **una invocación por sujeto, en serie**, de modo que cada una disponga de su propio registro. Tiene además una ventaja práctica en este entorno: con dos núcleos los sujetos se procesarían en serie de todas formas, así que no se pierde nada.

Existen otras dos vías, ambas peores. La primera es editar los sidecars para que cada sujeto tenga un identificador distinto, lo que implica modificar los datos crudos y, con ello, dejar de trabajar sobre el dataset tal como se publicó. La segunda es prescindir de `B0FieldIdentifier` y dejar que fMRIPrep se apoye solo en `IntendedFor`, que también está presente según se verificó en 3.3, pero eso exigiría igualmente editar los archivos para eliminar el campo conflictivo.

La lección general es doble. Primera, un dataset puede ser plenamente válido según la especificación y aun así activar un fallo en una herramienta concreta, lo que refuerza que la validación de la Sección 3 no sustituye a la ejecución real. Segunda, cuando una herramienta falla con un error que menciona una clave duplicada, conviene sospechar de un identificador compartido entre sujetos antes que de un problema en los datos de imagen.


### 5.5 Visualización de los informes de fMRIPrep

**Introducción conceptual.** fMRIPrep genera un informe HTML por sujeto que documenta cada decisión que la herramienta tomó y muestra el resultado de cada etapa. Es el instrumento principal de control de calidad del preprocesamiento, y su revisión no es opcional: los desarrolladores la consideran parte del procedimiento, no un extra. Esta celda lo incorpora al notebook con el mismo visor definido en 4.7.

**Diferencia esencial respecto a los informes de MRIQC.** Los de MRIQC evalúan los datos tal como salieron del escáner y responden a la pregunta de si merece la pena procesarlos. Los de fMRIPrep evalúan lo que la herramienta hizo con ellos y responden a una pregunta distinta: si el procesamiento funcionó. Un sujeto puede tener métricas de calidad excelentes en MRIQC y un registro fallido en fMRIPrep, y también lo contrario, un sujeto mediocre que se procesa sin problemas. Son controles complementarios y ninguno sustituye al otro.

**Qué contiene el informe y en qué orden revisarlo.**

1. **Resumen de la adquisición.** Confirma qué archivos se encontraron y qué parámetros se leyeron de los sidecar. Es el primer sitio donde se detecta que un fieldmap no se asoció al funcional que debía, o que un parámetro no estaba donde se esperaba.
2. **Referencia anatómica y extracción cerebral.** El contorno de la máscara sobre el T1w corregido de campo. Se juzga igual que en MRIQC, pero aquí la imagen ya ha pasado por la corrección N4.
3. **Segmentación tisular.** Los tres tejidos sobre el anatómico. En este notebook tiene un peso añadido, porque con `--fs-no-reconall` esta segmentación es la que alimenta el coste de frontera del corregistro.
4. **Normalización espacial.** Animación que alterna el anatómico del sujeto y la plantilla. Es la vista donde se decide si NORManat, calculado en la Sección 8, tiene un valor bajo por un fallo real o por la cobertura del campo de visión.
5. **Corrección de distorsión por susceptibilidad.** Antes y después de aplicar el mapa de campo. Se busca que los lóbulos frontal y temporal recuperen su forma y que el funcional deje de estar comprimido o estirado en la dirección de codificación de fase.
6. **Corregistro funcional al anatómico.** Animación entre el funcional de referencia y el anatómico. Las referencias útiles son los ventrículos y el borde cortical.
7. **Confounds.** El carpet plot con las trazas de movimiento, DVARS y señal global. Se lee igual que el de MRIQC, pero sobre datos ya preprocesados, de modo que las bandas que sobrevivan aquí son las que el preprocesamiento no pudo corregir, y por tanto las que el denoising de la Sección 10 tendrá que tratar.
8. **Metodología generada automáticamente.** Un párrafo con las versiones y los parámetros exactos, redactado por la propia herramienta y citable. Conviene leerlo, porque a veces revela que una etapa no se ejecutó por falta de un insumo.

**Qué hacer según lo que se encuentre.** Los criterios de 4.7 se aplican igual, con estas adiciones propias del preprocesamiento:

| Hallazgo en el informe de fMRIPrep | Actuación recomendada |
|---|---|
| La corrección de distorsión no aparece en el informe | El fieldmap no se asoció. Revisar `IntendedFor` o `B0FieldIdentifier` en los sidecar y volver a ejecutar. No es un fallo del sujeto sino de los metadatos |
| La distorsión persiste tras la corrección | Comprobar que la dirección de codificación de fase declarada coincide con la real. Una dirección invertida empeora la distorsión en lugar de corregirla |
| Corregistro funcional al anatómico desplazado | Revisar primero la segmentación, porque con `--fs-no-reconall` el coste de frontera depende de ella. Si la segmentación es correcta y el corregistro no, considerar la exclusión |
| Normalización con deformación local | Excluir el sujeto de los análisis que dependan de esa región. Cuantificar con las métricas de la Sección 8 antes de decidir |
| El carpet plot conserva bandas verticales intensas | Es lo esperable: el preprocesamiento no elimina el movimiento, solo lo estima. Lo trata el denoising de la Sección 10 |
| Aparecen avisos sobre volúmenes fuera del estado estacionario | Normal al inicio de la serie. Comprobar que el número detectado es plausible, entre cero y unos pocos volúmenes |
| El párrafo de metodología omite una etapa esperada | Indica que faltó un insumo. Buscar la causa en el registro antes de interpretar cualquier resultado |


In [ ]:
# 5.5  Visor de los informes HTML de fMRIPrep
# Reutiliza `mostrar_informe` y `mostrar_figuras_sueltas`, definidas en la celda 4.7.
import subprocess
from pathlib import Path

from IPython.display import HTML, display

if "mostrar_informe" not in globals():
    raise RuntimeError("Ejecute la celda 4.7, que define el visor de informes, "
                       "antes de esta.")

# El directorio se deduce si la variable no esta definida, por el mismo motivo que en
# 5.3b: al retomar el trabajo en otra sesion o despues de que el entorno se reinicie,
# los productos estan en disco pero las variables de sesion no.
directorio = globals().get("DIR_FMRIPREP")
if directorio is None:
    candidatos = [PATHS["derivados"] / "fmriprep-rapido",
                  PATHS["derivados"] / "fmriprep"]
    directorio = next((d for d in candidatos
                       if d.exists() and any(d.glob("sub-*.html"))), candidatos[0])
    print(f"DIR_FMRIPREP no estaba definida. Se usa {directorio}")
    print("Ejecute las celdas 5.3 y 5.3c para restablecer el estado del notebook.\n")

informes_fmriprep = sorted(directorio.glob("sub-*.html"))
print(f"Informes de fMRIPrep encontrados: {len(informes_fmriprep)} de "
      f"{len(CFG.id_sujetos)} esperados")
print(f"Directorio: {directorio}")

if not informes_fmriprep:
    # No se lanza excepcion: es normal consultar esta celda mientras fMRIPrep sigue
    # trabajando, y un error interrumpiria la ejecucion secuencial del notebook sin
    # que haya nada que corregir.
    print("\nfMRIPrep escribe el informe de cada sujeto al TERMINAR de procesarlo, de")
    print("modo que su ausencia no indica un fallo si el proceso sigue vivo. Compruebe")
    print("el estado con la celda 5.3b y vuelva a ejecutar esta cuando haya informes.")
else:
    faltan = [s for s in CFG.id_sujetos
              if not any(p.name.startswith(s) for p in informes_fmriprep)]
    if faltan:
        print(f"Todavia sin informe: {', '.join(s.replace('sub-', '') for s in faltan)}")

    for ruta in informes_fmriprep:
        sujeto = ruta.stem.split("_")[0]
        display(HTML(f"<h3 style='margin:22px 0 2px 0'>Sujeto "
                     f"{sujeto.replace('sub-', '')}</h3>"))
        empotrado = mostrar_informe(ruta, titulo=f"Informe de fMRIPrep: {ruta.name}")
        if not empotrado:
            mostrar_figuras_sueltas(directorio / sujeto / "figures", maximo=12)

    # La descripcion metodologica que fMRIPrep redacta es citable y conviene tenerla
    # accesible, porque es el texto que debe acompanar a una publicacion. Se muestra
    # aqui ademas de estar dentro del informe, para que quede en la salida de la
    # celda y siga disponible sin abrir el marco.
    citas = sorted(directorio.glob("logs/CITATION.md"))
    if citas:
        texto = citas[0].read_text(errors="replace")
        display(HTML("<h4 style='margin:18px 0 4px 0'>Descripcion metodologica "
                     "generada por fMRIPrep, apta para citar</h4>"))
        display(HTML(f"<div style='font-size:0.85em; line-height:1.4; "
                     f"max-height:340px; overflow:auto; border:1px solid #ccc; "
                     f"padding:8px'>{texto}</div>"))
        print(f"\nTexto completo en {citas[0]}")
    else:
        print("\nLa descripcion metodologica esta dentro del propio informe HTML, en")
        print("su seccion final.")

print(f"\nRespaldo persistente de los informes:")
print(f"   {(DIR_PERSISTENTE / 'derivatives' / directorio.name) if DIR_PERSISTENTE else directorio}")
print("\nADVERTENCIA SOBRE LA PERSISTENCIA DE ESTA VISTA. Los marcos empotrados son")
print("contenido servido en vivo y NO se guardan en el archivo del notebook: quien lo")
print("abra sin ejecutarlo los vera vacios. Las figuras estaticas equivalentes las")
print("generan las celdas 4.7b, 5.4, 6.3 y 7.4, y esas si persisten.")


**Análisis de los informes de fMRIPrep obtenidos.** Igual que en 4.7, este apartado registra lo observado en los informes concretos, y se distingue de la guía de lectura anterior porque aquella enseña a mirar y esta documenta el resultado.

**Advertencia que condiciona todo lo que sigue.** Los informes proceden de la ejecución en modo rápido descrita en 5.3c, que emplea registros con un número muy reducido de iteraciones. Las animaciones de normalización y de corregistro deben leerse con ese dato presente: un desajuste visible en ellas es atribuible a la configuración y no a los datos. Lo que sí resulta plenamente informativo son las etapas que `--sloppy` no altera, es decir la corrección de campo, la extracción cerebral, la segmentación, la corrección de distorsión y las variables de confusión.

**Resumen de la adquisición.** El informe confirma para cada sujeto un T1w y una serie funcional, y declara haber encontrado **un mapa de campo** disponible, estimado por el método de codificación de fase opuesta con `topup`. Es la confirmación de que la asociación de fieldmaps comprobada en 3.3 funcionó: fMRIPrep no solo encontró los archivos, sino que los asoció al funcional correcto y los usó. Si esa asociación hubiera fallado, la corrección se habría omitido en silencio y este apartado del informe estaría vacío.

**Extracción cerebral y segmentación.** El contorno de la máscara sigue el borde del cerebro sin incluir cráneo. La segmentación en tres tejidos, realizada con `fast` sobre el T1w tras la corrección de no uniformidad, delimita correctamente la frontera entre sustancia gris y blanca. Tiene aquí un peso añadido: con `--fs-no-reconall` esa segmentación es la que alimenta el coste de frontera del corregistro, de modo que un fallo suyo se propagaría al alineamiento entre modalidades.

**Corrección de distorsión por susceptibilidad.** La comparación antes y después muestra la recuperación de forma en los lóbulos frontal y temporal, que es donde la susceptibilidad magnética actúa con más intensidad. La dirección de codificación de fase declarada, `j` para el funcional y `j` frente a `j-` para los dos mapas de campo, es coherente con el sentido de la deformación observada. Una dirección invertida habría empeorado la distorsión en lugar de corregirla, y se habría visto de inmediato en esta vista.

**Corregistro funcional al anatómico.** El informe declara haberlo hecho con `mri_coreg` seguido de `flirt` con función de coste de registro basado en fronteras. Ese dato confirma sobre la propia herramienta algo que este notebook afirma en la Sección 5: ejecutar con `--fs-no-reconall` desactiva la reconstrucción de superficies y `bbregister`, pero **no** elimina el coste de frontera, que se obtiene por la vía de FSL apoyada en la segmentación.

**Variables de confusión.** El carpet plot del informe, con las trazas de movimiento y DVARS debajo, muestra el mismo patrón que se observó en los datos crudos: bandas verticales aisladas coincidentes con picos de movimiento, en número reducido. Que persistan tras el preprocesamiento es lo esperable y no un fallo: el preprocesamiento estima el movimiento y corrige la posición, pero no elimina el efecto que el movimiento tuvo sobre la intensidad de la señal. Eso es competencia del denoising de la Sección 10, y esta vista es la evidencia de que sigue habiendo trabajo pendiente para él.

**La descripción metodológica generada automáticamente**, que la celda muestra íntegra, merece leerse por dos motivos. El primero es práctico: es el texto que debe acompañar a una publicación, redactado por la herramienta y liberado para copiarse sin cambios. El segundo es de verificación, y en este proyecto resultó especialmente útil. Ese texto declara que los componentes de CompCor **se calculan por separado en las máscaras de sustancia blanca y de líquido cefalorraquídeo**, además de en la combinada, y que se retienen tantos como hagan falta para explicar el cincuenta por ciento de la varianza. Ambas afirmaciones confirman de forma independiente lo que las Secciones 9 y 10 establecieron: que los tres conjuntos se distinguen por el prefijo del nombre de columna, y que preguntar cuántos componentes hacen falta para alcanzar el cincuenta por ciento devuelve el total porque la herramienta ya aplicó ese criterio.

**Conclusión de la revisión.** Los dos sujetos disponibles completaron el flujo sin incidencias en las etapas que la configuración rápida no degrada. La calidad de la normalización no puede juzgarse aquí y se cuantifica en la Sección 8, con la salvedad de que sus valores estarán deprimidos por construcción mientras se trabaje en modo rápido.


**Buenas prácticas.** Inspeccionar el informe HTML de fMRIPrep sujeto por sujeto y no confiar únicamente en las métricas cuantitativas, el informe muestra cosas que ningún número resume, y ese es precisamente el argumento de Provins et al. (2023) al defender los dos puntos de control. Fijar de forma explícita el espacio de salida y su resolución en lugar de aceptar el valor por defecto, porque ese valor cambia entre versiones de la herramienta. Verificar que la corrección de distorsión aparece efectivamente en el informe cuando hay fieldmaps disponibles, dado que su omisión no genera ningún error.

**Errores frecuentes.** Interpretar los volúmenes tisulares que produce esta sección como medidas morfométricas: proceden de una segmentación pensada para generar máscaras de procesamiento. Juzgar la normalización por el alineamiento de la corteza, que es la estructura donde más legítimamente puede diferir un individuo de la plantilla, en lugar de por ventrículos y estructuras subcorticales. Lanzar fMRIPrep en paralelo con otro proceso pesado en una máquina de dos núcleos, con lo que el tiempo total resulta mayor que ejecutándolos en serie. Dar por buena una máscara cerebral sin mirar la corteza orbitofrontal y los polos temporales, que es donde falla característicamente.

**Recomendaciones.** Conservar el directorio de derivados y descartar el directorio de trabajo una vez terminado: el primero es el producto y el segundo son temporales que ocupan varias veces más espacio. Si el objetivo posterior incluyera análisis basado en superficies o morfometría, activar FreeSurfer implica obtener la licencia y asumir varias horas adicionales por sujeto, y también endurece los criterios de control de calidad sobre el T1w, porque artefactos tolerables para un análisis volumétrico invalidan las superficies reconstruidas.


## Sección 6. Preprocesamiento funcional

**Nota sobre la ejecución.** fMRIPrep procesa lo anatómico y lo funcional en una sola invocación, de modo que la ejecución lanzada en 5.3 cubre también esta sección. No hay que volver a lanzar nada: los productos funcionales aparecen en el mismo directorio de derivados conforme la herramienta avanza, y la celda 5.3b los cuenta por separado.

**Introducción conceptual.** La serie funcional llega cruda con cuatro problemas simultáneos que hay que resolver antes de poder estimar conectividad. El sujeto se mueve durante los siete minutos y medio de adquisición, de modo que un mismo vóxel no corresponde al mismo tejido a lo largo del tiempo. La secuencia de imagen ecoplanar sufre distorsión geométrica allí donde el campo magnético se desvía de su valor nominal, cerca de las interfaces entre aire y tejido. La serie funcional está en un espacio distinto del anatómico, con otra resolución y otro contraste. Y todo ello ocurre en un espacio individual que hay que llevar a una referencia común para poder comparar entre sujetos.

**Las etapas y su orden.** El orden no es arbitrario, y cada decisión tiene consecuencias:

| Etapa | Qué resuelve | Nota |
|---|---|---|
| Descarte de volúmenes iniciales | Los primeros volúmenes se adquieren antes de que la magnetización alcance el estado estacionario y tienen intensidad anómala | fMRIPrep los detecta automáticamente y los marca en la tabla de confounds como `non_steady_state_outlier` |
| Estimación y corrección de movimiento | Alinea todos los volúmenes a una referencia común | Produce los seis parámetros de realineamiento que alimentan las métricas de la Sección 7 |
| Corrección de tiempo de corte | Compensa que los cortes no se adquieren simultáneamente | **Omitida aquí**, por la decisión justificada en 3.5 |
| Corrección de distorsión por susceptibilidad | Devuelve la geometría a su forma anatómica correcta | Posible gracias a los fieldmaps de codificación de fase opuesta, verificados en 3.3 |
| Corregistro funcional a anatómico | Alinea la serie con el T1w del propio sujeto | Con coste de frontera, calculado por FLIRT apoyado en la segmentación de FAST |
| Normalización espacial | Lleva la serie al espacio estándar | Reutiliza la transformación calculada sobre el anatómico en la Sección 5 |

**La interpolación única, y por qué importa.** Las cuatro transformaciones espaciales de la tabla anterior, movimiento, distorsión, corregistro y normalización, podrían aplicarse una tras otra, remuestreando la imagen en cada paso. Ese es el enfoque de los flujos escritos a mano y tiene un coste acumulativo: cada interpolación promedia vóxeles vecinos y por tanto desenfoca, de modo que cuatro interpolaciones sucesivas degradan la resolución efectiva mucho más que una sola.

fMRIPrep compone las transformaciones en un único campo de desplazamiento y aplica **una sola interpolación** desde los datos originales hasta el espacio final. Es una de las diferencias metodológicas relevantes frente a los flujos tradicionales, y también frente al flujo por defecto de CONN, que aplica el realineamiento y la normalización como pasos separados. La consecuencia práctica es que los datos preprocesados por fMRIPrep conservan más detalle espacial para la misma cadena nominal de correcciones.

**Qué produce y qué no.** fMRIPrep entrega la serie funcional preprocesada, normalizada y con sus máscaras, junto con una tabla de confounds por sujeto que contiene parámetros de movimiento, desplazamiento de encuadre, DVARS, señales tisulares medias, componentes de aCompCor y de tCompCor, y marcas de volúmenes atípicos. Lo que **no** hace es aplicar esos confounds: la regresión y el filtrado son decisiones analíticas que la herramienta deja deliberadamente al usuario, y son el contenido de las Secciones 9 y 10. Esta separación entre preprocesamiento y denoising es la diferencia estructural más importante frente a CONN, que integra ambos en un mismo flujo.


### 6.2 Comparación etapa por etapa con CONN Toolbox

**Fundamento metodológico.** Aqui se compara cada etapa con lo que hace CONN Toolbox, y la comparación es instructiva porque las dos herramientas resuelven los mismos problemas con decisiones distintas y ambas defendibles. La descripción del flujo de CONN que sigue procede de Morfini et al. (2023), que documentan su pipeline mínimo por defecto.

| Etapa | CONN (pipeline mínimo por defecto) | fMRIPrep (configuración de este notebook) | Comentario |
|---|---|---|---|
| Corrección de movimiento | Realineamiento y *unwarp* de SPM, alineando todos los volúmenes al primero | Estimación con MCFLIRT de FSL, alineando a una referencia interna calculada del propio run | Alinear al primer volumen lo hace dependiente de la calidad de ese volumen concreto; una referencia promediada es más robusta. MRIQC, en cambio, estima el movimiento con `3dvolreg` de AFNI, por lo que sus parámetros no son numéricamente idénticos a los de fMRIPrep |
| Corrección de distorsión | Integrada en el *unwarp*, que estima la interacción entre movimiento y campo sin fieldmaps | `topup` de FSL a partir de fieldmaps de codificación de fase opuesta | Con fieldmaps medidos, la corrección se basa en un dato y no en una estimación. Morfini et al. no disponían de ellos y por eso eligieron el enfoque directo |
| Tiempo de corte | Aplicada por defecto | Omitida aquí, decisión de 3.5 | Morfini et al. también la omitieron, aunque por un motivo distinto: solo tenían la información para una parte de los sujetos y prefirieron homogeneidad analítica |
| Normalización funcional | **Directa** al espacio MNI, sin pasar por el anatómico | **Indirecta**: funcional al anatómico y de este a MNI | La normalización directa evita propagar errores del corregistro entre modalidades, y Calhoun et al. (2017) la defienden cuando no hay corrección de distorsión. La indirecta aprovecha el mayor detalle anatómico del T1w, y con la distorsión ya corregida el corregistro deja de ser el eslabón débil |
| Plantilla | IXI-549 en espacio MNI, salida a 2 mm | `MNI152NLin2009cAsym`, salida a 2 mm | Plantillas distintas del mismo espacio nominal. Conviene declarar cuál se usó: no son intercambiables sin más |
| Segmentación | Segmentación unificada de SPM, seis clases tisulares | FAST de FSL sobre el T1w corregido, tres clases | La segmentación unificada estima segmentación y normalización de forma conjunta |
| Remuestreo | Realineamiento y normalización como pasos separados | Composición de transformaciones e **interpolación única** | Es la diferencia con efecto más directo sobre la resolución efectiva del resultado |
| Suavizado espacial | Incluido en el preprocesamiento, núcleo de 8 mm | **No se aplica** | fMRIPrep deja el suavizado al usuario. Kumar et al. (2024) recomiendan un núcleo del doble del tamaño de vóxel, que aquí serían unos 5 mm |
| Detección de volúmenes atípicos | Integrada, con umbrales de 0.5 mm de desplazamiento y 3 desviaciones de cambio de señal global | Calcula las métricas y marca candidatos, pero no decide | La decisión queda en la Sección 7, con los mismos umbrales de referencia |
| Denoising | Integrado en el mismo flujo | **Separado**: entrega los confounds sin aplicarlos | Es la diferencia estructural más importante. Obliga a decidir de forma explícita, que es una virtud metodológica y también una fuente de error si no se documenta |

**Qué implementa este notebook y por qué.** Se adopta el flujo de fMRIPrep para el preprocesamiento y se implementan por separado, sobre sus salidas, las métricas de control de calidad y las estrategias de denoising que la literatura de conectividad funcional documenta con más detalle en el contexto de CONN. La razón no es preferencia por una herramienta sobre otra, sino que esas métricas y estrategias no dependen de la herramienta que generó los datos, se aplican por igual a salidas de fMRIPrep, de AFNI o de cualquier otro flujo, tal como los propios Morfini et al. señalan al afirmar que sus recomendaciones son agnósticas respecto del software empleado.

**Dónde la comparación deja de ser posible.** Dos diferencias impiden una equivalencia exacta entre ambos flujos, y conviene reconocerlo en lugar de forzar la comparación. La primera es la normalización directa frente a la indirecta, que produce campos de deformación distintos y por tanto series funcionales que no son idénticas vóxel a vóxel aunque compartan espacio nominal. La segunda es el suavizado, comparar métricas de calidad entre datos suavizados y sin suavizar es engañoso, porque el suavizado mejora artificialmente cualquier medida basada en relación señal ruido al promediar vóxeles vecinos. Es el mismo tipo de sesgo que se documentó en 4.4 con el tSNR calculado tras filtrar.


### 6.3 Inspección de los productos funcionales

Antes de usar los datos preprocesados conviene comprobar tres cosas que no requieren todavía ninguna métrica elaborada, y que si fallan invalidan todo lo que venga después.

1. La primera es el **inventario**: que existan todos los productos esperados para todos los sujetos. fMRIPrep puede completar un sujeto y fallar en otro sin que el proceso global aborte, de modo que contar archivos es una comprobación barata que evita descubrir la ausencia mucho más tarde.

2. La segunda es el **corregistro**, es decir si la serie funcional quedó bien alineada con la anatomía del propio sujeto. Se evalúa superponiendo los contornos del T1w sobre la referencia funcional y comprobando que coinciden en ventrículos y borde cortical, que son las estructuras con más contraste. Es también la comprobación que revela si la corrección de distorsión funcionó: sin ella, el lóbulo frontal inferior aparece sistemáticamente desplazado respecto del anatómico.

3. La tercera es la **cobertura**, que en este dataset tiene una importancia particular. La Sección 3.4 estableció que el campo de visión abarca 115 mm en el eje de cortes y no cubre el cerebro completo. La consecuencia práctica es que la máscara funcional de cada sujeto cubre una porción del cerebro, y que esas porciones no tienen por qué coincidir entre sujetos porque dependen de cómo quedó colocada cada cabeza. Cualquier análisis entre sujetos debe restringirse a la **intersección** de las máscaras funcionales, no a la máscara de la plantilla, porque de lo contrario se compararían regiones presentes en unos sujetos y ausentes en otros. La celda calcula esa intersección y la guarda para que las Secciones 8, 10 y 11 la utilicen.

**Sobre la tabla de confounds.** fMRIPrep entrega por sujeto una tabla con del orden de un centenar de columnas: parámetros de movimiento y sus derivadas y cuadrados, desplazamiento de encuadre, variantes de DVARS, señales medias de sustancia blanca, líquido cefalorraquídeo y global, componentes de aCompCor y tCompCor con su varianza explicada, regresores de coseno para la deriva, y marcas de volúmenes atípicos. La celda no las usa todavía, solo comprueba que están y muestra su estructura. Su uso es el contenido de las Secciones 7, 9 y 10.


In [ ]:
# 6.3  Inventario de productos funcionales, corregistro, cobertura y tSNR
import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import detrend

ESPACIO = CFG.plantilla
BLOQUE_VOLUMENES = 50     # volumenes por lectura, ver la funcion siguiente
VOXELES_TSNR = 20000      # submuestra para estimar el tSNR


def leer_voxeles_por_bloques(img, indices_planos, bloque=BLOQUE_VOLUMENES):
    """Serie temporal de unos voxeles concretos, leyendo por bloques de volumenes.

    Esta funcion la reutilizan las Secciones 7, 10 y 11, y su forma responde a un
    problema medido, no a una preferencia. Una version anterior leia un volumen por
    iteracion, con `img.dataobj[..., t]`, y resultaba inaceptablemente lenta. La
    causa es que los archivos NIfTI vienen comprimidos, y en un flujo comprimido no
    hay acceso aleatorio: para llegar al volumen t hay que descomprimir todo lo
    anterior, de modo que cuatrocientas lecturas sueltas descomprimen el archivo
    muchas veces.

    Leyendo bloques CONTIGUOS de volumenes se descomprime una sola vez de principio
    a fin. Con bloques de cincuenta volumenes en la rejilla de 2 mm el pico de
    memoria son unos doscientos megabytes, asumible, y se pasa de cuatrocientas
    lecturas a ocho.
    """
    n_vol = img.shape[3]
    salida = np.empty((len(indices_planos), n_vol), dtype=np.float32)
    for inicio in range(0, n_vol, bloque):
        fin = min(inicio + bloque, n_vol)
        trozo = np.asanyarray(img.dataobj[..., inicio:fin], dtype=np.float32)
        salida[:, inicio:fin] = trozo.reshape(-1, fin - inicio)[indices_planos]
        del trozo
    return salida


# Inventario de productos
filas = []
rutas_por_sujeto = {}
for sujeto in CFG.id_sujetos:
    productos = {
        "BOLD preprocesado": f"{sujeto}/**/func/*space-{ESPACIO}*_desc-preproc_bold.nii.gz",
        "Mascara funcional": f"{sujeto}/**/func/*space-{ESPACIO}*_desc-brain_mask.nii.gz",
        "Referencia funcional": f"{sujeto}/**/func/*space-{ESPACIO}*_boldref.nii.gz",
        "Confounds": f"{sujeto}/**/func/*_desc-confounds_timeseries.tsv",
        "T1w normalizado": f"{sujeto}/**/anat/*space-{ESPACIO}*_desc-preproc_T1w.nii.gz",
    }
    encontrados = {nombre: sorted(DIR_FMRIPREP.glob(patron))
                   for nombre, patron in productos.items()}
    filas.append({"Sujeto": sujeto.replace("sub-", ""),
                  **{nombre: len(v) for nombre, v in encontrados.items()}})
    rutas_por_sujeto[sujeto] = {nombre: (v[0] if v else None)
                                for nombre, v in encontrados.items()}

inventario = pd.DataFrame(filas)
inventario_funcional = inventario      # nombre que consume el informe de la Seccion 12
print("Inventario de productos de fMRIPrep (se espera 1 de cada por sujeto)")
print(inventario.to_string(index=False))

completos = [s for s in CFG.id_sujetos
             if all(v is not None for v in rutas_por_sujeto[s].values())]
print(f"\nSujetos con todos los productos: {len(completos)} de {len(CFG.id_sujetos)}")
if len(completos) < len(CFG.id_sujetos):
    faltan = set(CFG.id_sujetos) - set(completos)
    print(f"Incompletos: {', '.join(sorted(faltan))}. Revise el estado con 5.3b.")

if not completos:
    raise RuntimeError("Ningun sujeto tiene productos funcionales completos todavia.")

# Estructura de la tabla de confounds
confounds_ejemplo = pd.read_csv(rutas_por_sujeto[completos[0]]["Confounds"], sep="\t")

# Los tres conjuntos de aCompCor se distinguen por el PREFIJO y no por un metadato:
# `a_comp_cor_` es la mascara combinada de sustancia blanca y liquido, `w_comp_cor_`
# la de sustancia blanca sola y `c_comp_cor_` la de liquido sola. Agruparlos todos
# bajo `a_comp_cor` contaria solo una tercera parte de los componentes disponibles.
grupos = {
    "Movimiento (6 parametros)": [c for c in confounds_ejemplo.columns
                                  if c in ("trans_x", "trans_y", "trans_z",
                                           "rot_x", "rot_y", "rot_z")],
    "Movimiento (derivadas y cuadrados)": [c for c in confounds_ejemplo.columns
                                           if c.startswith(("trans_", "rot_"))
                                           and ("derivative" in c or "power2" in c)],
    "Desplazamiento y DVARS": [c for c in confounds_ejemplo.columns
                               if "framewise" in c or "dvars" in c.lower()
                               or c == "rmsd"],
    "Senales tisulares": [c for c in confounds_ejemplo.columns
                          if c in ("csf", "white_matter", "global_signal")],
    "Senales tisulares (derivadas)": [c for c in confounds_ejemplo.columns
                                      if c.startswith(("csf_", "white_matter_",
                                                       "global_signal_"))],
    "aCompCor (mascara combinada)": [c for c in confounds_ejemplo.columns
                                     if c.startswith("a_comp_cor")],
    "aCompCor (sustancia blanca)": [c for c in confounds_ejemplo.columns
                                    if c.startswith("w_comp_cor")],
    "aCompCor (liquido cefalorraquideo)": [c for c in confounds_ejemplo.columns
                                           if c.startswith("c_comp_cor")],
    "tCompCor": [c for c in confounds_ejemplo.columns
                 if c.startswith(("t_comp_cor", "tcompcor"))],
    "Corona del cerebro (edge)": [c for c in confounds_ejemplo.columns
                                  if c.startswith("edge_comp")],
    "Deriva (cosenos)": [c for c in confounds_ejemplo.columns if c.startswith("cosine")],
    "Volumenes atipicos": [c for c in confounds_ejemplo.columns
                           if c.startswith(("motion_outlier", "non_steady_state"))],
}
print(f"\nEstructura de la tabla de confounds ({confounds_ejemplo.shape[0]} filas, "
      f"{confounds_ejemplo.shape[1]} columnas)")
for grupo, columnas in grupos.items():
    print(f"  {grupo:38s} {len(columnas):3d}")

# Comprobacion de cobertura: toda columna debe caer en algun grupo. Sin ella, un
# cambio de nomenclatura entre versiones dejaria columnas sin contabilizar y la
# tabla daria una impresion incompleta sin que nada avisara.
cubiertas = {c for columnas in grupos.values() for c in columnas}
sin_grupo = [c for c in confounds_ejemplo.columns if c not in cubiertas]
print(f"  {'TOTAL agrupadas':38s} {len(cubiertas):3d}")
if sin_grupo:
    print(f"  Columnas sin grupo asignado ({len(sin_grupo)}): "
          f"{', '.join(sin_grupo[:12])}{' ...' if len(sin_grupo) > 12 else ''}")
else:
    print("  Todas las columnas estan asignadas a un grupo")

# Cobertura e interseccion de mascaras
mascaras = {}
for sujeto in completos:
    img = nib.load(rutas_por_sujeto[sujeto]["Mascara funcional"])
    mascaras[sujeto] = np.asanyarray(img.dataobj).astype(bool)
    referencia_afin, referencia_cabecera = img.affine, img.header

interseccion = np.logical_and.reduce(list(mascaras.values()))
union = np.logical_or.reduce(list(mascaras.values()))
vol_voxel_ml = float(np.prod(referencia_cabecera.get_zooms()[:3])) / 1000.0

cobertura = pd.DataFrame([
    {"Sujeto": s.replace("sub-", ""),
     "Voxeles": int(m.sum()),
     "Volumen (mL)": round(int(m.sum()) * vol_voxel_ml, 0),
     "% de la union": round(m.sum() / union.sum() * 100, 1)}
    for s, m in mascaras.items()])
print("\nCobertura de la mascara funcional por sujeto")
print(cobertura.to_string(index=False))
print(f"\nInterseccion de las {len(mascaras)} mascaras: {int(interseccion.sum())} voxeles "
      f"({round(int(interseccion.sum()) * vol_voxel_ml, 0)} mL), "
      f"{round(interseccion.sum() / union.sum() * 100, 1)} % de la union.")

# Se guarda para que las Secciones 8, 10 y 11 la reutilicen en lugar de recalcularla.
RUTA_INTERSECCION = PATHS["derivados"] / "mascara_interseccion.nii.gz"
nib.save(nib.Nifti1Image(interseccion.astype(np.uint8), referencia_afin),
         RUTA_INTERSECCION)
print(f"Mascara de interseccion guardada en {RUTA_INTERSECCION}")

# Senal a ruido temporal DESPUES del preprocesamiento, para contrastarla con la que
# 4.4 calculo sobre los datos crudos.
#
# Se calcula igual que alli: media dividida por desviacion tipica de la serie con la
# deriva lineal eliminada y SIN filtrar en banda. Ese detalle es esencial y fue un
# error corregido en una version anterior de este trabajo: el paso banda elimina la
# mayor parte de la varianza, de modo que calcular el tSNR sobre una serie filtrada
# lo infla en un orden de magnitud y produce cifras que no significan nada.
filas_tsnr = []
indices_interseccion = np.flatnonzero(interseccion.ravel())
rng = np.random.default_rng(CFG.semilla)
seleccion_tsnr = np.sort(rng.choice(indices_interseccion,
                                    size=min(VOXELES_TSNR, indices_interseccion.size),
                                    replace=False))

for sujeto in completos:
    img = nib.load(rutas_por_sujeto[sujeto]["BOLD preprocesado"])
    serie = leer_voxeles_por_bloques(img, seleccion_tsnr)
    medias = serie.mean(axis=1)
    residuos = detrend(serie, axis=1, type="linear")
    desviaciones = residuos.std(axis=1)
    validos = desviaciones > 0
    tsnr = medias[validos] / desviaciones[validos]
    filas_tsnr.append({
        "Sujeto": sujeto.replace("sub-", ""),
        "tSNR mediano (preprocesado)": round(float(np.median(tsnr)), 1),
        "Voxeles evaluados": int(validos.sum()),
    })
    del serie, residuos

tsnr_preprocesado = pd.DataFrame(filas_tsnr)

# Comparacion con el tSNR de los datos crudos, si la Seccion 4 se ejecuto.
if "tabla_tsnr" in globals() and not tabla_tsnr.empty:
    columna_crudo = next((c for c in tabla_tsnr.columns if "deriva" in c.lower()), None)
    if columna_crudo:
        comparacion_tsnr = tsnr_preprocesado.merge(
            tabla_tsnr[["Sujeto", columna_crudo]].rename(
                columns={columna_crudo: "tSNR mediano (crudo)"}),
            on="Sujeto", how="left")
        comparacion_tsnr["Cambio (%)"] = (
            100.0 * (comparacion_tsnr["tSNR mediano (preprocesado)"]
                     - comparacion_tsnr["tSNR mediano (crudo)"])
            / comparacion_tsnr["tSNR mediano (crudo)"]).round(1)
        print("\nSenal a ruido temporal antes y despues del preprocesamiento")
        print(comparacion_tsnr.to_string(index=False))
        comparacion_tsnr.to_csv(PATHS["reportes"] / "tsnr_antes_despues.tsv",
                                sep="\t", index=False)
    else:
        comparacion_tsnr = tsnr_preprocesado
        print("\nSenal a ruido temporal tras el preprocesamiento")
        print(tsnr_preprocesado.to_string(index=False))
else:
    comparacion_tsnr = tsnr_preprocesado
    print("\nSenal a ruido temporal tras el preprocesamiento")
    print(tsnr_preprocesado.to_string(index=False))
    print("Ejecute la celda 4.4 para poder compararla con la de los datos crudos.")

# Visualizacion del corregistro y de la cobertura
fig, axes = plt.subplots(len(completos), 3, figsize=(11, 3.4 * len(completos)),
                         squeeze=False)
for fila, sujeto in enumerate(completos):
    t1 = np.asanyarray(nib.load(rutas_por_sujeto[sujeto]["T1w normalizado"]).dataobj,
                       dtype=np.float32)
    ref = np.asanyarray(nib.load(rutas_por_sujeto[sujeto]["Referencia funcional"]).dataobj,
                        dtype=np.float32)

    z_total = ref.shape[2]
    for columna, fraccion in enumerate((0.35, 0.5, 0.65)):
        z = int(z_total * fraccion)
        ax = axes[fila][columna]
        ax.imshow(np.rot90(ref[:, :, z]), cmap="gray",
                  vmax=np.percentile(ref[ref > 0], 99))
        # Contorno del anatomico normalizado sobre la referencia funcional: si el
        # corregistro y la normalizacion son correctos, ambos coinciden.
        ax.contour(np.rot90(t1[:, :, z] > np.percentile(t1[t1 > 0], 40)),
                   levels=[0.5], colors="tab:orange", linewidths=0.6)
        # Contorno de la interseccion de mascaras, que delimita la region comun.
        ax.contour(np.rot90(interseccion[:, :, z]), levels=[0.5],
                   colors="tab:red", linewidths=0.8)
        ax.axis("off")
        if columna == 0:
            ax.text(0.02, 0.97, sujeto.replace("sub-", ""), transform=ax.transAxes,
                    color="white", fontsize=9, va="top")

fig.suptitle("Referencia funcional normalizada, con el contorno del anatomico "
             "(naranja) y la interseccion de mascaras (rojo)", fontsize=11)
plt.tight_layout()
if "guardar_figura" in globals():
    guardar_figura(fig, "corregistro_y_cobertura")
plt.show()


**Buenas prácticas.** Comprobar el inventario de productos antes de usarlos, fMRIPrep puede completar un sujeto y fallar en otro sin abortar el proceso global. Verificar que la corrección de distorsión aparece efectivamente aplicada cuando hay fieldmaps, porque su omisión es silenciosa. Calcular y conservar la intersección de las máscaras funcionales cuando la cobertura es parcial, en lugar de asumir que todos los sujetos cubren las mismas regiones.

**Errores frecuentes.** Usar la máscara de la plantilla en lugar de la intersección de las máscaras funcionales, con lo que se comparan regiones presentes en unos sujetos y ausentes en otros. Aplicar suavizado antes de calcular métricas de calidad y comparar esos valores con los de datos sin suavizar, error del mismo tipo que calcular el tSNR tras filtrar. Suponer que los confounds que entrega fMRIPrep ya están aplicados a la serie preprocesada: no lo están, y esa es una diferencia estructural con CONN que hay que tener presente. Repetir en la invocación de fMRIPrep una decisión ya tomada en otra sección, en lugar de leerla de la configuración, con el riesgo de que ambas diverjan.

**Recomendaciones.** Leer el informe HTML de cada sujeto completo antes de continuar, siguiendo el orden de prioridad de estructuras establecido en 1.9. Documentar la plantilla concreta empleada y no solo el espacio nominal: `MNI152NLin2009cAsym` e IXI-549 son ambos espacios MNI y no son intercambiables. Si se pretende comparar resultados con los de un estudio que usó CONN, tener presentes las dos diferencias que impiden la equivalencia exacta, la normalización directa frente a la indirecta y el suavizado.


## Sección 7. Evaluación de movimiento y de señal durante el preprocesamiento


### 7.2 Las métricas, sus definiciones y qué detecta cada una

**Introducción conceptual.** La Sección 4 evaluó la calidad de los datos crudos y la Sección 6 comprobó que el preprocesamiento produjo lo esperado. Esta sección responde a una pregunta distinta, cuánto movimiento y cuánta inestabilidad de señal quedan en los datos ya preprocesados, y qué volúmenes concretos están comprometidos. Es la información sobre la que se decide el censurado y, en última instancia, la inclusión o exclusión de cada sujeto.

**Fundamento metodológico.** Power et al. (2012) establecieron el resultado que motiva toda esta sección: el movimiento de cabeza introduce correlaciones espurias y sistemáticas en las estimaciones de conectividad, con un sesgo que depende de la distancia entre regiones, de modo que refuerza artificialmente las conexiones cortas y debilita las largas. El efecto no se elimina con la corrección de movimiento, porque esta reposiciona los volúmenes pero no restaura la señal que el movimiento alteró. Hace falta por tanto cuantificar el movimiento residual y actuar sobre él.

Las definiciones que siguen provienen de Morfini et al. (2023), que las documentan de forma precisa. Conviene subrayar algo que los propios autores señalan: estas métricas no son propias de ninguna herramienta concreta, se calculan igual sobre las salidas de cualquier flujo de preprocesamiento, y las recomendaciones asociadas se generalizan más allá del software empleado.

**Métricas por volumen (una serie temporal por sujeto):**

| Métrica | Definición | Qué detecta |
|---|---|---|
| Parámetros de realineamiento | Seis series: tres traslaciones en milímetros y tres rotaciones en radianes, respecto de la posición de referencia | Movimiento en sus seis grados de libertad. Las derivas lentas son benignas, los saltos abruptos no |
| Desplazamiento de encuadre (FD) | Cambio máximo de posición de seis puntos de control situados en el centro de cada cara de una caja envolvente del cerebro, sometidos a las mismas rotaciones y traslaciones que la cabeza | Movimiento entre volúmenes consecutivos, resumido en un solo número por volumen |
| DVARS | Raíz de la media del cuadrado del cambio de señal BOLD entre volúmenes consecutivos, sobre todo el cerebro | Cuánto cambió realmente la señal, con independencia de la causa |
| Cambio de señal global (GSchange) | Valor absoluto del cambio de la señal media de todo el cerebro entre volúmenes consecutivos, escalado a unidades estándar restando la mediana y dividiendo por 0.74 veces el rango intercuartílico | Eventos globales: movimiento, respiración profunda, cambios de vigilancia |
| Volúmenes atípicos (scrubbing) | Un regresor binario por cada volumen que supera el umbral de FD o de GSchange | Marca qué volúmenes concretos están comprometidos |

**Métricas resumen (un número por sujeto):**

| Métrica | Definición | Interpretación |
|---|---|---|
| MaxMotion | Máximo de la serie de FD sobre todos los volúmenes originales | Describe el peor instante, no el estado general |
| MeanMotion | Media de la serie de FD, calculada **solo sobre volúmenes no atípicos** | Movimiento residual tras el censurado |
| MeanGSchange | Media de GSchange sobre volúmenes no atípicos | Variabilidad global residual |
| InvalidScans | Número de volúmenes marcados como atípicos | Cuántos datos hay que descartar |
| ValidScans | Número de volúmenes que sobreviven | Cuántos datos quedan |
| PVS | Proporción de volúmenes válidos sobre el total | Normaliza lo anterior frente a duraciones distintas. Por debajo de 0.75 se considera extremo |

**Una distinción que importa: FD antes o después del censurado.** MaxMotion se calcula sobre todos los volúmenes y MeanMotion solo sobre los válidos. No es un descuido sino el diseño, el primero informa del estado de los datos **antes** del preprocesamiento, es decir de lo que ocurrió en el escáner, mientras que el segundo informa del estado **después** de descartar lo peor, es decir de con qué se va a trabajar realmente. Confundirlos lleva a conclusiones opuestas sobre el mismo sujeto.

**Dos definiciones distintas de FD.** fMRIPrep calcula el desplazamiento de encuadre siguiendo la definición de Power et al. (2012), que suma los valores absolutos de las derivadas de los seis parámetros, convirtiendo las rotaciones a milímetros mediante un radio de 50 mm. Morfini et al. usan la definición de caja envolvente descrita arriba. Las dos son razonables, están correlacionadas y **no dan el mismo número**, de modo que un umbral de 0.5 mm no significa exactamente lo mismo en una y en otra. La celda siguiente calcula ambas y las compara, en lugar de adoptar una y suponer que el umbral de la literatura se traslada sin más.


In [ ]:
# 7.1  Calculo de las metricas de movimiento y de senal
import numpy as np
import pandas as pd

# Caja envolvente del cerebro usada por la definicion de desplazamiento de
# encuadre de Morfini et al. (2023): 140 x 180 x 115 mm. Los seis puntos de
# control se situan en el centro de cada cara.
CAJA_MM = np.array([140.0, 180.0, 115.0])
PUNTOS_CONTROL = np.vstack([np.diag(CAJA_MM / 2.0), -np.diag(CAJA_MM / 2.0)])  # (6, 3)

# Umbrales de la definicion de Power, solo para poder comparar cuantos volumenes
# marcaria cada convencion. No gobiernan ninguna decision de este notebook.
UMBRAL_FD_POWER_MM = 0.5
UMBRAL_DVARS_ESTANDAR = 1.5


def matriz_rotacion(rx, ry, rz):
    """Rotacion rigida a partir de los tres angulos, en radianes."""
    cx, sx = np.cos(rx), np.sin(rx)
    cy, sy = np.cos(ry), np.sin(ry)
    cz, sz = np.cos(rz), np.sin(rz)
    return (np.array([[1, 0, 0], [0, cx, -sx], [0, sx, cx]])
            @ np.array([[cy, 0, sy], [0, 1, 0], [-sy, 0, cy]])
            @ np.array([[cz, -sz, 0], [sz, cz, 0], [0, 0, 1]]))


def fd_caja_envolvente(parametros):
    """Desplazamiento de encuadre segun la definicion de caja envolvente.

    Para cada par de volumenes consecutivos se aplica a los seis puntos de
    control la transformacion rigida de cada volumen y se toma el maximo de las
    distancias recorridas. Es la definicion de Morfini et al. (2023), distinta de
    la de Power et al. (2012) que usa fMRIPrep.
    """
    posiciones = np.stack([
        PUNTOS_CONTROL @ matriz_rotacion(*fila[3:]).T + fila[:3]
        for fila in parametros
    ])                                                  # (T, 6, 3)
    desplazamientos = np.linalg.norm(np.diff(posiciones, axis=0), axis=2)  # (T-1, 6)
    # El primer volumen no tiene anterior: se fija a cero por convencion, igual
    # que hacen fMRIPrep y CONN.
    return np.concatenate([[0.0], desplazamientos.max(axis=1)])


def escalar_a_unidades_estandar(serie):
    """Escalado robusto: resta la mediana y divide por 0.74 veces el rango
    intercuartilico. El factor 0.74 hace que, para una distribucion normal, el
    resultado quede en unidades de desviacion tipica. Se usa la mediana y el
    rango intercuartilico en lugar de media y desviacion porque son insensibles
    a los propios valores extremos que se quieren detectar."""
    iqr = np.subtract(*np.percentile(serie, [75, 25]))
    return (serie - np.median(serie)) / (0.74 * iqr) if iqr > 0 else np.zeros_like(serie)


series_qc = {}
filas_resumen = []

for sujeto in CFG.id_sujetos:
    rutas = sorted(DIR_FMRIPREP.glob(f"{sujeto}/**/func/*_desc-confounds_timeseries.tsv"))
    if not rutas:
        print(f"   sin tabla de confounds para {sujeto}, se omite")
        continue
    conf = pd.read_csv(rutas[0], sep="\t")

    # Parametros de realineamiento: traslaciones en mm, rotaciones en radianes.
    columnas_mov = ["trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]
    parametros = conf[columnas_mov].fillna(0.0).to_numpy()

    fd_power = conf["framewise_displacement"].fillna(0.0).to_numpy()   # definicion de fMRIPrep
    fd_caja = fd_caja_envolvente(parametros)                           # definicion de Morfini

    # DVARS: fMRIPrep publica varias variantes. Se usa la estandarizada.
    columna_dvars = next((c for c in ("std_dvars", "dvars") if c in conf.columns), None)
    dvars = conf[columna_dvars].fillna(0.0).to_numpy() if columna_dvars else np.zeros(len(conf))

    # Cambio de senal global, calculado y escalado como en la definicion de
    # referencia, a partir de la senal global que entrega fMRIPrep.
    #
    # El escalado NO lleva valor absoluto, y esto importa. La serie `gschange` ya es
    # no negativa, porque es el modulo de la diferencia entre volumenes
    # consecutivos. Al restarle su mediana y dividir por su rango intercuartilico se
    # obtiene una magnitud CON SIGNO, y asi la define la referencia, cuyo rango para
    # MeanGSchange va de menos infinito a mas infinito. Si se tomase el valor
    # absoluto de la serie ya escalada, un volumen con un cambio de senal
    # anormalmente PEQUENO quedaria convertido en una desviacion positiva grande y
    # seria marcado como atipico, que es justo lo contrario de lo que se pretende
    # detectar. Ademas MeanGSchange dejaria de rondar el cero y no seria comparable
    # con los valores publicados.
    senal_global = conf["global_signal"].to_numpy()
    gschange = np.concatenate([[0.0], np.abs(np.diff(senal_global))])
    gschange_std = escalar_a_unidades_estandar(gschange)

    # Volumenes atipicos: superan el umbral de FD O el de cambio de senal global.
    # Se usa la definicion de caja envolvente por coherencia con los umbrales de
    # la literatura de referencia.
    atipicos = (fd_caja > CFG.umbral_fd_mm) | (gschange_std > CFG.umbral_gschange_sd)
    validos = ~atipicos

    # Cuantos volumenes marcaria la otra convencion, con los umbrales por defecto de
    # fMRIPrep. Se calcula para poder comparar de forma explicita, porque las dos
    # definiciones NO son intercambiables y la diferencia se ve en la tabla.
    atipicos_power = (fd_power > UMBRAL_FD_POWER_MM) | (dvars > UMBRAL_DVARS_ESTANDAR)

    series_qc[sujeto] = pd.DataFrame({
        "fd_caja": fd_caja, "fd_power": fd_power, "dvars": dvars,
        "gschange": gschange, "gschange_std": gschange_std, "atipico": atipicos,
        "atipico_power": atipicos_power,
        **{c: parametros[:, i] for i, c in enumerate(columnas_mov)},
    })

    # Si un sujeto tuviese todos los volumenes marcados, promediar sobre la
    # seleccion vacia devolveria NaN con un aviso poco informativo. Se protege de
    # forma explicita para que el caso quede visible en la tabla.
    def media_de_validos(serie):
        return round(float(serie[validos].mean()), 3) if validos.any() else np.nan

    filas_resumen.append({
        "Sujeto": sujeto.replace("sub-", ""),
        "Volumenes": len(conf),
        "MaxMotion (mm)": round(float(fd_caja.max()), 3),
        "MeanMotion (mm)": media_de_validos(fd_caja),
        "MeanGSchange": media_de_validos(gschange_std),
        "InvalidScans": int(atipicos.sum()),
        "ValidScans": int(validos.sum()),
        "PVS": round(float(validos.mean()), 3),
        "InvalidScans (Power)": int(atipicos_power.sum()),
        "FD Power medio": round(float(fd_power.mean()), 3),
        "FD caja medio": round(float(fd_caja.mean()), 3),
        "corr(FD, DVARS)": round(float(np.corrcoef(fd_caja[1:], dvars[1:])[0, 1]), 3),
    })

if not filas_resumen:
    raise RuntimeError("No hay tablas de confounds. Ejecute fMRIPrep (celda 5.3).")

resumen_movimiento = pd.DataFrame(filas_resumen)
print("Metricas resumen por sujeto")
print(resumen_movimiento.to_string(index=False))

print(f"\nUmbrales aplicados: FD de caja envolvente > {CFG.umbral_fd_mm} mm, "
      f"cambio de senal global > {CFG.umbral_gschange_sd} desviaciones.")
print(f"La columna InvalidScans (Power) aplica en cambio FD de Power > "
      f"{UMBRAL_FD_POWER_MM} mm o DVARS estandarizado > {UMBRAL_DVARS_ESTANDAR}, que")
print("son los umbrales por defecto de fMRIPrep, y sirve solo de comparacion.")
print("PVS por debajo de 0.75 se considera valor extremo (Morfini et al., 2023).")

# Comparacion de las dos definiciones de desplazamiento de encuadre.
razon = (resumen_movimiento["FD caja medio"] / resumen_movimiento["FD Power medio"]).mean()
print(f"\nLa definicion de caja envolvente da valores {razon:.2f} veces los de la")
print("definicion de Power. Las dos son correctas y miden lo mismo con distinta")
print("convencion, de modo que un umbral de 0.5 mm NO significa lo mismo en ambas.")
print("Por eso las estrategias de censurado y de regresores de picos de la Seccion")
print("10 usan AMBAS la columna `atipico`, derivada de la definicion de caja, y no")
print("las columnas motion_outlier de fMRIPrep, que usan la de Power: mezclarlas")
print("haria que las dos estrategias tratasen conjuntos de volumenes distintos y")
print("dejasen de ser comparables entre si.")

resumen_movimiento.to_csv(PATHS["reportes"] / "resumen_movimiento.tsv",
                          sep="\t", index=False)
print(f"\nGuardado en {PATHS['reportes'] / 'resumen_movimiento.tsv'}")


### 7.3  Visualizaciones de movimiento y de senal


In [ ]:
# 7.3  Visualizaciones de movimiento y de senal
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sujetos = list(series_qc.keys())
etiquetas_sub = [s.replace("sub-", "") for s in sujetos]
n_sub = len(sujetos)

# Figura 1: series temporales por sujeto. Es la vista que permite localizar cuando
# ocurre cada evento y, sobre todo, comprobar si el desplazamiento de encuadre y
# DVARS coinciden en el tiempo, que es el diagnostico que distingue movimiento de
# artefacto de otro origen.
fig, axes = plt.subplots(n_sub, 1, figsize=(13, 3.2 * n_sub), squeeze=False, sharex=True)
for i, sujeto in enumerate(sujetos):
    s = series_qc[sujeto]
    ax = axes[i][0]
    ax.plot(s["fd_caja"], color="tab:blue", linewidth=0.8, label="FD (caja envolvente)")
    ax.axhline(CFG.umbral_fd_mm, color="tab:blue", linestyle=":", linewidth=0.8)
    ax.set_ylabel("FD (mm)", color="tab:blue")
    ax.tick_params(axis="y", labelcolor="tab:blue")

    # DVARS y cambio de senal global comparten un segundo eje, ya que ambos estan
    # en unidades estandarizadas y son comparables entre si.
    ax2 = ax.twinx()
    ax2.plot(s["dvars"], color="tab:red", linewidth=0.7, alpha=0.8, label="DVARS")
    ax2.plot(s["gschange_std"], color="tab:green", linewidth=0.7, alpha=0.8,
             label="GSchange (unidades estandar)")
    ax2.axhline(CFG.umbral_gschange_sd, color="tab:green", linestyle=":", linewidth=0.8)
    ax2.set_ylabel("Unidades estandar")

    # Sombreado de los volumenes marcados como atipicos.
    for indice in np.flatnonzero(s["atipico"].to_numpy()):
        ax.axvspan(indice - 0.5, indice + 0.5, color="grey", alpha=0.25, linewidth=0)

    ax.set_title(f"{etiquetas_sub[i]}: "
                 f"{int(s['atipico'].sum())} volumenes atipicos de {len(s)}",
                 fontsize=10, loc="left")
    if i == 0:
        lineas = ax.get_lines()[:1] + ax2.get_lines()[:2]
        ax.legend(lineas, [l.get_label() for l in lineas], fontsize=8,
                  ncol=3, frameon=False, loc="upper right")
axes[-1][0].set_xlabel("Volumen")
fig.suptitle("Series temporales de movimiento y de senal. Franjas grises: volumenes atipicos",
             fontsize=11)
plt.tight_layout()
plt.show()

# Figura 2: parametros de realineamiento. Separa traslaciones de rotaciones porque
# tienen unidades distintas y representarlas juntas oculta la de menor magnitud.
fig, axes = plt.subplots(n_sub, 2, figsize=(13, 2.6 * n_sub), squeeze=False, sharex=True)
for i, sujeto in enumerate(sujetos):
    s = series_qc[sujeto]
    for eje in ("x", "y", "z"):
        axes[i][0].plot(s[f"trans_{eje}"], linewidth=0.8, label=eje)
        # Rotaciones convertidas a grados: en radianes las cifras son tan pequenas
        # que resultan poco informativas a la vista.
        axes[i][1].plot(np.degrees(s[f"rot_{eje}"]), linewidth=0.8, label=eje)
    axes[i][0].set_ylabel(f"{etiquetas_sub[i]}\nTraslacion (mm)", fontsize=9)
    axes[i][1].set_ylabel("Rotacion (grados)", fontsize=9)
    if i == 0:
        axes[i][0].legend(fontsize=8, ncol=3, frameon=False)
        axes[i][0].set_title("Traslaciones", fontsize=10)
        axes[i][1].set_title("Rotaciones", fontsize=10)
axes[-1][0].set_xlabel("Volumen")
axes[-1][1].set_xlabel("Volumen")
plt.tight_layout()
plt.show()

# Figura 3: distribuciones, con histogramas, diagramas de caja y mapa de calor.
# Las series temporales muestran cuando ocurre cada evento; las distribuciones
# muestran si el sujeto tiene un problema puntual o sostenido, que es la distincion
# que decide entre censurar y excluir.
fig = plt.figure(figsize=(13, 8))
grid = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.28, height_ratios=[1, 1.2])

# Histogramas del desplazamiento de encuadre, con el eje de frecuencias en escala
# logaritmica porque la distribucion tiene una cola larga y en escala lineal solo
# se distingue el primer intervalo.
ax = fig.add_subplot(grid[0, 0])
for i, sujeto in enumerate(sujetos):
    ax.hist(series_qc[sujeto]["fd_caja"], bins=50, histtype="step",
            label=etiquetas_sub[i], linewidth=1.1)
ax.axvline(CFG.umbral_fd_mm, color="black", linestyle=":", linewidth=0.9)
ax.set_yscale("log")
ax.set_xlabel("FD (mm)")
ax.set_ylabel("Volumenes (log)")
ax.set_title("Distribucion de FD", fontsize=10, loc="left")
ax.legend(fontsize=8, frameon=False)

# Diagramas de caja. Las etiquetas del eje se fijan aparte y no mediante el
# argumento `labels` de boxplot, que matplotlib ha marcado como obsoleto: el
# objetivo es que el notebook no dependa de comportamientos que van a cambiar.
for columna, (clave, titulo) in enumerate((("fd_caja", "FD por sujeto"),
                                           ("dvars", "DVARS por sujeto")), start=1):
    ax = fig.add_subplot(grid[0, columna])
    ax.boxplot([series_qc[s][clave] for s in sujetos], showfliers=True,
               flierprops=dict(marker=".", markersize=3, alpha=0.4))
    ax.set_xticks(range(1, n_sub + 1))
    ax.set_xticklabels(etiquetas_sub)
    ax.set_ylabel("FD (mm)" if clave == "fd_caja" else "DVARS")
    if clave == "fd_caja":
        ax.axhline(CFG.umbral_fd_mm, color="black", linestyle=":", linewidth=0.9)
    ax.set_title(titulo, fontsize=10, loc="left")

# Mapa de calor del desplazamiento de encuadre, con sujetos en filas y volumenes
# en columnas. Con muchos sujetos es la vista que permite localizar de un vistazo
# si un problema afecta a un sujeto concreto, que aparece como fila, o a un
# instante concreto, que aparece como columna y apuntaria al equipo.
ax = fig.add_subplot(grid[1, :])
longitud = min(len(series_qc[s]) for s in sujetos)
matriz_fd = np.vstack([series_qc[s]["fd_caja"].to_numpy()[:longitud] for s in sujetos])
im = ax.imshow(matriz_fd, aspect="auto", cmap="magma",
               vmin=0, vmax=np.percentile(matriz_fd, 99))
ax.set_yticks(range(n_sub))
ax.set_yticklabels(etiquetas_sub)
ax.set_xlabel("Volumen")
ax.set_title("Mapa de calor de FD: filas son sujetos, columnas son volumenes",
             fontsize=10, loc="left")
fig.colorbar(im, ax=ax, fraction=0.03, label="FD (mm)")
plt.show()

# Tabla exportable con las series completas, para el informe final.
for sujeto, s in series_qc.items():
    s.to_csv(PATHS["reportes"] / f"series_qc_{sujeto}.tsv", sep="\t", index=False)
print(f"Series por volumen guardadas en {PATHS['reportes']}")


**Series temporales.** La comprobación central es si los picos de FD y los de DVARS coinciden en el tiempo. Un pico de FD sin pico de DVARS es movimiento que no llegó a corromper la señal, y no obliga a censurar. Un pico de DVARS sin pico de FD apunta a un artefacto de origen no motor, como una inestabilidad de bobina o un pulso de gradiente, y merece inspección visual del volumen concreto antes de decidir. La coincidencia de ambos es el caso claro de volumen comprometido.

**Parámetros de realineamiento.** Se separan traslaciones y rotaciones porque tienen unidades distintas y representarlas juntas oculta la de menor magnitud; las rotaciones se convierten a grados por legibilidad. Lo que hay que distinguir es la forma del patrón: una deriva lenta y monótona es un reposicionamiento gradual de la cabeza y resulta benigna, porque la corrección de movimiento la resuelve bien. Los saltos abruptos son los que corrompen volúmenes. Un patrón oscilatorio regular en torno a 0.2 o 0.3 Hz en los ejes y o z suele ser movimiento respiratorio aparente, un artefacto conocido de las secuencias multibanda con TR corto que no corresponde a movimiento real de la cabeza y que, por tanto, censurar no arregla.

**Distribuciones.** Las series muestran cuándo ocurre cada evento; las distribuciones muestran si el sujeto tiene un problema puntual o sostenido, que es la distinción que decide entre censurar y excluir. Un histograma concentrado en valores bajos con unos pocos valores extremos describe a un sujeto que estuvo quieto salvo en momentos concretos, y ese caso se resuelve censurando. Un histograma desplazado en bloque hacia valores altos describe movimiento sostenido, que ninguna cantidad de censurado arregla sin destruir la serie. El mapa de calor traslada esa lectura al conjunto: una fila entera brillante señala un sujeto problemático, mientras que una columna brillante compartida por varios sujetos apuntaría al equipo o al momento de la sesión y no al participante.

**Buenas prácticas.** Declarar la definición de desplazamiento de encuadre empleada junto con el umbral, porque un umbral de 0.5 mm no significa lo mismo en la convención de Power que en la de caja envolvente. Calcular MeanMotion solo sobre volúmenes válidos y MaxMotion sobre todos, y no confundir lo que informa cada uno. Conservar las series completas por volumen y no solo los resúmenes, porque el informe final necesita poder señalar qué volúmenes concretos se descartaron.

**Errores frecuentes.** Aplicar un umbral tomado de la literatura sin comprobar qué definición de FD usaba el trabajo de origen. Interpretar el movimiento respiratorio aparente de las secuencias multibanda como movimiento real y censurar en consecuencia, perdiendo datos sin ganar nada. Comparar el movimiento medio entre grupos experimentales sin verificar que las distribuciones tienen forma parecida, ya que dos sujetos con la misma media pueden tener perfiles de riesgo opuestos. Censurar y a la vez incluir regresores de picos para los mismos volúmenes, que es redundante y contabiliza dos veces el mismo coste en grados de libertad, error que se discute en la Sección 10.


### 7.4 Panel integrado de movimiento, señal tisular y volúmenes marcados

**Fundamento metodológico.** Las tres figuras de 7.3 muestran cada métrica por separado, y eso deja sin resolver la pregunta que de verdad importa: si un pico de movimiento produjo o no un cambio de señal. Para responderla hay que ver las series alineadas en el mismo eje temporal. Power (2017) propuso exactamente esta vista y argumentó que es la forma más eficiente de juzgar la calidad de una adquisición funcional, porque permite verificar la correspondencia temporal entre causa y efecto en lugar de inferirla de dos gráficos separados. fMRIPrep incluye una versión en su informe y MRIQC otra, pero ninguna de las dos usa los umbrales ni las definiciones de este notebook, de modo que se genera aquí con las propias.

**Qué contiene el panel, de arriba abajo.**

1. **Señales tisulares medias, estandarizadas y desplazadas en vertical.** Una traza por tejido: sustancia gris, sustancia blanca y líquido cefalorraquídeo. Se estandariza cada una por separado, porque sus escalas absolutas no son comparables, y se desplaza en vertical para poder superponerlas sin que se solapen. El desplazamiento es solo un recurso de presentación y no altera ninguna medida.
2. **Carpet plot con las filas agrupadas por tejido.** Cada fila es un vóxel y cada columna un volumen. La intensidad representa el cambio porcentual respecto a la media temporal de ese mismo vóxel, no el valor absoluto, porque de otro modo la imagen mostraría anatomía en lugar de dinámica. Las líneas horizontales separan los tres tejidos.
3. **Métricas de movimiento y de señal, con sus umbrales.** Desplazamiento de encuadre según la definición de caja envolvente, DVARS estandarizado y cambio de señal global en unidades típicas.

**El elemento que une los tres bloques** son las bandas verticales sombreadas, que marcan los volúmenes clasificados como atípicos por el criterio de 7.2. Aparecen en los tres paneles a la vez, y ahí reside el valor de la figura: permite comprobar de un vistazo si lo que el criterio marcó coincide con una alteración visible de la señal.

**Cómo se lee, en tres preguntas.**

**Primera, ¿coinciden las bandas verticales del carpet con los picos de movimiento?** Si coinciden, el criterio de censurado está funcionando: detecta alteraciones reales. Si el carpet muestra bandas donde las trazas no marcan nada, hay una fuente de artefacto que el movimiento no explica, y conviene sospechar de la reconstrucción o de un efecto fisiológico. Si las trazas marcan picos sin banda correspondiente en el carpet, el criterio está siendo conservador: descarta volúmenes cuya señal no se alteró de forma apreciable, lo cual cuesta grados de libertad sin beneficio.

**Segunda, ¿las tres señales tisulares se mueven juntas o por separado?** Que suban y bajen a la vez indica una fuente global, típicamente movimiento o respiración, que es lo que la regresión de señal global o los componentes de CompCor pueden tratar. Que solo se mueva la del líquido cefalorraquídeo apunta a pulsación cardiaca, más localizada en ventrículos y cisternas. Que la de sustancia gris se mueva de forma independiente de las otras dos es la situación deseable, porque es donde se espera la señal de origen neural, y de hecho es el argumento por el que aCompCor extrae sus componentes de la blanca y del líquido y no de la gris.

**Tercera, ¿el patrón es uniforme a lo largo del registro o se concentra al final?** Un deterioro progresivo, con más picos en la segunda mitad, es la firma del cansancio o la incomodidad del participante. Tiene una implicación práctica distinta de la del movimiento esporádico: sugiere acortar la duración de la adquisición en futuros protocolos, más que censurar en este.

**Qué hacer según lo que se encuentre.**

| Patrón observado | Actuación recomendada |
|---|---|
| Bandas del carpet alineadas con picos de movimiento, en número reducido | Censurar. Es el caso para el que el procedimiento está diseñado |
| Bandas alineadas con movimiento pero muy numerosas, con proporción de volúmenes válidos por debajo de 0.75 | Considerar la exclusión del sujeto. Censurar tanto deja una estimación sin grados de libertad suficientes |
| Bandas sin pico de movimiento asociado | No censurar sin más: el criterio no las detecta. Investigar la reconstrucción antes de decidir |
| Las tres señales tisulares acopladas de forma intensa | Evaluar la regresión de señal global en la Sección 10, con la advertencia sobre anticorrelaciones que allí se discute |
| Solo el líquido cefalorraquídeo oscila de forma marcada | Los componentes de aCompCor del líquido deberían capturarlo. Comprobar su varianza explicada en la Sección 9 |
| Deterioro progresivo hacia el final del registro | Documentarlo como limitación. Valorar el análisis restringido a la primera parte, declarando el criterio |
| Deriva lenta y monótona en las tres señales | Es lo que elimina el filtro paso banda. No requiere actuación específica |


In [ ]:
# 7.4  Panel integrado: senales tisulares, carpet plot y metricas de movimiento
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np

# Numero de filas del carpet por tejido. Se submuestrea porque dibujar cientos de
# miles de filas no aporta informacion y hace la figura ilegible y pesada.
FILAS_CARPET = {"GM": 400, "WM": 200, "CSF": 120}
# Voxeles por tejido para la media tisular. Con dos mil la media ya es estable.
VOXELES_MEDIA = 2000
LIMITE_CARPET = 2.0        # saturacion de la escala de color, en porcentaje
UMBRAL_PROBSEG = 0.5
BLOQUE_VOLUMENES = 50      # volumenes por lectura, ver la funcion siguiente

if "DIR_FIGURAS" not in globals():
    DIR_FIGURAS = PATHS["reportes"] / "figuras"
    DIR_FIGURAS.mkdir(parents=True, exist_ok=True)


def leer_voxeles_por_bloques(img, indices_planos, bloque=BLOQUE_VOLUMENES):
    """Serie temporal de unos voxeles concretos, leyendo por bloques de volumenes.

    Las Secciones 10 y 11 reutilizan esta funcion, y su forma responde a un problema
    medido y no a una preferencia. La version inicial leia un volumen por iteracion,
    con `img.dataobj[..., t]`, y resulto inaceptablemente lenta. La causa es que los
    archivos NIfTI vienen comprimidos, y en un flujo comprimido no existe acceso
    aleatorio: para llegar al volumen t hay que descomprimir todo lo anterior, de
    modo que cuatrocientas lecturas sueltas descomprimen el archivo muchas veces.

    Leyendo bloques CONTIGUOS de volumenes se descomprime una sola vez de principio
    a fin. Con bloques de cincuenta volumenes en la rejilla de 2 mm el pico de
    memoria son unos doscientos megabytes, asumible, y se pasa de cuatrocientas
    lecturas a ocho.
    """
    n_vol = img.shape[3]
    salida = np.empty((len(indices_planos), n_vol), dtype=np.float32)
    for inicio in range(0, n_vol, bloque):
        fin = min(inicio + bloque, n_vol)
        trozo = np.asanyarray(img.dataobj[..., inicio:fin], dtype=np.float32)
        salida[:, inicio:fin] = trozo.reshape(-1, fin - inicio)[indices_planos]
        del trozo
    return salida


def mascaras_tisulares_en_espacio_funcional(sujeto, forma_esperada):
    """Mascaras binarias de los tres tejidos en la rejilla del funcional.

    Se usan los mapas de probabilidad que fMRIPrep escribe en el espacio de salida,
    que comparten rejilla con el funcional remuestreado. Se comprueba la forma en
    lugar de suponerla: si no coincide, se informa y se devuelve vacio, porque
    indexar con una mascara de otra rejilla produciria un resultado sin sentido en
    lugar de un error.
    """
    mascaras = {}
    for tejido in ("GM", "WM", "CSF"):
        rutas = sorted(DIR_FMRIPREP.glob(
            f"{sujeto}/**/anat/*space-{CFG.plantilla}*label-{tejido}_probseg.nii.gz"))
        if not rutas:
            print(f"   {sujeto}: sin mapa de probabilidad de {tejido}")
            return {}
        datos = np.asanyarray(nib.load(rutas[0]).dataobj)
        if datos.shape != forma_esperada:
            print(f"   {sujeto}: la rejilla de {tejido} {datos.shape} no coincide "
                  f"con la del funcional {forma_esperada}, se omite el carpet")
            return {}
        mascaras[tejido] = datos > UMBRAL_PROBSEG
    return mascaras


rutas_paneles = []

for sujeto in CFG.id_sujetos:
    if sujeto not in series_qc:
        continue
    rutas_bold = sorted(DIR_FMRIPREP.glob(
        f"{sujeto}/**/func/*space-{CFG.plantilla}*_desc-preproc_bold.nii.gz"))
    if not rutas_bold:
        print(f"   {sujeto}: sin funcional preprocesado, se omite")
        continue

    img = nib.load(rutas_bold[0])
    tr = float(img.header.get_zooms()[3])
    n_vol = img.shape[3]
    mascaras = mascaras_tisulares_en_espacio_funcional(sujeto, img.shape[:3])
    if not mascaras:
        continue

    # Mascara cerebral del funcional, para no incluir voxeles sin senal.
    rutas_mascara = sorted(DIR_FMRIPREP.glob(
        f"{sujeto}/**/func/*space-{CFG.plantilla}*_desc-brain_mask.nii.gz"))
    cerebro = (np.asanyarray(nib.load(rutas_mascara[0]).dataobj).astype(bool)
               if rutas_mascara else np.ones(img.shape[:3], dtype=bool))

    # Seleccion de filas del carpet, por tejido, y de los voxeles necesarios para
    # las medias tisulares. Se leen todos de una sola pasada.
    rng = np.random.default_rng(CFG.semilla)
    filas_por_tejido, indices_medias = {}, {}
    for tejido, cuantas in FILAS_CARPET.items():
        disponibles = np.flatnonzero((mascaras[tejido] & cerebro).ravel())
        if disponibles.size == 0:
            print(f"   {sujeto}: mascara de {tejido} vacia tras intersecar con el "
                  f"cerebro, se omite")
            filas_por_tejido = {}
            break
        indices_medias[tejido] = (
            rng.choice(disponibles, size=min(VOXELES_MEDIA, disponibles.size),
                       replace=False))
        filas_por_tejido[tejido] = (
            rng.choice(disponibles, size=min(cuantas, disponibles.size), replace=False))
    if not filas_por_tejido:
        continue

    # Union de todos los indices que hay que leer, para una sola pasada por el 4D.
    todos = np.unique(np.concatenate(
        [v for v in filas_por_tejido.values()] + [v for v in indices_medias.values()]))
    print(f"   {sujeto}: leyendo {len(todos)} voxeles en bloques de "
          f"{BLOQUE_VOLUMENES} volumenes")
    lectura = leer_voxeles_por_bloques(img, todos)
    posicion = {valor: i for i, valor in enumerate(todos)}

    # Senales tisulares medias, estandarizadas por separado.
    senales = {}
    for tejido, indices in indices_medias.items():
        filas = [posicion[v] for v in indices]
        media = lectura[filas].mean(axis=0)
        desviacion = media.std()
        senales[tejido] = (media - media.mean()) / (desviacion if desviacion > 0 else 1.0)

    # Carpet: cambio porcentual de cada voxel respecto de su propia media temporal.
    bloques, separadores, etiquetas_tejido = [], [], []
    for tejido in ("GM", "WM", "CSF"):
        filas = [posicion[v] for v in filas_por_tejido[tejido]]
        bloque = lectura[filas]
        medias_voxel = bloque.mean(axis=1, keepdims=True)
        medias_voxel[np.abs(medias_voxel) < 1e-12] = 1.0
        bloques.append((bloque - medias_voxel) / medias_voxel * 100.0)
        etiquetas_tejido.append(tejido)
        separadores.append(sum(b.shape[0] for b in bloques))
    carpet = np.vstack(bloques)
    del lectura

    # Metricas de 7.2, recortadas a la longitud real de la serie.
    s_qc = series_qc[sujeto]
    fd = s_qc["fd_caja"].to_numpy()[:n_vol]
    dvars = s_qc["dvars"].to_numpy()[:n_vol]
    gsc = s_qc["gschange_std"].to_numpy()[:n_vol]
    atipicos = s_qc["atipico"].to_numpy()[:n_vol]
    tiempo = np.arange(n_vol) * tr

    fig, axes = plt.subplots(
        3, 1, figsize=(13, 9), sharex=True,
        gridspec_kw={"height_ratios": [1.1, 2.6, 1.6], "hspace": 0.08})

    def sombrear_atipicos(ax):
        """Bandas verticales en los volumenes marcados, en los tres paneles."""
        for indice in np.flatnonzero(atipicos):
            ax.axvspan(tiempo[indice] - tr / 2, tiempo[indice] + tr / 2,
                       color="tab:red", alpha=0.16, linewidth=0, zorder=0)

    # Panel 1: senales tisulares desplazadas en vertical.
    ax = axes[0]
    for i, (tejido, color) in enumerate((("GM", "tab:green"), ("WM", "tab:orange"),
                                         ("CSF", "tab:cyan"))):
        # El desplazamiento de 5 unidades tipicas separa las trazas sin solaparlas.
        ax.plot(tiempo, senales[tejido] + (2 - i) * 5.0, linewidth=0.7, color=color,
                label=tejido)
    sombrear_atipicos(ax)
    ax.set_yticks([10.0, 5.0, 0.0])
    ax.set_yticklabels(["GM", "WM", "CSF"], fontsize=9)
    ax.set_ylabel("Senal tisular\n(unidades tipicas, desplazadas)", fontsize=8)
    ax.set_title(f"Sujeto {sujeto.replace('sub-', '')}. Panel integrado de "
                 f"movimiento y senal. Las bandas rojas marcan los volumenes "
                 f"clasificados como atipicos", fontsize=10, loc="left")
    ax.legend(fontsize=8, frameon=False, ncol=3, loc="upper right")

    # Panel 2: carpet plot.
    ax = axes[1]
    ax.imshow(carpet, aspect="auto", cmap="gray", vmin=-LIMITE_CARPET,
              vmax=LIMITE_CARPET, interpolation="nearest",
              extent=[tiempo[0], tiempo[-1], carpet.shape[0], 0])
    for limite in separadores[:-1]:
        ax.axhline(limite, color="tab:red", linewidth=0.9)
    centros = [0] + separadores
    ax.set_yticks([(centros[i] + centros[i + 1]) / 2 for i in range(3)])
    ax.set_yticklabels(etiquetas_tejido, fontsize=9)
    ax.set_ylabel(f"Voxeles agrupados por tejido\n(escala mas menos "
                  f"{LIMITE_CARPET} por ciento)", fontsize=8)

    # Panel 3: metricas con sus umbrales.
    ax = axes[2]
    ax.plot(tiempo, fd, linewidth=0.8, color="tab:blue", label="FD caja (mm)")
    ax.axhline(CFG.umbral_fd_mm, color="tab:blue", linestyle="--", linewidth=0.8)
    ax.plot(tiempo, dvars, linewidth=0.8, color="tab:purple",
            label="DVARS estandarizado")
    ax.plot(tiempo, gsc, linewidth=0.8, color="tab:olive", alpha=0.9,
            label="GSchange (unidades tipicas)")
    ax.axhline(CFG.umbral_gschange_sd, color="tab:olive", linestyle="--",
               linewidth=0.8)
    sombrear_atipicos(ax)
    ax.set_xlabel("Tiempo (s)")
    ax.set_ylabel("Valor", fontsize=9)
    ax.legend(fontsize=8, frameon=False, ncol=3, loc="upper right")

    nombre = f"panel_integrado_{sujeto.replace('sub-', '')}"
    rutas_paneles.append(guardar_figura(fig, nombre))
    plt.show()

    # Resumen numerico que acompana a la figura, para no depender solo de la vista.
    n_atipicos = int(atipicos.sum())
    print(f"   {sujeto}: {n_atipicos} de {n_vol} volumenes marcados "
          f"({n_atipicos / n_vol:.1%}), proporcion valida "
          f"{1 - n_atipicos / n_vol:.3f}")
    acoplamiento = float(np.corrcoef(senales["GM"], senales["WM"])[0, 1])
    print(f"      correlacion entre senal de GM y de WM: {acoplamiento:.3f} "
          f"({'acoplamiento alto, revise la Seccion 10' if abs(acoplamiento) > 0.6 else 'acoplamiento moderado o bajo'})")
    if n_atipicos:
        # Cuantos de los marcados coinciden con un pico de movimiento y cuantos se
        # deben solo al cambio de senal global. Es la comprobacion de la primera
        # pregunta de lectura, hecha con numeros y no a ojo.
        por_movimiento = int(((fd > CFG.umbral_fd_mm) & atipicos).sum())
        solo_senal = n_atipicos - por_movimiento
        print(f"      por movimiento: {por_movimiento}, "
              f"solo por cambio de senal global: {solo_senal}")
        primera, segunda = atipicos[:n_vol // 2].sum(), atipicos[n_vol // 2:].sum()
        print(f"      reparto temporal: {int(primera)} en la primera mitad, "
              f"{int(segunda)} en la segunda"
              f"{'. Deterioro progresivo, vea la tabla de actuaciones' if segunda > 2 * max(primera, 1) else ''}")

if not rutas_paneles:
    print("No se genero ningun panel. Requiere los productos de fMRIPrep en el "
          "espacio de la plantilla y las metricas de 7.2.")
else:
    print("\nPaneles guardados:")
    for ruta in rutas_paneles:
        print(f"   {ruta}")
    if DIR_PERSISTENTE is not None:
        sincronizar_persistente()


**Cómo se lee el panel integrado.** Es la vista que responde a la pregunta que las figuras separadas de 7.3 no pueden contestar: si un pico de movimiento produjo o no un cambio real en la señal. Power (2017) propuso exactamente esta disposición y argumentó que es la forma más eficiente de juzgar la calidad de una adquisición funcional, porque permite verificar la correspondencia temporal entre causa y efecto en lugar de inferirla comparando dos gráficos.

**Los tres bloques, de arriba abajo.** Las señales medias de sustancia gris, blanca y líquido cefalorraquídeo, estandarizadas por separado y desplazadas en vertical para poder superponerlas sin que se solapen. En el centro, el carpet plot con las filas agrupadas por tejido y separadas por líneas rojas. Abajo, el desplazamiento de encuadre, DVARS y el cambio de señal global con sus umbrales. Atravesando los tres, las bandas verticales de los volúmenes clasificados como atípicos.

**Tres preguntas, en este orden.**

**Primera, ¿coinciden las bandas verticales del carpet con los picos de movimiento?** Si coinciden, el criterio de censurado está detectando alteraciones reales. Si el carpet muestra bandas donde las trazas no marcan nada, hay una fuente de artefacto que el movimiento no explica, y conviene sospechar de la reconstrucción o de un efecto fisiológico. Si las trazas marcan picos sin banda correspondiente, el criterio está siendo conservador: descarta volúmenes cuya señal no se alteró de forma apreciable, y eso cuesta grados de libertad sin beneficio. La celda cuantifica esa correspondencia contando cuántos volúmenes marcados vienen de movimiento y cuántos solo del cambio de señal global, para no depender del ojo.

**Segunda, ¿las tres señales tisulares se mueven juntas o por separado?** Que suban y bajen a la vez indica una fuente global, típicamente movimiento o respiración, que es lo que la regresión de señal global o los componentes de CompCor pueden tratar. Que solo oscile la del líquido cefalorraquídeo apunta a pulsación cardiaca, más localizada en ventrículos y cisternas. Que la de sustancia gris se mueva de forma independiente de las otras dos es la situación deseable, porque es donde se espera la señal de origen neural, y de hecho es el argumento por el que aCompCor extrae sus componentes de la blanca y del líquido y no de la gris. La celda reporta la correlación entre la señal de gris y la de blanca como resumen numérico de esta lectura.

**Tercera, ¿el patrón es uniforme a lo largo del registro o se concentra al final?** Un deterioro progresivo, con más volúmenes marcados en la segunda mitad, es la firma del cansancio o la incomodidad del participante. Tiene una implicación práctica distinta de la del movimiento esporádico: sugiere acortar la duración de la adquisición en futuros protocolos más que censurar en este.

**Qué hacer según lo que se encuentre.**

| Patrón observado | Actuación recomendada |
|---|---|
| Bandas alineadas con picos de movimiento, en número reducido | Censurar. Es el caso para el que el procedimiento está diseñado |
| Bandas alineadas con movimiento pero muy numerosas, con proporción de volúmenes válidos por debajo de 0.75 | Considerar la exclusión del sujeto. Censurar tanto deja una estimación sin grados de libertad suficientes |
| Bandas sin pico de movimiento asociado | No censurar sin más: el criterio no las detecta. Investigar la reconstrucción antes de decidir |
| Las tres señales tisulares acopladas de forma intensa | Evaluar la regresión de señal global en la Sección 10, con la advertencia sobre anticorrelaciones que allí se discute |
| Solo el líquido cefalorraquídeo oscila de forma marcada | Los componentes de aCompCor del líquido deberían capturarlo. Comprobar su varianza explicada en la Sección 9 |
| Deterioro progresivo hacia el final del registro | Documentarlo como limitación. Valorar el análisis restringido a la primera parte, declarando el criterio |
| Deriva lenta y monótona en las tres señales | Es lo que elimina el filtro paso banda. No requiere actuación específica |

**Conclusiones parciales de la Sección 7.** Quedan calculadas las métricas de movimiento y de señal con las dos definiciones de desplazamiento de encuadre que circulan en la literatura, y comprobado que no son intercambiables. Queda establecido qué volúmenes se consideran atípicos y por cuál de los dos criterios, lo que la Sección 10 usará tanto para el censurado como para los regresores de picos. Y queda documentada la correspondencia temporal entre movimiento y alteración de la señal, que es la evidencia sobre la que se apoya cualquier decisión de censurado.


## Sección 8. Evaluación de la normalización y del corregistro

### 8.1 Qué se mide y con qué definiciones

**Introducción conceptual.** Las Secciones 5 y 6 comprobaron visualmente que la normalización y el corregistro parecían correctos. Esta sección los cuantifica. La diferencia importa por dos razones: un número permite comparar sujetos entre sí y detectar al que se desvía, y permite además fijar un criterio de exclusión reproducible en lugar de depender del juicio de quien mira la imagen.

**Fundamento metodológico.** Todas las medidas de esta sección se apoyan en el mismo principio, cuantificar cuánto se solapan dos máscaras que, si el procesamiento fue correcto, deberían coincidir. Morfini et al. (2023) definen tres medidas de este tipo, y aquí se implementan junto con dos coeficientes de solapamiento complementarios.

**Los dos coeficientes de solapamiento.** Dadas dos máscaras binarias A y B, con |A| el número de vóxeles activos:

El **coeficiente de Dice** es dos veces la intersección dividida por la suma de los tamaños:

$$
D(A,B) = \frac{2\,|A \cap B|}{|A| + |B|}
$$

El **índice de Jaccard** es la intersección dividida por la unión:

$$
J(A,B) = \frac{|A \cap B|}{|A \cup B|}
$$

Ambos valen 1 con solapamiento perfecto y 0 sin solapamiento, y están relacionados de forma monótona por $J = D/(2-D)$, de modo que ordenan igual a los sujetos. No son intercambiables en magnitud: para un mismo par de máscaras, Jaccard siempre da un valor menor que Dice, y la diferencia crece cuanto peor es el solapamiento. Esa es su utilidad como control cruzado: Jaccard penaliza más las discrepancias y por eso resulta más sensible a un fallo moderado que Dice podría disimular. Reportar solo uno de los dos sin decir cuál es una fuente habitual de confusión al comparar con la literatura.

**Las tres medidas de calidad del registro.** Siguiendo las definiciones de Morfini et al. (2023):

| Medida | Qué compara | Qué fallo detecta |
|---|---|---|
| NORManat | Máscara de sustancia gris de la plantilla frente a la derivada del anatómico normalizado | Fallo de la normalización anatómica |
| NORMfunc | Máscara de sustancia gris de la plantilla frente a la derivada del funcional normalizado | Fallo de la normalización funcional, que arrastra también los errores del corregistro |
| AFO | Máscara de sustancia gris anatómica frente a la funcional, ambas del mismo sujeto | Fallo del corregistro entre modalidades, aislado de la normalización |


**Un detalle de la definición que no es evidente y que condiciona el resultado.** Las máscaras que se comparan no se umbralizan al mismo nivel de probabilidad, sino que **la segunda se umbraliza al nivel que produzca el mismo número de vóxeles que la primera**. La razón es que el coeficiente de Dice depende del tamaño de las máscaras, dos máscaras de tamaños muy distintos no pueden solaparse bien por construcción, con independencia de que estén bien alineadas. Igualar los tamaños elimina esa dependencia y deja que la medida refleje solo el alineamiento, que es lo que se quiere medir. Omitir ese paso produce valores sistemáticamente bajos que se interpretan como mal registro cuando en realidad son un artefacto del procedimiento.

**Solapamiento con las plantillas de los tres tejidos.** Además de la sustancia gris, que es la de interés para conectividad, se calcula el solapamiento con las plantillas de sustancia blanca y de líquido cefalorraquídeo. Sirve como diagnóstico diferencial: un solapamiento bajo en los tres tejidos apunta a un fallo global de la normalización, mientras que un solapamiento bajo solo en sustancia gris con los otros dos correctos apunta a un problema de segmentación y no de registro.

**Sobre la plantilla de referencia.** Morfini et al. usan la plantilla IXI-549 que trae SPM. Este notebook usa `MNI152NLin2009cAsym`, que es la que produce fMRIPrep y la que se fijó en la configuración. Ambas son espacios MNI y las medidas son comparables en su interpretación, pero **los valores absolutos no son directamente comparables entre trabajos que usen plantillas distintas**, porque la plantilla define la referencia contra la que se mide. Conviene declararla siempre al reportar estas cifras.


In [ ]:
# 8.2  Metricas de solapamiento: NORManat, NORMfunc, AFO, Dice y Jaccard
import numpy as np
import nibabel as nib
import pandas as pd

# Umbrales de probabilidad de la definicion de referencia (Morfini et al., 2023).
UMBRAL_PLANTILLA = 0.25   # mascara de la plantilla, para NORManat y NORMfunc
UMBRAL_ANATOMICO = 0.50   # mascara anatomica del sujeto, para AFO
UMBRAL_EXCLUIR_CSF = 0.30  # probabilidad de liquido por encima de la cual se excluye

TEJIDOS = {"GM": "GM", "WM": "WM", "CSF": "CSF"}


def dice(a, b):
    """Coeficiente de Dice entre dos mascaras booleanas."""
    suma = a.sum() + b.sum()
    return float(2.0 * np.logical_and(a, b).sum() / suma) if suma else np.nan


def jaccard(a, b):
    """Indice de Jaccard entre dos mascaras booleanas."""
    union = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / union) if union else np.nan


def umbralizar_a_tamano(mapa, n_objetivo, nombre=""):
    """Binariza un mapa conservando EXACTAMENTE n_objetivo voxeles.

    Es el paso que hace comparables dos mascaras de tamano distinto. El
    coeficiente de Dice depende del tamano: dos mascaras muy dispares no pueden
    solaparse bien por construccion, aunque esten perfectamente alineadas.
    Igualando los tamanos, la medida refleja solo el alineamiento. Omitir este
    paso produce valores bajos que se confunden con un mal registro.

    Se seleccionan los voxeles por INDICE y no aplicando un umbral por valor. La
    diferencia importa: los mapas de probabilidad de plantilla estan cuantizados, de
    modo que hay muchos voxeles con el valor exacto del corte, y `mapa >= corte`
    devolveria mas voxeles de los pedidos, sesgando el Dice al alza.

    Si el mapa no tiene suficientes voxeles no nulos, el objetivo se recorta y se
    AVISA. El aviso no es decorativo: con la cobertura parcial de campo de vision de
    este dataset, la mascara funcional puede ser menor que la de la plantilla, y
    entonces NORMfunc sale deprimido por falta de campo y no por mal registro.
    """
    planos = mapa.ravel()
    disponibles = int((planos > 0).sum())
    n_efectivo = int(min(n_objetivo, disponibles))
    if n_efectivo < n_objetivo:
        print(f"      AVISO en {nombre}: objetivo {int(n_objetivo)} voxeles, "
              f"disponibles {disponibles}. Se usa {n_efectivo}, un "
              f"{n_efectivo / n_objetivo:.1%} del objetivo. Un valor bajo de "
              f"solapamiento puede deberse a esta limitacion y no al registro.")
    if n_efectivo <= 0:
        return np.zeros(mapa.shape, dtype=bool), n_efectivo
    indices = np.argpartition(planos, -n_efectivo)[-n_efectivo:]
    mascara = np.zeros(planos.size, dtype=bool)
    mascara[indices] = True
    return mascara.reshape(mapa.shape), n_efectivo


def cargar_plantilla(tejido):
    """Mapa de probabilidad del tejido en la plantilla, via TemplateFlow."""
    import templateflow.api as tflow
    ruta = tflow.get(CFG.plantilla, resolution=2, label=tejido, suffix="probseg",
                     extension=".nii.gz")
    ruta = ruta[0] if isinstance(ruta, list) else ruta
    return np.asanyarray(nib.load(ruta).dataobj, dtype=np.float32)


try:
    import templateflow.api  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "templateflow", f"pandas=={pd.__version__}"], check=False)

plantillas = {t: cargar_plantilla(t) for t in TEJIDOS}
print("Plantillas cargadas desde TemplateFlow:")
for t, mapa in plantillas.items():
    print(f"  {t}: forma {mapa.shape}, voxeles por encima de "
          f"{UMBRAL_PLANTILLA}: {int((mapa > UMBRAL_PLANTILLA).sum())}")

filas = []
for sujeto in CFG.id_sujetos:
    patron_anat = f"{sujeto}/**/anat/*space-{CFG.plantilla}*label-{{t}}_probseg.nii.gz"
    probs_anat = {}
    for t in TEJIDOS:
        encontrados = sorted(DIR_FMRIPREP.glob(patron_anat.format(t=t)))
        if encontrados:
            probs_anat[t] = np.asanyarray(nib.load(encontrados[0]).dataobj,
                                          dtype=np.float32)
    ruta_bold = sorted(DIR_FMRIPREP.glob(
        f"{sujeto}/**/func/*space-{CFG.plantilla}*_desc-preproc_bold.nii.gz"))
    ruta_mascara = sorted(DIR_FMRIPREP.glob(
        f"{sujeto}/**/func/*space-{CFG.plantilla}*_desc-brain_mask.nii.gz"))

    if len(probs_anat) < 3 or not ruta_bold or not ruta_mascara:
        print(f"   {sujeto}: faltan productos, se omite")
        continue

    print(f"   {sujeto}")
    fila = {"Sujeto": sujeto.replace("sub-", "")}

    # NORManat y solapamiento por tejido: plantilla frente al anatomico del sujeto.
    for t in TEJIDOS:
        ref = plantillas[t] > UMBRAL_PLANTILLA
        sujeto_mask, _ = umbralizar_a_tamano(probs_anat[t], ref.sum(),
                                             f"{t} anatomico")
        fila[f"Dice {t} anat"] = round(dice(ref, sujeto_mask), 4)
        fila[f"Jaccard {t} anat"] = round(jaccard(ref, sujeto_mask), 4)
    fila["NORManat"] = fila["Dice GM anat"]

    # Mascara de sustancia gris derivada del FUNCIONAL.
    #
    # La definicion de referencia habla de una mascara de GM derivada de los datos
    # funcionales sin precisar como obtenerla. La aproximacion evidente, tomar los
    # voxeles de mayor intensidad media, es INSUFICIENTE por si sola y conviene
    # entender por que. En una secuencia potenciada en T2* la sustancia gris es mas
    # brillante que la blanca, lo cual es cierto, pero el liquido cefalorraquideo es
    # mas brillante todavia que la gris. El conjunto de los voxeles mas intensos
    # estaria por tanto dominado por ventriculos y cisternas, y NORMfunc y AFO
    # saldrian deprimidos por comparar sustancia gris de la plantilla con liquido
    # del sujeto, no por un fallo de registro.
    #
    # Por eso se excluyen antes los voxeles con probabilidad alta de liquido, que ya
    # esta cargada en `probs_anat`. Sigue siendo una aproximacion y se declara como
    # tal: no equivale a una segmentacion tisular del funcional.
    bold = nib.load(ruta_bold[0])
    mascara_func = np.asanyarray(nib.load(ruta_mascara[0]).dataobj).astype(bool)
    media_bold = np.zeros(bold.shape[:3], dtype=np.float32)
    # Se promedia por bloques de volumenes para no cargar la serie completa.
    n_vol = bold.shape[3]
    for inicio in range(0, n_vol, 50):
        media_bold += np.asanyarray(bold.dataobj[..., inicio:inicio + 50],
                                    dtype=np.float32).sum(axis=3)
    media_bold /= n_vol

    excluidos = mascara_func & (probs_anat["CSF"] > UMBRAL_EXCLUIR_CSF)
    dominio_func = mascara_func & ~excluidos
    media_bold[~dominio_func] = 0.0
    print(f"      voxeles del funcional: {int(mascara_func.sum())}, excluidos por "
          f"probabilidad de liquido superior a {UMBRAL_EXCLUIR_CSF}: "
          f"{int(excluidos.sum())} ({excluidos.sum() / max(mascara_func.sum(), 1):.1%})")

    ref_gm = plantillas["GM"] > UMBRAL_PLANTILLA
    gm_func, n_gm_func = umbralizar_a_tamano(media_bold, ref_gm.sum(),
                                            "GM funcional frente a plantilla")
    fila["NORMfunc"] = round(dice(ref_gm, gm_func), 4)
    fila["Jaccard GM func"] = round(jaccard(ref_gm, gm_func), 4)
    fila["Cobertura NORMfunc"] = round(n_gm_func / max(int(ref_gm.sum()), 1), 3)

    # AFO: mascara anatomica de GM al 50 % frente a la funcional del mismo tamano.
    gm_anat_50 = probs_anat["GM"] > UMBRAL_ANATOMICO
    gm_func_afo, n_afo = umbralizar_a_tamano(media_bold, gm_anat_50.sum(),
                                            "GM funcional frente a anatomico")
    fila["AFO"] = round(dice(gm_anat_50, gm_func_afo), 4)
    fila["Jaccard AFO"] = round(jaccard(gm_anat_50, gm_func_afo), 4)
    fila["Cobertura AFO"] = round(n_afo / max(int(gm_anat_50.sum()), 1), 3)

    filas.append(fila)
    del bold, media_bold

if not filas:
    raise RuntimeError(
        "No hay productos de fMRIPrep suficientes. Ejecute la Seccion 5 y compruebe "
        "el estado con la celda 5.3b antes de continuar.")

solapamientos = pd.DataFrame(filas)

print("\nMedidas de calidad del registro (Dice, mayor es mejor)")
print(solapamientos[["Sujeto", "NORManat", "NORMfunc", "AFO",
                     "Cobertura NORMfunc", "Cobertura AFO"]].to_string(index=False))
print("Las columnas de cobertura indican que fraccion del objetivo de voxeles se")
print("pudo alcanzar. Un valor claramente inferior a 1 significa que la metrica esta")
print("limitada por el campo de vision y no por el registro, y en ese caso el valor")
print("de Dice correspondiente NO debe leerse como calidad de normalizacion.")

# Solo las columnas que comparan con la PLANTILLA. Filtrar por el prefijo "Dice " o
# "Jaccard " incluiria tambien "Jaccard GM func" y "Jaccard AFO", que comparan otras
# cosas, y la tabla llevaria un rotulo falso.
print("\nSolapamiento con las plantillas de los tres tejidos")
columnas_tejido = [c for c in solapamientos.columns if c.endswith(" anat")]
print(solapamientos[["Sujeto"] + columnas_tejido].to_string(index=False))

print("\nSolapamiento anatomico funcional")
columnas_func = [c for c in solapamientos.columns
                 if c in ("NORMfunc", "Jaccard GM func", "AFO", "Jaccard AFO")]
print(solapamientos[["Sujeto"] + columnas_func].to_string(index=False))

print(f"\nPlantilla de referencia: {CFG.plantilla} a 2 mm.")
print("Los valores absolutos NO son comparables con trabajos que usen otra plantilla.")
print("Recuerde tambien que el Dice del liquido cefalorraquideo es sistematicamente")
print("menor que el de los otros dos tejidos, y que eso es lo esperable: el liquido")
print("ocupa espacios finos y muy variables entre personas.")

solapamientos.to_csv(PATHS["reportes"] / "solapamientos.tsv", sep="\t", index=False)
print(f"Guardado en {PATHS['reportes'] / 'solapamientos.tsv'}")


**Cómo leer estos resultados.** Los valores absolutos importan menos que tres lecturas relativas.

**Comparación entre sujetos.** Es la lectura principal, por lo establecido en 4.1, no hay umbrales transferibles entre datasets. Un sujeto cuyo NORManat o NORMfunc quede claramente por debajo de los demás, con el mismo protocolo, señala un fallo de normalización que hay que confirmar en el informe visual de fMRIPrep. El criterio de valores extremos aplicable es el mismo de 4.6, con el matiz de que aquí el problema está siempre en los valores bajos.

**Comparación entre las tres medidas del mismo sujeto.** Es lo que permite localizar en qué etapa está el fallo, y es la razón de calcular las tres:

| Patrón observado | Interpretación |
|---|---|
| Las tres altas | Normalización y corregistro correctos |
| NORManat alta, NORMfunc baja, AFO baja | El anatómico se normalizó bien pero el corregistro entre modalidades falló, y arrastra al funcional |
| NORManat baja y NORMfunc baja, AFO alta | La normalización falló para ambas modalidades, pero el corregistro entre ellas es correcto: las dos están mal alineadas de la misma manera |
| NORManat baja, NORMfunc alta | Combinación anómala que sugiere revisar el procedimiento antes que los datos |

**Comparación entre tejidos.** Un solapamiento bajo en los tres tejidos apunta a un fallo global del registro. Un solapamiento bajo solo en sustancia gris, con blanca y líquido cefalorraquídeo correctos, apunta a un problema de segmentación y no de alineamiento.

**Qué valores esperar.** Con un registro correcto, los coeficientes de Dice de sustancia gris y blanca suelen situarse en la parte alta del rango, mientras que el líquido cefalorraquídeo da valores más bajos de forma sistemática, porque los ventrículos varían mucho entre individuos y son la estructura donde la plantilla representa peor a un sujeto concreto. Un Dice bajo en líquido cefalorraquídeo con los otros dos altos no es un fallo. El índice de Jaccard será siempre menor que el Dice correspondiente, por la relación monótona descrita en 8.1, y esa diferencia crece cuanto peor es el solapamiento.

**Las columnas de cobertura y por qué se añadieron.** La tabla incluye `Cobertura NORMfunc` y `Cobertura AFO`, que indican qué fracción del número de vóxeles objetivo se pudo alcanzar. Existen porque este dataset tiene cobertura parcial de campo de visión, según se estableció en 3.4, con 48 cortes de 2.4 mm el funcional abarca 115.2 mm y no cubre el vértex. Cuando la máscara funcional es menor que la de referencia, el procedimiento de igualar tamaños no puede completarse y el coeficiente de Dice sale deprimido por falta de campo, no por mal registro. Un valor de cobertura claramente inferior a uno invalida por tanto la lectura de la métrica correspondiente como calidad de normalización, y la celda lo avisa de forma explícita en lugar de recortar en silencio.

**Una limitación de la máscara de sustancia gris funcional.** La definición de referencia no precisa cómo obtenerla, y la aproximación empleada aquí, tomar los vóxeles de mayor intensidad media del funcional, tiene un sesgo que conviene conocer. Es cierto que en una secuencia potenciada en T2* la sustancia gris es más brillante que la blanca, pero el líquido cefalorraquídeo es más brillante todavía que la gris, de modo que el conjunto de los vóxeles más intensos estaría dominado por ventrículos y cisternas. Por eso la celda excluye primero los vóxeles con probabilidad de líquido superior a 0.30 e informa de cuántos elimina. Sigue siendo una aproximación y no equivale a una segmentación tisular del funcional.

**Buenas prácticas.** Declarar la plantilla y su resolución junto con los valores, porque definen la referencia. Reportar Dice o Jaccard indicando cuál, ya que se confunden con frecuencia. Interpretar estas cifras junto al informe visual y no en su lugar: cuantifican el solapamiento pero no distinguen un desalineamiento global de una distorsión local, que el ojo sí detecta.

**Errores frecuentes.** Comparar las máscaras sin igualar previamente su número de vóxeles, lo que produce valores bajos por construcción que se confunden con mal registro. Usar el mismo umbral de probabilidad para la plantilla y para el sujeto, que es un caso particular del error anterior. Umbralizar por valor en lugar de seleccionar por índice, que con mapas de probabilidad cuantizados devuelve más vóxeles de los pedidos y sesga el Dice al alza. Comparar valores absolutos con los de un artículo que empleó otra plantilla. Concluir que la normalización falló a partir de un Dice bajo en líquido cefalorraquídeo, que es lo esperable.


## Sección 9. Extracción de componentes de ruido

### 9.1 Qué es CompCor y por qué funciona

**Introducción conceptual.** Hace falta una estimación del ruido fisiológico y de movimiento que no esté contaminada por la señal de interés. Los parámetros de realineamiento describen el movimiento de la cabeza pero no capturan el ruido cardíaco ni el respiratorio. Sin monitorización fisiológica, que este dataset no tiene según se estableció en 2.8, hay que estimar ese ruido a partir de los propios datos.

**Fundamento metodológico.** CompCor (Behzadi et al., 2007) parte de una observación simple, existen regiones del cerebro donde no cabe esperar señal de origen neural, en particular la sustancia blanca profunda y el interior de los ventrículos. La señal que fluctúa en esas regiones es ruido, y como el ruido fisiológico es global y afecta a todo el volumen, esa estimación sirve para corregir también la sustancia gris. Extraer los componentes principales de esas regiones resume el ruido en unas pocas series temporales que pueden regresarse.

Hay dos variantes, y conviene no confundirlas:

| Variante | De dónde extrae los componentes | Ventaja | Limitación |
|---|---|---|---|
| **aCompCor**, anatómica | Máscaras de sustancia blanca y de líquido cefalorraquídeo, definidas anatómicamente y erosionadas | La región está definida a priori, sin mirar los datos, lo que evita seleccionar vóxeles por su comportamiento temporal | Depende de que la segmentación y el corregistro sean correctos |
| **tCompCor**, temporal | Los vóxeles de mayor variabilidad temporal, con independencia de en qué tejido estén | No necesita segmentación | Selecciona vóxeles **por su varianza**, y la señal neural también produce varianza, de modo que puede capturar efecto de interés |

**Por qué las máscaras se erosionan.** Una máscara de sustancia blanca obtenida de la segmentación incluye vóxeles del borde con la sustancia gris, que por efecto de volumen parcial contienen mezcla de ambos tejidos y por tanto algo de señal neural. Erosionar la máscara un vóxel elimina ese borde y deja solo el interior, donde la contaminación es mínima. Es un paso barato con un efecto directo sobre la validez del método, sin él, aCompCor captura señal de interés y el denoising la elimina.

**Un detalle de la definición de referencia que fMRIPrep no aplica.** Morfini et al. (2023) especifican que los componentes principales de sustancia blanca y líquido cefalorraquídeo se calculan **después de descontar los efectos de movimiento y de los volúmenes atípicos**, es decir en un espacio ortogonal a los parámetros de realineamiento y a los regresores de censurado. La razón es evitar redundancia, si los componentes ya contienen el efecto del movimiento, al regresarlos junto con los parámetros de movimiento se estaría modelando dos veces la misma cosa, gastando grados de libertad sin ganancia.

fMRIPrep calcula los componentes de CompCor **sin** esa ortogonalización previa, y este notebook los emplea tal como los entrega. La variante ortogonalizada no se implementa aquí. Es una diferencia real entre las dos implementaciones y no un detalle de estilo, y conviene declararla al comunicar resultados.

**Qué entrega fMRIPrep y cómo se lee.** La tabla de confounds contiene columnas `a_comp_cor_XX` y `t_comp_cor_XX` numeradas por orden decreciente de varianza explicada, junto con un archivo JSON de metadatos que indica, para cada componente, la máscara de la que procede y la fracción de varianza que explica. Ese metadato es lo que permite elegir cuántos componentes usar con un criterio, en lugar de fijar un número arbitrario.

**Cuántos componentes usar.** Hay dos criterios y ambos son defendibles. Un **número fijo**, habitualmente cinco por tejido, que es lo que hacen Morfini et al., tiene la virtud de gastar el mismo número de grados de libertad en todos los sujetos y mantener así la comparabilidad. Un **umbral de varianza explicada**, habitualmente el 50 por ciento, adapta el gasto a cuánto ruido tiene realmente cada sujeto, lo que Muschelli et al. (2014) defienden, pero hace que cada sujeto conserve un número distinto de grados de libertad. Este notebook calcula ambos y adopta el número fijo por defecto, por la razón de comparabilidad, documentando la alternativa.


In [ ]:
# 9.2  Extraccion de los componentes de ruido y su varianza explicada
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

N_COMPONENTES_POR_TEJIDO = 5  # criterio de numero fijo (Morfini et al., 2023)
UMBRAL_VARIANZA = 0.50  # umbral que fMRIPrep ya aplica al truncar
UMBRAL_INTERMEDIO = 0.25  # umbral mas exigente, si informativo

# fMRIPrep calcula aCompCor sobre tres mascaras y distingue el resultado por el
# PREFIJO del nombre de columna, no por un metadato: `a_comp_cor_` corresponde a la
# mascara combinada de sustancia blanca y liquido cefalorraquideo, `w_comp_cor_` a
# la de sustancia blanca sola y `c_comp_cor_` a la de liquido sola.
#
# Esto importa mucho y es una fuente conocida de error silencioso. Filtrar las
# columnas `a_comp_cor_` por un metadato de mascara igual a "WM" o "CSF" no
# devuelve nada, porque todas ellas llevan mascara combinada, y el resultado es una
# tabla de ceros y una estrategia de denoising sin regresores tisulares, sin que se
# produzca ninguna excepcion. Para no depender de la version de la herramienta, la
# funcion siguiente DETECTA que convencion esta presente en los datos y lo informa.
PREFIJOS_TEJIDO = {"WM": "w_comp_cor_", "CSF": "c_comp_cor_", "combined": "a_comp_cor_"}


def ordenar_por_indice(nombres):
    """Ordena los componentes por su indice NUMERICO, no alfabeticamente.

    La numeracion lleva relleno de dos digitos, de modo que si un conjunto alcanza
    cien componentes el indice pasa a tres digitos y la ordenacion alfabetica
    coloca `a_comp_cor_100` antes de `a_comp_cor_11`. Como los componentes vienen
    ordenados por varianza decreciente, eso invertiria el criterio de seleccion y
    haria que la varianza acumulada dejase de ser monotona. No es una precaucion
    teorica: en estos datos hay 108 y 109 componentes, de modo que los indices
    llegan efectivamente a tres cifras.
    """
    return sorted(nombres, key=lambda c: int(c.rsplit("_", 1)[1]))


def componentes_de_tejido(conf, meta, tejido):
    """Componentes aCompCor de un tejido, con la convencion presente en los datos.

    Devuelve (nombres, descripcion_de_la_convencion).
    """
    prefijos_separados = any(c.startswith(("w_comp_cor_", "c_comp_cor_"))
                             for c in conf.columns)
    if prefijos_separados:
        nombres = [c for c in conf.columns if c.startswith(PREFIJOS_TEJIDO[tejido])]
        return ordenar_por_indice(nombres), f"prefijo {PREFIJOS_TEJIDO[tejido]}"
    # Respaldo para versiones que emiten un unico prefijo y distinguen la mascara
    # en el metadato del JSON.
    nombres = [c for c in conf.columns
               if c.startswith("a_comp_cor_") and meta.get(c, {}).get("Mask") == tejido]
    return ordenar_por_indice(nombres), f"a_comp_cor_ con Mask igual a {tejido}"


componentes = {}
filas_resumen = []
convenciones = set()
metadato_ausente = []

for sujeto in CFG.id_sujetos:
    rutas = sorted(DIR_FMRIPREP.glob(
        f"{sujeto}/**/func/*_desc-confounds_timeseries.tsv"))
    if not rutas:
        print(f"   {sujeto}: sin tabla de confounds, se omite")
        continue
    conf = pd.read_csv(rutas[0], sep="\t")

    # El JSON acompanante describe cada componente: de que mascara procede y que
    # fraccion de varianza explica. Sin ese metadato no se puede elegir cuantos
    # componentes usar con un criterio, solo fijar un numero a ciegas.
    ruta_json = rutas[0].with_suffix(".json")
    meta = json.loads(ruta_json.read_text()) if ruta_json.exists() else {}

    acompcor_wm, conv_wm = componentes_de_tejido(conf, meta, "WM")
    acompcor_csf, conv_csf = componentes_de_tejido(conf, meta, "CSF")
    acompcor_combinado, _ = componentes_de_tejido(conf, meta, "combined")
    tcompcor = ordenar_por_indice([c for c in conf.columns
                                   if c.startswith("t_comp_cor_")])
    convenciones.update({conv_wm, conv_csf})

    if not acompcor_wm or not acompcor_csf:
        print(f"   {sujeto}: AVISO, no se han localizado componentes de "
              f"{'WM' if not acompcor_wm else 'CSF'}. Las estrategias con CompCor "
              f"de la Seccion 10 se quedarian sin regresores tisulares. "
              f"Prefijos de tipo comp_cor presentes: "
              f"{sorted({c.split('comp_cor')[0] + 'comp_cor_' for c in conf.columns if 'comp_cor' in c})}")

    def varianzas(nombres):
        """Varianza explicada de cada componente, tal como la declara el metadato."""
        return np.array([meta.get(c, {}).get("VarianceExplained", np.nan)
                         for c in nombres], dtype=float)

    def n_para_umbral(nombres, umbral):
        """Cuantos componentes hacen falta para alcanzar el umbral de varianza.

        Devuelve NaN si el metadato no esta disponible. La comprobacion se hace
        sobre los valores CRUDOS y no sobre la suma acumulada: `nancumsum` trata los
        ausentes como cero, de modo que una serie enteramente ausente produciria una
        curva de ceros que es finita y pasaria una comprobacion hecha sobre ella.
        """
        valores = varianzas(nombres)
        if valores.size == 0 or not np.isfinite(valores).any():
            return np.nan
        acumulada = np.nancumsum(valores)
        alcanzan = np.flatnonzero(acumulada >= umbral)
        return int(alcanzan[0] + 1) if alcanzan.size else len(nombres)

    def varianza_con_n(nombres, n):
        """Varianza acumulada por los primeros n componentes, en porcentaje."""
        valores = varianzas(nombres)[:n]
        if valores.size == 0 or not np.isfinite(valores).any():
            return np.nan
        return round(100.0 * float(np.nansum(valores)), 1)

    if acompcor_wm and not np.isfinite(varianzas(acompcor_wm)).any():
        metadato_ausente.append(sujeto)

    componentes[sujeto] = {
        "conf": conf, "meta": meta,
        "wm": acompcor_wm, "csf": acompcor_csf,
        "combinado": acompcor_combinado, "tcompcor": tcompcor,
    }

    filas_resumen.append({
        "Sujeto": sujeto.replace("sub-", ""),
        "n WM": len(acompcor_wm),
        "n CSF": len(acompcor_csf),
        "n combinado": len(acompcor_combinado),
        "n tCompCor": len(tcompcor),
        # Lo verdaderamente informativo: cuanta varianza capturan los componentes
        # que SI se van a usar. La pregunta inversa, cuantos hacen falta para el 50
        # por ciento, no informa aqui porque fMRIPrep ya trunca en ese umbral.
        f"Var. {N_COMPONENTES_POR_TEJIDO} WM (%)": varianza_con_n(acompcor_wm, N_COMPONENTES_POR_TEJIDO),
        f"Var. {N_COMPONENTES_POR_TEJIDO} CSF (%)": varianza_con_n(acompcor_csf, N_COMPONENTES_POR_TEJIDO),
        f"WM para {UMBRAL_INTERMEDIO:.0%}": n_para_umbral(acompcor_wm, UMBRAL_INTERMEDIO),
        f"CSF para {UMBRAL_INTERMEDIO:.0%}": n_para_umbral(acompcor_csf, UMBRAL_INTERMEDIO),
        f"WM para {UMBRAL_VARIANZA:.0%}": n_para_umbral(acompcor_wm, UMBRAL_VARIANZA),
    })

if not filas_resumen:
    raise RuntimeError("No hay tablas de confounds. Ejecute fMRIPrep (celda 5.3).")

resumen_componentes = pd.DataFrame(filas_resumen)
print("Componentes disponibles y varianza que explican")
print(resumen_componentes.to_string(index=False))
print(f"\nConvencion de nombres detectada en los datos: {', '.join(sorted(convenciones))}")
if metadato_ausente:
    print(f"AVISO: sin metadato de varianza explicada para "
          f"{', '.join(metadato_ausente)}. Las columnas de varianza apareceran vacias")
    print("y la seleccion de componentes queda reducida a fijar un numero a ciegas.")

print(f"\nCriterio adoptado: {N_COMPONENTES_POR_TEJIDO} componentes por tejido, por")
print("comparabilidad entre sujetos.")
print("\nPOR QUE LA COLUMNA DEL 50 POR CIENTO DEVUELVE EL TOTAL DE COMPONENTES.")
print("No es un fallo del calculo. fMRIPrep ya aplica ese criterio por su cuenta:")
print("retiene tantos componentes como hagan falta para explicar el 50 por ciento de")
print("la varianza y descarta el resto. La suma acumulada alcanza por tanto el umbral")
print("justo en el ultimo componente disponible, y volver a preguntarlo es redundante.")
print("La pregunta util es la inversa, que responden las columnas de varianza: cuanta")
print("varianza capturan los componentes que realmente se usan. Si esa cifra es baja,")
print("el numero fijo esta dejando fuera una parte importante del ruido estimado.")

# Visualizacion: varianza explicada acumulada y series de los primeros componentes
sujetos = list(componentes.keys())
fig, axes = plt.subplots(len(sujetos), 2, figsize=(13, 2.8 * len(sujetos)),
                         squeeze=False, gridspec_kw={"width_ratios": [1, 2]})

for fila, sujeto in enumerate(sujetos):
    datos = componentes[sujeto]
    meta = datos["meta"]

    # Varianza explicada acumulada por tipo de componente.
    ax = axes[fila][0]
    for etiqueta, nombres, color in (("WM", datos["wm"], "tab:orange"),
                                     ("CSF", datos["csf"], "tab:cyan"),
                                     ("tCompCor", datos["tcompcor"], "tab:red")):
        valores = np.array([meta.get(c, {}).get("VarianceExplained", np.nan)
                            for c in nombres], dtype=float)
        if valores.size and np.isfinite(valores).any():
            ax.plot(range(1, len(valores) + 1), np.nancumsum(valores), marker="o",
                    markersize=3, linewidth=1, color=color, label=etiqueta)
    ax.axhline(UMBRAL_VARIANZA, color="grey", linestyle=":", linewidth=0.8)
    ax.axvline(N_COMPONENTES_POR_TEJIDO, color="black", linestyle="--", linewidth=0.8)
    ax.set_xlim(0, 25)
    ax.set_ylim(0, 0.6)
    ax.set_xlabel("Numero de componentes")
    ax.set_ylabel(f"{sujeto.replace('sub-', '')}\nVarianza acumulada", fontsize=9)
    if fila == 0:
        ax.set_title("Varianza explicada acumulada:"
                     f"{N_COMPONENTES_POR_TEJIDO} componentes", fontsize=10, loc="left")
        ax.legend(fontsize=8, frameon=False)

    # Series temporales de los primeros componentes de cada tejido.
    ax = axes[fila][1]
    conf = datos["conf"]
    for etiqueta, nombres, color in (("WM 1", datos["wm"], "tab:orange"),
                                     ("CSF 1", datos["csf"], "tab:cyan"),
                                     ("tCompCor 1", datos["tcompcor"], "tab:red")):
        if nombres:
            serie = conf[nombres[0]].to_numpy()
            # Normalizacion a desviacion tipica unidad solo para poder superponer
            # series de escalas distintas en el mismo eje.
            ax.plot(serie / np.nanstd(serie), linewidth=0.7, color=color,
                    label=etiqueta, alpha=0.85)
    ax.set_xlabel("Volumen")
    ax.set_ylabel("Unidades tipicas")
    if fila == 0:
        ax.set_title("Primer componente de cada tipo, normalizado", fontsize=10, loc="left")
        ax.legend(fontsize=8, ncol=3, frameon=False)

plt.tight_layout()
if "guardar_figura" in globals():
    guardar_figura(fig, "componentes_de_ruido")
plt.show()


**Cómo leer los componentes de ruido.** Tres lecturas, y cada una responde a una pregunta distinta.

**La curva de varianza acumulada.** Su forma es en sí misma un diagnóstico. Una curva empinada, que alcanza el 50 por ciento con dos o tres componentes, indica que el ruido está dominado por una o dos fuentes globales, típicamente respiración o un evento de movimiento sostenido. Una curva tendida, que necesita diez o más componentes, indica ruido repartido entre muchas fuentes independientes. Esa diferencia importa porque un número fijo de cinco componentes captura casi todo el ruido en el primer caso y una fracción modesta en el segundo, y por tanto la misma estrategia de denoising no es igualmente eficaz en los dos sujetos. Comparar las curvas entre sujetos es lo que revela esa desigualdad, que un número único no muestra.

**Las series temporales de los primeros componentes.** Lo que hay que buscar es su relación con lo ya conocido de las Secciones 4 y 7. Un primer componente de sustancia blanca o de líquido cefalorraquídeo que presente picos coincidentes con los del desplazamiento de encuadre está capturando movimiento, lo cual es correcto pero redundante con los parámetros de realineamiento, y es justamente el argumento a favor de la ortogonalización previa que se discutió en 9.1. Un componente con oscilación regular en torno a 0.2 o 0.3 Hz está capturando respiración, que es exactamente lo que se busca. Un componente de líquido cefalorraquídeo con oscilación más rápida captura pulsación cardíaca, coherente con lo que mostraron los mapas espaciales de 4.4b, donde la línea media y los ventrículos aparecían brillantes.

**La comparación entre aCompCor y tCompCor.** Si el primer componente temporal se parece mucho a los anatómicos, tCompCor está seleccionando vóxeles de las mismas regiones y no aporta información nueva. Si difiere sustancialmente, conviene preguntarse de dónde procede: puede estar capturando una fuente de ruido que las máscaras anatómicas no cubren, lo cual es su virtud, o puede estar capturando señal neural de regiones de alta varianza, que es su riesgo principal. La forma de distinguirlo es mirar el panel de máscaras del informe de fMRIPrep y comprobar si la región de la que tCompCor extrae se superpone con corteza, que es el criterio X de Provins et al. (2023).

**Una comprobación que la celda hace y conviene entender.** La salida informa de qué convención de nombres encontró en los datos. No es un detalle administrativo: fMRIPrep distingue las tres máscaras de aCompCor por el **prefijo** de la columna y no por un metadato. Los componentes de la máscara combinada se llaman `a_comp_cor_`, los de sustancia blanca `w_comp_cor_` y los de líquido cefalorraquídeo `c_comp_cor_`. Buscar componentes de sustancia blanca entre las columnas `a_comp_cor_` filtrando por un metadato de máscara no devuelve nada, porque todas ellas pertenecen a la máscara combinada, y el resultado sería un recuento de cero y unas estrategias de denoising sin regresores tisulares, todo ello sin producir ningún error. Es un fallo silencioso que este notebook cometió y corrigió, y la comprobación explícita existe para que no vuelva a pasar si cambia la versión de la herramienta.

**Buenas prácticas.** Declarar cuántos componentes se usaron y con qué criterio, porque el número afecta directamente a los grados de libertad. Verificar en el informe de fMRIPrep que las máscaras de las que se extraen los componentes no invaden corteza. Usar el metadato de varianza explicada en lugar de fijar un número a ciegas, aunque después se decida fijarlo por comparabilidad.

**Errores frecuentes.** Usar aCompCor sin comprobar la segmentación, con lo que se regresa señal neural y se elimina efecto de interés. **Combinar aCompCor con regresión de señal global** sin advertir que ambas capturan varianza compartida y que el solapamiento gasta grados de libertad sin ganancia proporcional. Suponer que más componentes es siempre mejor, cada uno cuesta un grado de libertad, y la Sección 11 muestra el efecto acumulado de ese coste. Y el ya descrito de suponer los nombres de las columnas en lugar de comprobarlos.


## Sección 10. Denoising y comparación de estrategias

### 10.1 Las estrategias, los regresores y un error que hay que evitar

**Introducción conceptual.** Llegados aquí se dispone de la serie preprocesada y de una tabla de confounds que aún no se ha aplicado todavía. El denoising consiste en decidir qué confounds regresar, aplicar esa regresión, y filtrar en la banda de frecuencias de interés. No existe una respuesta única, cada estrategia intercambia reducción de sesgo por pérdida de grados de libertad, y la elección debe justificarse y documentarse.

**Fundamento metodológico.** Ciric et al. (2017) compararon sistemáticamente estrategias de regresión de confounds y establecieron el marco que se sigue aquí, ninguna estrategia domina en todos los criterios, y la comparación debe hacerse con métricas que midan cosas distintas, en particular la eliminación del sesgo por movimiento y la preservación de la señal. Parkes et al. (2018) llegaron a conclusiones convergentes.

**Las cuatro estrategias que se comparan**, en orden creciente de agresividad:

| Estrategia | Regresores | Qué añade |
|---|---|---|
| Solo movimiento | 24 parámetros de movimiento, tendencia y constante | La corrección mínima defendible |
| Movimiento más CompCor | Lo anterior más 5 componentes de sustancia blanca y 5 de líquido cefalorraquídeo | Ruido fisiológico, que el movimiento no captura |
| Más censurado | Lo anterior, eliminando los volúmenes atípicos | Los volúmenes que ninguna regresión puede reparar |
| Más regresión de señal global | Lo anterior más la señal media de todo el cerebro | El ruido compartido por todo el volumen |

**Los 24 parámetros de movimiento.** Son los 6 parámetros de realineamiento, sus 6 derivadas temporales, y los cuadrados de esos 12. La justificación de incluir derivadas y cuadrados, que procede de Friston et al. (1996), es que el efecto del movimiento sobre la señal no es instantáneo ni lineal, la magnetización tarda en recuperarse, de modo que el movimiento de un volumen afecta también a los siguientes, y ese efecto no se describe bien con una relación lineal. Se comparan además las variantes de 6 y 12 parámetros para mostrar el coste y el beneficio de cada nivel.

**El error que hay que evitar, y que tiene nombre propio.** Hallquist et al. (2013) documentaron un problema que se comete con frecuencia y cuyo título describe bien la situación, la molestia de la regresión de ruidos. Si se regresan los confounds y **después** se filtra en banda, el filtrado reintroduce parte del ruido que la regresión había eliminado. El mecanismo es que la regresión deja residuos cuyo contenido espectral difiere del de la serie original, y el filtro aplicado a esos residuos no equivale a haber filtrado y regresado a la vez. La consecuencia práctica es que la serie resultante contiene ruido correlacionado con el movimiento que se creía eliminado.

La solución es aplicar **la regresión y el filtrado de forma simultánea**, filtrando también los propios regresores antes de la regresión, de modo que el modelo y los datos ocupen la misma banda. Es lo que hace `nilearn.signal.clean` cuando se le pasan los confounds y la banda en la misma llamada, y es la razón por la que este notebook no encadena las dos operaciones por separado.

**Otro error común.** Censurar los volúmenes atípicos y a la vez incluir regresores de picos para esos mismos volúmenes es redundante. Al eliminar un volumen, su regresor de pico queda idénticamente nulo y no explica nada, pero sigue contando como parámetro estimado y por tanto consume un grado de libertad. El resultado es que el mismo coste se paga dos veces. Las dos opciones son legítimas por separado y se tratan como estrategias distintas, censurar elimina los volúmenes, y los regresores de picos los modelan sin eliminarlos, lo que conserva la longitud de la serie a cambio de gastar un parámetro por volumen marcado.

**La banda de frecuencias.** De 0.008 a 0.09 Hz, valores fijados en la configuración y coherentes con Morfini et al. (2023). Con TR de 1.15 s la frecuencia de Nyquist es de 0.435 Hz, de modo que la banda retenida abarca alrededor del 19 por ciento del ancho disponible. Ese porcentaje es exactamente lo que determina la pérdida de grados de libertad por el filtrado, y es la razón por la que el tSNR calculado tras filtrar resulta engañoso, según se demostró en 4.4.

**Sobre la regresión de señal global.** Es la opción más discutida de la lista. Reduce eficazmente la varianza compartida por todo el volumen, que en buena parte es no neural, pero introduce un sesgo negativo en todas las correlaciones y genera anticorrelaciones cuya interpretación se debate desde hace años. Kumar et al. (2024) la clasifican como opcional en su consenso. Se incluye en la comparación para que el efecto sea visible, y no se adopta por defecto.


In [ ]:
# 10.2  Aplicacion de las estrategias de denoising y su evaluacion
import numpy as np
import nibabel as nib
import pandas as pd
from scipy.signal import butter, filtfilt

# La limpieza se implementa aqui de forma explicita en lugar de delegarla a
# `nilearn.signal.clean`, por dos razones. La primera es didactica: el paso critico
# es la simultaneidad entre filtrado y regresion, y verlo escrito evita el error
# que describen Hallquist et al. (2013). La segunda es practica: la version de
# nilearn preinstalada en Colab exige una version de pandas distinta de la que el
# kernel tiene cargada, y forzarla obligaria a reiniciar el entorno de ejecucion.

N_ACOMPCOR = 5          # componentes por tejido
BANDA = CFG.filtro_hz
N_NODOS = 1000          # voxeles del grafo de conectividad, ver 11.2
BLOQUE_VOXELES = 20000  # tamano de bloque para acotar el pico de memoria


def filtrar_banda(matriz, tr, banda, orden=2):
    """Filtro paso banda a lo largo del ultimo eje.

    El relleno de bordes por defecto de `filtfilt` son 15 muestras, muy inferiores
    al periodo del corte inferior: a 0.008 Hz el periodo es de 125 s, unas 109
    muestras con TR de 1.15 s. Con el relleno por defecto quedarian transitorios
    apreciables en los primeros y ultimos volumenes, asi que se dimensiona a tres
    veces el periodo del corte inferior, acotado por la longitud de la serie.
    """
    nyquist = 0.5 / tr
    b, a = butter(orden, [banda[0] / nyquist, banda[1] / nyquist], btype="band")
    relleno = min(int(3.0 / (banda[0] * tr)), matriz.shape[-1] - 1)
    filtrada = filtfilt(b, a, matriz, axis=-1, padlen=relleno)
    # `filtfilt` promociona a float64 porque los coeficientes lo son. Sin volver a
    # float32 el consumo de memoria se duplica en cada paso.
    return filtrada.astype(np.float32, copy=False)


def regresar_por_bloques(serie_f, diseno, bloque=BLOQUE_VOXELES):
    """Residuos de la regresion, calculados por bloques de voxeles.

    Se usa la pseudoinversa, equivalente a minimos cuadrados y estable frente a
    la colinealidad que aparece con 24 parametros de movimiento mas componentes de
    CompCor. El calculo se trocea porque la matriz de valores ajustados completa,
    de cientos de miles de voxeles por cientos de volumenes en doble precision,
    multiplicaria el pico de memoria sin necesidad.
    """
    proyector = np.linalg.pinv(diseno).astype(np.float32)      # (p, T)
    residuos = np.empty_like(serie_f)
    for inicio in range(0, serie_f.shape[0], bloque):
        fin = min(inicio + bloque, serie_f.shape[0])
        y = serie_f[inicio:fin].T                              # (T, v)
        residuos[inicio:fin] = (y - diseno @ (proyector @ y)).T
    return residuos


def limpiar(serie_filtrada, regresores, tr, banda, volumenes_validos=None):
    """Regresion de confounds sobre una serie YA filtrada en banda.

    El orden importa y no es intercambiable. Regresar primero y filtrar despues
    reintroduce ruido correlacionado con los regresores, porque los residuos de la
    regresion tienen un contenido espectral distinto del de la serie original y el
    filtro aplicado sobre ellos no equivale a haber hecho las dos cosas a la vez.
    La forma correcta es filtrar datos Y regresores en la misma banda, y regresar
    despues, de modo que modelo y datos ocupen el mismo espacio de frecuencias.

    La serie llega ya filtrada porque el filtrado se hace una sola vez por sujeto
    y se reutiliza en las siete estrategias; los regresores si se filtran aqui,
    que es barato porque son pocas columnas.

    `volumenes_validos` implementa el censurado: los volumenes marcados se
    eliminan de la estimacion. Se filtra ANTES de censurar, porque un filtro sobre
    una serie con huecos no esta definido.
    """
    reg_f = filtrar_banda(regresores.T, tr, banda).T

    if volumenes_validos is not None:
        serie_filtrada = serie_filtrada[:, volumenes_validos]
        reg_f = reg_f[volumenes_validos, :]

    # Termino constante para no forzar el paso por el origen.
    diseno = np.column_stack([reg_f, np.ones(reg_f.shape[0], dtype=np.float32)])
    return regresar_por_bloques(serie_filtrada, diseno), diseno.shape[1]


def construir_regresores(conf, comp_tejido, estrategia, atipicos):
    """Matriz de regresores de una estrategia. Devuelve (matriz, nombres)."""
    columnas = []

    mov6 = ["trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]
    if estrategia["movimiento"] == 6:
        columnas += mov6
    elif estrategia["movimiento"] == 12:
        columnas += mov6 + [f"{c}_derivative1" for c in mov6]
    elif estrategia["movimiento"] == 24:
        columnas += mov6 + [f"{c}_derivative1" for c in mov6]
        columnas += [f"{c}_power2" for c in mov6]
        columnas += [f"{c}_derivative1_power2" for c in mov6]

    if estrategia["compcor"]:
        # Los nombres de los componentes vienen ya resueltos por la celda 9.2, que
        # detecta la convencion de nombres presente en los datos y los ordena por
        # indice numerico. Repetir aqui el filtrado seria arriesgado: fMRIPrep
        # distingue las tres mascaras de aCompCor por el PREFIJO de la columna y no
        # por un metadato, de modo que buscar `a_comp_cor_` con mascara igual a "WM"
        # no devuelve nada y la estrategia se quedaria sin regresores tisulares sin
        # producir ninguna excepcion.
        columnas += comp_tejido["wm"][:N_ACOMPCOR]
        columnas += comp_tejido["csf"][:N_ACOMPCOR]

    if estrategia["senal_global"]:
        columnas.append("global_signal")

    presentes = [c for c in columnas if c in conf.columns]
    matriz = conf[presentes].fillna(0.0).to_numpy(dtype=np.float32)

    # Regresores de picos: modelan los volumenes atipicos SIN eliminarlos. No se
    # combinan con el censurado, porque seria redundante: al eliminar un volumen su
    # regresor queda identicamente nulo y aun asi consume un grado de libertad.
    #
    # Se construyen a partir del criterio propio de 7.2 y NO de las columnas
    # `motion_outlier` de fMRIPrep. La razon es que esas columnas usan los umbrales
    # por defecto de fMRIPrep, con la definicion de FD de Power, mientras que el
    # censurado de este notebook usa la definicion de caja envolvente y el umbral
    # de `CFG.umbral_fd_mm`, mas el criterio de cambio de senal global. Mezclarlos
    # marcaria conjuntos de volumenes distintos, y entonces las estrategias con
    # picos y con censurado no serian dos formas de tratar el MISMO problema, que
    # es justamente lo que la comparacion pretende aislar.
    if estrategia["picos"] and atipicos.any():
        picos = np.eye(len(atipicos), dtype=np.float32)[:, np.flatnonzero(atipicos)]
        matriz = np.column_stack([matriz, picos]) if matriz.size else picos
        presentes = presentes + [f"pico_{i:03d}" for i in np.flatnonzero(atipicos)]

    return matriz, presentes


ESTRATEGIAS = {
    "6HMP": dict(movimiento=6, compcor=False, censurar=False, senal_global=False, picos=False),
    "12HMP": dict(movimiento=12, compcor=False, censurar=False, senal_global=False, picos=False),
    "24HMP": dict(movimiento=24, compcor=False, censurar=False, senal_global=False, picos=False),
    "24HMP+CompCor": dict(movimiento=24, compcor=True, censurar=False, senal_global=False, picos=False),
    "24HMP+CompCor+Picos": dict(movimiento=24, compcor=True, censurar=False, senal_global=False, picos=True),
    "24HMP+CompCor+Censurado": dict(movimiento=24, compcor=True, censurar=True, senal_global=False, picos=False),
    "24HMP+CompCor+Censurado+GSR": dict(movimiento=24, compcor=True, censurar=True, senal_global=True, picos=False),
}

mascara_comun = np.asanyarray(nib.load(RUTA_INTERSECCION).dataobj).astype(bool)
print(f"Evaluacion restringida a la interseccion de mascaras: "
      f"{int(mascara_comun.sum())} voxeles")

# Nodos del grafo de conectividad. La definicion de referencia los toma de la
# mascara de sustancia gris de la plantilla, no del cerebro completo: incluir
# sustancia blanca y liquido cefalorraquideo diluye la distribucion de
# conectividad y desplaza GCOR. Si la plantilla no esta disponible o su rejilla no
# coincide con la de los datos, se recurre a la interseccion y se avisa.
mascara_nodos = mascara_comun
origen_nodos = "interseccion funcional (plantilla de GM no disponible)"
try:
    import templateflow.api as tflow
    ruta_gm = tflow.get(CFG.plantilla, resolution=2, label="GM", suffix="probseg",
                        extension=".nii.gz")
    gm = np.asanyarray(nib.load(str(ruta_gm[0] if isinstance(ruta_gm, list) else ruta_gm)
                                ).dataobj)
    if gm.shape == mascara_comun.shape:
        mascara_nodos = mascara_comun & (gm > 0.5)
        origen_nodos = "plantilla de sustancia gris al 50 por ciento"
    else:
        origen_nodos = (f"interseccion funcional (rejillas distintas: "
                        f"{gm.shape} frente a {mascara_comun.shape})")
except Exception as exc:
    origen_nodos = f"interseccion funcional ({type(exc).__name__})"

print(f"Nodos del grafo tomados de: {origen_nodos}")
print(f"Voxeles candidatos a nodo: {int(mascara_nodos.sum())}\n")

filas = []
series_nodos = {}     # solo los nodos del grafo, no las series completas

for sujeto in CFG.id_sujetos:
    ruta_bold = sorted(DIR_FMRIPREP.glob(
        f"{sujeto}/**/func/*space-{CFG.plantilla}*_desc-preproc_bold.nii.gz"))
    if not ruta_bold or sujeto not in componentes:
        print(f"   {sujeto}: faltan productos, se omite")
        continue

    img = nib.load(ruta_bold[0])
    tr = float(img.header.get_zooms()[3])
    serie = np.asanyarray(img.dataobj, dtype=np.float32)[mascara_comun]   # (V, T)
    n_vol = serie.shape[1]

    # Media global de la serie SIN filtrar. Hace falta para el escalado a media
    # global 100 de BOLDstd, y debe tomarse antes del paso banda porque este
    # elimina la componente continua y deja la media en cero.
    media_global = float(serie.mean())

    comp_tejido = componentes[sujeto]
    conf = comp_tejido["conf"]
    s_qc = series_qc[sujeto]
    atipicos = s_qc["atipico"].to_numpy()[:n_vol]
    validos = ~atipicos
    fd = s_qc["fd_caja"].to_numpy()[:n_vol]

    # Posiciones de los nodos DENTRO del vector de voxeles de la interseccion.
    nodos_dentro = np.flatnonzero(mascara_nodos[mascara_comun])
    rng = np.random.default_rng(CFG.semilla)
    posiciones_nodos = np.sort(rng.choice(nodos_dentro,
                                          size=min(N_NODOS, nodos_dentro.size),
                                          replace=False))
    # Submuestra independiente para GCOR, que por definicion se calcula sobre el
    # cerebro completo y no solo sobre sustancia gris.
    posiciones_gcor = rng.choice(serie.shape[0],
                                 size=min(1000, serie.shape[0]), replace=False)

    # Filtrado una sola vez por sujeto, reutilizado por las siete estrategias.
    serie_filtrada = filtrar_banda(serie, tr, BANDA)
    del serie

    for nombre, estrategia in ESTRATEGIAS.items():
        regresores, usados = construir_regresores(conf, comp_tejido, estrategia, atipicos)
        if regresores.size == 0:
            print(f"   {sujeto} / {nombre}: sin regresores disponibles, se omite")
            continue
        indice = validos if estrategia["censurar"] else None
        limpia, n_parametros = limpiar(serie_filtrada, regresores, tr, BANDA, indice)

        # Grados de libertad efectivos: volumenes usados menos parametros
        # estimados, multiplicado por la fraccion de la banda de Nyquist que el
        # filtro conserva. Es la definicion de Morfini et al. (2023).
        n_usados = int(indice.sum()) if indice is not None else n_vol
        fraccion_banda = (BANDA[1] - BANDA[0]) / (0.5 / tr)
        dof = (n_usados - n_parametros) * fraccion_banda

        # Correlacion residual entre movimiento y cambio de senal. Solo se usan
        # pares de volumenes ADYACENTES: tras censurar, dos volumenes consecutivos
        # en la serie recortada pueden no serlo en la original, y compararlos
        # produciria una diferencia espuria. Ese error se cometio en una version
        # anterior de este trabajo.
        indices = np.flatnonzero(indice) if indice is not None else np.arange(n_vol)
        adyacentes = np.flatnonzero(np.diff(indices) == 1)
        suficientes = adyacentes.size > 2

        # Referencia SIN regresion de confounds, calculada con la MISMA banda y el
        # MISMO subconjunto de pares que la serie limpia. Es imprescindible para
        # que la comparacion sea honesta: el paso banda por si solo reduce la
        # correlacion con un FD de banda ancha, y censurar excluye del calculo
        # precisamente los volumenes de FD alto. Si la referencia se tomase de la
        # serie sin filtrar y sobre todos los pares, buena parte de la mejora
        # aparente no seria atribuible a la regresion, que es lo que se quiere medir.
        base = serie_filtrada[:, indices] if indice is not None else serie_filtrada
        dvars_ref = np.sqrt((np.diff(base, axis=1) ** 2).mean(axis=0))
        corr_referencia = (float(np.corrcoef(fd[indices[adyacentes + 1]],
                                            dvars_ref[adyacentes])[0, 1])
                           if suficientes else np.nan)

        dvars_res = np.sqrt((np.diff(limpia, axis=1) ** 2).mean(axis=0))
        corr_residual = (float(np.corrcoef(fd[indices[adyacentes + 1]],
                                           dvars_res[adyacentes])[0, 1])
                         if suficientes else np.nan)

        # GCOR sobre una submuestra de voxeles: la matriz de correlacion completa
        # de cientos de miles de voxeles no cabe en memoria.
        z = limpia[posiciones_gcor] - limpia[posiciones_gcor].mean(axis=1, keepdims=True)
        z /= np.maximum(np.linalg.norm(z, axis=1, keepdims=True), 1e-12)
        matriz_corr = z @ z.T
        gcor = float((matriz_corr.sum() - np.trace(matriz_corr))
                     / (len(posiciones_gcor) * (len(posiciones_gcor) - 1)))

        # BOLDstd con escalado a media global 100, segun la definicion de
        # referencia. Sin ese escalado la metrica mide sobre todo la ganancia del
        # receptor, porque las salidas de fMRIPrep estan en unidades arbitrarias
        # del escaner, y deja de ser comparable entre sujetos.
        boldstd = float(np.mean((limpia / media_global * 100.0).std(axis=1)))

        filas.append({
            "Sujeto": sujeto.replace("sub-", ""),
            "Estrategia": nombre,
            "Regresores": n_parametros,
            "Volumenes usados": n_usados,
            "DOF efectivos": round(dof, 1),
            "corr FD-DVARS antes": round(corr_referencia, 3) if suficientes else np.nan,
            "corr FD-DVARS despues": round(corr_residual, 3) if suficientes else np.nan,
            "GCOR": round(gcor, 4),
            "BOLDstd": round(boldstd, 4),
        })
        # Se guardan SOLO los nodos del grafo. Guardar las 21 series completas
        # ocuparia entre 7 y 12 GB y agotaria la memoria del entorno.
        series_nodos[(sujeto, nombre)] = np.ascontiguousarray(limpia[posiciones_nodos])
        del limpia

    # Coordenadas de los nodos en milimetros, necesarias en 11.2 para la
    # dependencia de la distancia. Se guardan una vez, con la afin de la imagen.
    if "coordenadas_nodos" not in globals():
        indices_3d = np.array(np.nonzero(mascara_comun))          # (3, V)
        voxeles_nodo = indices_3d[:, posiciones_nodos].T          # (n, 3)
        coordenadas_nodos = nib.affines.apply_affine(img.affine, voxeles_nodo)

    del serie_filtrada

if not filas:
    raise RuntimeError("No hay datos preprocesados. Ejecute fMRIPrep (celda 5.3).")

comparacion_denoising = pd.DataFrame(filas)
print("Comparacion de estrategias de denoising")
print(comparacion_denoising.to_string(index=False))

comparacion_denoising.to_csv(PATHS["reportes"] / "comparacion_denoising.tsv",
                             sep="\t", index=False)
print(f"\nGuardado en {PATHS['reportes'] / 'comparacion_denoising.tsv'}")
print(f"Banda: {BANDA[0]} a {BANDA[1]} Hz. Fraccion de la banda de Nyquist "
      f"conservada: {(BANDA[1] - BANDA[0]) / (0.5 / tr):.1%}")
print(f"Nodos guardados para la Seccion 11: {len(posiciones_nodos)} por sujeto "
      f"y estrategia, tomados de {origen_nodos}.")


**Lo que costó.** `Regresores` es el número de columnas del modelo, incluido el término constante y, en las estrategias con picos, un regresor por volumen marcado. `Volumenes usados` baja solo en las estrategias con censurado. `DOF efectivos` combina ambas cosas con la fracción de la banda de frecuencias que el filtro conserva, y es la cifra que de verdad limita cualquier estimación posterior.

**Lo que se consiguió.** `corr FD-DVARS antes` y `corr FD-DVARS despues` miden si el movimiento sigue explicando la variación de la señal. Es importante entender qué es exactamente la columna del antes, no es la serie cruda, sino la misma serie filtrada en la misma banda y evaluada sobre los mismos pares de volúmenes, con la única diferencia de que no se ha regresado ninguna variable de confusión. Sin esa equivalencia la comparación no mediría el efecto de la regresión sino el del filtro.

**Lo que queda.** `GCOR` es la correlación media entre todos los pares de vóxeles de una submuestra, y describe cuánta estructura global permanece. `BOLDstd` es la desviación típica temporal de la señal tras escalarla a media global cien, y tiene dos extremos malos, alta indica ruido residual, muy baja indica que se eliminó también la señal.


### 10.2 Comparación visual de las estrategias

**Qué añade sobre la tabla.** La tabla permite comparar valores, pero no deja ver dos cosas que importan para decidir, si los sujetos se comportan igual entre sí, y cómo se relaciona lo que cada estrategia gana con lo que cuesta. Las cuatro vistas siguientes responden a eso.

El panel A muestra el movimiento residual por estrategia, con una línea por sujeto y la referencia marcada en horizontal. El panel B muestra los grados de libertad, es decir el precio. El panel C muestra la correlación global. Y el panel D los cruza, cada punto es una combinación de sujeto y estrategia, situada según lo que conserva en el eje horizontal y lo que le queda de artefacto en el vertical.

**Antes de mirar las figuras conviene saber qué se está buscando**, porque la tentación natural es elegir la estrategia que mejor puntúa en el panel A y esa es precisamente la trampa, la estrategia más agresiva casi siempre gana ahí, y no por ser mejor sino por construcción. Por eso la celda calcula además si las diferencias observadas son mayores que el error con el que se estiman, y por eso el panel D es el que decide.


### 10.3  Visualizacion comparativa de las estrategias de denoising


In [ ]:
# 10.3  Visualizacion comparativa de las estrategias de denoising
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

orden_estrategias = [e for e in ESTRATEGIAS
                     if e in set(comparacion_denoising["Estrategia"])]
sujetos_den = sorted(comparacion_denoising["Sujeto"].unique())

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Panel A: correlacion residual entre movimiento y cambio de senal.
ax = axes[0][0]
for sujeto in sujetos_den:
    sub = comparacion_denoising[comparacion_denoising["Sujeto"] == sujeto]
    sub = sub.set_index("Estrategia").reindex(orden_estrategias)
    ax.plot(range(len(orden_estrategias)), sub["corr FD-DVARS despues"],
            marker="o", markersize=4, linewidth=1, label=sujeto)
    ax.axhline(sub["corr FD-DVARS antes"].iloc[0], linestyle=":", linewidth=0.7,
               alpha=0.5)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(range(len(orden_estrategias)))
ax.set_xticklabels(orden_estrategias, rotation=35, ha="right", fontsize=7)
ax.set_ylabel("corr(FD, DVARS) residual")
ax.set_title("A. Movimiento residual", fontsize=10, loc="left")
ax.legend(fontsize=8, frameon=False)

# Panel B: grados de libertad efectivos.
ax = axes[0][1]
for sujeto in sujetos_den:
    sub = comparacion_denoising[comparacion_denoising["Sujeto"] == sujeto]
    sub = sub.set_index("Estrategia").reindex(orden_estrategias)
    ax.plot(range(len(orden_estrategias)), sub["DOF efectivos"],
            marker="o", markersize=4, linewidth=1, label=sujeto)
ax.set_xticks(range(len(orden_estrategias)))
ax.set_xticklabels(orden_estrategias, rotation=35, ha="right", fontsize=7)
ax.set_ylabel("Grados de libertad efectivos")
ax.set_title("B. Coste de cada estrategia", fontsize=10, loc="left")

# Panel C: correlacion global.
ax = axes[1][0]
for sujeto in sujetos_den:
    sub = comparacion_denoising[comparacion_denoising["Sujeto"] == sujeto]
    sub = sub.set_index("Estrategia").reindex(orden_estrategias)
    ax.plot(range(len(orden_estrategias)), sub["GCOR"],
            marker="o", markersize=4, linewidth=1, label=sujeto)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(range(len(orden_estrategias)))
ax.set_xticklabels(orden_estrategias, rotation=35, ha="right", fontsize=7)
ax.set_ylabel("GCOR")
ax.set_title("C. Correlacion global", fontsize=10, loc="left")

# Panel D: beneficio frente a coste.
ax = axes[1][1]
marcadores = ["o", "s", "^", "D", "v", "P", "X"]
for i, estrategia in enumerate(orden_estrategias):
    sub = comparacion_denoising[comparacion_denoising["Estrategia"] == estrategia]
    ax.scatter(sub["DOF efectivos"], sub["corr FD-DVARS despues"].abs(),
               marker=marcadores[i % len(marcadores)], s=45, label=estrategia)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Grados de libertad efectivos")
ax.set_ylabel("Movimiento residual (abs)")
ax.set_title("D. Beneficio frente a coste", fontsize=10, loc="left")
ax.legend(fontsize=7, frameon=False, loc="upper left")

plt.tight_layout()
if "guardar_figura" in globals():
    guardar_figura(fig, "comparacion_denoising")
plt.show()

resumen = (comparacion_denoising.groupby("Estrategia")
           .agg(residual=("corr FD-DVARS despues", lambda s: float(np.abs(s).mean())),
                referencia=("corr FD-DVARS antes", lambda s: float(np.abs(s).mean())),
                dof=("DOF efectivos", "mean"),
                volumenes=("Volumenes usados", "mean"))
           .reindex(orden_estrategias))

print("Resumen promediado entre sujetos")
print(resumen.round(4).to_string())

# Comprobacion previa a cualquier recomendacion: puede el criterio distinguir?
#
# Una correlacion estimada sobre n pares tiene un error tipico aproximado de
# 1 / raiz(n - 3). Si la dispersion entre estrategias es menor que ese error, las
# diferencias observadas son indistinguibles del ruido de muestreo y ordenar las
# estrategias por ese valor equivale a ordenarlas al azar.
n_pares = float(resumen["volumenes"].mean()) - 1.0
error_tipico = 1.0 / np.sqrt(max(n_pares - 3.0, 1.0))
dispersion = float(resumen["residual"].max() - resumen["residual"].min())
discrimina = dispersion > 2.0 * error_tipico

dof_maximo = resumen["dof"].max()
admisibles = resumen[resumen["dof"] >= 0.5 * dof_maximo]
hay_criterio = (discrimina and not admisibles.empty
                and admisibles["residual"].notna().any())
recomendada = admisibles["residual"].idxmin() if hay_criterio else None

print(f"\nCapacidad de discriminacion del criterio")
print(f"   pares por estimacion:                       {n_pares:.0f}")
print(f"   error tipico aproximado de una correlacion: {error_tipico:.3f}")
print(f"   dispersion observada entre estrategias:     {dispersion:.3f}")
print(f"   veredicto:                                  "
      f"{'discrimina' if discrimina else 'NO DISCRIMINA'}")

print(f"\nGrados de libertad efectivos: de {resumen['dof'].min():.1f} a "
      f"{resumen['dof'].max():.1f} sobre {resumen['volumenes'].mean():.0f} volumenes")
print(f"Umbral de admisibilidad (mitad del maximo): {0.5 * dof_maximo:.1f}")
print(f"Estrategias admisibles: {', '.join(admisibles.index)}")
print(f"\nRecomendacion: {recomendada if recomendada else 'ninguna, ver el texto siguiente'}")


**Cómo decidir la estrategia.** Cuatro criterios, y ninguno decide por sí solo.

**La correlación residual entre movimiento y cambio de señal es el criterio principal**, porque mide directamente lo que se quiere eliminar, si el movimiento sigue explicando la variación de la señal, el sesgo que documentaron Power et al. (2012) sigue presente y cualquier estimación de conectividad lo arrastrará. El objetivo es acercarse a cero. Un valor que se mantiene alto después de aCompCor indica movimiento severo que la regresión no puede reparar, y ahí la respuesta correcta es censurar o excluir al sujeto, no añadir más regresores.

**Los grados de libertad efectivos son el coste.** Con unos 400 volúmenes, decenas de regresores y una banda que conserva alrededor del 19 por ciento del ancho disponible, la cifra resultante es mucho menor de lo que sugiere el número de volúmenes. Un sujeto con grados de libertad muy inferiores a los del resto deja de ser comparable con el grupo, aunque sus otras métricas parezcan buenas.

**La correlación global debe bajar, pero con una advertencia importante.** Un valor cercano a cero después de regresión de señal global no demuestra que se haya eliminado ruido: la regresión de la señal media fuerza matemáticamente ese resultado, con independencia de que el ruido siga ahí en forma no global. Por eso la correlación global nunca se interpreta aislada, y por eso la regresión de señal global no se adopta por defecto pese a que en el panel correspondiente aparezca como la que más reduce esa métrica.

**La desviación típica del BOLD tiene dos extremos malos.** Un valor alto indica ruido residual. Un valor muy bajo indica sobre-limpieza: se eliminó también la señal de interés. Es la métrica que detecta el caso en que una estrategia parece excelente por todos los demás criterios porque, simplemente, ha dejado la serie casi plana.

**El panel D es donde se toma la decisión.** Interesa el punto más cercano al ángulo inferior derecho, con poco movimiento residual y muchos grados de libertad conservados. La estrategia más agresiva siempre gana en movimiento residual, y por eso el criterio automático impone además conservar al menos la mitad de los grados de libertad máximos, sin esa restricción, el procedimiento seleccionaría siempre la más agresiva por construcción, no por ser mejor.

**Por qué la celda comprueba antes si el criterio puede discriminar.** Una correlación estimada sobre `n` pares tiene un error típico aproximado de uno partido por la raíz de `n` menos tres. El criterio implementado exige que la dispersión entre estrategias supere **el doble** de ese error típico antes de emitir recomendación, un margen deliberadamente conservador. Si no lo supera, las diferencias observadas son indistinguibles del ruido de muestreo, y ordenar las estrategias por ese valor equivale a ordenarlas al azar. Comprobarlo antes de recomendar evita el error más fácil de cometer en esta sección: presentar como decisión metodológica lo que es una fluctuación.

**Qué ocurrió con estos datos, y es el resultado más instructivo de la sección.** El criterio **no discrimina**. Con unos 397 pares el error típico es de 0.050, de modo que el umbral de discriminación se sitúa en 0.100, y la dispersión completa entre las siete estrategias es de 0.048, es decir menor incluso que el error de una sola estimación. La celda por tanto no emite recomendación y explica por qué.

La causa está en la referencia. Antes de regresar ningún confound, la correlación entre movimiento y cambio de señal ya vale 0.043 una vez filtrada en banda, frente a 0.709 sobre la serie sin filtrar. El desplazamiento de encuadre es una señal de banda ancha y la banda de interés es estrecha, de modo que **el filtro paso banda por sí solo elimina casi toda la varianza con la que el movimiento correlacionaba**. Lo que queda después para que la regresión actúe es tan pequeño que las diferencias entre estrategias caen dentro del error.

Esto tiene una consecuencia que conviene subrayar porque es fácil de pasar por alto. Si la referencia se hubiera tomado de la serie sin filtrar, la tabla habría mostrado 0.709 antes y 0.03 después, y se habría concluido que el denoising producía una mejora enorme. Habría sido falso, la mejora la produce el filtro, no la regresión de variables de confusión. Comparar el antes y el después exige que ambos hayan pasado por el mismo tratamiento salvo por aquello cuyo efecto se quiere medir.

**El segundo resultado inesperado está en los grados de libertad.** Van de 67.5 a 74.1 sobre 400 volúmenes. Es decir, la diferencia entre la estrategia más simple, con seis regresores, y la más agresiva, con más de treinta más el censurado, son unos siete grados de libertad. El filtrado, en cambio, reduce de 400 a unos 74. La consecuencia práctica es que **el argumento de ahorrar grados de libertad usando pocos regresores tiene aquí mucho menos peso del que suele atribuírsele**, quien manda es la anchura de la banda, no el número de variables de confusión. Es también la razón por la que la restricción de conservar la mitad de los grados de libertad no llega a activarse con estos datos.

**Una discrepancia con la referencia, que el propio resultado acaba confirmando.** Morfini et al. (2023) advierten de forma explícita que medidas como la desviación típica del BOLD, los grados de libertad o el cambio de señal global están pensadas para comparar **sujetos** sometidos al mismo procesamiento, y que deben usarse con extrema cautela para comparar **procedimientos** entre sí. Las medidas que recomiendan para esa segunda tarea son QC-FC y la distribución de conectividad. Esta sección se aparta de esa recomendación por el tamaño de muestra, QC-FC correlaciona a través de sujetos y aquí solo hay dos con preprocesamiento disponible, de modo que la correlación no es débil sino **degenerada**: dos puntos definen siempre una recta y el coeficiente vale exactamente más uno o menos uno para toda arista, con independencia de los datos. Lo interesante es que el resultado obtenido termina dándoles la razón.

**Qué esperar de cada nivel en un caso donde el criterio sí discrimine.** Pasar de 6 a 12 y a 24 parámetros de movimiento reduce el movimiento residual con un coste moderado. Añadir aCompCor produce habitualmente la mayor caída de la correlación global, porque captura el ruido fisiológico que los parámetros de movimiento no describen. Censurar mejora el movimiento residual pero reduce los volúmenes disponibles y, sobre todo, hace que cada sujeto conserve un número distinto, lo que compromete la comparabilidad si las fracciones difieren mucho. Los regresores de picos evitan ese problema conservando la longitud de la serie, a cambio de gastar un parámetro por volumen marcado.

**Buenas prácticas.** Declarar la estrategia con todos sus componentes, incluidos la banda de filtrado y el umbral de censurado, porque dos trabajos que digan usar aCompCor pueden estar haciendo cosas muy distintas. Reportar los grados de libertad efectivos junto a los resultados de conectividad. Comparar estrategias con varias métricas y no con una. Y comprobar que la métrica elegida tiene resolución suficiente antes de basar una decisión en ella.

**Errores frecuentes.** Regresar los confounds y filtrar después, error documentado en 10.1 que reintroduce el ruido eliminado. Censurar y añadir regresores de picos para los mismos volúmenes, que paga dos veces el mismo coste. Elegir la estrategia que mejor puntúa en una sola métrica, casi siempre la más agresiva. Interpretar una correlación global próxima a cero tras regresión de señal global como prueba de limpieza. Tomar la referencia del antes sin aplicarle el mismo filtrado que al después, que produce una mejora aparente atribuible al filtro. Comparar el tSNR entre estrategias con distinta banda de filtrado, por lo demostrado en 4.4.


### 10.4 Carpet plot y espectro de la señal media para cada estrategia

**Por qué esta vista y no solo la tabla.** La comparación de 10.3 resume cada estrategia en unos pocos números, y eso oculta información que solo se ve mirando las series. Dos estrategias pueden dar una correlación residual entre movimiento y señal casi idéntica y sin embargo dejar los datos en estados muy distintos: una habiendo eliminado el artefacto y otra habiendo eliminado también la señal. El carpet plot muestra qué queda, y el espectro muestra qué se quitó y de dónde. Con estos datos esa distinción resulta especialmente pertinente, porque 10.3 acaba de establecer que su criterio numérico no discrimina entre estrategias.

**Qué contiene la figura.** Una figura por sujeto, organizada en filas alineadas sobre el mismo eje temporal:

En la parte superior, las trazas de desplazamiento de encuadre y de DVARS, con su umbral y con bandas rojas que marcan los volúmenes clasificados como atípicos. Es la referencia temporal contra la que se leen todos los carpet de abajo, y es la disposición que emplean los informes de fMRIPrep y de MRIQC.

Debajo, una fila por estrategia. A la izquierda el carpet plot de la serie limpia, sobre los mismos vóxeles de sustancia gris que forman el grafo de la Sección 11. A la derecha el espectro de potencia de la señal media, con la serie sin limpiar superpuesta como referencia y la banda de paso sombreada.

**Sobre la escala del carpet.** Se representa en unidades típicas de cada vóxel y no en cambio porcentual, que es la convención habitual. El motivo es que tras el paso banda la media temporal de cada vóxel queda prácticamente en cero, de modo que el cambio porcentual respecto de ella carece de escala estable y produce una imagen sin contraste. Normalizar por la desviación típica resuelve el problema y además hace comparables las siete filas entre sí, porque cada estrategia reduce la varianza en distinta medida y una escala absoluta las mostraría progresivamente más oscuras sin que eso signifique nada sobre la estructura del artefacto.

**Sobre el eje del espectro.** La frecuencia se representa en escala lineal desde cero y no en escala logarítmica. En escala logarítmica el cero no existe, y la banda de paso, que es estrecha y está pegada al origen, queda deformada y resulta difícil de situar. La potencia sí va en escala logarítmica, porque abarca varios órdenes de magnitud.


In [ ]:
# 10.4  Carpet plot y espectro de la senal media para cada estrategia
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd

# Saturacion de la escala de color, en unidades tipicas de cada voxel.
LIMITE_CARPET = 2.0

if "series_nodos" not in globals() or not series_nodos:
    raise RuntimeError("Ejecute la celda 10.2 antes de esta.")
if "leer_voxeles_por_bloques" not in globals():
    raise RuntimeError("Ejecute la celda 7.4, que define el lector por bloques de "
                       "volumenes que esta celda reutiliza.")
if "DIR_FIGURAS" not in globals():
    DIR_FIGURAS = PATHS["reportes"] / "figuras"
    DIR_FIGURAS.mkdir(parents=True, exist_ok=True)

indices_mascara_plana = np.flatnonzero(mascara_comun.ravel())
indices_nodos_planos = indices_mascara_plana[posiciones_nodos]


def espectro_de_potencia(serie, tr):
    """Espectro de potencia de una serie temporal unidimensional.

    Se centra la serie antes de transformar, porque la componente continua
    dominaria el espectro sin aportar nada: es la media, no una frecuencia.
    """
    centrada = serie - serie.mean()
    n = centrada.size
    frecuencias = np.fft.rfftfreq(n, d=tr)
    potencia = np.abs(np.fft.rfft(centrada)) ** 2 / n
    return frecuencias[1:], potencia[1:]


def carpet_en_unidades_tipicas(matriz):
    """Normaliza cada voxel por su propia desviacion tipica temporal.

    La escala convencional del carpet plot es el cambio porcentual respecto de la
    media temporal de cada voxel, adecuada sobre datos preprocesados sin filtrar.
    Aqui no sirve: tras el paso banda la media de cada voxel queda practicamente en
    cero, de modo que dividir por ella amplifica el ruido y produce una imagen sin
    contraste util. Normalizar por la desviacion tipica ademas hace comparables las
    siete estrategias entre si.
    """
    centrada = matriz - matriz.mean(axis=1, keepdims=True)
    desviaciones = np.maximum(matriz.std(axis=1, keepdims=True), 1e-12)
    return centrada / desviaciones


rutas_carpets = []
filas_espectro = []
nyquist = np.nan

for sujeto in CFG.id_sujetos:
    estrategias_del_sujeto = [e for e in ESTRATEGIAS if (sujeto, e) in series_nodos]
    if not estrategias_del_sujeto:
        continue

    rutas_bold = sorted(DIR_FMRIPREP.glob(
        f"{sujeto}/**/func/*space-{CFG.plantilla}*_desc-preproc_bold.nii.gz"))
    if not rutas_bold:
        print(f"   {sujeto}: sin funcional preprocesado, se omite")
        continue
    img = nib.load(rutas_bold[0])
    tr = float(img.header.get_zooms()[3])
    n_vol_total = img.shape[3]
    nyquist = 0.5 / tr

    referencia = leer_voxeles_por_bloques(img, indices_nodos_planos)
    media_referencia = referencia.mean(axis=0)
    frec_ref, pot_ref = espectro_de_potencia(media_referencia, tr)
    potencia_ref_en_banda = float(
        pot_ref[(frec_ref >= BANDA[0]) & (frec_ref <= BANDA[1])].sum())

    # Metricas de 7.2, dibujadas ALINEADAS sobre los carpet para poder comprobar la
    # correspondencia temporal entre un pico de movimiento y una banda vertical.
    s_qc = series_qc.get(sujeto)
    fd = s_qc["fd_caja"].to_numpy()[:n_vol_total] if s_qc is not None else None
    dvars = s_qc["dvars"].to_numpy()[:n_vol_total] if s_qc is not None else None
    atipicos = (s_qc["atipico"].to_numpy()[:n_vol_total] if s_qc is not None
                else np.zeros(n_vol_total, dtype=bool))
    tiempo_total = np.arange(n_vol_total) * tr

    n_filas = len(estrategias_del_sujeto)
    fig = plt.figure(figsize=(14, 1.7 * n_filas + 2.2))
    rejilla = fig.add_gridspec(n_filas + 1, 2, width_ratios=[2.6, 1],
                               height_ratios=[1.3] + [1] * n_filas,
                               hspace=0.18, wspace=0.16)

    # Fila superior: trazas de movimiento y de cambio de senal, con las bandas de
    # los volumenes marcados. Comparte eje temporal con los carpet de abajo.
    ax_trazas = fig.add_subplot(rejilla[0, 0])
    if fd is not None:
        ax_trazas.plot(tiempo_total, fd, linewidth=0.8, color="tab:blue")
        ax_trazas.axhline(CFG.umbral_fd_mm, color="tab:blue", linestyle="--",
                          linewidth=0.7)
        ax_trazas.set_ylabel("FD (mm)", fontsize=8, color="tab:blue")
        ax_trazas.tick_params(axis="y", labelsize=7, colors="tab:blue")
        eje_dvars = ax_trazas.twinx()
        eje_dvars.plot(tiempo_total, dvars, linewidth=0.8, color="tab:purple",
                       alpha=0.85)
        eje_dvars.set_ylabel("DVARS", fontsize=8, color="tab:purple")
        eje_dvars.tick_params(axis="y", labelsize=7, colors="tab:purple")
        for indice in np.flatnonzero(atipicos):
            ax_trazas.axvspan(tiempo_total[indice] - tr / 2,
                              tiempo_total[indice] + tr / 2,
                              color="tab:red", alpha=0.18, linewidth=0, zorder=0)
    ax_trazas.set_xlim(0, n_vol_total * tr)
    ax_trazas.tick_params(axis="x", labelbottom=False)
    ax_trazas.set_title(f"Sujeto {sujeto.replace('sub-', '')}", fontsize=10,
                        loc="left")

    for fila, estrategia in enumerate(estrategias_del_sujeto):
        limpia = series_nodos[(sujeto, estrategia)]
        n_vol_limpia = limpia.shape[1]

        ax = fig.add_subplot(rejilla[fila + 1, 0], sharex=ax_trazas)
        ax.imshow(carpet_en_unidades_tipicas(limpia), aspect="auto", cmap="gray",
                  vmin=-LIMITE_CARPET, vmax=LIMITE_CARPET,
                  interpolation="nearest",
                  extent=[0, n_vol_limpia * tr, limpia.shape[0], 0])
        ax.set_yticks([])
        ax.set_ylabel(estrategia, fontsize=7, rotation=0, ha="right", va="center")
        if fila == n_filas - 1:
            ax.set_xlabel("Tiempo (s)", fontsize=8)
            ax.tick_params(axis="x", labelsize=7)
        else:
            ax.tick_params(axis="x", labelbottom=False)

        # Espectro con eje de frecuencia LINEAL desde cero: en escala logaritmica el
        # cero no existe y la banda de paso, estrecha y pegada al origen, queda
        # deformada y resulta dificil de situar.
        ax = fig.add_subplot(rejilla[fila + 1, 1])
        media_limpia = limpia.mean(axis=0)
        frec, pot = espectro_de_potencia(media_limpia, tr)
        ax.semilogy(frec_ref, pot_ref, linewidth=0.7, color="grey", alpha=0.8,
                    label="sin limpiar")
        ax.semilogy(frec, pot, linewidth=0.8, color="tab:blue", label="limpia")
        ax.axvspan(BANDA[0], BANDA[1], color="tab:green", alpha=0.14, linewidth=0)
        ax.set_xlim(0, nyquist)
        ax.tick_params(labelsize=7)
        if fila == 0:
            ax.legend(fontsize=7, frameon=False)
        if fila == n_filas - 1:
            ax.set_xlabel("Frecuencia (Hz)", fontsize=8)
        else:
            ax.tick_params(axis="x", labelbottom=False)

        dentro = (frec >= BANDA[0]) & (frec <= BANDA[1])
        potencia_dentro = float(pot[dentro].sum())
        potencia_total = float(pot.sum())
        filas_espectro.append({
            "Sujeto": sujeto.replace("sub-", ""),
            "Estrategia": estrategia,
            "Potencia en banda (%)": round(100.0 * potencia_dentro / potencia_total, 1)
            if potencia_total > 0 else np.nan,
            "Potencia en banda / referencia": round(
                potencia_dentro / potencia_ref_en_banda, 3)
            if potencia_ref_en_banda > 0 else np.nan,
        })

    rutas_carpets.append(
        guardar_figura(fig, f"carpet_espectro_{sujeto.replace('sub-', '')}"))
    plt.show()
    del referencia

tabla_espectro = pd.DataFrame(filas_espectro)
print("Reparto de potencia respecto de la banda de paso")
print(tabla_espectro.to_string(index=False))
tabla_espectro.to_csv(PATHS["reportes"] / "espectro_denoising.tsv", sep="\t",
                      index=False)

print(f"\nEscala del carpet: unidades tipicas de cada voxel, saturada en "
      f"mas menos {LIMITE_CARPET}")
print(f"Banda de paso: {BANDA[0]} a {BANDA[1]} Hz. Nyquist: {nyquist:.3f} Hz")
for ruta in rutas_carpets:
    print(f"Figura guardada en {ruta}")
if DIR_PERSISTENTE is not None:
    sincronizar_persistente()


**Cómo se leen estas figuras.** Tres lecturas, en este orden.

**Primera, la correspondencia temporal entre las trazas de arriba y las bandas del carpet.** Es la razón de que ambas compartan el eje temporal. Una banda vertical en el carpet, es decir una columna más clara o más oscura que recorre toda su altura, significa que la intensidad cambió de golpe en muchos vóxeles a la vez, y eso no puede ser actividad neural porque ninguna respuesta neural es simultánea en todo el encéfalo. La comprobación decisiva es si esa banda cae sobre un pico de desplazamiento de encuadre:

| Observación | Interpretación |
|---|---|
| Banda que coincide con un pico de movimiento | Artefacto de movimiento, que es lo que el censurado trata |
| Banda sin pico de movimiento asociado | Sospechar la reconstrucción o un efecto fisiológico. El criterio de censurado no lo detecta |
| Pico de movimiento sin banda asociada | El criterio es conservador: descarta volúmenes cuya señal no se alteró de forma apreciable, y eso cuesta grados de libertad sin beneficio |

**Segunda, la evolución al bajar por las filas.** Si las bandas verticales desaparecen progresivamente al pasar a estrategias más agresivas, esas estrategias están tratando el artefacto. Lo que hay que vigilar es el momento en que, además de las bandas, desaparece también la textura de fondo: esa textura es señal, y su pérdida indica sobre-limpieza. Es el equivalente visual de lo que la desviación típica del BOLD detecta numéricamente en 10.3.

**Tercera, el espectro.** Cuatro observaciones, de la más básica a la más informativa.

La caída fuera de la banda de paso sombreada. Debe ser acusada. Si no la hay, el filtro no se aplicó o la banda no es la declarada, y conviene comprobarlo antes que ninguna otra cosa. La tabla de reparto de potencia lo cuantifica.

La potencia a muy baja frecuencia en la serie de referencia, que casi siempre domina el espectro porque recoge la deriva del equipo. Ver cuánta había ahí da la medida real de por qué el filtro paso alto no es opcional.

La presencia de picos estrechos, que son sospechosos porque la actividad de reposo tiene un espectro amplio y sin líneas.

La comparación entre estrategias dentro de la banda de paso, que es donde se detecta el exceso de regresión. Todas deberían coincidir aproximadamente en la forma del espectro dentro de la banda, porque lo que se pretende eliminar está mayoritariamente fuera de ella o es de banda ancha. Si una reduce de forma apreciable la potencia dentro de la banda respecto a las demás, está quitando varianza de la misma región del espectro donde vive la señal de interés, y eso es exactamente lo que se le reprocha a la regresión de señal global.



---
- Por qué mirar por encima de 0.1 Hz si la señal de interés está por debajo

La pregunta es pertinente y su respuesta es la parte menos intuitiva de esta sección. Si la banda de paso llega hasta 0.09 Hz y todo lo demás se elimina, parecería que el espectro por encima de esa frecuencia no aporta nada. Hay cuatro motivos por los que sí aporta, y el primero es el más importante.

**Primero, el ciclo cardiaco no se puede muestrear, y por eso no aparece donde debería.** El teorema del muestreo establece que una señal solo se representa correctamente si su frecuencia es inferior a la mitad de la frecuencia de muestreo. Con un TR de 1.15 s la frecuencia de muestreo es de 0.870 Hz y la de Nyquist queda en **0.435 Hz**. El ciclo cardiaco, que en reposo va aproximadamente de 0.8 a 1.5 Hz, está por encima de ese límite. No es que se muestree mal: es que no puede muestrearse en absoluto.

Lo que ocurre entonces se llama replegamiento, y consiste en que esa contribución aparece en el espectro a una frecuencia **falsa**, calculable como la distancia a un múltiplo de la frecuencia de muestreo. Con las cifras de este dataset:

| Frecuencia cardiaca real | Pulsaciones por minuto | Frecuencia aparente tras el replegamiento |
|---|---|---|
| 0.833 Hz | 50 | 0.037 Hz, **dentro de la banda de paso** |
| 0.917 Hz | 55 | 0.047 Hz, **dentro de la banda de paso** |
| 1.000 Hz | 60 | 0.130 Hz, fuera de la banda |
| 1.200 Hz | 72 | 0.330 Hz, fuera de la banda |
| 1.500 Hz | 90 | 0.239 Hz, fuera de la banda |

La consecuencia es incómoda y hay que entenderla bien: **en un participante con frecuencia cardiaca baja, en torno a 50 o 55 pulsaciones por minuto, el ruido cardiaco cae dentro de la banda de interés**, indistinguible de la señal de conectividad por cualquier criterio de frecuencia. Ningún filtro puede separarlos, porque ocupan el mismo lugar del espectro. Y como la frecuencia cardiaca varía entre personas, dos sujetos del mismo estudio pueden tener su contaminación cardiaca en lugares distintos del espectro, uno dentro de la banda y otro fuera.

Esta es la razón de fondo por la que existen métodos como aCompCor, y por la que el notebook los usa: el ruido fisiológico no se elimina filtrando, hay que estimarlo a partir de regiones donde no se espera señal neural y regresarlo. Mirar el espectro completo es lo que hace visible ese problema en lugar de dejarlo como una afirmación de manual.

**Segundo, la respiración sí se muestrea bien, y eso permite medirla.** A 0.2 o 0.3 Hz está por debajo de Nyquist, de modo que aparece en su frecuencia verdadera y el filtro la elimina limpiamente. Pero su altura es informativa aunque se vaya a eliminar: un pico respiratorio marcado indica un participante cuya fisiología modula la señal con fuerza, y eso predice más contaminación residual de baja frecuencia por una vía distinta. La respiración no solo produce oscilación a su propia frecuencia, también modifica de forma lenta el volumen respirado y la concentración de dióxido de carbono, y ambos alteran el flujo sanguíneo cerebral a frecuencias que sí caen dentro de la banda de interés. El pico de 0.3 Hz es por tanto un indicador indirecto de cuánta contaminación lenta cabe esperar.

**Tercero, es la comprobación de que el filtro hizo lo que dice.** Sin ver la región que se supone eliminada no hay forma de confirmar que se eliminó. Es la misma lógica por la que 4.3 examina el fondo de la imagen aunque sea aire: comprobar lo que debería estar vacío es lo que valida el procedimiento.

**Cuarto, la banda elegida es una convención discutida, no un hecho.** El límite superior de 0.08 a 0.1 Hz procede de los primeros trabajos de conectividad en reposo y se ha mantenido por inercia. Existe evidencia de fluctuaciones con estructura de red por encima de esa frecuencia, especialmente con adquisiciones de TR corto como esta, y mirar el espectro completo es lo que permite plantearse si la banda descarta señal además de ruido. Presentar el filtro como una verdad establecida en lugar de como una decisión sería lo contrario de lo que este notebook pretende.

**Las dos columnas de la tabla.** La primera dice qué fracción de la potencia total queda dentro de la banda de paso, y si el filtro funciona debe estar cerca del cien por cien en todas las estrategias. La segunda compara la potencia dentro de la banda con la que tenía la serie sin limpiar, y es la que detecta el exceso de regresión: un valor próximo a uno significa que la estrategia respetó la varianza de la banda de interés y solo quitó lo de fuera, mientras que un valor bajo significa que también se llevó varianza de dentro.

**Qué hacer según lo que se observe.**

| Observación | Actuación recomendada |
|---|---|
| El espectro no cae fuera de la banda declarada | El filtro no se aplicó. Revisar el código antes de interpretar ningún resultado posterior |
| Las bandas verticales persisten con todas las estrategias | El artefacto no es de movimiento. Volver al panel de 7.4 y a los informes de 4.7 |
| Las bandas desaparecen ya con seis parámetros de movimiento | No hace falta una estrategia agresiva. Elegir la más económica en grados de libertad |
| El carpet pierde toda textura con la estrategia elegida | Se está eliminando señal además de ruido. Retroceder a una estrategia menos agresiva |
| Una estrategia reduce la potencia dentro de la banda más que las demás | Sospechar de exceso de regresión. Contrastar con la distribución de conectividad de la Sección 11 |
| Pico estrecho en torno a 0.2 o 0.3 Hz | Respiración. El filtro lo elimina, pero su altura anticipa contaminación lenta residual |
| Ningún pico visible cerca de 1 Hz | No significa ausencia de ruido cardiaco, sino que está replegado. Ver la explicación anterior |
| El espectro de referencia está dominado por la frecuencia más baja | Normal. Es la deriva, y confirma la necesidad del filtro paso alto |

**Una limitación de esta vista que conviene tener presente.** El carpet se dibuja sobre los mil vóxeles de sustancia gris que la Sección 10 conservó, no sobre el cerebro completo, de modo que un artefacto restringido a una región pequeña puede no aparecer. Para esa clase de problema la vista adecuada es el panel integrado de 7.4, que agrupa por tejido y cubre más vóxeles, o los informes visuales de 4.7 y 5.5.

**Y una consecuencia práctica para el diseño de estudios.** Todo lo anterior es un argumento a favor de registrar la señal fisiológica durante la adquisición, con pulsioximetría y banda respiratoria. Con esos registros, métodos como RETROICOR modelan el ruido cardiaco y respiratorio a partir de su fase real, sin depender de que caiga o no en la banda muestreable. Este dataset no los incluye, y por eso la Sección 15 lo declara como limitación: la corrección recae por completo en aCompCor, que estima esas fuentes de los propios datos pero no las mide.


## Sección 11. Control de calidad posterior al denoising

### 11.1 Qué queda por comprobar y por qué no basta lo anterior

**Introducción conceptual.** La Sección 10 comparó estrategias con métricas calculadas sobre la serie temporal. Esta sección hace algo distinto y complementario, evalúa el efecto del denoising sobre **la estructura de conectividad**, que es el objeto final del análisis. La diferencia no es retórica. Una serie puede tener movimiento residual bajo y grados de libertad razonables, y aun así producir estimaciones de conectividad sesgadas si el ruido restante está distribuido de forma que afecte de manera sistemática a las correlaciones entre regiones.

**Fundamento metodológico.** Morfini et al. (2023) sitúan el control de calidad de los datos denoised como el último punto antes de cualquier análisis estadístico, con el argumento de que es el único momento en que se puede evaluar globalmente la idoneidad de los datos para lo que se va a hacer con ellos. Proponen dos familias de procedimientos, y ambas se implementan aquí.

**La distribución de conectividad de cada sujeto.** Se calcula la correlación entre las series de todos los pares de un conjunto fijo de vóxeles tomados al azar dentro de la máscara común, y se examina la distribución de esos valores. La lectura es directa: la presencia de ruido residual desplaza toda la distribución hacia valores positivos y altera su forma, y lo hace de manera muy variable entre sujetos. Una distribución centrada en un valor positivo pequeño, con la moda próxima a cero y colas ligeramente asimétricas hacia lo positivo, es lo esperable en datos limpios. Distribuciones marcadamente desplazadas, aplanadas o bimodales indican problemas, y su comparación entre sujetos es lo que revela si el denoising fue igualmente eficaz en todos.

**QC-FC, la asociación entre calidad y conectividad.** Es la medida más informativa de esta sección y merece explicarse con precisión, porque su lógica es indirecta. Para cada arista del grafo, es decir para cada par de vóxeles, se calcula la correlación **a través de los sujetos** entre la fuerza de conexión y una medida de calidad como el movimiento medio o la proporción de volúmenes válidos. Si el denoising funcionó, esas correlaciones deberían distribuirse alrededor de cero: la conectividad estimada no debería depender de cuánto se movió el participante. Si en cambio la distribución está desplazada, significa que los sujetos que se movieron más tienen sistemáticamente conectividades distintas, y entonces cualquier diferencia entre grupos que difieran en movimiento será espuria.

**La dependencia de la distancia.** Power et al. (2012) mostraron que el sesgo por movimiento no es uniforme: refuerza las conexiones cortas y debilita las largas. La consecuencia es que las correlaciones QC-FC, si existen, no se reparten al azar sino que dependen de la distancia entre las regiones. Medir esa dependencia añade información que la distribución de QC-FC por sí sola no da: una distribución centrada en cero pero con una relación clara con la distancia indica que hay sesgo, aunque se compense en promedio.

**Una advertencia que aquí es decisiva.** QC-FC es una medida **de dataset**, no de sujeto: se calcula correlacionando a través de los participantes. Aquí solo hay dos sujetos con preprocesamiento disponible, y con dos puntos la correlación no es simplemente imprecisa, es **degenerada**: dos puntos definen siempre una recta, de modo que el coeficiente vale exactamente más uno o menos uno en todas las aristas y el porcentaje de aristas con correlación absoluta alta es del cien por cien por construcción, sin que ese número diga nada sobre los datos. Los resultados de esta sección demuestran el procedimiento y son directamente aplicables a una muestra de decenas de sujetos sin ningún cambio, pero **no permiten concluir nada sobre la calidad de estos datos**. Presentarlos como si lo permitieran sería engañoso, y por eso la celda lo advierte en su propia salida y no solo en el texto.

**Las métricas resumen que se consolidan.** Además de lo anterior, se recogen las medidas que ya se calcularon en la Sección 10 para la estrategia elegida, comparándolas de forma explícita antes y después del denoising: grados de libertad efectivos, correlación global, desviación típica del BOLD, relación señal ruido temporal y correlación residual entre movimiento y cambio de señal. Esa comparación es la evidencia directa de que el denoising hizo lo que se esperaba de él.


In [ ]:
# 11.2  Distribuciones de conectividad, QC-FC y dependencia de la distancia
import time

import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt

_t0 = time.time()


def paso(mensaje):
    """Traza de progreso con marca de tiempo.

    Existe porque esta celda combina lectura de disco con calculo sobre cientos de
    miles de aristas, y sin trazas resulta imposible saber si esta trabajando o
    atascada. Diagnosticar un bloqueo sin ellas obliga a interrumpir a ciegas.
    """
    print(f"[{time.time() - _t0:6.1f} s] {mensaje}", flush=True)


ESTRATEGIA_POR_DEFECTO = "24HMP+CompCor+Censurado"
_rec = globals().get("recomendada")
ESTRATEGIA_ELEGIDA = _rec if isinstance(_rec, str) and pd.notna(_rec) else ESTRATEGIA_POR_DEFECTO
if ESTRATEGIA_ELEGIDA != _rec:
    print(f"La celda 10.3 no dejo una recomendacion utilizable. Se usa la estrategia "
          f"por defecto: {ESTRATEGIA_ELEGIDA}")

if "series_nodos" not in globals() or not series_nodos:
    raise RuntimeError("No hay series de nodos. Ejecute la celda 10.2 antes de esta.")
if "leer_voxeles_por_bloques" not in globals():
    raise RuntimeError("Ejecute la celda 6.3, que define el lector por bloques.")

coordenadas = coordenadas_nodos
n_nodos = coordenadas.shape[0]
paso(f"Estrategia evaluada: {ESTRATEGIA_ELEGIDA}. Nodos: {n_nodos}")

fila_i, fila_j = np.triu_indices(n_nodos, k=1)
distancias = np.linalg.norm(coordenadas[fila_i] - coordenadas[fila_j], axis=1)
paso(f"Aristas del grafo: {len(fila_i)}")

indices_mascara = np.flatnonzero(mascara_comun.ravel())
indices_nodos_planos = indices_mascara[posiciones_nodos]

# La serie de referencia se guarda en disco la primera vez que se calcula. Leer el
# volumen 4D comprimido es la unica operacion cara de esta celda, y sin esta cache
# cada reejecucion la repetiria. El archivo ocupa unos pocos megabytes.
DIR_CACHE_NODOS = PATHS["derivados"] / "cache-nodos"
DIR_CACHE_NODOS.mkdir(parents=True, exist_ok=True)


def correlaciones_de_aristas(serie):
    """Correlaciones entre todos los pares de filas de una matriz (V, T)."""
    z = serie - serie.mean(axis=1, keepdims=True)
    z /= np.maximum(np.linalg.norm(z, axis=1, keepdims=True), 1e-12)
    matriz = z @ z.T
    return matriz[fila_i, fila_j]


def referencia_de_nodos(sujeto):
    """Serie de los nodos SIN regresion de confounds, filtrada en la misma banda.

    Filtrar la referencia es imprescindible: la deriva lineal, compartida por casi
    todos los voxeles, empuja practicamente todas las aristas hacia la correlacion
    positiva, de modo que sin filtrarla el panel A mostraria una mejora que se debe
    al filtro y no a la regresion de confounds, que es lo que se quiere evaluar.
    """
    cache = DIR_CACHE_NODOS / f"{sujeto}_referencia_nodos.npy"
    if cache.exists():
        datos = np.load(cache)
        if datos.shape[0] == len(indices_nodos_planos):
            paso(f"   {sujeto}: referencia recuperada de la cache")
            return datos

    rutas = sorted(DIR_FMRIPREP.glob(
        f"{sujeto}/**/func/*space-{CFG.plantilla}*_desc-preproc_bold.nii.gz"))
    img = nib.load(rutas[0])
    tr = float(img.header.get_zooms()[3])
    paso(f"   {sujeto}: leyendo el volumen 4D por bloques...")
    bruta = leer_voxeles_por_bloques(img, indices_nodos_planos)
    paso(f"   {sujeto}: filtrando en banda...")
    bruta = filtrar_banda(bruta, tr, BANDA)
    np.save(cache, bruta)
    paso(f"   {sujeto}: referencia calculada y guardada en cache")
    return bruta


aristas_antes, aristas_despues = {}, {}
for sujeto in CFG.id_sujetos:
    clave = (sujeto, ESTRATEGIA_ELEGIDA)
    if clave not in series_nodos:
        continue
    bruta = referencia_de_nodos(sujeto)
    aristas_antes[sujeto] = correlaciones_de_aristas(bruta)
    aristas_despues[sujeto] = correlaciones_de_aristas(series_nodos[clave])
    del bruta
    paso(f"   {sujeto}: correlaciones calculadas")

if not aristas_despues:
    raise RuntimeError(
        f"Ninguna serie de nodos corresponde a la estrategia {ESTRATEGIA_ELEGIDA}. "
        f"Disponibles: {sorted({e for _, e in series_nodos})}")

sujetos_grafo = list(aristas_despues.keys())
paso(f"Sujetos con grafo: {len(sujetos_grafo)}")

# QC-FC: correlacion a traves de sujetos entre la fuerza de cada arista y una
# medida de calidad.
medidas_qc = {}
for etiqueta, columna in (("MeanMotion", "MeanMotion (mm)"),
                          ("InvalidScans", "InvalidScans"),
                          ("PVS", "PVS")):
    if columna in resumen_movimiento.columns:
        medidas_qc[etiqueta] = np.array([
            float(resumen_movimiento.loc[
                resumen_movimiento["Sujeto"] == s.replace("sub-", ""), columna].iloc[0])
            for s in sujetos_grafo])

if len(sujetos_grafo) < 3:
    print(f"\nAVISO: solo hay {len(sujetos_grafo)} sujetos con serie limpia. QC-FC es")
    print("una correlacion A TRAVES de sujetos y con menos de tres no esta definida.")
    print("Las columnas correspondientes apareceran vacias. La celda se ejecuta")
    print("igualmente para validar el procedimiento.\n")


def qc_fc(aristas_por_sujeto, valores_qc):
    """Correlacion, arista por arista, entre conectividad y medida de calidad."""
    matriz = np.vstack([aristas_por_sujeto[s] for s in sujetos_grafo])
    a = matriz - matriz.mean(axis=0)
    b = valores_qc - valores_qc.mean()
    denominador = (np.linalg.norm(a, axis=0) * np.linalg.norm(b))
    return np.divide(a.T @ b, denominador, out=np.zeros(matriz.shape[1]),
                     where=denominador > 1e-12)


filas_qcfc = []
qcfc_mapas = {}
for etiqueta, valores in medidas_qc.items():
    for momento, aristas in (("antes", aristas_antes), ("despues", aristas_despues)):
        r = qc_fc(aristas, valores)
        qcfc_mapas[(etiqueta, momento)] = r
        pendiente = float(np.polyfit(distancias, r, 1)[0])
        filas_qcfc.append({
            "Medida de calidad": etiqueta,
            "Momento": momento,
            "QC-FC mediana": round(float(np.median(r)), 4),
            "QC-FC media |r|": round(float(np.abs(r).mean()), 4),
            "Aristas con |r| alto (%)": round(float((np.abs(r) > 0.5).mean() * 100), 1),
            "Pendiente con la distancia": round(pendiente, 6),
        })
paso("QC-FC calculado")

if filas_qcfc:
    tabla_qcfc = pd.DataFrame(filas_qcfc)
    print("\nQC-FC y dependencia de la distancia, antes y despues del denoising")
    print(tabla_qcfc.to_string(index=False))
    tabla_qcfc.to_csv(PATHS["reportes"] / "qcfc.tsv", sep="\t", index=False)
else:
    tabla_qcfc = pd.DataFrame()
    print("\nNo se pudo calcular QC-FC: faltan medidas de calidad por sujeto.")

# Resumen de las distribuciones de conectividad, que si es interpretable con pocos
# sujetos porque se calcula dentro de cada uno y no a traves de ellos.
resumen_fc = pd.DataFrame([
    {"Sujeto": s.replace("sub-", ""),
     "Media antes": round(float(np.mean(aristas_antes[s])), 4),
     "Media despues": round(float(np.mean(aristas_despues[s])), 4),
     "Desviacion antes": round(float(np.std(aristas_antes[s])), 4),
     "Desviacion despues": round(float(np.std(aristas_despues[s])), 4)}
    for s in sujetos_grafo])
print("\nDistribucion de conectividad por sujeto")
print(resumen_fc.to_string(index=False))
resumen_fc.to_csv(PATHS["reportes"] / "distribucion_conectividad.tsv", sep="\t",
                  index=False)

# Visualizacion
paso("Generando figuras...")
fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))

ax = axes[0][0]
for sujeto in sujetos_grafo:
    ax.hist(aristas_antes[sujeto], bins=80, histtype="step", linewidth=1,
            density=True, linestyle=":", alpha=0.7)
    ax.hist(aristas_despues[sujeto], bins=80, histtype="step", linewidth=1.2,
            density=True, label=sujeto.replace("sub-", ""))
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Correlacion entre pares de voxeles")
ax.set_ylabel("Densidad")
ax.set_title("A. Distribucion de conectividad. Punteado antes, continuo despues",
             fontsize=10, loc="left")
ax.legend(fontsize=8, frameon=False)

ax = axes[0][1]
posiciones = np.arange(len(sujetos_grafo))
ax.errorbar(posiciones - 0.1, [np.mean(aristas_antes[s]) for s in sujetos_grafo],
            yerr=[np.std(aristas_antes[s]) for s in sujetos_grafo],
            fmt="o", capsize=4, label="antes")
ax.errorbar(posiciones + 0.1, [np.mean(aristas_despues[s]) for s in sujetos_grafo],
            yerr=[np.std(aristas_despues[s]) for s in sujetos_grafo],
            fmt="s", capsize=4, label="despues")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(posiciones)
ax.set_xticklabels([s.replace("sub-", "") for s in sujetos_grafo])
ax.set_ylabel("Media de la conectividad")
ax.set_title("B. Centro y dispersion", fontsize=10, loc="left")
ax.legend(fontsize=8, frameon=False)

ax = axes[1][0]
if qcfc_mapas:
    for (etiqueta, momento), r in qcfc_mapas.items():
        ax.hist(r, bins=60, histtype="step", density=True, linewidth=1.1,
                linestyle="-" if momento == "despues" else ":",
                label=f"{etiqueta} {momento}")
    ax.legend(fontsize=7, frameon=False, ncol=2)
else:
    ax.text(0.5, 0.5, "QC-FC no calculable con esta muestra", ha="center",
            va="center", transform=ax.transAxes, fontsize=10)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Correlacion QC-FC")
ax.set_ylabel("Densidad")
ax.set_title("C. QC-FC", fontsize=10, loc="left")

ax = axes[1][1]
intervalos = np.linspace(distancias.min(), distancias.max(), 30)
centros = (intervalos[:-1] + intervalos[1:]) / 2
indice_intervalo = np.clip(np.digitize(distancias, intervalos) - 1, 0, len(centros) - 1)
if qcfc_mapas:
    for (etiqueta, momento), r in qcfc_mapas.items():
        if momento != "despues":
            continue
        medias = [r[indice_intervalo == k].mean() if np.any(indice_intervalo == k) else np.nan
                  for k in range(len(centros))]
        ax.plot(centros, medias, marker="o", markersize=3, linewidth=1, label=etiqueta)
    ax.legend(fontsize=8, frameon=False)
else:
    ax.text(0.5, 0.5, "Requiere QC-FC", ha="center", va="center",
            transform=ax.transAxes, fontsize=10)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Distancia euclidea entre voxeles (mm)")
ax.set_ylabel("QC-FC medio")
ax.set_title("D. Dependencia de la distancia", fontsize=10, loc="left")

plt.tight_layout()
if "guardar_figura" in globals():
    guardar_figura(fig, "qcfc_y_conectividad")
plt.show()
if "respaldar_figuras" in globals():
    respaldar_figuras()
paso("Terminado")


**Cómo leer los cuatro paneles.**

**Panel A, distribuciones de conectividad.** Lo que hay que buscar es el desplazamiento del centro. Antes del denoising las distribuciones aparecen desplazadas hacia valores positivos, y con centros que difieren notablemente entre sujetos; después deberían quedar próximas a cero y parecidas entre sí. Ese doble efecto, acercarse a cero y homogeneizarse, es la señal de que el denoising funcionó. Una distribución que tras limpiar siga desplazada indica ruido residual global; una que quede muy estrecha y concentrada en cero puede indicar sobre-limpieza, y hay que confirmarlo con la desviación típica del BOLD de la Sección 10.

**Panel B, centro y dispersión.** Traduce el panel anterior a números y hace visible lo que más importa para un análisis de grupo: la **variabilidad entre sujetos** del centro de la distribución. Morfini et al. (2023) la usan como indicador de calidad del dataset, con el argumento de que si el centro de la distribución de conectividad varía mucho entre participantes, esa variabilidad se propaga a cualquier comparación entre grupos. La barra de error debe reducirse tras el denoising.

**Panel C, distribuciones de QC-FC.** Lo deseable es una distribución estrecha centrada en cero, y lo que hay que comprobar es que la línea continua, posterior al denoising, esté más concentrada alrededor de cero que la punteada. Una distribución desplazada indica que la conectividad estimada depende de cuánto se movió el participante. El porcentaje de aristas con correlación absoluta por encima de 0.5 es un resumen útil, pero con los dos sujetos disponibles ese porcentaje es del cien por cien por construcción: dos puntos definen siempre una recta, de modo que toda correlación vale exactamente más uno o menos uno con independencia de los datos.

**Panel D, dependencia de la distancia.** Es el panel más específico y el que detecta el sesgo característico del movimiento. Una línea horizontal en cero indica ausencia de sesgo dependiente de la distancia. Una línea con pendiente negativa, con valores positivos en distancias cortas y negativos en largas, es exactamente la firma que describieron Power et al. (2012): el movimiento refuerza las conexiones cortas y debilita las largas. Detectarla obliga a endurecer la estrategia de denoising o los criterios de exclusión, porque significa que el sesgo sobrevivió a la limpieza.

**Por qué se miran los cuatro juntos.** Cada panel puede parecer aceptable por separado mientras el conjunto revela un problema. El caso más instructivo es una distribución de QC-FC centrada en cero, panel C aparentemente correcto, acompañada de una pendiente clara con la distancia, panel D. Significa que hay sesgo pero que se compensa en promedio entre conexiones cortas y largas, de modo que la media no lo detecta. Interpretar solo el panel C llevaría a concluir que los datos están limpios cuando no lo están.

**Buenas prácticas.** Usar el mismo conjunto de vóxeles para todos los sujetos, sin lo cual las aristas no son comparables y QC-FC carece de sentido. Calcular QC-FC con varias medidas de calidad, porque capturan aspectos distintos del mismo problema. Reportar la dependencia de la distancia junto a la distribución de QC-FC y no una sola de las dos.

**Errores frecuentes.** Interpretar QC-FC como una medida de sujeto, cuando es una propiedad del dataset. Calcularlo con una muestra insuficiente y presentar el resultado como evidencia, que es exactamente lo que este notebook evita advirtiéndolo en la propia salida. Comparar valores de QC-FC entre estudios sin considerar que dependen del tamaño de muestra, del número de aristas y de la medida de calidad elegida. Concluir que el denoising fue eficaz porque la correlación global bajó, sin comprobar el efecto sobre la estructura de conectividad, que es lo que esta sección mide.


### 11.3 Análisis de los resultados obtenidos.

### Veredicto por sujeto

| Sujeto | Media FC antes | Media FC después | Desviación antes | Desviación después | Veredicto |
|---|---|---|---|---|---|
| 03286 | 0.0309 | 0.0076 | 0.1818 | 0.1586 | Denoising eficaz. Sin sobre-limpieza |
| 12813 | 0.0386 | 0.0076 | 0.1985 | 0.1519 | Denoising eficaz. Sin sobre-limpieza |
| 14229 | sin datos | sin datos | sin datos | sin datos | No preprocesado en esta ejecución |

Ninguno de los dos sujetos evaluables presenta un hallazgo que justifique excluirlo por esta sección.

### Las tres cifras que sostienen ese veredicto

**La media cayó a una cuarta parte**, de 0.031 y 0.039 a 0.008 en ambos. El desplazamiento positivo de la distribución de conectividad es la firma del ruido global: una fuente que afecta a todo el cerebro a la vez correlaciona entre sí a todos los pares de vóxeles, exista o no relación neural. Que caiga a 0.008 indica que esa componente se eliminó en su mayor parte.

**La dispersión bajó poco**, de 0.182 y 0.199 a 0.159 y 0.152, es decir entre un 13 y un 24 por ciento. Eso es lo correcto y no un resultado mediocre: la dispersión contiene la señal de interés, porque las diferencias reales de conectividad entre pares de regiones son lo que se quiere medir. Una caída drástica indicaría sobre-limpieza, y es la comprobación que descarta que la estrategia haya aplanado los datos.

**Los dos sujetos convergieron.** Antes diferían en la media, 0.031 frente a 0.039, un 25 por ciento de diferencia relativa. Después coinciden en 0.0076 hasta la cuarta cifra. En dispersión pasaron de diferir un 9 por ciento a diferir un 4. Es el resultado más informativo de los tres, porque reducir la variabilidad **entre** sujetos es lo que permite que una comparación de grupo no esté dominada por cuánto se movió cada participante.

### La tabla de QC-FC no dice nada, y hay que saber por qué

| Medida | Valor obtenido | Motivo |
|---|---|---|
| MeanMotion | mediana 1.0 antes, -1.0 después, 100 % de aristas | Con dos sujetos, la correlación entre dos puntos vale siempre más uno o menos uno |
| InvalidScans | 0.0 en todo | Ambos sujetos tienen exactamente 6 volúmenes inválidos: la medida no varía |
| PVS | 0.0 en todo | Ambos tienen proporción de válidos 0.985: la medida no varía |

Esos números no son un hallazgo ni un fallo, son aritmética. Dos puntos definen una recta de forma exacta, de modo que el coeficiente está determinado por el tamaño de muestra y no contiene información sobre los datos. El cambio de signo entre el antes y el después tampoco significa nada: indica solo que la ordenación relativa de los dos sujetos se invirtió en alguna arista, suceso trivial con dos puntos.

Los ceros de las otras dos filas tienen otra causa igual de mecánica: una correlación con una variable constante no está definida, y la celda devuelve cero de forma explícita en lugar de propagar un valor indeterminado.

**Cuántos sujetos harían falta.** Del orden de cuarenta a cincuenta para distinguir con confianza razonable una correlación de 0.3, que es la magnitud a la que la literatura sitúa el sesgo por movimiento. Los trabajos que emplean esta métrica de forma seria operan con muestras de ese tamaño o mayores.

### Qué se puede afirmar y qué no

**Se puede afirmar** que la estrategia empleada reduce la componente global de la conectividad, homogeneiza la distribución entre los dos sujetos disponibles y no aplana la señal.

**No se puede afirmar** que elimine el sesgo por movimiento, porque esa pregunta la responde QC-FC y aquí no está definida. Tampoco que la estrategia elegida sea la mejor, cuestión que 10.3 dejó abierta al comprobar que su criterio no discriminaba entre las siete.

**Advertencia sobre el origen de los datos.** Proceden de la ejecución rápida de 5.3c. Los nodos del grafo se toman de la máscara de sustancia gris de la plantilla, de modo que un registro que no converge los sitúa en posiciones anatómicas imprecisas. La estructura del resultado, es decir que el denoising centra la distribución y homogeneiza a los sujetos, es robusta frente a eso. Los valores concretos de las tablas no lo son, y no deben citarse como caracterización de este dataset.


## Sección 12. Informe final automatizado

### 12.1 Qué consolida y por qué se automatiza

**Introducción conceptual.** Hasta aquí cada sección produjo sus propias tablas y figuras. El informe final las reúne en un documento único que responde a la pregunta que motiva todo el flujo: qué sujetos son utilizables, con qué reservas, y con qué evidencia se sostiene esa decisión.

**Fundamento metodológico.** Morfini et al. (2023) cierran su trabajo con una afirmación que este notebook toma como principio: el control de calidad debe reportarse de forma sistemática junto a los resultados, igual que se reportan los detalles del procesamiento, porque es igual de necesario para interpretarlos y para replicarlos. Añaden que la diversidad de protocolos de control de calidad no es un problema si cada estudio documenta el suyo, del mismo modo que se aceptan enfoques analíticos distintos siempre que se declaren.

Provins et al. (2023) van en la misma dirección al proponer documentos de procedimientos normalizados mantenidos bajo control de versiones, con el argumento de que registrar las decisiones y sus motivos es lo que permite auditar el criterio y no solo el resultado.

**Por qué se automatiza.** Un informe generado por código tiene tres propiedades que uno escrito a mano no tiene. Se puede regenerar tras cualquier cambio de parámetro, de modo que nunca queda desactualizado respecto a los datos. No puede omitir selectivamente un sujeto o una métrica incómoda, porque recorre todo lo calculado. Y hace explícito el criterio, ya que las reglas de decisión están escritas en el código y son auditables, en lugar de residir en el juicio no documentado de quien redacta.

**Qué contiene.** El informe recoge, por sujeto, las métricas de calidad de los datos crudos, las de movimiento y señal, las de normalización y corregistro, y las posteriores al denoising, junto con el resultado de aplicar los criterios de exclusión declarados. Recoge además, a nivel de dataset, la consistencia de los parámetros de adquisición, las medidas que solo tienen sentido agregadas, y el resumen de decisiones metodológicas adoptadas con su justificación.

**Sobre los criterios de exclusión.** Se aplican los establecidos en las secciones anteriores y se declaran junto al resultado. La regla de fondo, que procede de 1.9 y se repite aquí porque es la que gobierna todo el informe, es que ninguna métrica excluye por sí sola: el señalamiento automático ordena el trabajo de revisión visual, y es esa revisión la que decide entre excluir, corregir o aceptar documentando. El informe distingue por tanto entre lo que el procedimiento señala y lo que el analista decide, y registra ambas cosas.


In [ ]:
# 12.1b  Comprobacion previa de los insumos del informe final
# El informe final agrega resultados producidos por las Secciones 4 a 11. Esta celda
# comprueba que existan ANTES de empezar a construirlo, por dos razones. La primera
# es practica: sin la comprobacion, la ausencia de una variable produce un
# NameError a mitad de la construccion, con un mensaje que no dice que seccion
# quedo sin ejecutar. La segunda es documental: la lista siguiente es la
# declaracion explicita de que depende el informe, que de otro modo habria que
# reconstruir leyendo el codigo.

INSUMOS_DEL_INFORME = {
    "iqm": ("4.5", "metricas de calidad de imagen de MRIQC"),
    "ranking": ("4.6", "ranking de sujetos y valores extremos"),
    "volumenes_tisulares": ("5.4", "volumenes tisulares del anatomico"),
    "inventario_funcional": ("6.3", "inventario de productos funcionales"),
    "resumen_movimiento": ("7.2", "metricas de movimiento y de senal"),
    "tabla_tsnr": ("4.4", "senal a ruido temporal"),
    "solapamientos": ("8.2", "metricas de normalizacion y corregistro"),
    "resumen_componentes": ("9.2", "componentes de ruido"),
    "comparacion_denoising": ("10.2", "comparacion de estrategias de denoising"),
    "tabla_qcfc": ("11.2", "QC-FC y dependencia de la distancia"),
}

disponibles, ausentes = [], []
for nombre, (celda, descripcion) in INSUMOS_DEL_INFORME.items():
    if nombre in globals() and globals()[nombre] is not None:
        disponibles.append(nombre)
    else:
        ausentes.append((nombre, celda, descripcion))

print(f"Insumos disponibles: {len(disponibles)} de {len(INSUMOS_DEL_INFORME)}")
if ausentes:
    print("\nAusentes. El informe se generara igualmente, con esas columnas vacias:")
    for nombre, celda, descripcion in ausentes:
        print(f"   {nombre:24s} lo produce la celda {celda:5s} ({descripcion})")
    print("\nEs deliberado que el informe no falle por esto: un informe parcial que")
    print("declara sus huecos es mas util que ninguno. Pero un informe con huecos NO")
    print("debe usarse para decidir exclusiones, porque el criterio se aplica sobre")
    print("las columnas presentes y podria absolver a un sujeto por falta de datos.")
else:
    print("Todas las secciones necesarias se han ejecutado.")

# La estrategia de denoising la fija la celda 11.2. Si no se ejecuto, se establece
# aqui el valor por defecto para que el informe pueda construirse, dejando
# constancia de que no procede de la comparacion de 10.3.
if "ESTRATEGIA_ELEGIDA" not in globals() or not isinstance(
        globals().get("ESTRATEGIA_ELEGIDA"), str):
    ESTRATEGIA_ELEGIDA = "24HMP+CompCor+Censurado"
    ESTRATEGIA_ES_POR_DEFECTO = True
    print(f"\nAVISO: la celda 11.2 no fijo la estrategia de denoising. Se usa el")
    print(f"valor por defecto {ESTRATEGIA_ELEGIDA}, que NO procede de la comparacion")
    print(f"de 10.3. El informe lo indicara.")
else:
    ESTRATEGIA_ES_POR_DEFECTO = False
    print(f"\nEstrategia de denoising que se reportara: {ESTRATEGIA_ELEGIDA}")


In [ ]:
# 12.2  Generacion del informe final por sujeto y de dataset
import numpy as np
import pandas as pd
from datetime import datetime, timezone

# Criterios de exclusion, declarados aqui de forma explicita para que sean
# auditables. Cada entrada indica la metrica, el sentido de la comparacion y el
# umbral, con su procedencia.
CRITERIOS = [
    dict(metrica="PVS", tabla="movimiento", columna="PVS", sentido="menor",
         umbral=0.75, fuente="Morfini et al. (2023), valor extremo bajo"),
    dict(metrica="MeanMotion", tabla="movimiento", columna="MeanMotion (mm)",
         sentido="mayor", umbral=CFG.umbral_fd_mm,
         fuente="umbral de censurado de este notebook, como referencia"),
    dict(metrica="tSNR", tabla="tsnr", columna="tSNR con deriva eliminada",
         sentido="menor", umbral=20.0,
         fuente="valor por debajo del cual la sensibilidad queda comprometida a 3T"),
]


def valor_de(tabla, sujeto_corto, columna):
    """Extrae un valor de una de las tablas producidas por las secciones anteriores."""
    tablas = {
        "movimiento": globals().get("resumen_movimiento"),
        "tsnr": globals().get("tabla_tsnr"),
        "solapamiento": globals().get("solapamientos"),
    }
    df = tablas.get(tabla)
    if df is None or columna not in df.columns:
        return np.nan
    fila = df[df["Sujeto"] == sujeto_corto]
    return float(fila[columna].iloc[0]) if not fila.empty else np.nan


filas_informe = []
for sujeto in CFG.id_sujetos:
    corto = sujeto.replace("sub-", "")
    fila = {"Sujeto": corto}

    # Calidad de los datos crudos, de la Seccion 4.
    if "iqm" in globals():
        for modalidad, columnas in (("T1w", ["cjv", "efc", "snr_gm", "inu_range"]),
                                    ("BOLD", ["tsnr", "fd_mean", "fd_perc", "gcor",
                                              "dvars_std"])):
            sub = iqm[(iqm["_modalidad"] == modalidad) & (iqm["_sujeto"] == corto)]
            for columna in columnas:
                if columna in sub.columns and not sub.empty:
                    fila[f"{modalidad} {columna}"] = round(float(sub[columna].iloc[0]), 4)

    # Movimiento y senal, de la Seccion 7.
    for columna in ("MaxMotion (mm)", "MeanMotion (mm)", "InvalidScans",
                    "ValidScans", "PVS"):
        fila[columna] = valor_de("movimiento", corto, columna)

    # Normalizacion y corregistro, de la Seccion 8.
    for columna in ("NORManat", "NORMfunc", "AFO"):
        fila[columna] = valor_de("solapamiento", corto, columna)

    # Posterior al denoising, de las Secciones 10 y 11.
    if "comparacion_denoising" in globals():
        sub = comparacion_denoising[
            (comparacion_denoising["Sujeto"] == corto)
            & (comparacion_denoising["Estrategia"] == ESTRATEGIA_ELEGIDA)]
        if not sub.empty:
            for columna in ("DOF efectivos", "GCOR", "BOLDstd",
                            "corr FD-DVARS antes", "corr FD-DVARS despues"):
                fila[columna] = sub[columna].iloc[0]

    # Aplicacion de los criterios de exclusion declarados.
    senalado_por = []
    for criterio in CRITERIOS:
        valor = valor_de(criterio["tabla"], corto, criterio["columna"])
        if valor != valor:
            continue
        supera = (valor < criterio["umbral"] if criterio["sentido"] == "menor"
                  else valor > criterio["umbral"])
        if supera:
            senalado_por.append(f"{criterio['metrica']} ({valor:.3f})")
    fila["Senalado por"] = ", ".join(senalado_por) if senalado_por else "ninguno"
    fila["Requiere revision visual"] = bool(senalado_por)

    filas_informe.append(fila)

informe_sujetos = pd.DataFrame(filas_informe)

print("INFORME FINAL POR SUJETO")
print(informe_sujetos.to_string(index=False))

print("\nCriterios aplicados")
for criterio in CRITERIOS:
    signo = "<" if criterio["sentido"] == "menor" else ">"
    print(f"  {criterio['metrica']:12s} {signo} {criterio['umbral']:<8} "
          f"{criterio['fuente']}")

senalados = informe_sujetos[informe_sujetos["Requiere revision visual"]]
print(f"\nSujetos senalados para revision visual dirigida: "
      f"{len(senalados)} de {len(informe_sujetos)}")
if not senalados.empty:
    print(senalados[["Sujeto", "Senalado por"]].to_string(index=False))
print("\nRecordatorio metodologico: el senalamiento automatico NO excluye. Ordena el")
print("trabajo de revision visual, y es esa revision la que decide entre excluir,")
print("corregir el procesamiento o aceptar documentando el hallazgo.")

# Resumen de dataset, con lo que solo tiene sentido agregado.
resumen_dataset = {
    "Fecha del informe": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"),
    "Dataset": CFG.dataset,
    "Sesion": CFG.id_sesion,
    "Sujetos": ", ".join(CFG.id_sujetos),
    "Plantilla": CFG.plantilla,
    "Correccion de tiempo de corte": "aplicada" if APLICAR_STC else "omitida (ver 3.5)",
    "Estrategia de denoising": ESTRATEGIA_ELEGIDA,
    "Banda de filtrado (Hz)": f"{CFG.filtro_hz[0]} a {CFG.filtro_hz[1]}",
    "Umbral de FD (mm)": CFG.umbral_fd_mm,
    "Umbral de cambio de senal global (sd)": CFG.umbral_gschange_sd,
    "Fondo suprimido": "si" if globals().get("FONDO_SUPRIMIDO") else "no",
    "Nivel de ejecucion": NIVEL,
    "Backend": detectar_backend(),
}
print("\nRESUMEN DE DATASET")
print(pd.Series(resumen_dataset).to_string())

if "tabla_qcfc" in globals():
    print("\nQC-FC (medida de dataset, no de sujeto)")
    print(tabla_qcfc.to_string(index=False))

# Exportacion. El informe se guarda en TSV para reutilizarlo y en HTML para leerlo.
informe_sujetos.to_csv(PATHS["reportes"] / "informe_final.tsv", sep="\t", index=False)
pd.Series(resumen_dataset).to_csv(PATHS["reportes"] / "informe_dataset.tsv", sep="\t")

partes_html = [
    "<h1>Informe de control de calidad de rs-fMRI</h1>",
    f"<p>Generado el {resumen_dataset['Fecha del informe']}</p>",
    "<h2>Resumen del dataset</h2>",
    pd.Series(resumen_dataset).to_frame("Valor").to_html(),
    "<h2>Informe por sujeto</h2>",
    informe_sujetos.to_html(index=False),
    "<h2>Criterios de exclusion aplicados</h2>",
    pd.DataFrame(CRITERIOS).to_html(index=False),
]
if "tabla_qcfc" in globals():
    partes_html += ["<h2>QC-FC</h2>", tabla_qcfc.to_html(index=False)]
if "comparacion_denoising" in globals():
    partes_html += ["<h2>Comparacion de estrategias de denoising</h2>",
                    comparacion_denoising.to_html(index=False)]

ruta_html = PATHS["reportes"] / "informe_final.html"
ruta_html.write_text("\n".join(partes_html), encoding="utf-8")
print(f"\nInforme exportado a {ruta_html}")
print(f"Tablas exportadas a {PATHS['reportes']}")


**Cómo se lee el informe final.** Tiene tres partes y cada una responde a una pregunta distinta.

**La tabla por sujeto** reúne en una sola fila lo que las Secciones 4 a 11 calcularon por separado: calidad de los datos crudos, movimiento y señal, normalización y corregistro, y resultado del denoising con la estrategia empleada. La columna decisiva es la última, `Señalado por`, que enumera qué criterios superó cada sujeto. Un sujeto sin ninguno no requiere revisión dirigida; uno con varios la requiere con prioridad.

**Los criterios aplicados** se imprimen a continuación, con su umbral y su procedencia. Esa impresión no es decorativa: es lo que permite auditar la decisión. Un lector que discrepe del umbral puede ver exactamente cuál se usó y de dónde sale, y recalcular con otro. Un informe que solo diera el veredicto obligaría a confiar.

**El resumen de dataset** recoge lo que no tiene sentido por sujeto: la sesión, la plantilla, la banda de filtrado, los umbrales, la estrategia de denoising, si la corrección de tiempo de corte se aplicó, y el modo de preprocesamiento empleado. Es la descripción que debe acompañar a cualquier resultado derivado de estos datos.

**La regla que gobierna todo el informe, y conviene repetirla aquí.** El señalamiento automático **no excluye a nadie**. Ordena el trabajo de revisión visual, que es el paso siguiente, y es esa revisión la que decide entre excluir, corregir el procesamiento o aceptar documentando el hallazgo. La distinción importa porque los tres desenlaces son distintos y solo uno de ellos implica perder un sujeto: un corregistro fallido casi siempre se arregla reprocesando, y una pérdida de señal en una región concreta suele resolverse excluyendo esa región y no al participante.

**Qué hacer con cada resultado.**

| Situación | Actuación |
|---|---|
| Ningún sujeto señalado | Revisar de todos modos los informes visuales de al menos un sujeto, para confirmar que las métricas no están enmascarando un problema que ninguna captura |
| Un sujeto señalado por una sola métrica | Abrir su informe individual y decidir. Una sola métrica fuera de rango suele ser variabilidad normal |
| Un sujeto señalado por varias métricas a la vez | Es el patrón que justifica la exclusión, pero solo tras confirmar visualmente cuál es el problema |
| Un sujeto señalado por movimiento con proporción de volúmenes válidos por debajo de 0.75 | Excluir, salvo que el diseño tolere una estimación con muy pocos grados de libertad |
| Columnas vacías en el informe | La celda 12.1b indica qué sección no se ejecutó. Un informe con huecos no debe usarse para decidir exclusiones, porque el criterio se aplica solo sobre lo presente y podría absolver a un sujeto por falta de datos |

**Los archivos exportados.** El informe se guarda en formato de tabla, para reutilizarlo en otro análisis, y en formato de documento, para leerlo o adjuntarlo. Ambos quedan en el directorio de informes y se respaldan en el almacén persistente, de modo que están disponibles sin necesidad de volver a ejecutar el notebook.

**Una limitación de este informe con la muestra actual.** Los dos sujetos con preprocesamiento disponible no permiten aplicar criterios relativos con ningún poder: el rango intercuartílico se estima con dos puntos, de modo que ambos ocupan por definición los extremos y el criterio es degenerado. Los umbrales absolutos que se aplican, como la proporción de volúmenes válidos, sí son interpretables. Con decenas de sujetos el informe cobra su valor completo y el procedimiento no cambia.



## Sección 13. Comparación sistemática con CONN Toolbox

### 13.1 Qué hace cada herramienta y qué implementa este notebook

**Propósito de esta sección.** La Sección 6.2 comparó etapa por etapa el preprocesamiento. Aquí la comparación se extiende a todo el flujo, incluidos el denoising y el control de calidad, y se justifica cada elección. El objetivo no es determinar qué herramienta es mejor, porque la pregunta está mal planteada: resuelven el mismo problema con decisiones distintas, y lo que importa es entender qué se gana y qué se pierde con cada una.

**Los tres enfoques, resumidos.** CONN es un entorno integrado que cubre desde el preprocesamiento hasta el análisis estadístico de conectividad, construido sobre SPM y MATLAB. fMRIPrep es un preprocesador que deliberadamente se detiene antes del denoising, con la filosofía de producir datos preparados para cualquier análisis posterior sin comprometerse con ninguno. Este notebook combina el segundo con una implementación propia de las etapas que el primero integra.

| Ámbito | CONN | fMRIPrep | Este notebook |
|---|---|---|---|
| Entorno | MATLAB con licencia, sobre SPM | Python, contenedor con ANTs, AFNI, FSL, FreeSurfer | Python, contenedores de Neurodesk mas código propio |
| Preprocesamiento | Integrado | Su única función | Delegado a fMRIPrep |
| Control de calidad de datos crudos | Inspección visual guiada mas medidas automáticas | No lo hace | Delegado a MRIQC, mas implementación propia de lo que no cubre |
| Denoising | Integrado, con estrategia por defecto | No lo hace, entrega confounds sin aplicar | Implementación propia, comparando siete estrategias |
| Métricas de calidad | Conjunto propio, documentado en Morfini et al. | Confounds y reportes visuales | Implementación propia sobre las salidas de fMRIPrep |
| Análisis de conectividad | Integrado | No lo hace | Fuera del alcance definido |
| Reproducibilidad | Scripts de MATLAB | Contenedor versionado mas descripción textual automática | Contenedor versionado mas manifiesto propio, Sección 14 |

**Las cuatro diferencias metodológicas que importan.**

**Primera, la normalización directa frente a la indirecta.** CONN proyecta el funcional directamente al espacio estándar, sin pasar por el anatómico. fMRIPrep lo lleva primero al anatómico del propio sujeto y de ahí a la plantilla. La vía directa evita propagar los errores del corregistro entre modalidades, y Calhoun et al. (2017) la defienden cuando no hay corrección de distorsión disponible, que es la situación en la que se encontraban Morfini et al. La vía indirecta aprovecha el mayor detalle anatómico del T1w, y con la distorsión ya corregida mediante fieldmaps medidos, como ocurre en este dataset según se verificó en 3.3, el corregistro deja de ser el eslabón débil. **Justificación de la elección:** disponer de fieldmaps de codificación de fase opuesta cambia el balance a favor de la vía indirecta, y por eso se adopta.

**Segunda, la interpolación única.** CONN aplica el realineamiento y la normalización como pasos separados, cada uno con su remuestreo. fMRIPrep compone todas las transformaciones y realiza una sola interpolación desde los datos originales. Cada interpolación promedia vóxeles vecinos y por tanto desenfoca, de modo que el efecto es acumulativo. **Justificación:** la ventaja es incondicional y no tiene contrapartida, más allá de la complejidad de implementación, que la herramienta ya resuelve.

**Tercera, la separación entre preprocesamiento y denoising.** Es la diferencia estructural más importante. CONN los integra, lo que hace el flujo más simple y menos propenso a errores de usuario, con el coste de que la estrategia de denoising queda menos visible y menos fácil de variar. fMRIPrep los separa, lo que obliga a decidir de forma explícita y facilita comparar estrategias, con el riesgo de que un usuario poco atento suponga que los confounds ya están aplicados cuando no lo están. **Justificación:** en una guía docente la separación es preferible, porque hace visible una decisión que de otro modo quedaría oculta en un valor por defecto. En un contexto de producción con muchos sujetos, el argumento se invierte.

**Cuarta, la ortogonalización de los componentes de ruido.** Morfini et al. calculan los componentes de CompCor en un espacio ortogonal a los parámetros de movimiento y a los regresores de censurado, para evitar modelar dos veces el mismo efecto. fMRIPrep no lo hace. Es una diferencia real que afecta al número efectivo de regresores independientes, y quedó documentada en 9.1. **Justificación:** se usan los componentes tal como los entrega fMRIPrep, por transparencia respecto a lo que la herramienta produce, dejando constancia de la diferencia para que sea auditable.

**Qué se toma de CONN sin usar CONN.** Las métricas de control de calidad y la filosofía de evaluación. Los propios autores señalan que sus recomendaciones son agnósticas respecto del software y que medidas como NORManat, NORMfunc o el solapamiento anatómico funcional pueden calcularse con independencia del programa que generó los datos. Este notebook lo comprueba en la práctica: las implementa sobre salidas de fMRIPrep, y ese es el sentido de la afirmación de la Sección 1 de que estas métricas no son propias de ninguna herramienta.

**Qué no se replica, y por qué.** No se reproduce la estrategia de denoising por defecto de CONN como tal, sino que se compara un conjunto de estrategias entre las que figura una equivalente en espíritu. No se reproduce el suavizado espacial de 8 mm que CONN aplica en el preprocesamiento, porque fMRIPrep deja esa decisión al usuario y porque comparar métricas de calidad entre datos suavizados y sin suavizar es engañoso, según se argumentó en 6.2. Y no se aborda el análisis de conectividad, que queda fuera del alcance definido en la Sección 1.

**Limitación de esta comparación.** Está construida sobre la descripción que Morfini et al. (2023) hacen del flujo por defecto de CONN, no sobre una ejecución paralela de ambas herramientas sobre los mismos datos. Una comparación empírica exigiría procesar el mismo dataset con las dos y contrastar los resultados, que es un trabajo distinto y de mayor alcance. Lo que aquí se ofrece es una comparación de decisiones metodológicas documentadas, útil para entender y justificar las elecciones, no una evaluación de rendimiento relativo.


## Sección 14. Reproducibilidad

### 14.1 Qué hace falta para que otra persona obtenga lo mismo

**Introducción conceptual.** Reproducir un análisis de neuroimagen exige más de lo que suele documentarse. No basta con nombrar las herramientas: hacen falta sus versiones exactas, las de sus dependencias internas, los parámetros empleados, la identidad verificable de los datos de entrada, y las semillas de cualquier procedimiento con componente aleatorio. Esta sección produce todo eso en un único archivo.

**Fundamento metodológico.** Botvinik-Nezer et al. (2020) mostraron que setenta equipos analizando los mismos datos con libertad metodológica llegaban a conclusiones distintas, y Warrington et al. (2023) cuantificaron que la elección del procedimiento de análisis puede introducir variabilidad del orden de la variabilidad biológica entre sujetos. Ambos resultados apuntan a la misma conclusión práctica: la descripción del procesamiento no es un anexo administrativo del trabajo, es parte del resultado.

**Los cinco componentes del manifiesto y por qué cada uno.**

**Versiones interrogadas, no declaradas.** El manifiesto pregunta a cada binario su versión en ejecución en lugar de anotar la que se creía usar. La razón se documentó en 2.2 y se comprobó en la práctica: el contenedor de MRIQC identificado como 24.0.2 aloja un binario que se identifica como una versión de desarrollo posterior. Declarar la del contenedor habría sido inexacto. La celda avisa de forma explícita cuando la versión reportada no coincide con la solicitada.

**Versiones de los paquetes de Python realmente cargados.** Se leen de los metadatos de instalación, no de una lista escrita a mano, por el mismo motivo. Este notebook tuvo además un incidente relacionado, documentado en 3.3, en el que la versión en disco y la cargada en memoria divergieron, con un fallo que apareció mucho después y sin relación aparente con su causa.

**Sumas de verificación de las entradas.** Es lo que permite comprobar que un análisis futuro parte exactamente de los mismos datos. Cumple la función que habría aportado DataLad y que se decidió no usar en 2.9: allí se argumentó que la trazabilidad se cubriría por esta vía, y aquí es donde se cumple ese compromiso.

**Parámetros tomados del objeto de configuración.** No se transcriben a mano, se extraen de `CFG` y de las variables de decisión que fijaron las secciones anteriores. Así no pueden divergir de lo realmente ejecutado, que es el modo habitual en que las descripciones de métodos se vuelven falsas sin que nadie lo advierta.

**Semillas.** Fijadas en 2.8 y registradas aquí, de modo que los procedimientos con componente aleatorio, como la selección de vóxeles del grafo en la Sección 11, produzcan el mismo resultado en cada ejecución.

**Los derivados como dataset.** Se escribe además un archivo de descripción siguiendo la especificación de derivados de BIDS, que declara qué herramientas generaron los resultados y de qué dataset de origen proceden. Eso convierte el directorio de salida en un dataset citable y encadenable, no en una carpeta de archivos sueltos.

**Lo que este manifiesto no captura.** Conviene ser explícito sobre sus límites. No captura el estado del hardware, que en Colab varía entre sesiones y afecta al tiempo pero no al resultado. No captura las decisiones tomadas durante la inspección visual, que por naturaleza son juicios y quedan registradas en el informe de la Sección 12. Y no garantiza que una ejecución futura produzca resultados idénticos bit a bit, porque las bibliotecas numéricas pueden variar en el orden de las operaciones de coma flotante entre versiones y arquitecturas; garantiza que se pueda reconstruir el mismo procedimiento con los mismos datos y parámetros, que es lo que la reproducibilidad exige de forma razonable.


In [ ]:
# 14.2  Manifiesto de reproducibilidad: versiones, sumas de verificacion y parametros
import hashlib
import json
import platform
import subprocess
import sys
from dataclasses import asdict
from datetime import datetime, timezone

import numpy as np
import pandas as pd


def suma_verificacion(ruta, bloque=1 << 20):
    """SHA-256 de un archivo, leido por bloques para no cargarlo en memoria."""
    resumen = hashlib.sha256()
    with open(ruta, "rb") as archivo:
        for trozo in iter(lambda: archivo.read(bloque), b""):
            resumen.update(trozo)
    return resumen.hexdigest()


# Versiones de las herramientas externas, INTERROGADAS en ejecucion y no
# declaradas de memoria. Es el punto que se argumento en 2.2: el nombre del
# contenedor puede no coincidir con la version del binario que contiene, y
# declarar la version que se creia usar es un fallo de reproducibilidad.
versiones_externas = {}
for herramienta in CONTENEDORES:
    try:
        codigo, salida = ejecutar_herramienta(herramienta, ["--version"],
                                              mostrar_registro=False)
        versiones_externas[herramienta] = {
            "modulo solicitado": f"{herramienta}/{CONTENEDORES[herramienta]}",
            "version reportada por el binario": salida.strip().splitlines()[-1]
            if salida.strip() else "sin respuesta",
            "codigo de salida": codigo,
        }
    except Exception as exc:
        versiones_externas[herramienta] = {
            "modulo solicitado": f"{herramienta}/{CONTENEDORES[herramienta]}",
            "version reportada por el binario": f"no disponible: {type(exc).__name__}",
            "codigo de salida": None,
        }

print("Versiones de las herramientas externas")
for herramienta, datos in versiones_externas.items():
    print(f"  {herramienta}")
    for clave, valor in datos.items():
        print(f"      {clave}: {valor}")
    solicitada = CONTENEDORES[herramienta]
    reportada = str(datos["version reportada por el binario"])
    if solicitada not in reportada:
        print(f"      ATENCION: la version reportada no contiene la solicitada "
              f"({solicitada}). Declare la reportada, no la del contenedor.")

# Versiones de los binarios que hacen el trabajo real, que viven DENTRO de los
# contenedores. Interrogarlos uno por uno seria costoso, pero fMRIPrep ya las
# registra en su propio `dataset_description.json`, de modo que se leen de ahi. Sin
# este paso el manifiesto declararia MRIQC y fMRIPrep pero omitiria ANTS, FSL,
# AFNI y FreeSurfer, que son los que ejecutan el registro, la segmentacion y la
# correccion de distorsion.
herramientas_internas = []
for subdirectorio in ("fmriprep", "mriqc"):
    ruta_desc = PATHS["derivados"] / subdirectorio / "dataset_description.json"
    if not ruta_desc.exists():
        continue
    try:
        descripcion_externa = json.loads(ruta_desc.read_text())
    except Exception:
        continue
    for generador in descripcion_externa.get("GeneratedBy", []):
        entrada = {"Origen": subdirectorio,
                   "Nombre": generador.get("Name"),
                   "Version": generador.get("Version")}
        contenedor = generador.get("Container") or {}
        if contenedor:
            entrada["Contenedor"] = contenedor.get("Tag") or contenedor.get("Type")
        herramientas_internas.append(entrada)

if herramientas_internas:
    print("\nHerramientas registradas por los propios derivados")
    print(pd.DataFrame(herramientas_internas).to_string(index=False))
else:
    print("\nNo se localizaron los dataset_description.json de fMRIPrep ni de MRIQC.")
    print("Las versiones de ANTS, FSL, AFNI y FreeSurfer estan en el informe HTML de")
    print("fMRIPrep, en su seccion de metodologia, y deben transcribirse a mano si se")
    print("necesita el manifiesto completo.")

# Versiones de los paquetes de Python realmente cargados.
import importlib.metadata as metadatos
paquetes = ["numpy", "scipy", "pandas", "matplotlib", "nibabel", "scikit-image",
            "scikit-learn", "templateflow", "nilearn", "psutil"]
versiones_python = {}
for nombre in paquetes:
    try:
        versiones_python[nombre] = metadatos.version(nombre)
    except metadatos.PackageNotFoundError:
        versiones_python[nombre] = "no instalado"

print("\nVersiones de los paquetes de Python")
print(pd.Series(versiones_python).to_string())

# Sumas de verificacion de las entradas. Es lo que permite comprobar que un
# analisis futuro parte de los mismos datos, y sustituye a la trazabilidad que
# habria aportado DataLad, segun se argumento en 2.9.
filas_checksum = []
for sujeto in CFG.id_sujetos:
    base = DIR_BIDS / sujeto / CFG.id_sesion
    if not base.exists():
        print(f"   aviso: no existe {base}, no se pueden calcular sus sumas")
        continue
    for ruta in sorted(base.rglob("*.nii.gz")):
        filas_checksum.append({
            "Archivo": str(ruta.relative_to(DIR_BIDS)),
            "Bytes": ruta.stat().st_size,
            "SHA-256": suma_verificacion(ruta),
        })

checksums = pd.DataFrame(filas_checksum)
if checksums.empty:
    # Sin esta comprobacion, `checksums["SHA-256"]` lanzaria KeyError sobre un
    # DataFrame sin columnas, y lo haria al final de todo el trabajo.
    print("\nNo se encontro ninguna imagen de entrada. El manifiesto se genera sin")
    print("sumas de verificacion, lo que limita la trazabilidad: quedaria sin poder")
    print("comprobar que una ejecucion futura parte de los mismos datos.")
else:
    print(f"\nSumas de verificacion calculadas para {len(checksums)} imagenes")
    print(checksums.assign(**{"SHA-256": checksums["SHA-256"].str[:16] + "..."})
          .to_string(index=False))
    checksums.to_csv(PATHS["reportes"] / "sumas_verificacion.tsv", sep="\t",
                     index=False)

# Parametros del analisis, tomados del objeto de configuracion para que no puedan
# divergir de lo realmente ejecutado.
parametros = asdict(CFG)
# `asdict` recoge los campos del dataclass pero no sus propiedades, de modo que los
# identificadores en formato BIDS se anaden de forma explicita.
parametros["id_sujetos"] = list(CFG.id_sujetos)
parametros["id_sesion"] = CFG.id_sesion
parametros["aplicar_correccion_tiempo_de_corte"] = bool(globals().get("APLICAR_STC", False))
parametros["estrategia_de_denoising"] = globals().get("ESTRATEGIA_ELEGIDA", "no definida")
parametros["estrategia_por_defecto"] = bool(globals().get("ESTRATEGIA_ES_POR_DEFECTO", False))
parametros["origen_de_los_nodos"] = globals().get("origen_nodos", "no definido")
parametros["fondo_suprimido"] = bool(globals().get("FONDO_SUPRIMIDO", False))
parametros["nivel_de_ejecucion"] = globals().get("NIVEL", "no definido")
parametros["backend"] = detectar_backend()

print("\nParametros del analisis")
print(pd.Series({k: str(v) for k, v in parametros.items()}).to_string())

# `ram_gb` puede ser None si la deteccion de recursos no pudo leerlo. La comparacion
# `x == x` detecta NaN pero da verdadero con None, y `float(None)` lanza TypeError.
ram_registrada = (round(float(ram_gb), 1)
                  if isinstance(ram_gb, (int, float)) and ram_gb == ram_gb else None)

manifiesto = {
    "generado": datetime.now(timezone.utc).isoformat(),
    "entorno": {
        "plataforma": platform.platform(),
        "python": sys.version,
        "nucleos": n_cpu,
        "ram_gb": ram_registrada,
        "en_colab": IN_COLAB,
    },
    "herramientas_externas": versiones_externas,
    "herramientas_internas": herramientas_internas,
    "paquetes_python": versiones_python,
    "parametros": {k: (list(v) if isinstance(v, tuple) else v)
                   for k, v in parametros.items()},
    "semillas": {"numpy": CFG.semilla, "random": CFG.semilla},
    "entradas": filas_checksum,
}

# `default=str` evita que el volcado falle por un objeto no serializable, por
# ejemplo un Path o un escalar de numpy que se cuele en la configuracion. Sin el, el
# fallo llegaria despues de haber hecho todo el trabajo.
ruta_manifiesto = PATHS["reportes"] / "manifiesto_reproducibilidad.json"
ruta_manifiesto.write_text(
    json.dumps(manifiesto, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8")

# Descripcion del dataset de derivados, siguiendo la especificacion
# BIDS-Derivatives. Se escribe en un subdirectorio PROPIO y no en la raiz de
# `derivatives`, que no es en si misma un dataset derivado sino el contenedor de
# varios: cada herramienta tiene el suyo, y mezclarlos confundiria la procedencia.
DIR_QC_PROPIO = PATHS["derivados"] / "rsfmri-qc"
DIR_QC_PROPIO.mkdir(parents=True, exist_ok=True)

generadores = [
    {"Name": "Notebook de control de calidad de rs-fMRI",
     "Description": "Metricas de calidad, extraccion de ruido y denoising "
                    "implementados en Python sobre las salidas de fMRIPrep"},
]
for herramienta, datos in versiones_externas.items():
    generadores.append({
        "Name": herramienta,
        "Version": str(datos["version reportada por el binario"]),
        "Container": {"Type": "apptainer",
                      "Tag": f"{herramienta}/{CONTENEDORES[herramienta]}"},
    })
generadores.extend(herramientas_internas)

descripcion = {
    "Name": "Preprocesamiento y control de calidad de rs-fMRI",
    "BIDSVersion": "1.8.0",
    "DatasetType": "derivative",
    "GeneratedBy": generadores,
    "SourceDatasets": [{"DOI": "doi:10.18112/openneuro.ds004712.v2.0.1",
                        "URL": "https://openneuro.org/datasets/ds004712"}],
}
(DIR_QC_PROPIO / "dataset_description.json").write_text(
    json.dumps(descripcion, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8")

print(f"\nManifiesto guardado en {ruta_manifiesto}")
print(f"Descripcion de derivados guardada en "
      f"{DIR_QC_PROPIO / 'dataset_description.json'}")

# Inventario de todo lo producido, para que quien reciba el material sepa que hay
# sin tener que ejecutar nada. Es el cierre del compromiso de que los resultados
# queden almacenados y no solo mostrados.
print("\nINVENTARIO DE PRODUCTOS")
for etiqueta, directorio, patron in (
        ("Tablas", PATHS["reportes"], "*.tsv"),
        ("Figuras PNG", PATHS["reportes"] / "figuras", "*.png"),
        ("Figuras PDF", PATHS["reportes"] / "figuras", "*.pdf"),
        ("Informes HTML propios", PATHS["reportes"], "*.html"),
        ("Informes de MRIQC", PATHS["derivados"] / "mriqc", "*.html"),
        ("Informes de fMRIPrep", PATHS["derivados"] / "fmriprep", "sub-*.html"),
        ("Manifiestos JSON", PATHS["reportes"], "*.json")):
    if not directorio.exists():
        print(f"   {etiqueta:24s} directorio inexistente")
        continue
    archivos = sorted(directorio.glob(patron))
    total_mb = sum(a.stat().st_size for a in archivos) / 1e6
    print(f"   {etiqueta:24s} {len(archivos):3d} archivos, {total_mb:7.1f} MB")

# Respaldo final en el almacen persistente, para que nada de lo anterior dependa de
# que la maquina virtual siga viva.
if DIR_PERSISTENTE is not None:
    sincronizar_persistente()
    print(f"\nTodo respaldado en {DIR_PERSISTENTE}")
    print("Quien reciba esa carpeta dispone de las tablas, las figuras, los informes")
    print("y el manifiesto sin necesidad de ejecutar el notebook.")


**Qué contiene el manifiesto y cómo se usa.** El archivo generado reúne cinco bloques: las versiones interrogadas a los binarios en ejecución, las versiones de las herramientas internas que fMRIPrep declara en sus propios derivados, las versiones de los paquetes de Python realmente cargados, los parámetros extraídos del objeto de configuración, y las sumas de verificación de las imágenes de entrada. El inventario final enumera todo lo producido, con su tamaño.

**Cómo comprobar que otra ejecución parte de los mismos datos.** Las sumas de verificación son el instrumento para eso. Basta con recalcularlas sobre las imágenes de entrada y compararlas con las del manifiesto: si coinciden, los datos son idénticos bit a bit, y cualquier diferencia en los resultados procede del procedimiento y no de los datos. Es la comprobación que permite distinguir un problema de datos de un problema de análisis, y sin ella esa distinción es una conjetura.

**Qué mirar en la salida.** Dos avisos merecen atención. El primero aparece cuando la versión que reporta un binario no coincide con la del contenedor solicitado, situación que se dio en este proyecto y que obliga a declarar la reportada. El segundo, cuando no se encuentran las imágenes de entrada y el manifiesto se genera sin sumas de verificación, lo que limita la trazabilidad y conviene resolver antes de dar el trabajo por cerrado.

**El modo de preprocesamiento queda registrado**, y esa es la razón de que exista la variable correspondiente. Mientras los resultados procedan de la ejecución rápida, el manifiesto lo declara, de modo que la advertencia sobre su calidad viaja con los datos y no depende de que alguien recuerde haber leído la Sección 5.

**Lo que este manifiesto no captura**, y conviene ser explícito sobre sus límites. No captura el estado del hardware, que varía entre sesiones y afecta al tiempo pero no al resultado. No captura las decisiones tomadas durante la inspección visual, que por naturaleza son juicios y quedan registradas en el informe de la Sección 12. Y no garantiza que una ejecución futura produzca resultados idénticos bit a bit, porque las bibliotecas numéricas pueden variar en el orden de las operaciones de coma flotante entre versiones y arquitecturas. Lo que garantiza es que se pueda reconstruir el mismo procedimiento con los mismos datos y parámetros, que es lo que la reproducibilidad exige de forma razonable.



## Sección 15. Limitaciones y referencias

### 15.1 Lo que este notebook no hace, y por qué importa

Toda limitación que se enumera aquí es una restricción sobre lo que se puede concluir. Se presentan agrupadas por su origen, porque no todas tienen el mismo remedio.

**Limitaciones del dataset, que ninguna decisión de análisis puede corregir.**

*Tres sujetos.* Es una elección deliberada de alcance docente, pero tiene consecuencias que se han señalado en cada sección afectada. El criterio de valores extremos por rango intercuartílico no discrimina con tres puntos. QC-FC y la dependencia de la distancia carecen de poder estadístico, porque son medidas de dataset calculadas a través de sujetos. Lo que estas secciones demuestran es el procedimiento, aplicable sin cambios a decenas de sujetos.

*Fondo suprimido.* Comprobado empíricamente en 4.3, con más del 80 por ciento de ceros exactos en regiones exteriores al cerebro. Invalida la interpretación de cinco métricas de calidad de imagen que se calculan y se reportan marcadas como no aplicables. No es una rareza: la supresión de fondo es habitual en reconstrucciones de fabricante y el desfigurado es obligatorio en datos compartidos.

*Cobertura parcial del campo de visión.* Con 48 cortes de 2.4 mm el campo abarca 115.2 mm y no cubre el cerebro completo. Todo análisis entre sujetos queda restringido a la intersección de las máscaras funcionales, calculada en 6.3.

*Ausencia de `SliceTiming`.* Documentada en 3.3 y resuelta en 3.5 omitiendo la corrección, con la justificación cuantitativa correspondiente. Si se dispusiera del orden real de adquisición, la decisión podría revisarse.

*Ausencia de monitorización fisiológica.* Sin registros de pulso ni respiración no se pueden aplicar métodos de corrección basados en señal medida. La aproximación recae por completo en aCompCor, que los estima de los propios datos pero no los sustituye.

*Fijación visual no documentada.* Los participantes mantuvieron los ojos abiertos, pero el protocolo no documenta fijación en un punto. Kumar et al. (2024) la recomiendan para reducir la probabilidad de que el participante se duerma, con evidencia de cambios medibles en la conectividad durante el sueño.

**Limitaciones de las decisiones de procesamiento, que sí serían modificables.**

*Sin reconstrucción de superficies.* Se ejecuta con `--fs-no-reconall`, lo que descarta el análisis vertex-wise y la morfometría cortical. La licencia de FreeSurfer es obligatoria en ambos casos, según se comprobó y se documentó en 2.4.

*Sin informes verbose ni descomposición en componentes independientes.* MRIQC se ejecutó sin `--verbose-reports` ni `--ica`. La primera ausencia tiene consecuencia metodológica: la vista saturada del fondo del anatómico pertenece a los informes verbose, y el protocolo de Provins et al. (2023) se apoya en el panel de fondo. La segunda impide evaluar su criterio F, los efectos de historia de espín.

*Componentes de CompCor sin ortogonalizar.* fMRIPrep no descuenta los efectos de movimiento antes de calcular los componentes, a diferencia de la definición de referencia. Documentado en 9.1.

*Sin suavizado espacial.* fMRIPrep deja esa decisión al usuario y aquí no se aplica, lo que hace que las métricas no sean comparables con las de flujos que sí suavizan.

*Comparación con CONN basada en documentación.* La Sección 13 compara decisiones metodológicas descritas en la literatura, no resultados de una ejecución paralela de ambas herramientas sobre los mismos datos.

**Limitaciones de fondo del campo, que no dependen de este trabajo.**

*No hay verdad de referencia.* No existe una medida independiente de cuál es la señal BOLD real, de modo que toda evaluación de calidad es indirecta y comparativa. Es la limitación que Morfini et al. (2023) sitúan en primer lugar al discutir la suya.

*Señal y ruido no son categorías separadas.* Se comportan como un continuo. Regresar componentes etiquetados como no neurales elimina inevitablemente algo de señal de origen neural, y no hay forma de cuantificar cuánto sin una verdad de referencia.

*Los criterios de exclusión son relativos.* Se definen respecto a la propia muestra y no en términos absolutos, lo que dificulta la estandarización. El argumento a favor, que este notebook comparte, es que esa flexibilidad es necesaria dada la heterogeneidad de protocolos, y que el problema se mitiga si el procedimiento de control de calidad se reporta de forma sistemática.

### 15.2 Referencias


- Morfini, F., Whitfield-Gabrieli, S., Nieto-Castañón, A. (2023). Functional connectivity MRI quality control procedures in CONN. *Frontiers in Neuroscience*, 17:1092125. doi:10.3389/fnins.2023.1092125
- Provins, C., MacNicol, E., Seeley, S.H., Hagmann, P., Esteban, O. (2023). Quality control in functional MRI studies with MRIQC and fMRIPrep. *Frontiers in Neuroimaging*, 1:1073734. doi:10.3389/fnimg.2022.1073734
- Kumar, V.A., Lee, J., Liu, H.-L., et al. (2024). Recommended resting-state fMRI acquisition and preprocessing steps for preoperative mapping of language, motor, and visual areas in adult and pediatric patients with brain tumors and epilepsy. *AJNR American Journal of Neuroradiology*, 45:139-148. doi:10.3174/ajnr.A8067
- Warrington, S., Ntata, A., Mougin, O., et al. (2023). A resource for development and comparison of multimodal brain 3T MRI harmonisation approaches. *Imaging Neuroscience*, 1. doi:10.1162/imag_a_00042
- Esteban, O., Markiewicz, C.J., Blair, R.W., et al. (2019). fMRIPrep: a robust preprocessing pipeline for functional MRI. *Nature Methods*, 16:111-116.
- Esteban, O., Birman, D., Schaer, M., Koyejo, O.O., Poldrack, R.A., Gorgolewski, K.J. (2017). MRIQC: Advancing the automatic prediction of image quality in MRI from unseen sites. *PLoS ONE*, 12:e0184661.
- Gorgolewski, K.J., Auer, T., Calhoun, V.D., et al. (2016). The brain imaging data structure, a format for organizing and describing outputs of neuroimaging experiments. *Scientific Data*, 3:160044.
- Ciric, R., Thompson, W.H., Lorenz, R., et al. (2022). TemplateFlow: FAIR-sharing of multi-scale, multi-species brain models. *Nature Methods*, 19:1568-1571.
- Whitfield-Gabrieli, S., Nieto-Castañón, A. (2012). Conn: a functional connectivity toolbox for correlated and anticorrelated brain networks. *Brain Connectivity*, 2:125-141.
- Cox, R.W. (1996). AFNI: software for analysis and visualization of functional magnetic resonance neuroimages. *Computers and Biomedical Research*, 29:162-173.
- Jenkinson, M., Beckmann, C.F., Behrens, T.E.J., Woolrich, M.W., Smith, S.M. (2012). FSL. *NeuroImage*, 62:782-790.
- Li, X., Morgan, P.S., Ashburner, J., Smith, J., Rorden, C. (2016). The first step for neuroimaging data analysis: DICOM to NIfTI conversion. *Journal of Neuroscience Methods*, 264:47-56.
- Williams, B., Lindner, M. (2020). pyfMRIqc: a software package for raw fMRI data quality assurance. *Journal of Open Research Software*, 8:23.
- Power, J.D., Barnes, K.A., Snyder, A.Z., Schlaggar, B.L., Petersen, S.E. (2012). Spurious but systematic correlations in functional connectivity MRI networks arise from subject motion. *NeuroImage*, 59:2142-2154.
- Power, J.D., Mitra, A., Laumann, T.O., Snyder, A.Z., Schlaggar, B.L., Petersen, S.E. (2014). Methods to detect, characterize, and remove motion artifact in resting state fMRI. *NeuroImage*, 84:320-341.
- Power, J.D. (2017). A simple but useful way to assess fMRI scan qualities. *NeuroImage*, 154:150-158.
- Behzadi, Y., Restom, K., Liau, J., Liu, T.T. (2007). A component based noise correction method (CompCor) for BOLD and perfusion based fMRI. *NeuroImage*, 37:90-101.
- Muschelli, J., Nebel, M.B., Caffo, B.S., et al. (2014). Reduction of motion-related artifacts in resting state fMRI using aCompCor. *NeuroImage*, 96:22-35.
- Chai, X.J., Castañón, A.N., Öngür, D., Whitfield-Gabrieli, S. (2012). Anticorrelations in resting state networks without global signal regression. *NeuroImage*, 59:1420-1428.
- Hallquist, M.N., Hwang, K., Luna, B. (2013). The nuisance of nuisance regression: spectral misspecification in a common approach to resting-state fMRI preprocessing reintroduces noise and obscures functional connectivity. *NeuroImage*, 82:208-225.
- Ciric, R., Wolf, D.H., Power, J.D., et al. (2017). Benchmarking of participant-level confound regression strategies for the control of motion artifact in studies of functional connectivity. *NeuroImage*, 154:174-187.
- Parkes, L., Fulcher, B., Yücel, M., Fornito, A. (2018). An evaluation of the efficacy, reliability, and sensitivity of motion correction strategies for resting-state functional MRI. *NeuroImage*, 171:415-436.
- Murphy, K., Fox, M.D. (2017). Towards a consensus regarding global signal regression for resting state functional connectivity MRI. *NeuroImage*, 154:169-173.
- Friston, K.J., Williams, S., Howard, R., Frackowiak, R.S.J., Turner, R. (1996). Movement-related effects in fMRI time-series. *Magnetic Resonance in Medicine*, 35:346-355.
- Saad, Z.S., Reynolds, R.C., Jo, H.J., et al. (2013). Correcting brain-wide correlation differences in resting-state FMRI. *Brain Connectivity*, 3:339-352.
- Caballero-Gaudes, C., Reynolds, R.C. (2017). Methods for cleaning the BOLD fMRI signal. *NeuroImage*, 154:128-149.
- Nichols, T. (2017). Notes on creating a standardized version of DVARS. arXiv:1704.01469.
- Dietrich, O., Raya, J.G., Reeder, S.B., Reiser, M.F., Schoenberg, S.O. (2007). Measurement of signal-to-noise ratios in MR images. *Journal of Magnetic Resonance Imaging*, 26:375-385.
- Mortamet, B., Bernstein, M.A., Jack, C.R., et al. (2009). Automatic quality assessment in structural brain magnetic resonance imaging. *Magnetic Resonance in Medicine*, 62:365-372.
- Atkinson, D., Hill, D.L.G., Stoyle, P.N.R., Summers, P.E., Keevil, S.F. (1997). Automatic correction of motion artifacts in magnetic resonance images using an entropy focus criterion. *IEEE Transactions on Medical Imaging*, 16:903-910.
- Tustison, N.J., Avants, B.B., Cook, P.A., et al. (2010). N4ITK: improved N3 bias correction. *IEEE Transactions on Medical Imaging*, 29:1310-1320.
- Avants, B.B., Epstein, C.L., Grossman, M., Gee, J.C. (2008). Symmetric diffeomorphic image registration with cross-correlation. *Medical Image Analysis*, 12:26-41.
- Greve, D.N., Fischl, B. (2009). Accurate and robust brain image alignment using boundary-based registration. *NeuroImage*, 48:63-72.
- Klein, A., Andersson, J., Ardekani, B.A., et al. (2009). Evaluation of 14 nonlinear deformation algorithms applied to human brain MRI registration. *NeuroImage*, 46:786-802.
- Andersson, J.L.R., Skare, S., Ashburner, J. (2003). How to correct susceptibility distortions in spin-echo echo-planar images. *NeuroImage*, 20:870-888.
- Calhoun, V.D., Wager, T.D., Krishnan, A., et al. (2017). The impact of T1 versus EPI spatial normalization templates for fMRI data analyses. *Human Brain Mapping*, 38:5331-5342.
- Fonov, V.S., Evans, A.C., McKinstry, R.C., Almli, C.R., Collins, D.L. (2009). Unbiased non-linear average age-appropriate brain templates from birth to adulthood. *NeuroImage*, 47:S102.
- Ashburner, J., Friston, K.J. (2005). Unified segmentation. *NeuroImage*, 26:839-851.
- Glen, D.R., Taylor, P.A., Buchsbaum, B.R., Cox, R.W., Reynolds, R.C. (2020). Beware (surprisingly common) left-right flips in your MRI data. *Frontiers in Neuroinformatics*, 14:18.
- Parker, D., Liu, X., Razlighi, Q.R. (2017). Optimal slice timing correction and its interaction with fMRI parameters and artifacts. *Medical Image Analysis*, 35:434-445.
- Botvinik-Nezer, R., Holzmeister, F., Camerer, C.F., et al. (2020). Variability in the analysis of a single neuroimaging dataset by many teams. *Nature*, 582:84-88.
- Marcus, D.S., Harms, M.P., Snyder, A.Z., et al. (2013). Human Connectome Project informatics: quality control, database services, and data visualization. *NeuroImage*, 80:202-219.
- Alfaro-Almagro, F., Jenkinson, M., Bangerter, N.K., et al. (2018). Image processing and quality control for the first 10,000 brain imaging datasets from UK Biobank. *NeuroImage*, 166:400-424.
- Taylor, P.A., Etzel, J.A., Glen, D., Reynolds, R.C., Moraczewski, D., Basavaraj, A. (2022). FMRI Open QC Project. https://osf.io/qaesm/
